In [ ]:
# Plain-language: This cell loads the shared tools and sets up a stable notebook environment so the rest of the analysis runs consistently.
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone

import argparse
import hashlib
import html
import importlib
import importlib
import importlib.metadata
import importlib.util
import io
import json
import random
import random
import os
import platform
import re
import subprocess
import sys
import tempfile
import types
from contextlib import redirect_stdout, redirect_stderr

os.environ.setdefault('MPLCONFIGDIR', tempfile.mkdtemp(prefix='dice-mpl-'))
os.environ.setdefault('MPLBACKEND', 'Agg')
for _name in [
    'OPENBLAS_NUM_THREADS',
    'OMP_NUM_THREADS',
    'MKL_NUM_THREADS',
    'NUMEXPR_NUM_THREADS',
    'VECLIB_MAXIMUM_THREADS',
    'BLIS_NUM_THREADS',
]:
    os.environ.setdefault(_name, '1')

import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import HTML, Image, Markdown, display
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_rgb
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, Patch, Rectangle
from matplotlib.ticker import FormatStrFormatter, FuncFormatter, MaxNLocator, PercentFormatter
from matplotlib.transforms import blended_transform_factory
from sklearn.metrics import average_precision_score, roc_auc_score

NOTEBOOK_RANDOM_SEED = 0
random.seed(NOTEBOOK_RANDOM_SEED)
np.random.seed(NOTEBOOK_RANDOM_SEED)
IPythonImage = Image

NOTEBOOK_IMPORTS = [
    'from IPython.display import HTML, Image, Markdown, display',
    'from contextlib import redirect_stdout, redirect_stderr',
    'from datetime import datetime, timezone',
    'from matplotlib.cm import ScalarMappable',
    'from matplotlib.colors import LinearSegmentedColormap, Normalize, to_rgb',
    'from matplotlib.lines import Line2D',
    'from matplotlib.patches import FancyBboxPatch, Patch, Rectangle',
    'from matplotlib.ticker import FormatStrFormatter, FuncFormatter, MaxNLocator, PercentFormatter',
    'from matplotlib.transforms import blended_transform_factory',
    'from pathlib import Path',
    'from sklearn.metrics import average_precision_score, roc_auc_score',
    'from time import perf_counter',
    'import argparse',
    'import hashlib',
    'import html',
    'import importlib',
    'import importlib.metadata',
    'import importlib.util',
    'import io',
    'import json',
    'import matplotlib.patheffects as pe',
    'import matplotlib.pyplot as plt',
    'import numpy as np',
    'import os',
    'import pandas as pd',
    'import platform',
    'import random',
    'import re',
    'import subprocess',
    'import sys',
    'import tempfile',
    'import types',
]
EMBEDDED_PIPELINE_IMPORTS = [
    'from __future__ import annotations',
    'from dataclasses import dataclass',
    'from pathlib import Path',
    'from sklearn.metrics import average_precision_score, balanced_accuracy_score, confusion_matrix, f1_score, precision_recall_curve, roc_auc_score, roc_curve',
    'from sklearn.metrics import average_precision_score, roc_auc_score',
    'from typing import Dict, Iterable, List, Tuple',
    'from typing import Dict, List, Sequence, Tuple',
    'import argparse',
    'import json',
    'import matplotlib',
    'import matplotlib.pyplot as plt',
    'import numpy as np',
    'import pandas as pd'
]
RESULT_SAVE_LOCATIONS = {
    'analysis_root': 'DATASET_ROOT/results_analysis',
    'global_results_per_profile': 'DATASET_ROOT/results_dice_full or DATASET_ROOT/results_dice_full_<profile>',
    'holdout_results_per_profile': 'DATASET_ROOT/results_dice_full_holdout or DATASET_ROOT/results_dice_full_<profile>_holdout',
    'tuning_sweeps_per_profile': 'DATASET_ROOT/results_dice_tuning or DATASET_ROOT/results_dice_tuning_<profile>',
    'paper_bundle_root': 'DATASET_ROOT/results_itc_paper/<profile>',
    'appendix_bundle_root': 'DATASET_ROOT/results_itc_appendix/<profile>',
    'comparison_exports': 'DATASET_ROOT/results_itc_paper/comparison',
    'reproducibility_manifest': 'DATASET_ROOT/results_portable/run_manifest.json',
}

# DICE ITC Results Notebook

This notebook is the paper-facing notebook for DICE. It runs the released analysis pipeline, regenerates the main and appendix artifacts, and exports the tables and figures used in the draft.


## Scope of This Notebook

This notebook is organized so the GitHub render reads like a compact ITC paper story before it expands into deeper analysis.

The front of the notebook now does four things first:
- gives a reader guide and paper-safe headline claims,
- compares the mixed and full observability profiles,
- summarizes diagnosis, localization, and hardware-facing evidence, and
- states the recommended DICE sweep plan for a complete evaluation.

After that, the notebook proceeds in a more traditional order: setup, main monitoring, reliability, cross-workload robustness, design-space sweeps, localization, uncertainty, appendix bundles, grounded LLM triage, and reproducibility.

If you only need the shortest paper path, read in this order:
1. **Reader Guide and Main Claims**
2. **Quick Paper-Safe Metrics**
3. **Mixed vs Full Deployment Summary**
4. **Diagnosis, Localization, and Hardware Scorecard**
5. **Section 2. Main DICE Performance**
6. **Section 3. Reliability Without Per-Workload Tuning**
7. **Section 5. Cross-Workload Transfer Robustness and Drift Proxy**
8. **Section 6. DICE-Specific Design-Space Evaluation**


## Notebook-Local Helper Functions

These helpers keep the later paper-facing cells readable.
They do not run experiments by themselves; they only support rendering, path resolution, and paper artifact organization.


In [ ]:
# Plain-language: This cell finds the DICE repo and dataset, then defines reusable helper functions and output paths.
def _looks_like_repo_root(base: Path) -> bool:
    return (base / 'data generation').exists() and (base / 'environment.yml').exists()


def resolve_repo_root(start: Path) -> Path:
    candidates = []
    seen = set()

    def add_candidate(p: Path | None) -> None:
        if p is None:
            return
        try:
            rp = p.expanduser().resolve()
        except FileNotFoundError:
            rp = p.expanduser()
        key = str(rp)
        if key not in seen:
            seen.add(key)
            candidates.append(rp)

    add_candidate(start)
    for base in [start, *start.parents]:
        add_candidate(base)

    env_hint = os.environ.get('DICE_REPO_ROOT')
    if env_hint:
        add_candidate(Path(env_hint))

    home = Path.home()
    for base in [
        home / 'Documents' / 'New project' / 'DICE',
        home / 'Documents' / 'New project' / 'DICE-latest-sync',
        home / 'DICE',
        home / 'Downloads' / 'DICE',
        home / 'Downloads' / 'DICE-latest-sync',
    ]:
        add_candidate(base)

    for root in [home, home / 'Documents', home / 'Documents' / 'New project', home / 'Downloads']:
        if root.exists():
            for child in root.iterdir():
                if child.is_dir() and 'dice' in child.name.lower():
                    add_candidate(child)

    for base in candidates:
        if _looks_like_repo_root(base):
            return base

    raise RuntimeError(
        'Could not locate the DICE repository root. Launch the notebook from a DICE checkout or set DICE_REPO_ROOT.'
    )


def portable_env() -> dict[str, str]:
    env = os.environ.copy()
    env['MPLCONFIGDIR'] = env.get('MPLCONFIGDIR', tempfile.mkdtemp(prefix='dice-mpl-'))
    env['MPLBACKEND'] = 'Agg'
    env['OPENBLAS_NUM_THREADS'] = '1'
    env['OMP_NUM_THREADS'] = '1'
    env['MKL_NUM_THREADS'] = '1'
    env['NUMEXPR_NUM_THREADS'] = '1'
    env['VECLIB_MAXIMUM_THREADS'] = '1'
    env['BLIS_NUM_THREADS'] = '1'
    env['PYTHONHASHSEED'] = '0'
    return env


REPO_ROOT = resolve_repo_root(Path.cwd().resolve())
DATASET_ROOT = REPO_ROOT / 'data generation' / 'dataset' / 'ITC_M2Pro_DATA'
FEATURE_PROFILES_TO_SWEEP = ['mixed', 'full']  # Sweep both core-backed and full Tier-1/Tier-2 profiles.
PRIMARY_PAPER_PROFILE = 'mixed'  # The draft-facing headline tables and abstract claims use the mixed deployment profile.
DISPLAY_FEATURE_PROFILE = PRIMARY_PAPER_PROFILE  # Downstream paper figures default to the draft-facing profile unless changed.
OUT = DATASET_ROOT / 'results_analysis'
FIG = OUT / 'figures'


def _normalize_feature_profiles(profiles: list[str] | tuple[str, ...]) -> list[str]:
    ordered = []
    for profile in profiles:
        if profile not in {'mixed', 'full'}:
            raise ValueError(f'Unsupported feature profile: {profile}')
        if profile not in ordered:
            ordered.append(profile)
    if not ordered:
        raise ValueError('FEATURE_PROFILES_TO_SWEEP must contain at least one profile.')
    return ordered


FEATURE_PROFILES_TO_SWEEP = _normalize_feature_profiles(FEATURE_PROFILES_TO_SWEEP)
if PRIMARY_PAPER_PROFILE not in FEATURE_PROFILES_TO_SWEEP:
    PRIMARY_PAPER_PROFILE = FEATURE_PROFILES_TO_SWEEP[0]
if DISPLAY_FEATURE_PROFILE not in FEATURE_PROFILES_TO_SWEEP:
    DISPLAY_FEATURE_PROFILE = FEATURE_PROFILES_TO_SWEEP[-1]


def profile_out_dirs(profile: str) -> tuple[Path, Path]:
    if profile == 'mixed':
        return DATASET_ROOT / 'results_dice_full', DATASET_ROOT / 'results_dice_full_holdout'
    return (
        DATASET_ROOT / f'results_dice_full_{profile}',
        DATASET_ROOT / f'results_dice_full_{profile}_holdout',
    )


PROFILE_OUT_DIRS = {
    profile: {
        'global': profile_out_dirs(profile)[0],
        'holdout': profile_out_dirs(profile)[1],
    }
    for profile in FEATURE_PROFILES_TO_SWEEP
}
OUT_FULL = PROFILE_OUT_DIRS[DISPLAY_FEATURE_PROFILE]['global']
OUT_HOLDOUT = PROFILE_OUT_DIRS[DISPLAY_FEATURE_PROFILE]['holdout']
OUT_PAPER = DATASET_ROOT / 'results_itc_paper'
OUT_APPENDIX = DATASET_ROOT / 'results_itc_appendix'
MANIFEST = DATASET_ROOT / 'results_portable' / 'run_manifest.json'

print('REPO_ROOT           :', REPO_ROOT)
print('DATASET_ROOT        :', DATASET_ROOT)
print('SWEEP_PROFILES      :', FEATURE_PROFILES_TO_SWEEP)
print('PRIMARY_PAPER_PROFILE:', PRIMARY_PAPER_PROFILE)
print('DISPLAY_PROFILE     :', DISPLAY_FEATURE_PROFILE)
for profile in FEATURE_PROFILES_TO_SWEEP:
    print(f'OUT[{profile}] global  :', PROFILE_OUT_DIRS[profile]['global'])
    print(f'OUT[{profile}] holdout :', PROFILE_OUT_DIRS[profile]['holdout'])

RESULT_EXPORT_SUFFIXES = ('.png', '.csv')
RESULT_EXPORT_ROOTS = [
    OUT,
    *[PROFILE_OUT_DIRS[profile]['global'] for profile in FEATURE_PROFILES_TO_SWEEP],
    *[PROFILE_OUT_DIRS[profile]['holdout'] for profile in FEATURE_PROFILES_TO_SWEEP],
    OUT_PAPER,
    OUT_APPENDIX,
    DATASET_ROOT / 'results_portable',
]
RESULT_ARTIFACT_MANIFEST_CSV = DATASET_ROOT / 'results_portable' / 'notebook_saved_artifacts.csv'
RESULT_ARTIFACT_MANIFEST_JSON = DATASET_ROOT / 'results_portable' / 'notebook_saved_artifacts.json'


def repo_relative_path(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(REPO_ROOT.resolve()))
    except ValueError:
        return str(path.resolve())


def iter_saved_result_files(
    roots: list[Path] | tuple[Path, ...] = RESULT_EXPORT_ROOTS,
    suffixes: tuple[str, ...] = RESULT_EXPORT_SUFFIXES,
):
    seen = set()
    for root in roots:
        if not root.exists():
            continue
        for suffix in suffixes:
            for path in sorted(root.rglob(f'*{suffix}')):
                try:
                    resolved = path.resolve()
                except FileNotFoundError:
                    continue
                if resolved in seen or not path.is_file():
                    continue
                seen.add(resolved)
                yield resolved


def build_saved_artifact_manifest(
    roots: list[Path] | tuple[Path, ...] = RESULT_EXPORT_ROOTS,
    suffixes: tuple[str, ...] = RESULT_EXPORT_SUFFIXES,
) -> pd.DataFrame:
    columns = ['relative_path', 'absolute_path', 'suffix', 'bytes']
    rows = []
    for path in iter_saved_result_files(roots=roots, suffixes=suffixes):
        try:
            stat = path.stat()
        except FileNotFoundError:
            continue
        rows.append(
            {
                'relative_path': repo_relative_path(path),
                'absolute_path': str(path),
                'suffix': path.suffix.lower(),
                'bytes': int(stat.st_size),
            }
        )
    return pd.DataFrame(rows, columns=columns).sort_values(['suffix', 'relative_path']).reset_index(drop=True)


def write_saved_artifact_manifest(
    csv_path: Path = RESULT_ARTIFACT_MANIFEST_CSV,
    json_path: Path = RESULT_ARTIFACT_MANIFEST_JSON,
) -> pd.DataFrame:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    artifact_df = build_saved_artifact_manifest()
    artifact_df.to_csv(csv_path, index=False)
    json_path.write_text(artifact_df.to_json(orient='records', indent=2) + '\n')
    return artifact_df


def load_grounded_llm_triage_helpers(repo_root: Path) -> dict[str, object]:
    helper_names = [
        'discover_latest_llm_outputs',
        'evaluate_llm_grounding_outputs',
        'export_llm_case_cards',
        'export_llm_diagnostic_model_catalog',
        'export_llm_diagnostic_prompt_bundle',
        'runtime_availability',
    ]
    try:
        module = importlib.import_module('tools.grounded_llm_triage')
    except Exception:
        llm_tool_path = repo_root / 'tools' / 'grounded_llm_triage.py'
        spec = importlib.util.spec_from_file_location('dice_grounded_llm_triage', llm_tool_path)
        if spec is None or spec.loader is None:
            raise ImportError(f'Could not load grounded_llm_triage from {llm_tool_path}')
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
    return {name: getattr(module, name) for name in helper_names}


## Execution Guard

The notebook should be launched from the intended DICE clone, not from a stale copy in `Trash` or another transient folder.

The next cell validates the repository location, confirms the released dataset is available, and prepares the main paper and appendix output folders.


In [ ]:
# Plain-language: This cell checks that we opened the right project and prepares the folders where the paper-ready outputs will be saved.
if '.Trash' in str(REPO_ROOT):
    raise RuntimeError(
        'This notebook was launched from a Trash clone. Reopen it from your intended DICE repository checkout.'
    )
assert DATASET_ROOT.exists(), f'Missing dataset root: {DATASET_ROOT}'

PAPER_FULL = OUT_PAPER / 'full'
PAPER_FIG = PAPER_FULL / 'figures'
APPENDIX_FULL = OUT_APPENDIX / 'full'
NOTEBOOK_RUNTIME = OUT_PAPER / 'runtime_summary.json'

for path in [OUT_PAPER, OUT_APPENDIX, PAPER_FULL, PAPER_FIG, APPENDIX_FULL]:
    path.mkdir(parents=True, exist_ok=True)

print('Validated repository root :', REPO_ROOT)
print('Validated dataset root    :', DATASET_ROOT)
print('Main paper output folder  :', PAPER_FULL)
print('Appendix output folder    :', APPENDIX_FULL)


## Embedded DICE Engine

This cell embeds the released DICE analysis and training/evaluation pipeline directly inside the notebook.
No external Python runner is required.

Important:
- this is where the notebook defines the notebook-local DICE engine;
- the actual experiment begins in the next section, **Run End-to-End**;
- the notebook also includes a small patch cell right after this one so the embedded engine exports block traces for the virtual-system overlay.


In [ ]:
# Plain-language: This cell loads the notebook-local DICE engine so later cells can run the full method without calling an external runner.
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 16,
    "axes.titlesize": 20,
    "axes.titleweight": "bold",
    "axes.labelsize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 15,
    "legend.frameon": True,
    "legend.borderaxespad": 0.8,
})

NOTEBOOK_GENERATE_RESULTS_ANALYSIS_SOURCE = '#!/usr/bin/env python3\n"""\nGenerate paper-ready Results/Analysis artifacts from DICE tiered dataset.\n\nOutputs:\n- CSV tables (overall metrics, stressor metrics, workload summaries, feature inventory)\n- LaTeX tables ready for Overleaf\n- PNG figures (AF-index trajectories, separability heatmaps, score distributions)\n- Markdown summary with key values to paste into paper draft\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Dict, Iterable, List, Tuple\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nREPO_ROOT = PROJECT_ROOT\nDEFAULT_DATASET_ROOT = REPO_ROOT / "data generation" / "dataset" / "ITC_M2Pro_DATA"\n\n\nWORKLOADS = ["BROWSER", "VIDEO_SW", "PY_AI", "PY_STATS"]\nSTRESSORS = ["NOMINAL", "CACHE", "TLB", "BRANCH", "MEMBW", "ATOMIC"]\nANOMALIES = [s for s in STRESSORS if s != "NOMINAL"]\n\nTIER_FILE = {\n    "tier0": "tier0_full_5hz.csv",\n    "tier1_alt": "tier1_alt_core_5hz.csv",\n    "tier2": "tier2_core_5hz.csv",\n}\n\nIGNORE_COLS = {\n    "idx",\n    "ts_unix_s",\n    "t_rel_s",\n    "timestamp",\n    "time",\n    "ts",\n}\n\nTIER_PRETTY = {\n    "tier0": "Tier-0",\n    "tier1_alt": "Tier-1",\n    "tier2": "Tier-2",\n}\n\nCOLOR_BY_STRESSOR = {\n    "NOMINAL": "#000000",\n    "ATOMIC": "#e57373",\n    "BRANCH": "#66bb6a",\n    "CACHE": "#f6a04d",\n    "MEMBW": "#b39ddb",\n    "TLB": "#bcaaa4",\n}\n\n\n@dataclass(frozen=True)\nclass TierData:\n    tier: str\n    features: List[str]\n    run_df: pd.DataFrame\n    timeseries: Dict[str, Dict[str, np.ndarray]]\n    case_quality: pd.DataFrame\n    union_features: List[str]\n\n\ndef case_id(workload: str, stressor: str) -> str:\n    return f"{workload}__{stressor}"\n\n\ndef robust_scale(x: np.ndarray) -> float:\n    x = np.asarray(x, dtype=float)\n    x = x[np.isfinite(x)]\n    if x.size == 0:\n        return 1.0\n    q75, q25 = np.percentile(x, [75, 25])\n    iqr = q75 - q25\n    if iqr > 1e-12:\n        return float(iqr / 1.349)\n    med = np.median(x)\n    mad = np.median(np.abs(x - med))\n    if mad > 1e-12:\n        return float(1.4826 * mad)\n    std = float(np.std(x))\n    if std > 1e-12:\n        return std\n    return 1.0\n\n\ndef safe_auc(y_true: np.ndarray, y_score: np.ndarray) -> float:\n    y_true = np.asarray(y_true, dtype=int)\n    y_score = np.asarray(y_score, dtype=float)\n    if len(np.unique(y_true)) < 2:\n        return float("nan")\n    return float(roc_auc_score(y_true, y_score))\n\n\ndef safe_ap(y_true: np.ndarray, y_score: np.ndarray) -> float:\n    y_true = np.asarray(y_true, dtype=int)\n    y_score = np.asarray(y_score, dtype=float)\n    if len(np.unique(y_true)) < 2:\n        return float("nan")\n    return float(average_precision_score(y_true, y_score))\n\n\ndef downsample_to_1hz(df: pd.DataFrame, source_hz: int = 5) -> pd.DataFrame:\n    if len(df) < source_hz:\n        return df.copy()\n    n = (len(df) // source_hz) * source_hz\n    out = df.iloc[:n].copy()\n    grp = np.arange(n) // source_hz\n    return out.groupby(grp, sort=False).mean(numeric_only=True)\n\n\ndef read_case_csv(root: Path, tier: str, workload: str, stressor: str) -> pd.DataFrame:\n    p = root / tier / case_id(workload, stressor) / TIER_FILE[tier]\n    if not p.exists():\n        raise FileNotFoundError(f"Missing case file: {p}")\n    df = pd.read_csv(p)\n    for c in df.columns:\n        if pd.api.types.is_numeric_dtype(df[c]):\n            df[c] = pd.to_numeric(df[c], errors="coerce")\n    return df\n\n\ndef numeric_feature_columns(df: pd.DataFrame) -> List[str]:\n    cols = []\n    for c in df.columns:\n        if c in IGNORE_COLS:\n            continue\n        if pd.api.types.is_numeric_dtype(df[c]):\n            cols.append(c)\n    return cols\n\n\ndef discover_features(root: Path, tier: str, source_hz: int = 5) -> Tuple[List[str], List[str], pd.DataFrame]:\n    common = None\n    union = set()\n    rows = []\n    for w in WORKLOADS:\n        for s in STRESSORS:\n            p = root / tier / case_id(w, s) / TIER_FILE[tier]\n            df = pd.read_csv(p)\n            cols = set(numeric_feature_columns(df))\n            union |= cols\n            common = cols if common is None else (common & cols)\n            rows.append(\n                {\n                    "tier": tier,\n                    "case_id": case_id(w, s),\n                    "workload": w,\n                    "stressor": s,\n                    "rows_5hz": int(len(df)),\n                    "cols_total": int(df.shape[1]),\n                    "numeric_cols": int(len(cols)),\n                    "nan_fraction": float(df.isna().mean().mean()),\n                    "file_bytes": int(p.stat().st_size),\n                }\n            )\n    common_list = sorted(common) if common else []\n    union_list = sorted(union)\n\n    # Drop globally near-constant channels from common list.\n    keep = []\n    for f in common_list:\n        vals = []\n        for w in WORKLOADS:\n            for s in STRESSORS:\n                d = downsample_to_1hz(read_case_csv(root, tier, w, s), source_hz=source_hz)\n                vals.append(d[f].to_numpy(dtype=float))\n        x = np.concatenate(vals)\n        if np.nanstd(x) > 1e-10:\n            keep.append(f)\n    return keep, union_list, pd.DataFrame(rows)\n\n\ndef build_tier_data(root: Path, tier: str, source_hz: int = 5) -> TierData:\n    features, union_features, quality = discover_features(root, tier, source_hz=source_hz)\n    run_rows = []\n    timeseries = {w: {} for w in WORKLOADS}\n\n    for w in WORKLOADS:\n        ds = {s: downsample_to_1hz(read_case_csv(root, tier, w, s), source_hz=source_hz) for s in STRESSORS}\n        n = min(len(v) for v in ds.values())\n        arr = {\n            s: ds[s].iloc[:n][features].to_numpy(dtype=float, copy=True)\n            for s in STRESSORS\n        }\n        baseline = arr["NOMINAL"]\n        med = np.nanmedian(baseline, axis=0)\n        scale = np.array([robust_scale(baseline[:, j]) for j in range(baseline.shape[1])], dtype=float)\n        scale[scale <= 1e-12] = 1.0\n\n        for s in STRESSORS:\n            z = np.abs((arr[s] - med) / (scale + 1e-12))\n            score_ts = np.nanmean(z, axis=1)\n            timeseries[w][s] = score_ts\n            run_rows.append(\n                {\n                    "tier": tier,\n                    "workload": w,\n                    "stressor": s,\n                    "label": 0 if s == "NOMINAL" else 1,\n                    "run_score_median": float(np.nanmedian(score_ts)),\n                    "run_score_mean": float(np.nanmean(score_ts)),\n                    "run_score_p95": float(np.nanpercentile(score_ts, 95)),\n                    "samples_1hz": int(len(score_ts)),\n                }\n            )\n\n    return TierData(\n        tier=tier,\n        features=features,\n        run_df=pd.DataFrame(run_rows),\n        timeseries=timeseries,\n        case_quality=quality,\n        union_features=union_features,\n    )\n\n\ndef make_overall_metrics(tier_data: Iterable[TierData]) -> pd.DataFrame:\n    rows = []\n    for td in tier_data:\n        df = td.run_df.copy()\n        y = df["label"].to_numpy(dtype=int)\n        s = df["run_score_median"].to_numpy(dtype=float)\n\n        nom = df[df["label"] == 0]["run_score_median"].to_numpy(dtype=float)\n        anm = df[df["label"] == 1]["run_score_median"].to_numpy(dtype=float)\n        tau95 = float(np.quantile(nom, 0.95))\n        rows.append(\n            {\n                "tier": td.tier,\n                "tier_name": TIER_PRETTY[td.tier],\n                "n_features_common": int(len(td.features)),\n                "n_features_union": int(len(td.union_features)),\n                "n_cases": int(len(df)),\n                "roc_auc": safe_auc(y, s),\n                "pr_auc": safe_ap(y, s),\n                "median_nominal": float(np.median(nom)),\n                "median_anomaly": float(np.median(anm)),\n                "anom_nom_ratio": float(np.median(anm) / (np.median(nom) + 1e-12)),\n                "threshold_q95_nominal": tau95,\n                "fpr_at_q95": float(np.mean(nom > tau95)),\n                "tpr_at_q95": float(np.mean(anm > tau95)),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef make_stressor_metrics(tier_data: Iterable[TierData]) -> pd.DataFrame:\n    rows = []\n    for td in tier_data:\n        df = td.run_df.copy()\n        neg = df[df["stressor"] == "NOMINAL"][["workload", "run_score_median"]].set_index("workload")\n        for a in ANOMALIES:\n            pos = df[df["stressor"] == a][["workload", "run_score_median"]].set_index("workload")\n            merged = neg.join(pos, lsuffix="_neg", rsuffix="_pos", how="inner")\n            y_true = np.array([0] * len(merged) + [1] * len(merged), dtype=int)\n            y_score = np.concatenate(\n                [\n                    merged["run_score_median_neg"].to_numpy(dtype=float),\n                    merged["run_score_median_pos"].to_numpy(dtype=float),\n                ]\n            )\n            rows.append(\n                {\n                    "tier": td.tier,\n                    "tier_name": TIER_PRETTY[td.tier],\n                    "stressor": a,\n                    "n_pos": int(len(merged)),\n                    "n_neg": int(len(merged)),\n                    "roc_auc": safe_auc(y_true, y_score),\n                    "pr_auc": safe_ap(y_true, y_score),\n                    "median_neg": float(np.median(merged["run_score_median_neg"])),\n                    "median_pos": float(np.median(merged["run_score_median_pos"])),\n                    "pos_neg_ratio": float(\n                        np.median(merged["run_score_median_pos"])\n                        / (np.median(merged["run_score_median_neg"]) + 1e-12)\n                    ),\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef make_workload_summary(tier_data: Iterable[TierData]) -> pd.DataFrame:\n    rows = []\n    for td in tier_data:\n        df = td.run_df.copy()\n        for w in WORKLOADS:\n            d = df[df["workload"] == w]\n            nom = d[d["stressor"] == "NOMINAL"]["run_score_median"].iloc[0]\n            anm = d[d["stressor"] != "NOMINAL"]["run_score_median"].to_numpy(dtype=float)\n            rows.append(\n                {\n                    "tier": td.tier,\n                    "tier_name": TIER_PRETTY[td.tier],\n                    "workload": w,\n                    "nominal_score": float(nom),\n                    "anomaly_median_score": float(np.median(anm)),\n                    "anomaly_nominal_ratio": float(np.median(anm) / (float(nom) + 1e-12)),\n                    "anomaly_p95_score": float(np.percentile(anm, 95)),\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef table_to_latex(df: pd.DataFrame, caption: str, label: str) -> str:\n    rendered = df.to_latex(index=False, escape=False, float_format=lambda x: f"{x:.4f}")\n    return (\n        "\\\\begin{table}[t]\\n"\n        "\\\\centering\\n"\n        f"\\\\caption{{{caption}}}\\n"\n        f"\\\\label{{{label}}}\\n"\n        "\\\\footnotesize\\n"\n        f"{rendered}\\n"\n        "\\\\end{table}\\n"\n    )\n\n\ndef save_metric_tables(\n    out_dir: Path,\n    overall: pd.DataFrame,\n    stressor: pd.DataFrame,\n    workload: pd.DataFrame,\n    features: pd.DataFrame,\n    quality: pd.DataFrame,\n    runs: pd.DataFrame,\n) -> None:\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    overall_out = overall.sort_values("tier")\n    stressor_out = stressor.sort_values(["tier", "stressor"])\n    workload_out = workload.sort_values(["tier", "workload"])\n    features_out = features.sort_values("tier")\n    quality_out = quality.sort_values(["tier", "case_id"])\n    runs_out = runs.sort_values(["tier", "workload", "stressor"])\n\n    overall_out.to_csv(out_dir / "table_overall_metrics.csv", index=False)\n    stressor_out.to_csv(out_dir / "table_stressor_metrics.csv", index=False)\n    workload_out.to_csv(out_dir / "table_workload_summary.csv", index=False)\n    features_out.to_csv(out_dir / "table_feature_inventory.csv", index=False)\n    quality_out.to_csv(out_dir / "table_case_quality.csv", index=False)\n    runs_out.to_csv(out_dir / "table_run_scores.csv", index=False)\n\n    overall_tex = overall_out[\n        [\n            "tier_name",\n            "n_features_common",\n            "roc_auc",\n            "pr_auc",\n            "median_nominal",\n            "median_anomaly",\n            "anom_nom_ratio",\n        ]\n    ].rename(\n        columns={\n            "tier_name": "Tier",\n            "n_features_common": "Common Features",\n            "roc_auc": "ROC-AUC",\n            "pr_auc": "AUC-PR",\n            "median_nominal": "Median(Nominal)",\n            "median_anomaly": "Median(Anomaly)",\n            "anom_nom_ratio": "Anomaly/Nominal",\n        }\n    )\n\n    stressor_tex = stressor_out[\n        ["tier_name", "stressor", "roc_auc", "pr_auc", "pos_neg_ratio"]\n    ].rename(\n        columns={\n            "tier_name": "Tier",\n            "stressor": "Stressor",\n            "roc_auc": "ROC-AUC",\n            "pr_auc": "AUC-PR",\n            "pos_neg_ratio": "Pos/Neg Score Ratio",\n        }\n    )\n\n    (out_dir / "table_overall_metrics.tex").write_text(\n        table_to_latex(\n            overall_tex,\n            "Run-level anomaly separability by telemetry tier (AF-index score).",\n            "tab:dice_overall_metrics",\n        )\n    )\n    (out_dir / "table_stressor_metrics.tex").write_text(\n        table_to_latex(\n            stressor_tex,\n            "Per-stressor separability by tier (four workloads pooled per stressor).",\n            "tab:dice_stressor_metrics",\n        )\n    )\n\n\ndef plot_af_timeseries(out_dir: Path, tier_data: Iterable[TierData]) -> List[str]:\n    out_paths = []\n    for td in tier_data:\n        fig, axes = plt.subplots(len(WORKLOADS), 1, figsize=(16, 13), sharex=True)\n        if len(WORKLOADS) == 1:\n            axes = [axes]\n\n        for i, w in enumerate(WORKLOADS):\n            ax = axes[i]\n            nom = td.timeseries[w]["NOMINAL"]\n            x = np.arange(len(nom), dtype=float) / 60.0  # minutes (1Hz grid)\n\n            stack = np.vstack([td.timeseries[w][a] for a in ANOMALIES])\n            anom_mean = np.mean(stack, axis=0)\n            anom_min = np.min(stack, axis=0)\n            anom_max = np.max(stack, axis=0)\n\n            ax.plot(x, nom, color="black", linewidth=2.4, label="Benign (NOMINAL)")\n            for a in ANOMALIES:\n                ax.plot(\n                    x,\n                    td.timeseries[w][a],\n                    color=COLOR_BY_STRESSOR[a],\n                    alpha=0.6,\n                    linewidth=1.0,\n                    label=a,\n                )\n            ax.plot(x, anom_mean, color="#c62828", linewidth=2.2, label="Anomaly mean")\n            ax.fill_between(x, anom_min, anom_max, color="#ef5350", alpha=0.18, label="Anomaly range")\n            ax.set_ylabel("AF Index", fontsize=14)\n            ax.set_xlabel("Time (minutes)", fontsize=14)\n            ax.set_title(w, fontsize=16, fontweight="bold")\n            ax.grid(alpha=0.25)\n            ax.tick_params(axis="both", labelsize=12)\n\n        h, l = axes[0].get_legend_handles_labels()\n        dedup = dict(zip(l, h))\n        fig.legend(\n            dedup.values(),\n            dedup.keys(),\n            loc="lower center",\n            ncol=4,\n            frameon=True,\n            fontsize=12,\n            bbox_to_anchor=(0.5, -0.01),\n        )\n        fig.suptitle(f"All-feature AF-index trajectories | {TIER_PRETTY[td.tier]}", fontsize=20, y=0.995)\n        fig.tight_layout(rect=[0, 0.10, 1, 0.95])\n\n        out = out_dir / f"fig_af_timeseries_{td.tier}.png"\n        fig.savefig(out, dpi=240, bbox_inches="tight")\n        plt.close(fig)\n        out_paths.append(str(out))\n    return out_paths\n\n\ndef plot_auc_heatmaps(out_dir: Path, stressor: pd.DataFrame) -> List[str]:\n    out_paths = []\n    for metric, title, fname in [\n        ("roc_auc", "ROC-AUC by tier and stressor", "fig_heatmap_roc_auc.png"),\n        ("pr_auc", "AUC-PR by tier and stressor", "fig_heatmap_pr_auc.png"),\n    ]:\n        piv = stressor.pivot(index="stressor", columns="tier_name", values=metric).loc[ANOMALIES]\n        cols = [c for c in ["Tier-0", "Tier-1", "Tier-2"] if c in piv.columns]\n        piv = piv[cols]\n\n        fig, ax = plt.subplots(figsize=(8.5, 4.5))\n        im = ax.imshow(piv.to_numpy(dtype=float), vmin=0.5, vmax=1.0, cmap="viridis")\n        ax.set_xticks(np.arange(len(piv.columns)))\n        ax.set_xticklabels(piv.columns, fontsize=12)\n        ax.set_yticks(np.arange(len(piv.index)))\n        ax.set_yticklabels(piv.index, fontsize=12)\n        ax.set_title(title, fontsize=16, fontweight="bold")\n        for i in range(len(piv.index)):\n            for j in range(len(piv.columns)):\n                v = float(piv.iloc[i, j])\n                ax.text(j, i, f"{v:.3f}", ha="center", va="center", color="white", fontsize=11)\n        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)\n        cbar.ax.set_ylabel(metric.upper(), rotation=90, fontsize=11)\n        fig.tight_layout()\n        out = out_dir / fname\n        fig.savefig(out, dpi=240, bbox_inches="tight")\n        plt.close(fig)\n        out_paths.append(str(out))\n    return out_paths\n\n\ndef plot_run_score_distributions(out_dir: Path, runs: pd.DataFrame) -> str:\n    tiers = ["tier0", "tier1_alt", "tier2"]\n    fig, axes = plt.subplots(1, len(tiers), figsize=(14.5, 4.6), sharey=False)\n    if len(tiers) == 1:\n        axes = [axes]\n\n    for i, t in enumerate(tiers):\n        ax = axes[i]\n        d = runs[runs["tier"] == t]\n        nom = d[d["label"] == 0]["run_score_median"].to_numpy(dtype=float)\n        anm = d[d["label"] == 1]["run_score_median"].to_numpy(dtype=float)\n        bp = ax.boxplot([nom, anm], tick_labels=["Benign", "Anomaly"], patch_artist=True)\n        for patch, color in zip(bp["boxes"], ["#9e9e9e", "#ef9a9a"]):\n            patch.set_facecolor(color)\n            patch.set_alpha(0.8)\n        ax.scatter(np.repeat(1, len(nom)), nom, color="black", s=24, alpha=0.8)\n        ax.scatter(np.repeat(2, len(anm)), anm, color="#c62828", s=24, alpha=0.7)\n        ax.set_title(TIER_PRETTY[t], fontsize=14, fontweight="bold")\n        ax.set_ylabel("Run AF Index (median)", fontsize=12)\n        ax.grid(alpha=0.22)\n        ax.tick_params(axis="both", labelsize=11)\n\n    fig.suptitle("Run-level AF-index score distributions", fontsize=18, y=1.02)\n    fig.tight_layout()\n    out = out_dir / "fig_run_score_distributions.png"\n    fig.savefig(out, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n    return str(out)\n\n\ndef build_feature_inventory(tier_data: Iterable[TierData]) -> pd.DataFrame:\n    rows = []\n    for td in tier_data:\n        rows.append(\n            {\n                "tier": td.tier,\n                "tier_name": TIER_PRETTY[td.tier],\n                "n_features_common": len(td.features),\n                "n_features_union": len(td.union_features),\n                "common_features_json": json.dumps(td.features),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef write_markdown_summary(\n    out_dir: Path,\n    overall: pd.DataFrame,\n    stressor: pd.DataFrame,\n    fig_paths: List[str],\n) -> None:\n    best_tier = overall.sort_values("pr_auc", ascending=False).iloc[0]\n    weakest = (\n        stressor.groupby("stressor")[["roc_auc", "pr_auc"]]\n        .mean()\n        .sort_values("pr_auc", ascending=True)\n        .head(2)\n        .index.tolist()\n    )\n    strongest = (\n        stressor.groupby("stressor")[["roc_auc", "pr_auc"]]\n        .mean()\n        .sort_values("pr_auc", ascending=False)\n        .head(3)\n        .index.tolist()\n    )\n    lines = []\n    lines.append("# DICE Results/Analysis Auto-Summary")\n    lines.append("")\n    lines.append("## Key Findings")\n    lines.append(\n        f"- Best run-level AUC-PR tier: **{best_tier[\'tier_name\']}** "\n        f"(AUC-PR={best_tier[\'pr_auc\']:.4f}, ROC-AUC={best_tier[\'roc_auc\']:.4f})."\n    )\n    lines.append(f"- Strongest stressors (mean AUC-PR across tiers): **{\', \'.join(strongest)}**.")\n    lines.append(f"- Hardest stressors (mean AUC-PR across tiers): **{\', \'.join(weakest)}**.")\n    lines.append("")\n    lines.append("## Suggested Results Narrative")\n    lines.append(\n        "Across the 24-run Apple dataset, AF-index separation is consistently visible between nominal and "\n        "anomalous runs in all telemetry tiers. Tier-aware scoring indicates that anomaly/nominal score ratios "\n        "remain above 1.0 in every tier, confirming stable separability under the fixed collection protocol. "\n        "Per-stressor analysis shows stronger separation for ATOMIC, CACHE, and MEMBW, while BRANCH and TLB "\n        "remain comparatively harder due to weaker host-visible signatures. These observations match the "\n        "expected mechanism-level difficulty ordering in software-driven stressors."\n    )\n    lines.append("")\n    lines.append("## Generated Figures")\n    for p in fig_paths:\n        lines.append(f"- `{p}`")\n    lines.append("")\n    lines.append("## Generated Tables")\n    for p in [\n        out_dir / "table_overall_metrics.csv",\n        out_dir / "table_stressor_metrics.csv",\n        out_dir / "table_workload_summary.csv",\n        out_dir / "table_feature_inventory.csv",\n        out_dir / "table_overall_metrics.tex",\n        out_dir / "table_stressor_metrics.tex",\n    ]:\n        lines.append(f"- `{p}`")\n    (out_dir / "RESULTS_SUMMARY.md").write_text("\\n".join(lines) + "\\n")\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\n        "--root",\n        type=Path,\n        default=DEFAULT_DATASET_ROOT,\n        help="Dataset root containing tier0, tier1_alt, tier2 folders.",\n    )\n    ap.add_argument(\n        "--out_dir",\n        type=Path,\n        default=None,\n        help="Output directory for results tables/figures (default: <root>/results_analysis).",\n    )\n    ap.add_argument("--source_hz", type=int, default=5, help="Source sampling Hz used for downsampling.")\n    args = ap.parse_args()\n\n    root = args.root.expanduser().resolve()\n    out_dir = (args.out_dir.expanduser().resolve() if args.out_dir else (root / "results_analysis"))\n    out_dir.mkdir(parents=True, exist_ok=True)\n    fig_dir = out_dir / "figures"\n    fig_dir.mkdir(parents=True, exist_ok=True)\n\n    tier_data = [build_tier_data(root, t, source_hz=args.source_hz) for t in ["tier0", "tier1_alt", "tier2"]]\n\n    runs = pd.concat([td.run_df for td in tier_data], ignore_index=True)\n    quality = pd.concat([td.case_quality for td in tier_data], ignore_index=True)\n    features = build_feature_inventory(tier_data)\n    overall = make_overall_metrics(tier_data)\n    stressor = make_stressor_metrics(tier_data)\n    workload = make_workload_summary(tier_data)\n\n    save_metric_tables(out_dir, overall, stressor, workload, features, quality, runs)\n\n    figs = []\n    figs.extend(plot_af_timeseries(fig_dir, tier_data))\n    figs.extend(plot_auc_heatmaps(fig_dir, stressor))\n    figs.append(plot_run_score_distributions(fig_dir, runs))\n\n    write_markdown_summary(out_dir, overall, stressor, figs)\n\n    print(f"[OK] Results generated at: {out_dir}")\n    print("[OK] Figures:")\n    for p in figs:\n        print(f" - {p}")\n    print("[OK] Tables:")\n    print(f" - {out_dir / \'table_overall_metrics.csv\'}")\n    print(f" - {out_dir / \'table_stressor_metrics.csv\'}")\n    print(f" - {out_dir / \'table_workload_summary.csv\'}")\n    print(f" - {out_dir / \'table_overall_metrics.tex\'}")\n    print(f" - {out_dir / \'table_stressor_metrics.tex\'}")\n\n\nif __name__ == "__main__":\n    main()\n'
NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE = '#!/usr/bin/env python3\n"""\nFull retrain/evaluation for DICE micro-twin + split-conformal pipeline.\n\nProtocol:\n- Use Tier-0 / Tier-1-alt / Tier-2 clean dataset (5000 rows @ 5Hz per run).\n- Align to 1Hz via mean pooling.\n- Train only on benign runs (NOMINAL) with cross-workload transfer folds.\n- Fit linear micro-twin dynamics in normalized feature space.\n- Build residual signatures on decision blocks.\n- Calibrate conformal threshold on benign calibration blocks.\n- Evaluate run-level labels (Benign vs Anomaly) via persistent block alerts.\n\nOutputs:\n- CSV metrics tables and per-case predictions\n- LaTeX table snippets for paper\n- ROC/PR and score distribution figures\n- Markdown summary for direct paste into Results section\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom time import perf_counter\nfrom typing import Dict, List, Sequence, Tuple\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom matplotlib.colors import Normalize\nfrom matplotlib.patches import Polygon\nfrom sklearn.metrics import (\n    average_precision_score,\n    confusion_matrix,\n    f1_score,\n    precision_recall_curve,\n    roc_auc_score,\n    roc_curve,\n)\n\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nREPO_ROOT = PROJECT_ROOT\nDEFAULT_DATASET_ROOT = REPO_ROOT / "data generation" / "dataset" / "ITC_M2Pro_DATA"\n\n\nWORKLOADS = ["BROWSER", "VIDEO_SW", "PY_AI", "PY_STATS"]\nSTRESSORS = ["NOMINAL", "CACHE", "TLB", "BRANCH", "MEMBW", "ATOMIC"]\nANOMALIES = [s for s in STRESSORS if s != "NOMINAL"]\nSTRESSOR_FAMILY = {\n    "NOMINAL": "benign",\n    "CACHE": "memory_pressure",\n    "TLB": "memory_pressure",\n    "MEMBW": "memory_pressure",\n    "BRANCH": "control_flow",\n    "ATOMIC": "synchronization",\n}\nDIAGNOSIS_FAMILIES = ["memory_pressure", "control_flow", "synchronization"]\n\nIGNORE_COLS = {"idx", "ts_unix_s", "t_rel_s", "timestamp", "time", "ts"}\n\nFEATURE_PROFILES = {\n    "mixed": {\n        "tier0": "tier0_full_5hz.csv",\n        "tier1_alt": "tier1_alt_core_5hz.csv",\n        "tier2": "tier2_core_5hz.csv",\n    },\n    "full": {\n        "tier0": "tier0_full_5hz.csv",\n        "tier1_alt": "tier1_alt_full_5hz.csv",\n        "tier2": "tier2_full_5hz.csv",\n    },\n}\n\nCONFIGS = {\n    "tier0": ["tier0"],\n    "tier0_tier1": ["tier0", "tier1_alt"],\n    "tier0_tier1_tier2": ["tier0", "tier1_alt", "tier2"],\n}\n\nDIAG_TOP_K = 5\nDIAG_POST_ALERT_BLOCKS = 5\nDIAG_POST_ALERT_MIN_BLOCKS = 3\nDIAG_MONOTONIC_TOKENS = (\n    "uptime",\n    "syscall",\n    "ctx_switch",\n    "soft_interrupt",\n    "hard_interrupt",\n)\nDIAG_GENERIC_MEMORY_STATE_TOKENS = (\n    "mem_free_bytes",\n    "mem_inactive_bytes",\n    "mem_available_bytes",\n    "mem_percent",\n    "swap_percent",\n    "swap_free_bytes",\n    "swap_used_bytes",\n)\nDIAG_GENERIC_MEMORY_STATE_FACTOR = 0.35\nMECHANISM_GROUPS = [\n    "compute",\n    "memory_io",\n    "thermal_power",\n    "scheduler_runtime",\n    "platform_pressure",\n]\n\nTITLE_SIZE = 15\nLABEL_SIZE = 13\nTICK_SIZE = 11\nLEGEND_SIZE = 11\nANNOTATION_SIZE = 10\nCONFIG_PRETTY = {\n    "tier0": "Tier-0",\n    "tier0_tier1": "Tier-0/1",\n    "tier0_tier1_tier2": "Tier-0/1/2",\n}\nCONFIG_COLORS = {\n    "tier0": "#355070",\n    "tier0_tier1": "#2A9D8F",\n    "tier0_tier1_tier2": "#E76F51",\n}\nSTRESSOR_COLORS = {\n    "ATOMIC": "#E76F51",\n    "BRANCH": "#43AA8B",\n    "CACHE": "#577590",\n    "MEMBW": "#F4A261",\n    "TLB": "#8D5A97",\n}\nSQRT3 = float(np.sqrt(3.0))\nSQRT2 = float(np.sqrt(2.0))\nDIAG_FAMILY_GATE_QUANTILE = 0.35\nDIAG_CONFIDENCE_GATE_QUANTILE = 0.45\n\n\n@dataclass(frozen=True)\nclass CaseRef:\n    workload: str\n    stressor: str\n\n    @property\n    def case_id(self) -> str:\n        return f"{self.workload}__{self.stressor}"\n\n    @property\n    def label(self) -> int:\n        return 0 if self.stressor == "NOMINAL" else 1\n\n\n@dataclass\nclass ModelBundle:\n    feature_names: List[str]\n    median: np.ndarray\n    scale: np.ndarray\n    A: np.ndarray\n    weights: np.ndarray\n    diagnosis_weights: np.ndarray\n    cal_scores: np.ndarray\n    tau: float\n\n\ndef all_cases() -> List[CaseRef]:\n    return [CaseRef(w, s) for w in WORKLOADS for s in STRESSORS]\n\n\ndef robust_scale_1d(x: np.ndarray) -> float:\n    x = np.asarray(x, dtype=float)\n    x = x[np.isfinite(x)]\n    if x.size == 0:\n        return 1.0\n    q75, q25 = np.percentile(x, [75, 25])\n    iqr = q75 - q25\n    if iqr > 1e-12:\n        return float(iqr / 1.349)\n    med = np.median(x)\n    mad = np.median(np.abs(x - med))\n    if mad > 1e-12:\n        return float(1.4826 * mad)\n    std = float(np.std(x))\n    if std > 1e-12:\n        return std\n    return 1.0\n\n\ndef robust_fit_matrix(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:\n    med = np.nanmedian(X, axis=0)\n    scale = np.zeros(X.shape[1], dtype=float)\n    for j in range(X.shape[1]):\n        scale[j] = robust_scale_1d(X[:, j])\n    scale[scale <= 1e-12] = 1.0\n    return med, scale\n\n\ndef safe_auc(y_true: np.ndarray, score: np.ndarray) -> float:\n    if len(np.unique(y_true)) < 2:\n        return float("nan")\n    return float(roc_auc_score(y_true, score))\n\n\ndef safe_ap(y_true: np.ndarray, score: np.ndarray) -> float:\n    if len(np.unique(y_true)) < 2:\n        return float("nan")\n    return float(average_precision_score(y_true, score))\n\n\ndef default_results_dir(root: Path, protocol: str, feature_profile: str) -> Path:\n    if feature_profile == "mixed":\n        return root / ("results_dice_full_holdout" if protocol == "workload_holdout" else "results_dice_full")\n    suffix = f"results_dice_full_{feature_profile}"\n    if protocol == "workload_holdout":\n        suffix = f"{suffix}_holdout"\n    return root / suffix\n\n\ndef case_path(root: Path, tier: str, case: CaseRef, tier_files: Dict[str, str]) -> Path:\n    return root / tier / case.case_id / tier_files[tier]\n\n\ndef read_df(path: Path) -> pd.DataFrame:\n    if not path.exists():\n        raise FileNotFoundError(f"Missing file: {path}")\n    df = pd.read_csv(path)\n    for c in df.columns:\n        if pd.api.types.is_numeric_dtype(df[c]):\n            df[c] = pd.to_numeric(df[c], errors="coerce")\n    return df\n\n\ndef numeric_features(df: pd.DataFrame) -> List[str]:\n    out = []\n    for c in df.columns:\n        if c in IGNORE_COLS:\n            continue\n        if pd.api.types.is_numeric_dtype(df[c]):\n            out.append(c)\n    return out\n\n\ndef downsample_1hz(df: pd.DataFrame, source_hz: int = 5) -> pd.DataFrame:\n    if len(df) < source_hz:\n        return df.copy()\n    n = (len(df) // source_hz) * source_hz\n    tmp = df.iloc[:n].copy()\n    grp = np.arange(n) // source_hz\n    return tmp.groupby(grp, sort=False).mean(numeric_only=True)\n\n\ndef common_features_per_tier(root: Path, tier: str, tier_files: Dict[str, str]) -> List[str]:\n    common = None\n    for case in all_cases():\n        df = read_df(case_path(root, tier, case, tier_files))\n        cols = set(numeric_features(df))\n        common = cols if common is None else (common & cols)\n    common_list = sorted(common) if common else []\n\n    # Drop globally constant features.\n    keep = []\n    for f in common_list:\n        vals = []\n        for case in all_cases():\n            d = downsample_1hz(read_df(case_path(root, tier, case, tier_files)))\n            vals.append(d[f].to_numpy(dtype=float))\n        x = np.concatenate(vals)\n        if np.nanstd(x) > 1e-10:\n            keep.append(f)\n    return keep\n\n\ndef build_case_matrix(\n    root: Path,\n    case: CaseRef,\n    tiers: Sequence[str],\n    feature_map: Dict[str, List[str]],\n    tier_files: Dict[str, str],\n    source_hz: int = 5,\n) -> Tuple[np.ndarray, List[str]]:\n    mats = []\n    names = []\n    lengths = []\n    for t in tiers:\n        df = downsample_1hz(read_df(case_path(root, t, case, tier_files)), source_hz=source_hz)\n        feats = feature_map[t]\n        arr = df[feats].to_numpy(dtype=float)\n        mats.append(arr)\n        lengths.append(arr.shape[0])\n        names.extend([f"{t}:{f}" for f in feats])\n\n    n = min(lengths)\n    mats = [m[:n] for m in mats]\n    X = np.concatenate(mats, axis=1)\n    return X, names\n\n\ndef fit_linear_dynamics(X_runs: List[np.ndarray], ridge_lambda: float = 1e-3) -> np.ndarray:\n    X_prev = []\n    X_next = []\n    for X in X_runs:\n        if len(X) < 2:\n            continue\n        X_prev.append(X[:-1])\n        X_next.append(X[1:])\n    if not X_prev:\n        raise RuntimeError("Not enough samples to fit dynamics.")\n    P = np.vstack(X_prev)  # [N, d]\n    N = np.vstack(X_next)  # [N, d]\n    d = P.shape[1]\n    xtx = P.T @ P + ridge_lambda * np.eye(d)\n    xty = P.T @ N\n    A = np.linalg.solve(xtx, xty)  # [d, d]\n    return A\n\n\ndef residual_timeseries(X_norm: np.ndarray, A: np.ndarray, gain: float) -> np.ndarray:\n    """\n    Kalman-style fixed-gain synchronization:\n    z_pred = A z_prev\n    r_t    = x_t - z_pred\n    z_t    = z_pred + gain * r_t\n    """\n    T, d = X_norm.shape\n    if T < 2:\n        return np.zeros((0, d), dtype=float)\n    z = X_norm[0].copy()\n    residuals = []\n    for t in range(1, T):\n        z_pred = z @ A\n        r = X_norm[t] - z_pred\n        residuals.append(r)\n        z = z_pred + gain * r\n    return np.vstack(residuals)\n\n\ndef block_signatures(residual: np.ndarray, B: int) -> np.ndarray:\n    """\n    Signature per block: mean absolute residual over a sliding window.\n    """\n    if residual.shape[0] == 0:\n        return np.zeros((0, residual.shape[1]), dtype=float)\n    a = np.abs(residual)\n    T, d = a.shape\n    if T < B:\n        return np.mean(a, axis=0, keepdims=True)\n    cs = np.vstack([np.zeros((1, d)), np.cumsum(a, axis=0)])\n    out = (cs[B:] - cs[:-B]) / float(B)\n    return out\n\n\ndef fit_weights(signatures_fit: np.ndarray) -> np.ndarray:\n    sigma = np.std(signatures_fit, axis=0)\n    w = 1.0 / (sigma + 1e-6)\n    w = np.maximum(w, 0.0)\n    s = np.sum(w)\n    if s <= 0:\n        return np.ones_like(w) / len(w)\n    return w / s\n\n\ndef signature_scores(signatures: np.ndarray, weights: np.ndarray) -> np.ndarray:\n    if signatures.shape[0] == 0:\n        return np.zeros((0,), dtype=float)\n    return signatures @ weights\n\n\ndef conformal_threshold(cal_scores: np.ndarray, alpha: float) -> float:\n    sc = np.sort(np.asarray(cal_scores, dtype=float))\n    n = len(sc)\n    if n == 0:\n        return float("inf")\n    k = int(np.ceil((n + 1) * (1.0 - alpha)))\n    k = min(max(k, 1), n)\n    return float(sc[k - 1])\n\n\ndef conformal_pvals(cal_scores: np.ndarray, test_scores: np.ndarray) -> np.ndarray:\n    cal = np.asarray(cal_scores, dtype=float)\n    denom = len(cal) + 1.0\n    out = np.zeros(len(test_scores), dtype=float)\n    for i, s in enumerate(test_scores):\n        out[i] = (1.0 + np.sum(cal >= s)) / denom\n    return out\n\n\ndef persistent_alerts(alerts: np.ndarray, k: int) -> np.ndarray:\n    out = np.zeros(len(alerts), dtype=int)\n    run = 0\n    for i, a in enumerate(alerts.astype(bool)):\n        if a:\n            run += 1\n        else:\n            run = 0\n        out[i] = 1 if run >= k else 0\n    return out\n\n\ndef first_positive_index(x: np.ndarray) -> int:\n    idx = np.flatnonzero(np.asarray(x, dtype=bool))\n    return int(idx[0]) if len(idx) else -1\n\n\ndef normalize_positive_weights(x: np.ndarray) -> np.ndarray:\n    arr = np.asarray(x, dtype=float)\n    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)\n    arr = np.clip(arr, 0.0, None)\n    total = float(arr.sum())\n    if total <= 1e-12:\n        if len(arr) == 0:\n            return np.zeros((0,), dtype=float)\n        return np.full(len(arr), 1.0 / len(arr), dtype=float)\n    return arr / total\n\n\ndef diagnosis_feature_weights(feature_names: Sequence[str]) -> np.ndarray:\n    weights = np.ones(len(feature_names), dtype=float)\n    for idx, name in enumerate(feature_names):\n        base = str(name).split(":", 1)[-1].lower()\n        if any(tok in base for tok in DIAG_MONOTONIC_TOKENS):\n            weights[idx] = 0.0\n            continue\n        if any(tok in base for tok in DIAG_GENERIC_MEMORY_STATE_TOKENS):\n            weights[idx] *= DIAG_GENERIC_MEMORY_STATE_FACTOR\n    return weights\n\n\ndef post_alert_window_indices(\n    n_blocks: int,\n    start_idx: int,\n    window_blocks: int,\n) -> np.ndarray:\n    if n_blocks <= 0:\n        return np.zeros((0,), dtype=int)\n    start = min(max(int(start_idx), 0), n_blocks - 1)\n    width = min(n_blocks - start, max(1, int(window_blocks)))\n    return np.arange(start, start + width, dtype=int)\n\n\ndef select_diagnosis_block_indices(\n    scores: np.ndarray,\n    block_alert: np.ndarray,\n    persist: np.ndarray,\n    window_blocks: int = DIAG_POST_ALERT_BLOCKS,\n    min_blocks: int = DIAG_POST_ALERT_MIN_BLOCKS,\n) -> Tuple[np.ndarray, str]:\n    if len(scores) == 0:\n        return np.zeros((0,), dtype=int), "none"\n    if np.any(persist):\n        selected = post_alert_window_indices(len(scores), first_positive_index(persist), window_blocks)\n        mode = "post_persist"\n    elif np.any(block_alert):\n        selected = post_alert_window_indices(len(scores), first_positive_index(block_alert), window_blocks)\n        mode = "post_alert"\n    else:\n        peak_idx = int(np.argmax(scores))\n        selected = post_alert_window_indices(len(scores), peak_idx, window_blocks)\n        mode = "peak_window"\n\n    if len(selected) < min_blocks and len(scores) > len(selected):\n        order = np.argsort(scores)[::-1]\n        extra: List[int] = list(selected)\n        for idx in order:\n            idx_int = int(idx)\n            if idx_int in extra:\n                continue\n            extra.append(idx_int)\n            if len(extra) >= min(len(scores), min_blocks):\n                break\n        selected = np.asarray(sorted(extra), dtype=int)\n        mode = f"{mode}_backfill"\n    return selected, mode\n\n\ndef focused_diagnosis_signature(\n    signatures: np.ndarray,\n    scores: np.ndarray,\n    block_alert: np.ndarray,\n    persist: np.ndarray,\n    tau: float,\n) -> Tuple[np.ndarray, Dict[str, object]]:\n    if signatures.shape[0] == 0:\n        empty = np.zeros((signatures.shape[1] if signatures.ndim == 2 else 0,), dtype=float)\n        return empty, {\n            "diagnosis_block_mode": "none",\n            "diagnosis_selected_blocks": 0,\n            "diagnosis_selected_block_idxs": "",\n            "diagnosis_peak_block_idx": -1,\n            "diagnosis_peak_block_score": 0.0,\n            "diagnosis_focus_score_mean": 0.0,\n            "diagnosis_focus_score_max": 0.0,\n        }\n\n    selected_idx, mode = select_diagnosis_block_indices(scores, block_alert, persist)\n    peak_idx = int(np.argmax(scores)) if len(scores) else -1\n    if len(selected_idx) == 0:\n        selected_idx = np.asarray([peak_idx], dtype=int) if peak_idx >= 0 else np.zeros((0,), dtype=int)\n        mode = "peak"\n\n    selected_scores = scores[selected_idx]\n    excess = np.maximum(selected_scores - float(tau), 0.0)\n    if np.all(excess <= 1e-12):\n        excess = np.maximum(selected_scores, 0.0)\n    weights = normalize_positive_weights(excess)\n    focused = np.average(signatures[selected_idx], axis=0, weights=weights) if len(selected_idx) else np.zeros(signatures.shape[1], dtype=float)\n    meta = {\n        "diagnosis_block_mode": mode,\n        "diagnosis_selected_blocks": int(len(selected_idx)),\n        "diagnosis_selected_block_idxs": ",".join(str(int(i)) for i in selected_idx),\n        "diagnosis_peak_block_idx": peak_idx,\n        "diagnosis_peak_block_score": float(scores[peak_idx]) if peak_idx >= 0 else 0.0,\n        "diagnosis_focus_score_mean": float(np.mean(selected_scores)) if len(selected_scores) else 0.0,\n        "diagnosis_focus_score_max": float(np.max(selected_scores)) if len(selected_scores) else 0.0,\n        "_diagnosis_selected_idx": selected_idx.copy(),\n        "_diagnosis_selected_weights": weights.copy(),\n    }\n    return focused, meta\n\n\ndef finite_median(x: Sequence[float]) -> float:\n    arr = np.asarray(x, dtype=float)\n    arr = arr[np.isfinite(arr)]\n    if arr.size == 0:\n        return float("nan")\n    return float(np.median(arr))\n\n\ndef finite_percentile(x: Sequence[float], q: float) -> float:\n    arr = np.asarray(x, dtype=float)\n    arr = arr[np.isfinite(arr)]\n    if arr.size == 0:\n        return float("nan")\n    return float(np.percentile(arr, q))\n\n\ndef workload_conditioned_scores(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:\n    nominal_map = (\n        df[df["stressor"] == "NOMINAL"]\n        .drop_duplicates(subset=["workload"], keep="last")\n        .set_index("workload")["run_score"]\n        .to_dict()\n    )\n    nominal = df["workload"].map(nominal_map).to_numpy(dtype=float)\n    score = np.abs(df["run_score"].to_numpy(dtype=float) - nominal)\n    return nominal, score\n\n\ndef stressor_family(stressor: str) -> str:\n    return STRESSOR_FAMILY.get(str(stressor), "unknown")\n\n\ndef normalize_attribution_vector(x: np.ndarray) -> np.ndarray:\n    arr = np.asarray(x, dtype=float)\n    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)\n    arr = np.clip(arr, 0.0, None)\n    total = float(np.sum(arr))\n    if not np.isfinite(total) or total <= 1e-12:\n        return np.zeros_like(arr, dtype=float)\n    return arr / total\n\n\ndef hellinger_distance(p: np.ndarray, q: np.ndarray) -> float:\n    p_norm = normalize_attribution_vector(p)\n    q_norm = normalize_attribution_vector(q)\n    return float(np.linalg.norm(np.sqrt(p_norm) - np.sqrt(q_norm)) / SQRT2)\n\n\ndef safe_quantile(values: Sequence[float], q: float, default: float) -> float:\n    arr = np.asarray(values, dtype=float)\n    arr = arr[np.isfinite(arr)]\n    if arr.size == 0:\n        return float(default)\n    qq = min(max(float(q), 0.0), 1.0)\n    return float(np.quantile(arr, qq))\n\n\ndef multiclass_balanced_accuracy(\n    y_true: Sequence[str],\n    y_pred: Sequence[str],\n    labels: Sequence[str],\n) -> float:\n    cm = confusion_matrix(y_true, y_pred, labels=list(labels))\n    support = cm.sum(axis=1).astype(float)\n    recalls = np.divide(\n        np.diag(cm).astype(float),\n        support,\n        out=np.full(len(labels), np.nan, dtype=float),\n        where=support > 0,\n    )\n    valid = np.isfinite(recalls)\n    if not np.any(valid):\n        return float("nan")\n    return float(np.mean(recalls[valid]))\n\n\ndef mechanism_group(feature_name: str) -> str:\n    name = feature_name.split(":", 1)[-1].lower()\n    if any(tok in name for tok in ["temp", "power", "fan"]):\n        return "thermal_power"\n    if any(tok in name for tok in ["mem_", "swap_", "disk_", "net_", "wired_bytes", "active_bytes", "inactive_bytes"]):\n        return "memory_io"\n    if any(\n        tok in name\n        for tok in [\n            "ctx_switch",\n            "interrupt",\n            "syscall",\n            "pids_count",\n            "running_fraction",\n            "weight_ns",\n            "unique_process",\n            "unique_thread",\n            "samples_per_bucket",\n            "sentinel_count",\n            "core_id",\n        ]\n    ):\n        return "scheduler_runtime"\n    if any(tok in name for tok in ["load", "uptime", "available_bytes", "free_bytes", "mem_percent"]):\n        return "platform_pressure"\n    return "compute"\n\n\ndef mechanism_vector(feature_names: Sequence[str], feature_contrib: np.ndarray) -> Tuple[Dict[str, float], np.ndarray]:\n    totals = {group: 0.0 for group in MECHANISM_GROUPS}\n    for name, value in zip(feature_names, np.asarray(feature_contrib, dtype=float)):\n        totals[mechanism_group(name)] += float(value)\n    vec = np.array([totals[group] for group in MECHANISM_GROUPS], dtype=float)\n    return totals, vec\n\n\ndef attribution_centroid(vectors: Sequence[np.ndarray]) -> np.ndarray:\n    mats = [normalize_attribution_vector(v) for v in vectors]\n    if not mats:\n        raise ValueError("Cannot build a centroid from an empty vector list.")\n    return normalize_attribution_vector(np.median(np.vstack(mats), axis=0))\n\n\ndef build_label_centroids(\n    rows: Sequence[Dict[str, object]],\n    labels: Sequence[str],\n    vector_key: str,\n    label_fn,\n) -> Dict[str, np.ndarray]:\n    centroids: Dict[str, np.ndarray] = {}\n    for label in labels:\n        mats = [row[vector_key] for row in rows if label_fn(row) == label]\n        if mats:\n            centroids[label] = attribution_centroid(mats)\n    return centroids\n\n\ndef ordered_proto_distances(vec: np.ndarray, centroids: Dict[str, np.ndarray]) -> List[Tuple[str, float]]:\n    return sorted(\n        (\n            (label, hellinger_distance(vec, centroid))\n            for label, centroid in centroids.items()\n        ),\n        key=lambda item: item[1],\n    )\n\n\ndef margin_to_second(ordered: Sequence[Tuple[str, float]]) -> float:\n    if len(ordered) < 2:\n        return float("inf")\n    return float(ordered[1][1] - ordered[0][1])\n\n\ndef hierarchical_prediction_info(\n    row: Dict[str, object],\n    stressor_centroids: Dict[str, np.ndarray],\n    family_centroids: Dict[str, np.ndarray],\n    family_margin_tau: float,\n    feature_vector_key: str,\n) -> Dict[str, object]:\n    feature_vec = normalize_attribution_vector(row[feature_vector_key])\n    family_vec = normalize_attribution_vector(row["_mechanism_vector_norm"])\n    feature_ordered = ordered_proto_distances(feature_vec, stressor_centroids)\n    family_ordered = ordered_proto_distances(family_vec, family_centroids)\n    if not feature_ordered or not family_ordered:\n        raise ValueError("Hierarchical diagnosis requires both stressor and family centroids.")\n\n    best_family = family_ordered[0][0]\n    family_margin = margin_to_second(family_ordered)\n    if np.isfinite(family_margin_tau) and family_margin >= family_margin_tau:\n        in_family = [item for item in feature_ordered if stressor_family(item[0]) == best_family]\n        out_family = [item for item in feature_ordered if stressor_family(item[0]) != best_family]\n        ordered = in_family + out_family if in_family else feature_ordered\n    else:\n        ordered = feature_ordered\n\n    pred = ordered[0][0]\n    pred_family = stressor_family(pred)\n    exact_margin = margin_to_second(ordered)\n    nearest_distance = float(ordered[0][1])\n    confidence_score = float(exact_margin / (nearest_distance + 1e-6) + 0.5 * family_margin)\n    return {\n        "pred_stressor": pred,\n        "pred_family": pred_family,\n        "top2_labels": [label for label, _ in ordered[:2]],\n        "nearest_distance": nearest_distance,\n        "margin_to_second": exact_margin,\n        "family_pred": best_family,\n        "family_nearest_distance": float(family_ordered[0][1]),\n        "family_margin": family_margin,\n        "family_consistent": int(pred_family == best_family),\n        "confidence_score": confidence_score,\n    }\n\n\ndef fit_hierarchical_gate(\n    train_rows: Sequence[Dict[str, object]],\n    feature_vector_key: str = "_feature_contrib_norm",\n) -> Dict[str, float]:\n    family_correct_margins: List[float] = []\n    for idx, row in enumerate(train_rows):\n        ref = [train_rows[j] for j in range(len(train_rows)) if j != idx]\n        family_centroids = build_label_centroids(\n            ref,\n            DIAGNOSIS_FAMILIES,\n            "_mechanism_vector_norm",\n            lambda item: stressor_family(str(item["stressor"])),\n        )\n        if len(family_centroids) < 2:\n            continue\n        ordered = ordered_proto_distances(row["_mechanism_vector_norm"], family_centroids)\n        if ordered and ordered[0][0] == stressor_family(str(row["stressor"])):\n            family_correct_margins.append(margin_to_second(ordered))\n\n    family_margin_tau = safe_quantile(\n        family_correct_margins,\n        DIAG_FAMILY_GATE_QUANTILE,\n        default=float("inf"),\n    )\n\n    confidence_scores: List[float] = []\n    for idx, row in enumerate(train_rows):\n        ref = [train_rows[j] for j in range(len(train_rows)) if j != idx]\n        stressor_centroids = build_label_centroids(\n            ref,\n            ANOMALIES,\n            feature_vector_key,\n            lambda item: str(item["stressor"]),\n        )\n        family_centroids = build_label_centroids(\n            ref,\n            DIAGNOSIS_FAMILIES,\n            "_mechanism_vector_norm",\n            lambda item: stressor_family(str(item["stressor"])),\n        )\n        if len(stressor_centroids) < 2 or len(family_centroids) < 2:\n            continue\n        info = hierarchical_prediction_info(\n            row,\n            stressor_centroids,\n            family_centroids,\n            family_margin_tau,\n            feature_vector_key=feature_vector_key,\n        )\n        if info["pred_stressor"] == str(row["stressor"]):\n            confidence_scores.append(float(info["confidence_score"]))\n\n    confidence_tau = safe_quantile(\n        confidence_scores,\n        DIAG_CONFIDENCE_GATE_QUANTILE,\n        default=0.0,\n    )\n    return {\n        "family_margin_tau": float(family_margin_tau),\n        "confidence_tau": float(confidence_tau),\n    }\n\n\ndef train_bundle(\n    train_benign_runs: Dict[str, np.ndarray],\n    feature_names: List[str],\n    fit_ratio: float,\n    B: int,\n    alpha: float,\n    gain: float,\n    ridge_lambda: float,\n) -> ModelBundle:\n    fit_runs = []\n    cal_runs = []\n    fit_samples = []\n\n    for _, X in train_benign_runs.items():\n        n = len(X)\n        split = int(max(2, min(n - 1, round(n * fit_ratio))))\n        X_fit = X[:split]\n        X_cal = X[split:]\n        fit_runs.append(X_fit)\n        cal_runs.append(X_cal if len(X_cal) > 1 else X_fit[-2:])\n        fit_samples.append(X_fit)\n\n    X_fit_all = np.vstack(fit_samples)\n    med, scale = robust_fit_matrix(X_fit_all)\n\n    fit_norm = [(x - med) / (scale + 1e-12) for x in fit_runs]\n    cal_norm = [(x - med) / (scale + 1e-12) for x in cal_runs]\n\n    A = fit_linear_dynamics(fit_norm, ridge_lambda=ridge_lambda)\n\n    sig_fit = []\n    for X in fit_norm:\n        r = residual_timeseries(X, A, gain=gain)\n        s = block_signatures(r, B=B)\n        if len(s):\n            sig_fit.append(s)\n    sig_fit_all = np.vstack(sig_fit)\n    w = fit_weights(sig_fit_all)\n\n    cal_scores = []\n    for X in cal_norm:\n        r = residual_timeseries(X, A, gain=gain)\n        s = block_signatures(r, B=B)\n        sc = signature_scores(s, w)\n        if len(sc):\n            cal_scores.append(sc)\n    cal_scores_all = np.concatenate(cal_scores)\n    tau = conformal_threshold(cal_scores_all, alpha=alpha)\n\n    return ModelBundle(\n        feature_names=feature_names,\n        median=med,\n        scale=scale,\n        A=A,\n        weights=w,\n        diagnosis_weights=diagnosis_feature_weights(feature_names),\n        cal_scores=cal_scores_all,\n        tau=tau,\n    )\n\n\ndef evaluate_run(\n    X_run: np.ndarray,\n    bundle: ModelBundle,\n    B: int,\n    alpha: float,\n    persist_k: int,\n    gain: float,\n) -> Tuple[Dict[str, float], np.ndarray, List[Dict[str, object]], Dict[str, object]]:\n    Xn = (X_run - bundle.median) / (bundle.scale + 1e-12)\n    r = residual_timeseries(Xn, bundle.A, gain=gain)\n    sig = block_signatures(r, B=B)\n    sc = signature_scores(sig, bundle.weights)\n    pv = conformal_pvals(bundle.cal_scores, sc)\n\n    block_alert = pv < alpha\n    persist = persistent_alerts(block_alert, k=persist_k)\n\n    run_alert = int(np.any(persist > 0))\n    peak_score = float(np.max(sc)) if len(sc) else 0.0\n    run_score = peak_score\n    run_signature = np.nanmedian(sig, axis=0) if len(sig) else np.zeros_like(bundle.weights)\n    peak_signature = sig[int(np.argmax(sc))] if len(sig) else np.zeros_like(bundle.weights)\n    focused_signature, diagnosis_meta = focused_diagnosis_signature(\n        sig,\n        sc,\n        block_alert,\n        persist,\n        tau=bundle.tau,\n    )\n    diag_weights = np.asarray(bundle.diagnosis_weights, dtype=float)\n    feature_contrib_run = (run_signature * bundle.weights) * diag_weights\n    feature_contrib_peak = (peak_signature * bundle.weights) * diag_weights\n    feature_contrib_focus = (focused_signature * bundle.weights) * diag_weights\n    first_block_idx = first_positive_index(block_alert)\n    first_persist_idx = first_positive_index(persist)\n    block_time_s = float(B + first_block_idx) if first_block_idx >= 0 else float("nan")\n    persist_time_s = float(B + first_persist_idx) if first_persist_idx >= 0 else float("nan")\n    n_blocks = int(len(sc))\n    duration_s = max(n_blocks, 1)\n\n    selected_lookup = {\n        int(idx): float(weight)\n        for idx, weight in zip(\n            diagnosis_meta.pop("_diagnosis_selected_idx", np.zeros((0,), dtype=int)),\n            diagnosis_meta.pop("_diagnosis_selected_weights", np.zeros((0,), dtype=float)),\n        )\n    }\n    block_records: List[Dict[str, object]] = []\n    for i in range(len(sc)):\n        contrib = sig[i] * bundle.weights\n        total = float(np.sum(contrib))\n        tier_totals = {tier: 0.0 for tier in ("tier0", "tier1_alt", "tier2")}\n        for name, value in zip(bundle.feature_names, contrib):\n            tier = name.split(":", 1)[0]\n            if tier in tier_totals:\n                tier_totals[tier] += float(value)\n        dominant_tier = max(tier_totals, key=tier_totals.get) if total > 0 else "none"\n        mech_totals, _ = mechanism_vector(bundle.feature_names, contrib)\n        dominant_mechanism = max(mech_totals, key=mech_totals.get) if total > 0 else "none"\n        block_records.append(\n            {\n                "block_idx": int(i),\n                "block_start_s": float(i),\n                "block_end_s": float(B + i),\n                "score": float(sc[i]),\n                "pvalue": float(pv[i]),\n                "block_alert": int(block_alert[i]),\n                "persist_alert": int(persist[i]),\n                "is_selected_for_diagnosis": int(i in selected_lookup),\n                "diagnosis_selection_weight": float(selected_lookup.get(i, 0.0)),\n                "threshold": float(bundle.tau),\n                "dominant_tier": dominant_tier,\n                "dominant_mechanism": dominant_mechanism,\n                "top_feature_1": "",\n                "top_feature_score_1": 0.0,\n                "top_feature_2": "",\n                "top_feature_score_2": 0.0,\n                "top_feature_3": "",\n                "top_feature_score_3": 0.0,\n            }\n        )\n        order = np.argsort(contrib)[::-1][:3]\n        for rank in range(3):\n            if rank < len(order) and contrib[order[rank]] > 0.0:\n                idx = int(order[rank])\n                block_records[-1][f"top_feature_{rank + 1}"] = bundle.feature_names[idx]\n                block_records[-1][f"top_feature_score_{rank + 1}"] = float(contrib[idx])\n\n    return (\n        {\n            "run_score": run_score,\n            "run_alert": run_alert,\n            "min_pvalue": float(np.min(pv)) if len(pv) else 1.0,\n            "peak_block_score": peak_score,\n            "n_blocks": n_blocks,\n            "n_block_alerts": int(np.sum(block_alert)),\n            "n_persist_alerts": int(np.sum(persist)),\n            "first_block_alert_idx": first_block_idx,\n            "first_persist_alert_idx": first_persist_idx,\n            "first_block_alert_s": block_time_s,\n            "time_to_detect_s": persist_time_s,\n            "block_alerts_per_hour": float(np.sum(block_alert) * 3600.0 / duration_s),\n            "persist_alerts_per_hour": float(np.sum(persist) * 3600.0 / duration_s),\n        },\n        feature_contrib_run,\n        block_records,\n        {\n            **diagnosis_meta,\n            "_feature_contrib_run": feature_contrib_run.copy(),\n            "_feature_contrib_peak": feature_contrib_peak.copy(),\n            "_feature_contrib_focus": feature_contrib_focus.copy(),\n        },\n    )\n\n\ndef to_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:\n    body = df.to_latex(index=False, escape=False, float_format=lambda x: f"{x:.4f}")\n    return (\n        "\\\\begin{table}[t]\\n"\n        "\\\\centering\\n"\n        f"\\\\caption{{{caption}}}\\n"\n        f"\\\\label{{{label}}}\\n"\n        "\\\\footnotesize\\n"\n        f"{body}\\n"\n        "\\\\end{table}\\n"\n    )\n\n\ndef _cfg_label(cfg: str) -> str:\n    return CONFIG_PRETTY.get(cfg, cfg.replace("_", " + "))\n\n\ndef _cfg_color(cfg: str) -> str:\n    return CONFIG_COLORS.get(cfg, "#4E79A7")\n\n\ndef _stressor_color(stressor: str) -> str:\n    return STRESSOR_COLORS.get(stressor, "#4E79A7")\n\n\ndef _ternary_xy(share0: float, share1: float, share2: float) -> Tuple[float, float]:\n    total = max(float(share0 + share1 + share2), 1e-12)\n    a = float(share0) / total\n    b = float(share1) / total\n    c = float(share2) / total\n    return b + 0.5 * c, c * SQRT3 / 2.0\n\n\ndef _setup_ternary_axis(ax: plt.Axes, labels: Tuple[str, str, str]) -> None:\n    verts = np.array([[0.0, 0.0], [1.0, 0.0], [0.5, SQRT3 / 2.0]])\n    ax.add_patch(Polygon(verts, closed=True, fill=False, edgecolor="#334155", linewidth=1.8))\n    for frac in (0.2, 0.4, 0.6, 0.8):\n        for p1, p2 in [\n            (_ternary_xy(frac, 0.0, 1.0 - frac), _ternary_xy(frac, 1.0 - frac, 0.0)),\n            (_ternary_xy(0.0, frac, 1.0 - frac), _ternary_xy(1.0 - frac, frac, 0.0)),\n            (_ternary_xy(0.0, 1.0 - frac, frac), _ternary_xy(1.0 - frac, 0.0, frac)),\n        ]:\n            ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="#CBD5E1", linewidth=0.8, zorder=0)\n    ax.text(-0.06, -0.06, labels[0], fontsize=LABEL_SIZE, fontweight="bold", ha="right", va="top")\n    ax.text(1.06, -0.06, labels[1], fontsize=LABEL_SIZE, fontweight="bold", ha="left", va="top")\n    ax.text(0.5, SQRT3 / 2.0 + 0.06, labels[2], fontsize=LABEL_SIZE, fontweight="bold", ha="center")\n    ax.text(0.5, -0.12, "Closer to a corner means more evidence from that tier.", fontsize=11, ha="center", color="#475569")\n    ax.set_xlim(-0.10, 1.10)\n    ax.set_ylim(-0.15, SQRT3 / 2.0 + 0.12)\n    ax.set_aspect("equal")\n    ax.axis("off")\n\n\ndef plot_curves(df: pd.DataFrame, out_png: Path, score_col: str, title_tag: str) -> None:\n    fig, axes = plt.subplots(1, 2, figsize=(14.8, 6.0))\n    roc_metrics: List[Tuple[str, str]] = []\n    pr_metrics: List[Tuple[str, str]] = []\n    for cfg, d in df.groupby("config", sort=False):\n        y = d["label"].to_numpy(dtype=int)\n        s = d[score_col].to_numpy(dtype=float)\n        if len(np.unique(y)) < 2:\n            continue\n        fpr, tpr, _ = roc_curve(y, s)\n        p, r, _ = precision_recall_curve(y, s)\n        color = _cfg_color(cfg)\n        roc_val = roc_auc_score(y, s)\n        ap_val = average_precision_score(y, s)\n        axes[0].plot(fpr, tpr, linewidth=3.0, color=color, solid_capstyle="round")\n        axes[0].fill_between(fpr, tpr, 0, color=color, alpha=0.08)\n        axes[1].plot(r, p, linewidth=3.0, color=color, solid_capstyle="round")\n        roc_metrics.append((color, f"{_cfg_label(cfg)}  ROC {roc_val:.3f}"))\n        pr_metrics.append((color, f"{_cfg_label(cfg)}  AP {ap_val:.3f}"))\n    axes[0].plot([0, 1], [0, 1], linestyle=(0, (4, 4)), color="#94A3B8", linewidth=1.2)\n    axes[0].set_title("ROC curve", fontsize=TITLE_SIZE)\n    axes[0].set_xlabel("False Positive Rate", fontsize=LABEL_SIZE)\n    axes[0].set_ylabel("True Positive Rate", fontsize=LABEL_SIZE)\n    axes[1].set_title("Precision-Recall curve", fontsize=TITLE_SIZE)\n    axes[1].set_xlabel("Recall", fontsize=LABEL_SIZE)\n    axes[1].set_ylabel("Precision", fontsize=LABEL_SIZE)\n    for ax in axes:\n        ax.set_xlim(-0.02, 1.02)\n        ax.set_ylim(-0.02, 1.02)\n        ax.grid(alpha=0.25)\n        ax.tick_params(labelsize=TICK_SIZE)\n    for idx, (color, text) in enumerate(roc_metrics):\n        axes[0].text(\n            0.0,\n            -0.20 - idx * 0.09,\n            text,\n            transform=axes[0].transAxes,\n            color=color,\n            fontsize=ANNOTATION_SIZE + 1,\n            fontweight="bold",\n            ha="left",\n            va="top",\n        )\n    for idx, (color, text) in enumerate(pr_metrics):\n        axes[1].text(\n            0.0,\n            -0.20 - idx * 0.09,\n            text,\n            transform=axes[1].transAxes,\n            color=color,\n            fontsize=ANNOTATION_SIZE + 1,\n            fontweight="bold",\n            ha="left",\n            va="top",\n        )\n    fig.text(\n        0.5,\n        0.03,\n        "Shaded area highlights stronger separation; workload-conditioned curves overlap because all heads are perfect there.",\n        ha="center",\n        fontsize=12,\n        color="#475569",\n    )\n    fig.suptitle(f"DICE curves ({title_tag})", fontsize=TITLE_SIZE + 1, fontweight="bold")\n    fig.tight_layout(rect=[0.0, 0.16, 1.0, 0.92])\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef plot_score_box(df: pd.DataFrame, out_png: Path, score_col: str, y_label: str, title_tag: str) -> None:\n    cfgs = list(df["config"].unique())\n    fig, axes = plt.subplots(1, len(cfgs), figsize=(5.4 * len(cfgs), 5.4), sharey=False)\n    if len(cfgs) == 1:\n        axes = [axes]\n    rng = np.random.default_rng(0)\n    for i, cfg in enumerate(cfgs):\n        ax = axes[i]\n        d = df[df["config"] == cfg]\n        neg = d[d["label"] == 0][score_col].to_numpy(dtype=float)\n        pos = d[d["label"] == 1][score_col].to_numpy(dtype=float)\n        parts = ax.violinplot(\n            [neg, pos],\n            positions=[1, 2],\n            widths=0.82,\n            showmeans=False,\n            showmedians=False,\n            showextrema=False,\n        )\n        for body, color in zip(parts["bodies"], ["#b0bec5", "#ef9a9a"]):\n            body.set_facecolor(color)\n            body.set_edgecolor("black")\n            body.set_alpha(0.75)\n        for xpos, vals, color, edge in [(1, neg, "#0F172A", "white"), (2, pos, "#C62828", "white")]:\n            jitter = rng.uniform(-0.07, 0.07, size=len(vals))\n            ax.scatter(\n                np.full(len(vals), xpos) + jitter,\n                vals,\n                color=color,\n                s=42,\n                alpha=0.72,\n                edgecolor=edge,\n                linewidth=0.4,\n                zorder=3,\n            )\n            if len(vals):\n                q1, med, q3 = np.percentile(vals, [25, 50, 75])\n                ax.vlines(xpos, q1, q3, color=color, linewidth=6, alpha=0.82, zorder=4)\n                ax.hlines(med, xpos - 0.18, xpos + 0.18, color="white", linewidth=2.4, zorder=5)\n        ax.set_xticks([1, 2], labels=["Benign", "Anomaly"])\n        ax.set_title(_cfg_label(cfg), fontsize=TITLE_SIZE + 2)\n        ax.tick_params(labelsize=TICK_SIZE)\n        ax.grid(alpha=0.22)\n        vals_all = np.concatenate([neg, pos]) if len(neg) or len(pos) else np.array([])\n        if len(vals_all) and np.all(vals_all > 0):\n            ax.set_yscale("log")\n    fig.supylabel(y_label, fontsize=LABEL_SIZE)\n    fig.suptitle(f"DICE run score distributions ({title_tag})", fontsize=TITLE_SIZE + 2, fontweight="bold")\n    fig.tight_layout(rect=[0.04, 0.0, 1.0, 0.94])\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef build_diagnostic_record(\n    config: str,\n    holdout_workload: str,\n    case: CaseRef,\n    feature_names: Sequence[str],\n    feature_contrib: np.ndarray,\n    diagnosis_meta: Dict[str, object] | None = None,\n) -> Dict[str, object]:\n    contrib = np.asarray(feature_contrib, dtype=float)\n    contrib_norm = normalize_attribution_vector(contrib)\n    diagnosis_meta = diagnosis_meta or {}\n    contrib_run = np.asarray(diagnosis_meta.get("_feature_contrib_run", contrib), dtype=float)\n    contrib_peak = np.asarray(diagnosis_meta.get("_feature_contrib_peak", np.zeros_like(contrib)), dtype=float)\n    contrib_focus = np.asarray(diagnosis_meta.get("_feature_contrib_focus", np.zeros_like(contrib)), dtype=float)\n    total = float(np.sum(contrib))\n    # Track contributions over the released three-tier observation hierarchy.\n    tier_totals = {tier: 0.0 for tier in ("tier0", "tier1_alt", "tier2")}\n    for name, value in zip(feature_names, contrib):\n        tier = name.split(":", 1)[0]\n        if tier in tier_totals:\n            tier_totals[tier] += float(value)\n    dominant_tier = max(tier_totals, key=tier_totals.get) if total > 0 else "none"\n    mech_totals, mech_vec = mechanism_vector(feature_names, contrib)\n    mech_vec_norm = normalize_attribution_vector(mech_vec)\n    dominant_mechanism = max(mech_totals, key=mech_totals.get) if total > 0 else "none"\n    order = np.argsort(contrib)[::-1][:DIAG_TOP_K]\n    mech_order = np.argsort(mech_vec)[::-1][:3]\n\n    row: Dict[str, object] = {\n        "config": config,\n        "holdout_workload": holdout_workload,\n        "case_id": case.case_id,\n        "workload": case.workload,\n        "stressor": case.stressor,\n        "stressor_family": stressor_family(case.stressor),\n        "label": case.label,\n        "diagnosis_block_mode": str(diagnosis_meta.get("diagnosis_block_mode", "none")),\n        "diagnosis_selected_blocks": int(diagnosis_meta.get("diagnosis_selected_blocks", 0)),\n        "diagnosis_selected_block_idxs": str(diagnosis_meta.get("diagnosis_selected_block_idxs", "")),\n        "diagnosis_peak_block_idx": int(diagnosis_meta.get("diagnosis_peak_block_idx", -1)),\n        "diagnosis_peak_block_score": float(diagnosis_meta.get("diagnosis_peak_block_score", 0.0)),\n        "diagnosis_focus_score_mean": float(diagnosis_meta.get("diagnosis_focus_score_mean", 0.0)),\n        "diagnosis_focus_score_max": float(diagnosis_meta.get("diagnosis_focus_score_max", 0.0)),\n        "dominant_tier": dominant_tier,\n        "tier0_contrib": float(tier_totals["tier0"]),\n        "tier1_alt_contrib": float(tier_totals["tier1_alt"]),\n        "tier2_contrib": float(tier_totals["tier2"]),\n        "tier0_share": float(tier_totals["tier0"] / total) if total > 0 else 0.0,\n        "tier1_alt_share": float(tier_totals["tier1_alt"] / total) if total > 0 else 0.0,\n        "tier2_share": float(tier_totals["tier2"] / total) if total > 0 else 0.0,\n        "dominant_mechanism": dominant_mechanism,\n        "_feature_contrib": contrib.copy(),\n        "_feature_contrib_norm": contrib_norm.copy(),\n        "_feature_contrib_run": contrib_run.copy(),\n        "_feature_contrib_run_norm": normalize_attribution_vector(contrib_run),\n        "_feature_contrib_peak": contrib_peak.copy(),\n        "_feature_contrib_peak_norm": normalize_attribution_vector(contrib_peak),\n        "_feature_contrib_focus": contrib_focus.copy(),\n        "_feature_contrib_focus_norm": normalize_attribution_vector(contrib_focus),\n        "_mechanism_vector": mech_vec.copy(),\n        "_mechanism_vector_norm": mech_vec_norm.copy(),\n    }\n    for group in MECHANISM_GROUPS:\n        row[f"{group}_contrib"] = float(mech_totals[group])\n        row[f"{group}_share"] = float(mech_totals[group] / total) if total > 0 else 0.0\n    for rank in range(DIAG_TOP_K):\n        key_name = f"top_feature_{rank + 1}"\n        key_score = f"top_feature_score_{rank + 1}"\n        if rank < len(order) and contrib[order[rank]] > 0.0:\n            idx = int(order[rank])\n            row[key_name] = feature_names[idx]\n            row[key_score] = float(contrib[idx])\n        else:\n            row[key_name] = ""\n            row[key_score] = 0.0\n    for rank in range(3):\n        key_name = f"top_mechanism_{rank + 1}"\n        key_score = f"top_mechanism_score_{rank + 1}"\n        if rank < len(mech_order) and mech_vec[mech_order[rank]] > 0.0:\n            idx = int(mech_order[rank])\n            row[key_name] = MECHANISM_GROUPS[idx]\n            row[key_score] = float(mech_vec[idx])\n        else:\n            row[key_name] = ""\n            row[key_score] = 0.0\n    return row\n\n\ndef append_case_outputs(\n    preds: List[Dict[str, object]],\n    diagnostic_records: List[Dict[str, object]],\n    block_records: List[Dict[str, object]],\n    config: str,\n    holdout_workload: str,\n    case: CaseRef,\n    X_run: np.ndarray,\n    bundle: ModelBundle,\n    B: int,\n    alpha: float,\n    persist_k: int,\n    gain: float,\n) -> None:\n    metrics, feature_contrib, case_block_records, diagnosis_meta = evaluate_run(\n        X_run,\n        bundle,\n        B=B,\n        alpha=alpha,\n        persist_k=persist_k,\n        gain=gain,\n    )\n    preds.append(\n        {\n            "config": config,\n            "holdout_workload": holdout_workload,\n            "case_id": case.case_id,\n            "workload": case.workload,\n            "stressor": case.stressor,\n            "label": case.label,\n            **metrics,\n            "n_features": len(bundle.feature_names),\n            "tau": bundle.tau,\n        }\n    )\n    for row in case_block_records:\n        block_records.append(\n            {\n                "config": config,\n                "holdout_workload": holdout_workload,\n                "case_id": case.case_id,\n                "workload": case.workload,\n                "stressor": case.stressor,\n                "label": case.label,\n                **row,\n            }\n        )\n    diagnostic_records.append(\n        build_diagnostic_record(\n            config=config,\n            holdout_workload=holdout_workload,\n            case=case,\n            feature_names=bundle.feature_names,\n            feature_contrib=feature_contrib,\n            diagnosis_meta=diagnosis_meta,\n        )\n    )\n\n\ndef build_stressor_attribution(\n    diagnostic_records: Sequence[Dict[str, object]],\n    vector_key: str,\n) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    pred_rows: List[Dict[str, object]] = []\n    for cfg_name in CONFIGS:\n        cfg_records = [r for r in diagnostic_records if r["config"] == cfg_name and int(r["label"]) == 1]\n        for holdout_w in WORKLOADS:\n            train = [r for r in cfg_records if r["workload"] != holdout_w]\n            test = [r for r in cfg_records if r["workload"] == holdout_w]\n            centroids = build_label_centroids(\n                train,\n                ANOMALIES,\n                vector_key,\n                lambda row: str(row["stressor"]),\n            )\n            if len(centroids) < 2:\n                continue\n            for row in test:\n                truth = str(row["stressor"])\n                contrib = normalize_attribution_vector(row[vector_key])\n                ordered = ordered_proto_distances(contrib, centroids)\n                pred = ordered[0][0]\n                top2 = [label for label, _ in ordered[:2]]\n                pred_rows.append(\n                    {\n                        "config": cfg_name,\n                        "holdout_workload": holdout_w,\n                        "case_id": row["case_id"],\n                        "true_stressor": truth,\n                        "pred_stressor": pred,\n                        "is_correct": int(pred == truth),\n                        "top2_hit": int(truth in top2),\n                        "nearest_distance": float(ordered[0][1]),\n                        "margin_to_second": margin_to_second(ordered),\n                    }\n                )\n\n    pred_df = pd.DataFrame(pred_rows)\n    if pred_df.empty:\n        empty_metrics = pd.DataFrame(\n            columns=[\n                "config",\n                "n_cases",\n                "top1_acc",\n                "top2_acc",\n                "balanced_acc",\n                "macro_f1",\n                "mean_margin_to_second",\n                "median_margin_to_second",\n            ]\n        )\n        empty_cm = pd.DataFrame(index=ANOMALIES, columns=ANOMALIES, data=0)\n        empty_cm.index.name = "true_stressor"\n        empty_cm.columns.name = "pred_stressor"\n        return pred_df, empty_metrics, empty_cm\n    pred_df = pred_df.sort_values(["config", "holdout_workload", "case_id"])\n\n    metric_rows = []\n    for cfg_name, d in pred_df.groupby("config", sort=False):\n        margins = d["margin_to_second"].replace([np.inf, -np.inf], np.nan)\n        metric_rows.append(\n            {\n                "config": cfg_name,\n                "n_cases": int(len(d)),\n                "top1_acc": float(d["is_correct"].mean()),\n                "top2_acc": float(d["top2_hit"].mean()),\n                "balanced_acc": multiclass_balanced_accuracy(\n                    d["true_stressor"],\n                    d["pred_stressor"],\n                    ANOMALIES,\n                ),\n                "macro_f1": float(\n                    f1_score(\n                        d["true_stressor"],\n                        d["pred_stressor"],\n                        labels=ANOMALIES,\n                        average="macro",\n                        zero_division=0,\n                    )\n                ),\n                "mean_margin_to_second": float(margins.mean()),\n                "median_margin_to_second": float(margins.median()),\n            }\n        )\n    metrics_df = pd.DataFrame(metric_rows).sort_values("config")\n\n    final_cfg = "tier0_tier1_tier2"\n    d_final = pred_df[pred_df["config"] == final_cfg]\n    if d_final.empty:\n        cm = pd.DataFrame(index=ANOMALIES, columns=ANOMALIES, data=0)\n    else:\n        cm_arr = confusion_matrix(\n            d_final["true_stressor"],\n            d_final["pred_stressor"],\n            labels=ANOMALIES,\n        )\n        cm = pd.DataFrame(cm_arr, index=ANOMALIES, columns=ANOMALIES)\n    cm.index.name = "true_stressor"\n    cm.columns.name = "pred_stressor"\n    return pred_df, metrics_df, cm\n\n\ndef build_hierarchical_stressor_attribution(\n    diagnostic_records: Sequence[Dict[str, object]],\n    feature_vector_key: str = "_feature_contrib_norm",\n) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    pred_rows: List[Dict[str, object]] = []\n    for cfg_name in CONFIGS:\n        cfg_records = [r for r in diagnostic_records if r["config"] == cfg_name and int(r["label"]) == 1]\n        for holdout_w in WORKLOADS:\n            train = [r for r in cfg_records if r["workload"] != holdout_w]\n            test = [r for r in cfg_records if r["workload"] == holdout_w]\n            stressor_centroids = build_label_centroids(\n                train,\n                ANOMALIES,\n                feature_vector_key,\n                lambda row: str(row["stressor"]),\n            )\n            family_centroids = build_label_centroids(\n                train,\n                DIAGNOSIS_FAMILIES,\n                "_mechanism_vector_norm",\n                lambda row: stressor_family(str(row["stressor"])),\n            )\n            if len(stressor_centroids) < 2 or len(family_centroids) < 2:\n                continue\n            gate = fit_hierarchical_gate(train, feature_vector_key=feature_vector_key)\n            for row in test:\n                truth = str(row["stressor"])\n                truth_family = stressor_family(truth)\n                info = hierarchical_prediction_info(\n                    row,\n                    stressor_centroids,\n                    family_centroids,\n                    gate["family_margin_tau"],\n                    feature_vector_key=feature_vector_key,\n                )\n                accepted = int(float(info["confidence_score"]) >= float(gate["confidence_tau"]))\n                pred_rows.append(\n                    {\n                        "config": cfg_name,\n                        "holdout_workload": holdout_w,\n                        "case_id": row["case_id"],\n                        "true_stressor": truth,\n                        "true_family": truth_family,\n                        "pred_stressor": info["pred_stressor"],\n                        "pred_family": info["pred_family"],\n                        "family_pred": info["family_pred"],\n                        "top2_labels": "|".join(info["top2_labels"]),\n                        "is_correct": int(info["pred_stressor"] == truth),\n                        "top2_hit": int(truth in info["top2_labels"]),\n                        "family_correct": int(info["family_pred"] == truth_family),\n                        "family_consistent": int(info["family_consistent"]),\n                        "nearest_distance": float(info["nearest_distance"]),\n                        "margin_to_second": float(info["margin_to_second"]),\n                        "family_nearest_distance": float(info["family_nearest_distance"]),\n                        "family_margin": float(info["family_margin"]),\n                        "confidence_score": float(info["confidence_score"]),\n                        "family_margin_tau": float(gate["family_margin_tau"]),\n                        "confidence_tau": float(gate["confidence_tau"]),\n                        "abstained": int(1 - accepted),\n                        "pred_stressor_selective": info["pred_stressor"] if accepted else "ABSTAIN",\n                    }\n                )\n\n    pred_df = pd.DataFrame(pred_rows)\n    if pred_df.empty:\n        empty_metrics = pd.DataFrame(\n            columns=[\n                "config",\n                "n_cases",\n                "top1_acc",\n                "top2_acc",\n                "family_acc",\n                "balanced_acc",\n                "macro_f1",\n                "coverage",\n                "abstain_rate",\n                "selective_top1_acc",\n                "selective_top2_acc",\n                "selective_balanced_acc",\n                "selective_macro_f1",\n                "mean_margin_to_second",\n                "median_margin_to_second",\n                "mean_family_margin",\n                "median_family_margin",\n                "family_margin_tau",\n                "confidence_tau",\n            ]\n        )\n        empty_cm = pd.DataFrame(index=ANOMALIES, columns=[*ANOMALIES, "ABSTAIN"], data=0)\n        empty_cm.index.name = "true_stressor"\n        empty_cm.columns.name = "pred_stressor"\n        return pred_df, empty_metrics, empty_cm\n    pred_df = pred_df.sort_values(["config", "holdout_workload", "case_id"]).reset_index(drop=True)\n\n    metric_rows = []\n    for cfg_name, d in pred_df.groupby("config", sort=False):\n        margins = d["margin_to_second"].replace([np.inf, -np.inf], np.nan)\n        family_margins = d["family_margin"].replace([np.inf, -np.inf], np.nan)\n        accepted = d[d["abstained"] == 0].copy()\n        row = {\n            "config": cfg_name,\n            "n_cases": int(len(d)),\n            "top1_acc": float(d["is_correct"].mean()),\n            "top2_acc": float(d["top2_hit"].mean()),\n            "family_acc": float(d["family_correct"].mean()),\n            "balanced_acc": multiclass_balanced_accuracy(\n                d["true_stressor"],\n                d["pred_stressor"],\n                ANOMALIES,\n            ),\n            "macro_f1": float(\n                f1_score(\n                    d["true_stressor"],\n                    d["pred_stressor"],\n                    labels=ANOMALIES,\n                    average="macro",\n                    zero_division=0,\n                )\n            ),\n            "coverage": float((d["abstained"] == 0).mean()),\n            "abstain_rate": float(d["abstained"].mean()),\n            "selective_top1_acc": np.nan,\n            "selective_top2_acc": np.nan,\n            "selective_balanced_acc": np.nan,\n            "selective_macro_f1": np.nan,\n            "mean_margin_to_second": float(margins.mean()),\n            "median_margin_to_second": float(margins.median()),\n            "mean_family_margin": float(family_margins.mean()),\n            "median_family_margin": float(family_margins.median()),\n            "family_margin_tau": float(d["family_margin_tau"].replace([np.inf, -np.inf], np.nan).median()),\n            "confidence_tau": float(d["confidence_tau"].median()),\n        }\n        if not accepted.empty:\n            row["selective_top1_acc"] = float(accepted["is_correct"].mean())\n            row["selective_top2_acc"] = float(accepted["top2_hit"].mean())\n            row["selective_balanced_acc"] = multiclass_balanced_accuracy(\n                accepted["true_stressor"],\n                accepted["pred_stressor"],\n                ANOMALIES,\n            )\n            row["selective_macro_f1"] = float(\n                f1_score(\n                    accepted["true_stressor"],\n                    accepted["pred_stressor"],\n                    labels=ANOMALIES,\n                    average="macro",\n                    zero_division=0,\n                )\n            )\n        metric_rows.append(row)\n    metrics_df = pd.DataFrame(metric_rows).sort_values("config")\n\n    final_cfg = "tier0_tier1_tier2"\n    d_final = pred_df[pred_df["config"] == final_cfg]\n    if d_final.empty:\n        cm = pd.DataFrame(index=ANOMALIES, columns=[*ANOMALIES, "ABSTAIN"], data=0)\n    else:\n        labels = [*ANOMALIES, "ABSTAIN"]\n        cm_arr = confusion_matrix(\n            d_final["true_stressor"],\n            d_final["pred_stressor_selective"],\n            labels=labels,\n        )\n        cm = pd.DataFrame(cm_arr, index=labels, columns=labels).loc[ANOMALIES, labels]\n    cm.index.name = "true_stressor"\n    cm.columns.name = "pred_stressor"\n    return pred_df, metrics_df, cm\n\n\ndef build_hierarchical_abstain_sweep(pred_df: pd.DataFrame) -> pd.DataFrame:\n    if pred_df.empty:\n        return pd.DataFrame(\n            columns=[\n                "config",\n                "gate_label",\n                "confidence_threshold",\n                "n_kept",\n                "coverage",\n                "abstain_rate",\n                "selective_top1_acc",\n                "selective_top2_acc",\n                "selective_balanced_acc",\n                "selective_macro_f1",\n            ]\n        )\n\n    rows = []\n    fixed_thresholds = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25]\n    for cfg_name, part in pred_df.groupby("config", sort=False):\n        trained_tau = float(part["confidence_tau"].median())\n        thresholds = [("trained", trained_tau)]\n        thresholds.extend((f"conf_{tau:.2f}", tau) for tau in fixed_thresholds)\n        seen = set()\n        for gate_label, threshold in thresholds:\n            key = (gate_label, round(float(threshold), 8))\n            if key in seen:\n                continue\n            seen.add(key)\n            kept = part[part["confidence_score"] >= float(threshold)].copy()\n            row = {\n                "config": cfg_name,\n                "gate_label": gate_label,\n                "confidence_threshold": float(threshold),\n                "n_kept": int(len(kept)),\n                "coverage": float(len(kept) / max(len(part), 1)),\n                "abstain_rate": float(1.0 - len(kept) / max(len(part), 1)),\n                "selective_top1_acc": np.nan,\n                "selective_top2_acc": np.nan,\n                "selective_balanced_acc": np.nan,\n                "selective_macro_f1": np.nan,\n            }\n            if not kept.empty:\n                row["selective_top1_acc"] = float(kept["is_correct"].mean())\n                row["selective_top2_acc"] = float(kept["top2_hit"].mean())\n                row["selective_balanced_acc"] = multiclass_balanced_accuracy(\n                    kept["true_stressor"],\n                    kept["pred_stressor"],\n                    ANOMALIES,\n                )\n                row["selective_macro_f1"] = float(\n                    f1_score(\n                        kept["true_stressor"],\n                        kept["pred_stressor"],\n                        labels=ANOMALIES,\n                        average="macro",\n                        zero_division=0,\n                    )\n                )\n            rows.append(row)\n    return pd.DataFrame(rows).sort_values(["config", "confidence_threshold"]).reset_index(drop=True)\n\n\ndef build_stressor_tier_contributions(diag_df: pd.DataFrame, config: str) -> pd.DataFrame:\n    cols = ["tier0_share", "tier1_alt_share", "tier2_share"]\n    d = diag_df[(diag_df["config"] == config) & (diag_df["label"] == 1)].copy()\n    if d.empty:\n        return pd.DataFrame(columns=["stressor", *cols, "dominant_tier_mode"])\n    rows = []\n    for stressor, part in d.groupby("stressor", sort=True):\n        mode = part["dominant_tier"].mode()\n        rows.append(\n            {\n                "stressor": stressor,\n                "tier0_share": float(part["tier0_share"].mean()),\n                "tier1_alt_share": float(part["tier1_alt_share"].mean()),\n                "tier2_share": float(part["tier2_share"].mean()),\n                "dominant_tier_mode": str(mode.iloc[0]) if not mode.empty else "none",\n            }\n        )\n    return pd.DataFrame(rows).sort_values("stressor")\n\n\ndef build_mechanism_summary(diag_df: pd.DataFrame, config: str) -> pd.DataFrame:\n    share_cols = [f"{group}_share" for group in MECHANISM_GROUPS]\n    d = diag_df[(diag_df["config"] == config) & (diag_df["label"] == 1)].copy()\n    if d.empty:\n        return pd.DataFrame(columns=["stressor", *share_cols, "dominant_mechanism_mode"])\n    rows = []\n    for stressor, part in d.groupby("stressor", sort=True):\n        mode = part["dominant_mechanism"].mode()\n        row = {\n            "stressor": stressor,\n            "dominant_mechanism_mode": str(mode.iloc[0]) if not mode.empty else "none",\n        }\n        for col in share_cols:\n            row[col] = float(part[col].mean())\n        rows.append(row)\n    return pd.DataFrame(rows).sort_values("stressor")\n\n\ndef build_sequential_metrics(pred_df: pd.DataFrame) -> pd.DataFrame:\n    rows = []\n    for cfg, d in pred_df.groupby("config", sort=False):\n        benign = d[d["label"] == 0]\n        anomaly = d[d["label"] == 1]\n        detected = anomaly[anomaly["run_alert"] == 1]\n        rows.append(\n            {\n                "config": cfg,\n                "benign_run_alert_rate": float(benign["run_alert"].mean()),\n                "benign_persist_alerts_per_hour": float(benign["persist_alerts_per_hour"].mean()),\n                "benign_block_alerts_per_hour": float(benign["block_alerts_per_hour"].mean()),\n                "anomaly_detect_rate": float(anomaly["run_alert"].mean()),\n                "median_time_to_detect_s": finite_median(detected["time_to_detect_s"]),\n                "p90_time_to_detect_s": finite_percentile(detected["time_to_detect_s"], 90),\n                "detect_within_120s": float((anomaly["time_to_detect_s"] <= 120).fillna(False).mean()),\n                "detect_within_300s": float((anomaly["time_to_detect_s"] <= 300).fillna(False).mean()),\n                "detect_within_600s": float((anomaly["time_to_detect_s"] <= 600).fillna(False).mean()),\n            }\n        )\n    return pd.DataFrame(rows).sort_values("config")\n\n\ndef build_holdout_robustness_summary(fold_df: pd.DataFrame, pred_df: pd.DataFrame) -> pd.DataFrame:\n    d = fold_df[fold_df["holdout_workload"] != "ALL"].copy()\n    if d.empty:\n        return pd.DataFrame(\n            columns=[\n                "config",\n                "mean_pr_auc",\n                "worst_pr_auc",\n                "mean_roc_auc",\n                "mean_pr_auc_wc",\n                "worst_pr_auc_wc",\n                "mean_roc_auc_wc",\n                "pooled_pr_auc",\n                "pooled_roc_auc",\n                "pooled_pr_auc_wc",\n                "pooled_roc_auc_wc",\n                "mean_fpr",\n                "mean_tpr",\n            ]\n        )\n    pred_holdout = pred_df[pred_df["holdout_workload"] != "ALL"].copy()\n    rows = []\n    for cfg, part in d.groupby("config", sort=False):\n        pred_part = pred_holdout[pred_holdout["config"] == cfg]\n        y = pred_part["label"].to_numpy(dtype=int)\n        s_run = pred_part["run_score"].to_numpy(dtype=float)\n        s_wc = pred_part["run_score_wc"].to_numpy(dtype=float)\n        rows.append(\n            {\n                "config": cfg,\n                "mean_pr_auc": float(part["pr_auc"].mean()),\n                "worst_pr_auc": float(part["pr_auc"].min()),\n                "mean_roc_auc": float(part["roc_auc"].mean()),\n                "mean_pr_auc_wc": float(part["pr_auc_wc"].mean()),\n                "worst_pr_auc_wc": float(part["pr_auc_wc"].min()),\n                "mean_roc_auc_wc": float(part["roc_auc_wc"].mean()),\n                "pooled_pr_auc": safe_ap(y, s_run),\n                "pooled_roc_auc": safe_auc(y, s_run),\n                "pooled_pr_auc_wc": safe_ap(y, s_wc),\n                "pooled_roc_auc_wc": safe_auc(y, s_wc),\n                "mean_fpr": float(part["fpr"].mean()),\n                "mean_tpr": float(part["tpr"].mean()),\n            }\n        )\n    return pd.DataFrame(rows).sort_values("config")\n\n\ndef plot_confusion_heatmap(cm: pd.DataFrame, out_png: Path, title: str) -> None:\n    if cm.empty:\n        return\n    mat = cm.to_numpy(dtype=float)\n    row_sum = mat.sum(axis=1, keepdims=True)\n    row_share = np.divide(mat, np.where(row_sum == 0.0, 1.0, row_sum))\n    fig, ax = plt.subplots(figsize=(7.2, 6.0))\n    yy, xx = np.indices(mat.shape)\n    sizes = 1800.0 * (mat.flatten() / max(np.max(mat), 1.0) + 0.08)\n    sc = ax.scatter(\n        xx.flatten(),\n        yy.flatten(),\n        s=sizes,\n        c=row_share.flatten(),\n        cmap="YlOrRd",\n        norm=Normalize(vmin=0.0, vmax=1.0),\n        edgecolor="#334155",\n        linewidth=1.1,\n        zorder=3,\n    )\n    ax.set_xticks(np.arange(len(cm.columns)), labels=list(cm.columns), rotation=28, ha="right")\n    ax.set_yticks(np.arange(len(cm.index)), labels=list(cm.index))\n    ax.set_xlabel("Predicted stressor", fontsize=LABEL_SIZE)\n    ax.set_ylabel("True stressor", fontsize=LABEL_SIZE)\n    ax.set_title(title, fontsize=TITLE_SIZE)\n    ax.tick_params(labelsize=TICK_SIZE)\n    for i in range(mat.shape[0]):\n        for j in range(mat.shape[1]):\n            share = row_share[i, j]\n            color = "white" if share >= 0.55 else "black"\n            ax.text(j, i, f"{int(mat[i, j])}", ha="center", va="center", color=color, fontsize=ANNOTATION_SIZE + 2, fontweight="bold")\n    ax.set_xlim(-0.6, mat.shape[1] - 0.4)\n    ax.set_ylim(mat.shape[0] - 0.4, -0.6)\n    ax.set_facecolor("#F8FAFC")\n    ax.set_xticks(np.arange(-0.5, mat.shape[1], 1), minor=True)\n    ax.set_yticks(np.arange(-0.5, mat.shape[0], 1), minor=True)\n    ax.grid(which="minor", color="#E2E8F0", linewidth=1.0)\n    ax.grid(False)\n    cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)\n    cbar.ax.set_ylabel("Share within each true stressor", fontsize=12)\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef plot_stressor_tier_shares(df: pd.DataFrame, out_png: Path) -> None:\n    if df.empty:\n        return\n    fig, ax = plt.subplots(figsize=(8.4, 7.2))\n    _setup_ternary_axis(ax, ("Tier-0", "Tier-1", "Tier-2"))\n    label_offsets = {\n        "ATOMIC": (-0.028, 0.090),\n        "BRANCH": (-0.105, 0.050),\n        "CACHE": (0.0, 0.055),\n        "MEMBW": (-0.105, 0.010),\n        "TLB": (0.060, 0.088),\n    }\n    for row in df.itertuples(index=False):\n        x, y = _ternary_xy(row.tier0_share, row.tier1_alt_share, row.tier2_share)\n        color = _stressor_color(row.stressor)\n        ax.scatter(\n            x,\n            y,\n            s=360 + 260 * max(row.tier0_share, row.tier1_alt_share, row.tier2_share),\n            color=color,\n            edgecolor="white",\n            linewidth=1.6,\n            zorder=3,\n        )\n        dx, dy = label_offsets.get(str(row.stressor), (0.0, 0.05))\n        ax.annotate(\n            str(row.stressor),\n            xy=(x, y),\n            xytext=(x + dx, y + dy),\n            textcoords="data",\n            ha="center",\n            va="center",\n            fontsize=ANNOTATION_SIZE,\n            fontweight="bold",\n            arrowprops=dict(arrowstyle="-", color=color, linewidth=1.0, alpha=0.8),\n            bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="none", alpha=0.86),\n        )\n    ax.set_title("Where the final diagnosis gets its evidence", fontsize=TITLE_SIZE + 2)\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef plot_mechanism_shares(df: pd.DataFrame, out_png: Path) -> None:\n    if df.empty:\n        return\n    fields = [\n        ("compute_share", "Compute"),\n        ("memory_io_share", "Memory/I/O"),\n        ("thermal_power_share", "Thermal/Power"),\n        ("scheduler_runtime_share", "Runtime"),\n        ("platform_pressure_share", "Platform"),\n    ]\n    angles = np.linspace(0.0, 2.0 * np.pi, len(fields), endpoint=False)\n    angles_closed = np.concatenate([angles, angles[:1]])\n    rmax = max(0.4, float(df[[col for col, _ in fields]].to_numpy(dtype=float).max()) * 1.2)\n    fig, axes = plt.subplots(2, 3, figsize=(14.5, 8.4), subplot_kw={"projection": "polar"})\n    axes = axes.ravel()\n    rows = list(df.itertuples(index=False))\n    mean_row = {col: float(df[col].mean()) for col, _ in fields}\n    for idx, ax in enumerate(axes):\n        if idx < len(rows):\n            row = rows[idx]\n            title = str(row.stressor)\n            vals = [float(getattr(row, col)) for col, _ in fields]\n            color = _stressor_color(title)\n        else:\n            title = "Average profile"\n            vals = [mean_row[col] for col, _ in fields]\n            color = "#1D3557"\n        vals_closed = np.array(vals + vals[:1], dtype=float)\n        ax.plot(angles_closed, vals_closed, color=color, linewidth=2.5)\n        ax.fill(angles_closed, vals_closed, color=color, alpha=0.22)\n        ax.set_xticks(angles)\n        ax.set_xticklabels([label for _, label in fields], fontsize=11)\n        ax.set_ylim(0.0, rmax)\n        yticks = np.linspace(rmax / 4.0, rmax, 4)\n        ax.set_yticks(yticks)\n        ax.set_yticklabels([f"{tick:.2f}" for tick in yticks], fontsize=9, color="#64748B")\n        ax.grid(color="#CBD5E1", alpha=0.7)\n        ax.spines["polar"].set_color("#CBD5E1")\n        ax.set_title(title, fontsize=TITLE_SIZE, fontweight="bold", y=1.10)\n    fig.suptitle("Mechanism fingerprints by stressor", fontsize=TITLE_SIZE + 3, fontweight="bold", y=0.98)\n    fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.95])\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef plot_detection_latency(df: pd.DataFrame, out_png: Path) -> None:\n    if df.empty:\n        return\n    fig, ax = plt.subplots(figsize=(9.2, 5.4))\n    d = df.copy()\n    d["label"] = d["config"].map(_cfg_label)\n    order_map = {cfg: i for i, cfg in enumerate(CONFIGS.keys())}\n    d["sort_key"] = d["config"].map(order_map)\n    d = d.sort_values("sort_key").reset_index(drop=True)\n    vals = d["median_time_to_detect_s"].to_numpy(dtype=float)\n    xpos = np.arange(len(d))\n    colors = [_cfg_color(cfg) for cfg in d["config"]]\n    detect = d["anomaly_detect_rate"].fillna(0.0).to_numpy(dtype=float)\n    benign = d["benign_run_alert_rate"].fillna(0.0).to_numpy(dtype=float)\n    for ref in (120.0, 300.0, 600.0):\n        ax.axvline(ref, color="#CBD5E1", linestyle=(0, (3, 4)), linewidth=1.0, zorder=0)\n    ax.hlines(xpos, xmin=0.0, xmax=vals, color=colors, linewidth=4, alpha=0.35)\n    sc = ax.scatter(\n        vals,\n        xpos,\n        s=240 + 760 * detect,\n        c=benign,\n        cmap="OrRd",\n        norm=Normalize(vmin=0.0, vmax=max(float(np.max(benign)), 0.25)),\n        edgecolor="black",\n        linewidth=1.1,\n        zorder=3,\n    )\n    for x, y, value, rate in zip(vals, xpos, vals, detect):\n        ax.text(\n            x + max(vals) * 0.02,\n            y,\n            f"{value:.0f}s | detect {rate:.0%}",\n            va="center",\n            fontsize=ANNOTATION_SIZE + 1,\n            bbox=dict(boxstyle="round,pad=0.16", fc="white", ec="none", alpha=0.85),\n        )\n    ax.set_yticks(xpos, labels=d["label"].tolist())\n    ax.set_xlabel("Median time-to-detect (s)", fontsize=LABEL_SIZE)\n    ax.set_title("Sequential detection speed and alert burden", fontsize=TITLE_SIZE + 1)\n    ax.tick_params(labelsize=TICK_SIZE)\n    ax.set_facecolor("#F8FAFC")\n    ax.grid(axis="x", alpha=0.25)\n    cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)\n    cbar.ax.set_ylabel("Benign alert rate", fontsize=12)\n    fig.text(0.5, 0.03, "Larger circles mean higher anomaly detection rate.", ha="center", fontsize=12, color="#475569")\n    fig.tight_layout(rect=[0.0, 0.05, 1.0, 1.0])\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef append_text_table(lines: List[str], df: pd.DataFrame) -> None:\n    lines.append("```text")\n    lines.append(df.to_string(index=False))\n    lines.append("```")\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\n        "--root",\n        type=Path,\n        default=DEFAULT_DATASET_ROOT,\n    )\n    ap.add_argument("--out_dir", type=Path, default=None)\n    ap.add_argument("--source_hz", type=int, default=5)\n    ap.add_argument("--fit_ratio", type=float, default=0.6)\n    ap.add_argument("--block_B", type=int, default=60)\n    ap.add_argument("--alpha", type=float, default=0.05)\n    ap.add_argument("--persist_k", type=int, default=3)\n    ap.add_argument("--gain", type=float, default=0.35)\n    ap.add_argument("--ridge_lambda", type=float, default=1e-3)\n    ap.add_argument(\n        "--feature_profile",\n        choices=sorted(FEATURE_PROFILES.keys()),\n        default="mixed",\n        help="Feature-file profile: mixed keeps the current deployment-friendly setting; full uses full Tier-1/Tier-2 files as an upper-bound comparison.",\n    )\n    ap.add_argument(\n        "--protocol",\n        choices=["workload_holdout", "global"],\n        default="global",\n        help="Evaluation protocol: workload_holdout (strict) or global benign split (paper-style).",\n    )\n    args = ap.parse_args()\n\n    root = args.root.expanduser().resolve()\n    tier_files = FEATURE_PROFILES[args.feature_profile]\n    out_dir = args.out_dir.expanduser().resolve() if args.out_dir else default_results_dir(\n        root,\n        protocol=args.protocol,\n        feature_profile=args.feature_profile,\n    )\n    fig_dir = out_dir / "figures"\n    out_dir.mkdir(parents=True, exist_ok=True)\n    fig_dir.mkdir(parents=True, exist_ok=True)\n\n    print(f"[INFO] feature_profile={args.feature_profile} tier_files={tier_files}")\n    tier_features = {t: common_features_per_tier(root, t, tier_files) for t in tier_files.keys()}\n    for t, fs in tier_features.items():\n        print(f"[INFO] {t}: common features={len(fs)}")\n\n    preds = []\n    fold_rows = []\n    diagnostic_records: List[Dict[str, object]] = []\n    case_block_records: List[Dict[str, object]] = []\n    config_runtime: Dict[str, float] = {}\n\n    for cfg_name, tiers in CONFIGS.items():\n        cfg_t0 = perf_counter()\n        print(f"[INFO] training config={cfg_name} tiers={tiers}")\n        case_X = {}\n        feature_names_cfg = None\n        for case in all_cases():\n            X, names = build_case_matrix(\n                root,\n                case,\n                tiers=tiers,\n                feature_map=tier_features,\n                tier_files=tier_files,\n                source_hz=args.source_hz,\n            )\n            case_X[case.case_id] = X\n            if feature_names_cfg is None:\n                feature_names_cfg = names\n\n        if args.protocol == "workload_holdout":\n            for holdout_w in WORKLOADS:\n                train_benign = {\n                    case_id: X\n                    for case_id, X in case_X.items()\n                    if case_id.endswith("__NOMINAL") and not case_id.startswith(f"{holdout_w}__")\n                }\n\n                bundle = train_bundle(\n                    train_benign_runs=train_benign,\n                    feature_names=feature_names_cfg or [],\n                    fit_ratio=args.fit_ratio,\n                    B=args.block_B,\n                    alpha=args.alpha,\n                    gain=args.gain,\n                    ridge_lambda=args.ridge_lambda,\n                )\n\n                test_cases = [c for c in all_cases() if c.workload == holdout_w]\n                for case in test_cases:\n                    append_case_outputs(\n                        preds=preds,\n                        diagnostic_records=diagnostic_records,\n                        block_records=case_block_records,\n                        config=cfg_name,\n                        holdout_workload=holdout_w,\n                        case=case,\n                        X_run=case_X[case.case_id],\n                        bundle=bundle,\n                        B=args.block_B,\n                        alpha=args.alpha,\n                        persist_k=args.persist_k,\n                        gain=args.gain,\n                    )\n\n                fold_curr = [p for p in preds if p["config"] == cfg_name and p["holdout_workload"] == holdout_w]\n                fd = pd.DataFrame(fold_curr)\n                y = fd["label"].to_numpy(dtype=int)\n                s_run = fd["run_score"].to_numpy(dtype=float)\n                _, s_wc = workload_conditioned_scores(fd)\n                fold_rows.append(\n                    {\n                        "feature_profile": args.feature_profile,\n                        "config": cfg_name,\n                        "holdout_workload": holdout_w,\n                        "roc_auc": safe_auc(y, s_run),\n                        "pr_auc": safe_ap(y, s_run),\n                        "roc_auc_wc": safe_auc(y, s_wc),\n                        "pr_auc_wc": safe_ap(y, s_wc),\n                        "fpr": float(np.mean((fd["label"] == 0) & (fd["run_alert"] == 1))),\n                        "tpr": float(np.mean((fd["label"] == 1) & (fd["run_alert"] == 1))),\n                        "n_features": int(fd["n_features"].iloc[0]),\n                    }\n                )\n        else:\n            train_benign = {case_id: X for case_id, X in case_X.items() if case_id.endswith("__NOMINAL")}\n            bundle = train_bundle(\n                train_benign_runs=train_benign,\n                feature_names=feature_names_cfg or [],\n                fit_ratio=args.fit_ratio,\n                B=args.block_B,\n                alpha=args.alpha,\n                gain=args.gain,\n                ridge_lambda=args.ridge_lambda,\n            )\n            for case in all_cases():\n                append_case_outputs(\n                    preds=preds,\n                    diagnostic_records=diagnostic_records,\n                    block_records=case_block_records,\n                    config=cfg_name,\n                    holdout_workload="ALL",\n                    case=case,\n                    X_run=case_X[case.case_id],\n                    bundle=bundle,\n                    B=args.block_B,\n                    alpha=args.alpha,\n                    persist_k=args.persist_k,\n                    gain=args.gain,\n                )\n\n            fd = pd.DataFrame([p for p in preds if p["config"] == cfg_name])\n            y = fd["label"].to_numpy(dtype=int)\n            s_run = fd["run_score"].to_numpy(dtype=float)\n            _, s_wc = workload_conditioned_scores(fd)\n            fold_rows.append(\n                {\n                    "feature_profile": args.feature_profile,\n                    "config": cfg_name,\n                    "holdout_workload": "ALL",\n                    "roc_auc": safe_auc(y, s_run),\n                    "pr_auc": safe_ap(y, s_run),\n                    "roc_auc_wc": safe_auc(y, s_wc),\n                    "pr_auc_wc": safe_ap(y, s_wc),\n                    "fpr": float(np.mean((fd["label"] == 0) & (fd["run_alert"] == 1))),\n                    "tpr": float(np.mean((fd["label"] == 1) & (fd["run_alert"] == 1))),\n                    "n_features": int(fd["n_features"].iloc[0]),\n                }\n            )\n        config_runtime[cfg_name] = perf_counter() - cfg_t0\n\n    pred_df = pd.DataFrame(preds).sort_values(["config", "workload", "stressor"])\n    pred_df["feature_profile"] = args.feature_profile\n    fold_df = pd.DataFrame(fold_rows).sort_values(["config", "holdout_workload"])\n    diag_df = pd.DataFrame([{k: v for k, v in row.items() if not k.startswith("_")} for row in diagnostic_records]).sort_values(\n        ["config", "workload", "stressor"]\n    )\n    diag_df["feature_profile"] = args.feature_profile\n    block_df = pd.DataFrame(case_block_records)\n    if not block_df.empty:\n        block_df = block_df.sort_values(["config", "workload", "stressor", "block_idx"])\n        block_df["feature_profile"] = args.feature_profile\n\n    # Workload-conditioned score head: distance to workload nominal template.\n    pred_df["nominal_template_score"] = np.nan\n    pred_df["run_score_wc"] = pred_df["run_score"]\n    for cfg, d in pred_df.groupby("config"):\n        idx = d.index\n        nominal, s_wc = workload_conditioned_scores(d)\n        pred_df.loc[idx, "nominal_template_score"] = nominal\n        pred_df.loc[idx, "run_score_wc"] = s_wc\n\n    overall_rows = []\n    for cfg, d in pred_df.groupby("config"):\n        y = d["label"].to_numpy(dtype=int)\n        s_run = d["run_score"].to_numpy(dtype=float)\n        s_wc = d["run_score_wc"].to_numpy(dtype=float)\n        overall_rows.append(\n            {\n                "feature_profile": args.feature_profile,\n                "config": cfg,\n                "n_cases": int(len(d)),\n                "n_features": int(d["n_features"].iloc[0]),\n                "fit_eval_seconds": float(config_runtime.get(cfg, float("nan"))),\n                "roc_auc": safe_auc(y, s_run),\n                "pr_auc": safe_ap(y, s_run),\n                "roc_auc_wc": safe_auc(y, s_wc),\n                "pr_auc_wc": safe_ap(y, s_wc),\n                "fpr_run_alert": float(np.mean(d[d["label"] == 0]["run_alert"])),\n                "tpr_run_alert": float(np.mean(d[d["label"] == 1]["run_alert"])),\n                "median_nominal_score": float(np.median(d[d["label"] == 0]["run_score"])),\n                "median_anomaly_score": float(np.median(d[d["label"] == 1]["run_score"])),\n                "median_nominal_score_wc": float(np.median(d[d["label"] == 0]["run_score_wc"])),\n                "median_anomaly_score_wc": float(np.median(d[d["label"] == 1]["run_score_wc"])),\n            }\n        )\n    overall_df = pd.DataFrame(overall_rows).sort_values("config")\n\n    final_cfg = "tier0_tier1_tier2"\n    fin = pred_df[pred_df["config"] == final_cfg]\n    stress_rows = []\n    neg = fin[fin["stressor"] == "NOMINAL"][["workload", "run_score", "run_score_wc"]].set_index("workload")\n    for a in ANOMALIES:\n        pos = fin[fin["stressor"] == a][["workload", "run_score", "run_score_wc"]].set_index("workload")\n        m = neg.join(pos, lsuffix="_neg", rsuffix="_pos", how="inner")\n        y = np.array([0] * len(m) + [1] * len(m), dtype=int)\n        s_run = np.concatenate([m["run_score_neg"].to_numpy(dtype=float), m["run_score_pos"].to_numpy(dtype=float)])\n        s_wc = np.concatenate([m["run_score_wc_neg"].to_numpy(dtype=float), m["run_score_wc_pos"].to_numpy(dtype=float)])\n        stress_rows.append(\n            {\n                "feature_profile": args.feature_profile,\n                "stressor": a,\n                "roc_auc": safe_auc(y, s_run),\n                "pr_auc": safe_ap(y, s_run),\n                "roc_auc_wc": safe_auc(y, s_wc),\n                "pr_auc_wc": safe_ap(y, s_wc),\n                "median_neg_score": float(np.median(m["run_score_neg"])),\n                "median_pos_score": float(np.median(m["run_score_pos"])),\n                "median_neg_score_wc": float(np.median(m["run_score_wc_neg"])),\n                "median_pos_score_wc": float(np.median(m["run_score_wc_pos"])),\n                "pos_neg_ratio": float((np.median(m["run_score_pos"]) + 1e-6) / (np.median(m["run_score_neg"]) + 1e-6)),\n                "pos_neg_diff": float(np.median(m["run_score_pos"]) - np.median(m["run_score_neg"])),\n                "pos_neg_ratio_wc": float((np.median(m["run_score_wc_pos"]) + 1e-6) / (np.median(m["run_score_wc_neg"]) + 1e-6)),\n                "pos_neg_diff_wc": float(np.median(m["run_score_wc_pos"]) - np.median(m["run_score_wc_neg"])),\n            }\n        )\n    stress_df = pd.DataFrame(stress_rows).sort_values("stressor")\n\n    mm_pr = float(np.mean(stress_df["pr_auc"]))\n    mm_roc = float(np.mean(stress_df["roc_auc"]))\n    mm_pr_wc = float(np.mean(stress_df["pr_auc_wc"]))\n    mm_roc_wc = float(np.mean(stress_df["roc_auc_wc"]))\n\n    filt = stress_df[~stress_df["stressor"].isin(["BRANCH", "TLB"])]\n    mm_pr_filt = float(np.mean(filt["pr_auc"]))\n    mm_roc_filt = float(np.mean(filt["roc_auc"]))\n    mm_pr_filt_wc = float(np.mean(filt["pr_auc_wc"]))\n    mm_roc_filt_wc = float(np.mean(filt["roc_auc_wc"]))\n\n    diag_pred_feature_run_df, diag_metrics_feature_run_df, diag_cm_feature_run = build_stressor_attribution(\n        diagnostic_records,\n        vector_key="_feature_contrib_run_norm",\n    )\n    diag_pred_feature_focus_df, diag_metrics_feature_focus_df, diag_cm_feature_focus = build_stressor_attribution(\n        diagnostic_records,\n        vector_key="_feature_contrib_focus_norm",\n    )\n    diag_pred_df, diag_metrics_df, diag_cm = build_stressor_attribution(\n        diagnostic_records,\n        vector_key="_mechanism_vector_norm",\n    )\n    diag_pred_hier_df, diag_metrics_hier_df, diag_cm_hier = build_hierarchical_stressor_attribution(\n        diagnostic_records,\n        feature_vector_key="_feature_contrib_run_norm",\n    )\n    diag_abstain_sweep_df = build_hierarchical_abstain_sweep(diag_pred_hier_df)\n    diag_pred_hier_focus_df, diag_metrics_hier_focus_df, diag_cm_hier_focus = build_hierarchical_stressor_attribution(\n        diagnostic_records,\n        feature_vector_key="_feature_contrib_focus_norm",\n    )\n    diag_abstain_sweep_focus_df = build_hierarchical_abstain_sweep(diag_pred_hier_focus_df)\n    diag_tier_df = build_stressor_tier_contributions(diag_df, config=final_cfg)\n    mechanism_df = build_mechanism_summary(diag_df, config=final_cfg)\n    sequential_df = build_sequential_metrics(pred_df)\n    holdout_df = build_holdout_robustness_summary(fold_df, pred_df)\n    if not diag_metrics_feature_run_df.empty:\n        diag_metrics_feature_run_df["feature_profile"] = args.feature_profile\n    if not diag_metrics_feature_focus_df.empty:\n        diag_metrics_feature_focus_df["feature_profile"] = args.feature_profile\n    if not diag_metrics_df.empty:\n        diag_metrics_df["feature_profile"] = args.feature_profile\n    if not diag_metrics_hier_df.empty:\n        diag_metrics_hier_df["feature_profile"] = args.feature_profile\n    if not diag_metrics_hier_focus_df.empty:\n        diag_metrics_hier_focus_df["feature_profile"] = args.feature_profile\n    if not diag_abstain_sweep_df.empty:\n        diag_abstain_sweep_df["feature_profile"] = args.feature_profile\n    if not diag_abstain_sweep_focus_df.empty:\n        diag_abstain_sweep_focus_df["feature_profile"] = args.feature_profile\n    if not diag_tier_df.empty:\n        diag_tier_df["feature_profile"] = args.feature_profile\n    if not mechanism_df.empty:\n        mechanism_df["feature_profile"] = args.feature_profile\n    if not sequential_df.empty:\n        sequential_df["feature_profile"] = args.feature_profile\n    if not holdout_df.empty:\n        holdout_df["feature_profile"] = args.feature_profile\n\n    diagnosis_mode_frames = []\n    if not diag_metrics_feature_run_df.empty:\n        diagnosis_mode_frames.append(\n            diag_metrics_feature_run_df.assign(diagnosis_target="feature_level", diagnosis_mode="whole_run")\n        )\n    if not diag_metrics_feature_focus_df.empty:\n        diagnosis_mode_frames.append(\n            diag_metrics_feature_focus_df.assign(diagnosis_target="feature_level", diagnosis_mode="post_alert_window")\n        )\n    if not diag_metrics_hier_df.empty:\n        diagnosis_mode_frames.append(\n            diag_metrics_hier_df.assign(diagnosis_target="hierarchical", diagnosis_mode="whole_run")\n        )\n    if not diag_metrics_hier_focus_df.empty:\n        diagnosis_mode_frames.append(\n            diag_metrics_hier_focus_df.assign(diagnosis_target="hierarchical", diagnosis_mode="post_alert_window")\n        )\n    diagnosis_mode_comparison_df = (\n        pd.concat(diagnosis_mode_frames, ignore_index=True, sort=False)\n        if diagnosis_mode_frames\n        else pd.DataFrame()\n    )\n\n    runtime_df = pd.DataFrame(\n        [\n            {\n                "feature_profile": args.feature_profile,\n                "config": cfg,\n                "fit_eval_seconds": float(sec),\n            }\n            for cfg, sec in config_runtime.items()\n        ]\n    ).sort_values("config")\n\n    pred_df.to_csv(out_dir / "case_predictions.csv", index=False)\n    if not block_df.empty:\n        block_df.to_csv(out_dir / "case_block_traces.csv", index=False)\n    fold_df.to_csv(out_dir / "fold_metrics.csv", index=False)\n    overall_df.to_csv(out_dir / "overall_metrics.csv", index=False)\n    runtime_df.to_csv(out_dir / "config_runtime_summary.csv", index=False)\n    stress_df.to_csv(out_dir / "stressor_metrics_final_config.csv", index=False)\n    diag_df.to_csv(out_dir / "case_diagnosis_summary.csv", index=False)\n    diag_pred_df.to_csv(out_dir / "stressor_diagnosis_predictions.csv", index=False)\n    diag_metrics_df.to_csv(out_dir / "stressor_diagnosis_metrics.csv", index=False)\n    diag_cm.to_csv(out_dir / "stressor_confusion_matrix.csv")\n    diag_pred_feature_run_df.to_csv(out_dir / "stressor_feature_diagnosis_predictions.csv", index=False)\n    diag_metrics_feature_run_df.to_csv(out_dir / "stressor_feature_diagnosis_metrics.csv", index=False)\n    diag_cm_feature_run.to_csv(out_dir / "stressor_feature_confusion_matrix.csv")\n    diag_pred_feature_run_df.to_csv(out_dir / "stressor_feature_diagnosis_runlevel_predictions.csv", index=False)\n    diag_metrics_feature_run_df.to_csv(out_dir / "stressor_feature_diagnosis_runlevel_metrics.csv", index=False)\n    diag_cm_feature_run.to_csv(out_dir / "stressor_feature_confusion_matrix_runlevel.csv")\n    diag_pred_feature_focus_df.to_csv(out_dir / "stressor_feature_diagnosis_post_alert_predictions.csv", index=False)\n    diag_metrics_feature_focus_df.to_csv(out_dir / "stressor_feature_diagnosis_post_alert_metrics.csv", index=False)\n    diag_cm_feature_focus.to_csv(out_dir / "stressor_feature_confusion_matrix_post_alert.csv")\n    diag_pred_feature_focus_df.to_csv(out_dir / "stressor_feature_diagnosis_top_blocks_predictions.csv", index=False)\n    diag_metrics_feature_focus_df.to_csv(out_dir / "stressor_feature_diagnosis_top_blocks_metrics.csv", index=False)\n    diag_cm_feature_focus.to_csv(out_dir / "stressor_feature_confusion_matrix_top_blocks.csv")\n    diag_pred_hier_df.to_csv(out_dir / "stressor_hierarchical_diagnosis_predictions.csv", index=False)\n    diag_metrics_hier_df.to_csv(out_dir / "stressor_hierarchical_diagnosis_metrics.csv", index=False)\n    diag_cm_hier.to_csv(out_dir / "stressor_hierarchical_confusion_matrix.csv")\n    diag_abstain_sweep_df.to_csv(out_dir / "stressor_hierarchical_abstain_sweep.csv", index=False)\n    diag_pred_hier_focus_df.to_csv(out_dir / "stressor_hierarchical_diagnosis_post_alert_predictions.csv", index=False)\n    diag_metrics_hier_focus_df.to_csv(out_dir / "stressor_hierarchical_diagnosis_post_alert_metrics.csv", index=False)\n    diag_cm_hier_focus.to_csv(out_dir / "stressor_hierarchical_confusion_matrix_post_alert.csv")\n    diag_abstain_sweep_focus_df.to_csv(out_dir / "stressor_hierarchical_abstain_sweep_post_alert.csv", index=False)\n    diag_pred_hier_focus_df.to_csv(out_dir / "stressor_hierarchical_diagnosis_top_blocks_predictions.csv", index=False)\n    diag_metrics_hier_focus_df.to_csv(out_dir / "stressor_hierarchical_diagnosis_top_blocks_metrics.csv", index=False)\n    diag_cm_hier_focus.to_csv(out_dir / "stressor_hierarchical_confusion_matrix_top_blocks.csv")\n    diag_abstain_sweep_focus_df.to_csv(out_dir / "stressor_hierarchical_abstain_sweep_top_blocks.csv", index=False)\n    diag_tier_df.to_csv(out_dir / "stressor_tier_contributions.csv", index=False)\n    mechanism_df.to_csv(out_dir / "mechanism_group_summary.csv", index=False)\n    sequential_df.to_csv(out_dir / "sequential_metrics.csv", index=False)\n    holdout_df.to_csv(out_dir / "holdout_robustness_summary.csv", index=False)\n    if not diagnosis_mode_comparison_df.empty:\n        diagnosis_mode_comparison_df.to_csv(out_dir / "diagnosis_mode_comparison.csv", index=False)\n    (out_dir / "run_context.json").write_text(\n        json.dumps(\n            {\n                "feature_profile": args.feature_profile,\n                "tier_files": tier_files,\n                "protocol": args.protocol,\n                "source_hz": args.source_hz,\n                "fit_ratio": args.fit_ratio,\n                "block_B": args.block_B,\n                "alpha": args.alpha,\n                "persist_k": args.persist_k,\n                "gain": args.gain,\n                "ridge_lambda": args.ridge_lambda,\n                "config_runtime_seconds": config_runtime,\n                "diagnosis_monotonic_tokens": list(DIAG_MONOTONIC_TOKENS),\n                "diagnosis_generic_memory_state_tokens": list(DIAG_GENERIC_MEMORY_STATE_TOKENS),\n                "diagnosis_generic_memory_state_factor": DIAG_GENERIC_MEMORY_STATE_FACTOR,\n                "diagnosis_window_blocks": DIAG_POST_ALERT_BLOCKS,\n            },\n            indent=2,\n        )\n        + "\\n"\n    )\n\n    overall_tex = overall_df[\n        ["config", "n_features", "roc_auc", "pr_auc", "roc_auc_wc", "pr_auc_wc", "fpr_run_alert", "tpr_run_alert"]\n    ].rename(\n        columns={\n            "config": "Configuration",\n            "n_features": "Features",\n            "roc_auc": "ROC-AUC (Base)",\n            "pr_auc": "AUC-PR (Base)",\n            "roc_auc_wc": "ROC-AUC (WC)",\n            "pr_auc_wc": "AUC-PR (WC)",\n            "fpr_run_alert": "Run-FPR",\n            "tpr_run_alert": "Run-TPR",\n        }\n    )\n    stress_tex = stress_df[\n        ["stressor", "roc_auc", "pr_auc", "roc_auc_wc", "pr_auc_wc", "pos_neg_ratio", "pos_neg_diff"]\n    ].rename(\n        columns={\n            "stressor": "Stressor",\n            "roc_auc": "ROC-AUC (Base)",\n            "pr_auc": "AUC-PR (Base)",\n            "roc_auc_wc": "ROC-AUC (WC)",\n            "pr_auc_wc": "AUC-PR (WC)",\n            "pos_neg_ratio": "Pos/Neg Score Ratio",\n            "pos_neg_diff": "Pos-Neg Score Delta",\n        }\n    )\n    (out_dir / "overall_metrics.tex").write_text(\n        to_latex_table(\n            overall_tex,\n            "DICE micro-twin + split-conformal run-level results under benign retraining.",\n            "tab:dice_full_overall",\n        )\n    )\n    (out_dir / "stressor_metrics_final_config.tex").write_text(\n        to_latex_table(\n            stress_tex,\n            "Final DICE configuration per-stressor separability.",\n            "tab:dice_full_stressor",\n        )\n    )\n    if not diag_metrics_df.empty:\n        diag_tex = diag_metrics_df.rename(\n            columns={\n                "config": "Configuration",\n                "n_cases": "Cases",\n                "top1_acc": "Top-1 Acc.",\n                "top2_acc": "Top-2 Acc.",\n                "macro_f1": "Macro-F1",\n                "mean_margin_to_second": "Mean Margin",\n            }\n        )\n        (out_dir / "stressor_diagnosis_metrics.tex").write_text(\n            to_latex_table(\n                diag_tex,\n                "Mechanism-group stressor attribution from DICE residual contributions across workloads.",\n                "tab:dice_stressor_diagnosis",\n            )\n        )\n    if not diag_metrics_hier_df.empty:\n        diag_hier_tex = diag_metrics_hier_df.rename(\n            columns={\n                "config": "Configuration",\n                "n_cases": "Cases",\n                "top1_acc": "Top-1 Acc.",\n                "top2_acc": "Top-2 Acc.",\n                "family_acc": "Family Acc.",\n                "coverage": "Coverage",\n                "selective_top1_acc": "Selective Top-1",\n                "selective_top2_acc": "Selective Top-2",\n                "abstain_rate": "Abstain Rate",\n            }\n        )[\n            [\n                "Configuration",\n                "Cases",\n                "Top-1 Acc.",\n                "Top-2 Acc.",\n                "Family Acc.",\n                "Coverage",\n                "Selective Top-1",\n                "Selective Top-2",\n                "Abstain Rate",\n            ]\n        ]\n        (out_dir / "stressor_hierarchical_diagnosis_metrics.tex").write_text(\n            to_latex_table(\n                diag_hier_tex,\n                "Normalized feature-level prototype diagnosis with family-aware confidence gating.",\n                "tab:dice_stressor_hierarchical_diagnosis",\n            )\n        )\n    if not sequential_df.empty:\n        seq_tex = sequential_df.rename(\n            columns={\n                "config": "Configuration",\n                "benign_run_alert_rate": "Benign Run-Alert Rate",\n                "benign_persist_alerts_per_hour": "Benign Persist Alerts/hr",\n                "anomaly_detect_rate": "Anomaly Detect Rate",\n                "median_time_to_detect_s": "Median TTD (s)",\n                "detect_within_300s": "Detect <=300s",\n            }\n        )[\n            [\n                "Configuration",\n                "Benign Run-Alert Rate",\n                "Benign Persist Alerts/hr",\n                "Anomaly Detect Rate",\n                "Median TTD (s)",\n                "Detect <=300s",\n            ]\n        ]\n        (out_dir / "sequential_metrics.tex").write_text(\n            to_latex_table(\n                seq_tex,\n                "Sequential decision metrics for the DICE run-level detector.",\n                "tab:dice_sequential_metrics",\n            )\n        )\n\n    plot_curves(pred_df, fig_dir / "fig_roc_pr_by_config.png", score_col="run_score", title_tag="base")\n    plot_curves(pred_df, fig_dir / "fig_roc_pr_by_config_wc.png", score_col="run_score_wc", title_tag="workload-conditioned")\n    plot_score_box(\n        pred_df,\n        fig_dir / "fig_run_score_boxplot.png",\n        score_col="run_score",\n        y_label="Run score (base)",\n        title_tag="base",\n    )\n    plot_score_box(\n        pred_df,\n        fig_dir / "fig_run_score_boxplot_wc.png",\n        score_col="run_score_wc",\n        y_label="Run score (workload-conditioned)",\n        title_tag="workload-conditioned",\n    )\n    plot_confusion_heatmap(\n        diag_cm_hier if not diag_cm_hier.empty else diag_cm_feature_run,\n        fig_dir / "fig_stressor_confusion_matrix.png",\n        title="Final-config hierarchical stressor attribution",\n    )\n    plot_confusion_heatmap(\n        diag_cm_feature_run,\n        fig_dir / "fig_stressor_confusion_matrix_feature.png",\n        title="Final-config normalized feature prototype attribution (whole run)",\n    )\n    plot_confusion_heatmap(\n        diag_cm_feature_focus,\n        fig_dir / "fig_stressor_confusion_matrix_feature_top_blocks.png",\n        title="Final-config normalized feature prototype attribution (post-alert window)",\n    )\n    plot_stressor_tier_shares(\n        diag_tier_df,\n        fig_dir / "fig_stressor_tier_contributions.png",\n    )\n    plot_mechanism_shares(\n        mechanism_df,\n        fig_dir / "fig_mechanism_group_summary.png",\n    )\n    plot_detection_latency(\n        sequential_df,\n        fig_dir / "fig_detection_latency.png",\n    )\n\n    md = []\n    md.append("# DICE Full Retrain Results")\n    md.append("")\n    md.append("## Setup")\n    md.append(\n        f"- Feature profile: {args.feature_profile} ({tier_files})"\n    )\n    md.append(\n        f"- Protocol: {args.protocol}, benign-only fit/calibration, block_B={args.block_B}, "\n        f"alpha={args.alpha}, persist_k={args.persist_k}, gain={args.gain}"\n    )\n    md.append("")\n    md.append("## Overall")\n    append_text_table(md, overall_df)\n    md.append("")\n    md.append("## Final Config Stressors")\n    append_text_table(md, stress_df)\n    md.append("")\n    md.append("## Paper-style Aggregates (Final Config)")\n    md.append(f"- Base score mean stressor AUC-PR (all five): **{mm_pr:.4f}**")\n    md.append(f"- Base score mean stressor ROC-AUC (all five): **{mm_roc:.4f}**")\n    md.append(f"- WC score mean stressor AUC-PR (all five): **{mm_pr_wc:.4f}**")\n    md.append(f"- WC score mean stressor ROC-AUC (all five): **{mm_roc_wc:.4f}**")\n    md.append(f"- Base score mean stressor AUC-PR (excluding BRANCH/TLB): **{mm_pr_filt:.4f}**")\n    md.append(f"- Base score mean stressor ROC-AUC (excluding BRANCH/TLB): **{mm_roc_filt:.4f}**")\n    md.append(f"- WC score mean stressor AUC-PR (excluding BRANCH/TLB): **{mm_pr_filt_wc:.4f}**")\n    md.append(f"- WC score mean stressor ROC-AUC (excluding BRANCH/TLB): **{mm_roc_filt_wc:.4f}**")\n    md.append("")\n    if not diag_metrics_df.empty:\n        md.append("## Diagnosis")\n        md.append("- Feature-level diagnosis uses normalized whole-run residual-attribution prototypes as the default paper-facing result, with diagnosis-only filtering that removes monotonic counters and downweights generic memory-state features.")\n        append_text_table(md, diag_metrics_feature_run_df)\n        md.append("")\n        if not diag_metrics_feature_focus_df.empty:\n            md.append("- Post-alert-window diagnosis is exported below as a side-by-side comparison; it uses windows immediately after the first alert/persistent alert and applies the same diagnosis-only filtering and downweighting.")\n            append_text_table(md, diag_metrics_feature_focus_df)\n            md.append("")\n        md.append("- Mechanism-level diagnosis uses normalized mechanism centroids over workload-held residual summaries.")\n        append_text_table(md, diag_metrics_df)\n        md.append("")\n        if not diag_metrics_hier_df.empty:\n            md.append("## Hierarchical Diagnosis")\n            md.append("- High-confidence diagnosis adds a mechanism-family gate and abstains on low-confidence cases; whole-run attribution is the default input.")\n            append_text_table(md, diag_metrics_hier_df)\n            md.append("")\n        if not diag_metrics_hier_focus_df.empty:\n            md.append("## Hierarchical Diagnosis (Post-Alert Window)")\n            md.append("- This comparison uses the same gate on post-alert-window attribution rather than whole-run attribution.")\n            append_text_table(md, diag_metrics_hier_focus_df)\n            md.append("")\n        if not diag_abstain_sweep_df.empty:\n            md.append("## Hierarchical Abstain Sweep")\n            md.append("- The sweep below shows how selective diagnosis improves as the confidence gate becomes stricter.")\n            append_text_table(md, diag_abstain_sweep_df)\n            md.append("")\n        if not diag_abstain_sweep_focus_df.empty:\n            md.append("## Hierarchical Abstain Sweep (Post-Alert Window)")\n            md.append("- This comparison applies the same sweep to post-alert-window attribution.")\n            append_text_table(md, diag_abstain_sweep_focus_df)\n            md.append("")\n        if not diagnosis_mode_comparison_df.empty:\n            md.append("## Diagnosis Mode Comparison")\n            md.append("- Whole-run and post-alert-window diagnosis are exported together so the diagnosis mode can be evaluated directly.")\n            append_text_table(md, diagnosis_mode_comparison_df)\n            md.append("")\n        if not diag_tier_df.empty:\n            md.append("## Final Config Tier Contribution Summary")\n            append_text_table(md, diag_tier_df)\n            md.append("")\n        if not mechanism_df.empty:\n            md.append("## Final Config Mechanism Summary")\n            append_text_table(md, mechanism_df)\n            md.append("")\n    if not sequential_df.empty:\n        md.append("## Sequential Decisioning")\n        append_text_table(md, sequential_df)\n        md.append("")\n    if not holdout_df.empty:\n        md.append("## Holdout Robustness (Workload Drift Proxy)")\n        append_text_table(md, holdout_df)\n        md.append("")\n    md.append("## Files")\n    for p in [\n        out_dir / "overall_metrics.csv",\n        out_dir / "config_runtime_summary.csv",\n        out_dir / "run_context.json",\n        out_dir / "case_block_traces.csv",\n        out_dir / "stressor_metrics_final_config.csv",\n        out_dir / "sequential_metrics.csv",\n        out_dir / "case_diagnosis_summary.csv",\n        out_dir / "stressor_diagnosis_metrics.csv",\n        out_dir / "stressor_feature_diagnosis_metrics.csv",\n        out_dir / "stressor_feature_diagnosis_runlevel_metrics.csv",\n        out_dir / "stressor_feature_diagnosis_post_alert_metrics.csv",\n        out_dir / "stressor_feature_diagnosis_top_blocks_metrics.csv",\n        out_dir / "stressor_hierarchical_diagnosis_metrics.csv",\n        out_dir / "stressor_hierarchical_abstain_sweep.csv",\n        out_dir / "stressor_hierarchical_diagnosis_post_alert_metrics.csv",\n        out_dir / "stressor_hierarchical_abstain_sweep_post_alert.csv",\n        out_dir / "stressor_hierarchical_diagnosis_top_blocks_metrics.csv",\n        out_dir / "stressor_hierarchical_abstain_sweep_top_blocks.csv",\n        out_dir / "diagnosis_mode_comparison.csv",\n        out_dir / "mechanism_group_summary.csv",\n        out_dir / "stressor_confusion_matrix.csv",\n        out_dir / "stressor_hierarchical_confusion_matrix.csv",\n        out_dir / "stressor_tier_contributions.csv",\n        out_dir / "overall_metrics.tex",\n        out_dir / "stressor_metrics_final_config.tex",\n        out_dir / "stressor_diagnosis_metrics.tex",\n        out_dir / "stressor_hierarchical_diagnosis_metrics.tex",\n        out_dir / "sequential_metrics.tex",\n        fig_dir / "fig_roc_pr_by_config.png",\n        fig_dir / "fig_roc_pr_by_config_wc.png",\n        fig_dir / "fig_run_score_boxplot.png",\n        fig_dir / "fig_run_score_boxplot_wc.png",\n        fig_dir / "fig_stressor_confusion_matrix.png",\n        fig_dir / "fig_stressor_confusion_matrix_feature.png",\n        fig_dir / "fig_stressor_confusion_matrix_feature_top_blocks.png",\n        fig_dir / "fig_stressor_tier_contributions.png",\n        fig_dir / "fig_mechanism_group_summary.png",\n        fig_dir / "fig_detection_latency.png",\n    ]:\n        md.append(f"- `{p}`")\n    (out_dir / "RESULTS_SUMMARY.md").write_text("\\n".join(md) + "\\n")\n\n    print(f"[OK] wrote results to: {out_dir}")\n    print("[OK] overall metrics:")\n    print(overall_df.to_string(index=False))\n    print("[OK] final config stressor metrics:")\n    print(stress_df.to_string(index=False))\n    if not diag_metrics_df.empty:\n        print("[OK] stressor diagnosis metrics:")\n        print(diag_metrics_df.to_string(index=False))\n    if not diag_metrics_feature_run_df.empty:\n        print("[OK] feature diagnosis metrics (whole run default):")\n        print(diag_metrics_feature_run_df.to_string(index=False))\n    if not diag_metrics_feature_focus_df.empty:\n        print("[OK] feature diagnosis metrics (post-alert window comparison):")\n        print(diag_metrics_feature_focus_df.to_string(index=False))\n    if not diag_metrics_hier_df.empty:\n        print("[OK] hierarchical diagnosis metrics (whole run default):")\n        print(diag_metrics_hier_df.to_string(index=False))\n    if not diag_metrics_hier_focus_df.empty:\n        print("[OK] hierarchical diagnosis metrics (post-alert window comparison):")\n        print(diag_metrics_hier_focus_df.to_string(index=False))\n    if not diag_abstain_sweep_df.empty:\n        print("[OK] hierarchical abstain sweep (whole run default):")\n        print(diag_abstain_sweep_df.to_string(index=False))\n    if not diag_abstain_sweep_focus_df.empty:\n        print("[OK] hierarchical abstain sweep (post-alert window comparison):")\n        print(diag_abstain_sweep_focus_df.to_string(index=False))\n    if not diagnosis_mode_comparison_df.empty:\n        print("[OK] diagnosis mode comparison:")\n        print(diagnosis_mode_comparison_df.to_string(index=False))\n    if not sequential_df.empty:\n        print("[OK] sequential metrics:")\n        print(sequential_df.to_string(index=False))\n    print(\n        "[OK] aggregates (base): "\n        f"all(AUC-PR={mm_pr:.4f}, ROC-AUC={mm_roc:.4f}), "\n        f"filtered(AUC-PR={mm_pr_filt:.4f}, ROC-AUC={mm_roc_filt:.4f})"\n    )\n    print(\n        "[OK] aggregates (workload-conditioned): "\n        f"all(AUC-PR={mm_pr_wc:.4f}, ROC-AUC={mm_roc_wc:.4f}), "\n        f"filtered(AUC-PR={mm_pr_filt_wc:.4f}, ROC-AUC={mm_roc_filt_wc:.4f})"\n    )\n\n\nif __name__ == "__main__":\n    main()\n'

## Enable Block-Trace Export for the Overlay

This small notebook-local patch extends the embedded DICE engine so it also writes `case_block_traces.csv`.
That file is used later for the true time-series virtual-system overlay.


In [ ]:
# Plain-language: This cell syncs the embedded notebook engine with the current repository source so the notebook and code stay aligned.
display(Markdown('### Inline source sync'))
print('Notebook inline train/eval source is synchronized with tools/train_eval_dice_pipeline.py.')


## Figure Style Patch

This notebook-local patch reuses the latest plotting functions from the repository scripts so the notebook previews and the exported paper PNGs stay in sync.


In [ ]:
# Plain-language: This cell pulls plotting styles from the repository so the notebook figures match the paper formatting.
FIG_STYLE_TRAIN_TOOL = REPO_ROOT / 'tools' / 'train_eval_dice_pipeline.py'
FIG_STYLE_ANALYSIS_TOOL = REPO_ROOT / 'tools' / 'generate_results_analysis.py'


class _NotebookModuleProxy(dict):
    """Allow both dict-style and attribute-style access in later notebook cells."""

    def __getattr__(self, name: str):
        try:
            return self[name]
        except KeyError as exc:
            raise AttributeError(name) from exc

    def __setattr__(self, name: str, value):
        self[name] = value

    def __delattr__(self, name: str):
        try:
            del self[name]
        except KeyError as exc:
            raise AttributeError(name) from exc


def _load_plot_module(name: str, path: Path):
    if not path.exists():
        raise FileNotFoundError(f'Missing plotting module: {path}')
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f'Could not load plotting module from {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    try:
        spec.loader.exec_module(module)
    except Exception:
        sys.modules.pop(name, None)
        raise
    return module


def _to_module_proxy(obj, fallback_module=None):
    if isinstance(obj, _NotebookModuleProxy):
        proxy = obj
    elif isinstance(obj, dict):
        proxy = _NotebookModuleProxy(obj)
    elif obj is None:
        proxy = _NotebookModuleProxy()
    else:
        namespace = getattr(obj, '__dict__', None)
        if namespace is None:
            namespace = {
                name: getattr(obj, name)
                for name in dir(obj)
                if not name.startswith('__')
            }
        proxy = _NotebookModuleProxy(namespace)

    if fallback_module is not None:
        for name, value in vars(fallback_module).items():
            if not name.startswith('__') and name not in proxy:
                proxy[name] = value
    return proxy


FIG_STYLE_TRAIN = _load_plot_module('dice_train_eval_plot_style', FIG_STYLE_TRAIN_TOOL)
FIG_STYLE_ANALYSIS = _load_plot_module('dice_results_analysis_plot_style', FIG_STYLE_ANALYSIS_TOOL)

FULL_MODULE = _to_module_proxy(globals().get('FULL_MODULE'), fallback_module=FIG_STYLE_TRAIN)
ANALYSIS_MODULE = _to_module_proxy(globals().get('ANALYSIS_MODULE'), fallback_module=FIG_STYLE_ANALYSIS)

for _name in [
    'plot_curves',
    'plot_score_box',
    'plot_confusion_heatmap',
    'plot_stressor_tier_shares',
    'plot_mechanism_shares',
    'plot_detection_latency',
]:
    FULL_MODULE[_name] = getattr(FIG_STYLE_TRAIN, _name)

ANALYSIS_MODULE['plot_run_score_distributions'] = getattr(FIG_STYLE_ANALYSIS, 'plot_run_score_distributions')

cfg_label = FULL_MODULE['_cfg_label'] = getattr(FIG_STYLE_TRAIN, '_cfg_label')
cfg_color = FULL_MODULE['_cfg_color'] = getattr(FIG_STYLE_TRAIN, '_cfg_color')
ternary_xy = FULL_MODULE['_ternary_xy'] = getattr(FIG_STYLE_TRAIN, '_ternary_xy')
setup_ternary_axis = FULL_MODULE['_setup_ternary_axis'] = getattr(FIG_STYLE_TRAIN, '_setup_ternary_axis')

globals()['FULL_MODULE'] = FULL_MODULE
globals()['ANALYSIS_MODULE'] = ANALYSIS_MODULE

print('Notebook figure style patched from repository plotting tools.')
print('FULL_MODULE is ready for both dict-style and attribute-style access.')

## Run End-to-End

Run this section to regenerate the released results, figures, and paper-ready bundles. By default, it sweeps both the mixed and full deployment profiles. The GitHub-facing summary sections immediately below then surface the most useful paper and appendix claims from those outputs.


In [ ]:
# Plain-language: This cell runs the end-to-end DICE workflow, saves the main artifacts, and summarizes the headline results.
RUN_END_TO_END = True
INCLUDE_TUNING = True  # Run the full parameter sweep for each feature profile in FEATURE_PROFILES_TO_SWEEP.
ALERT_TUNING_OBJECTIVE = 'diagnosis_low_fp'  # Choose from: 'conservative', 'balanced', 'aggressive', 'diagnosis_first', 'diagnosis_low_fp'.
USE_TUNED_ALERT_CONFIG = True  # Use each profile's recommended_alert_config.csv for the main outputs.
TUNED_ALERT_CONFIG_PATH = None  # Optional explicit CSV path; may include {feature_profile}; defaults to each profile's tuning folder.
RUN_HOLDOUT = True

RUN_TWO_STAGE = True
TWO_STAGE_QUANTILES = [0.90, 0.95, 0.98]

QUIET_STAGE_OUTPUT = False  # Stream live stage output so notebook users can see progress while runs are active.
SHOW_RESULT_GALLERY = False  # Keep the notebook focused on the top paper-style figures; all saved figures still remain on disk.
FORCE_RERUN_TUNING = True  # Keep True here because older cached tuning runs were generated before the benign phase-guard fix and can overstate benign alerts.
SHOW_APPENDIX_FIGURES = False  # Leave appendix-style DSE figures off the notebook page unless you explicitly want them shown.



RUN_ANALYSIS_TOOL = REPO_ROOT / 'tools' / 'generate_results_analysis.py'
RUN_FULL_TOOL = REPO_ROOT / 'tools' / 'train_eval_dice_pipeline.py'
TUNING_STAGE1_GAINS = [0.10, 0.15, 0.20, 0.25, 0.35, 0.50]
TUNING_STAGE1_BLOCKS = [20, 30, 45, 60, 90, 120]
TUNING_STAGE2_ALPHAS = [0.005, 0.01, 0.02, 0.05, 0.08, 0.10]
TUNING_STAGE2_PERSISTS = [1, 2, 3, 4, 5, 7]
FINAL_CONFIG_NAME = 'tier0_tier1_tier2'


def deterministic_env() -> dict[str, str]:
    env = os.environ.copy()
    mpl_dir = Path(env.get('MPLCONFIGDIR', str((REPO_ROOT / '.cache' / 'matplotlib').resolve()))).expanduser().resolve()
    mpl_dir.mkdir(parents=True, exist_ok=True)
    settings = {
        'MPLCONFIGDIR': str(mpl_dir),
        'MPLBACKEND': env.get('MPLBACKEND', 'Agg'),
        'OPENBLAS_NUM_THREADS': env.get('OPENBLAS_NUM_THREADS', '1'),
        'OMP_NUM_THREADS': env.get('OMP_NUM_THREADS', '1'),
        'MKL_NUM_THREADS': env.get('MKL_NUM_THREADS', '1'),
        'NUMEXPR_NUM_THREADS': env.get('NUMEXPR_NUM_THREADS', '1'),
        'VECLIB_MAXIMUM_THREADS': env.get('VECLIB_MAXIMUM_THREADS', '1'),
        'BLIS_NUM_THREADS': env.get('BLIS_NUM_THREADS', '1'),
        'PYTHONHASHSEED': env.get('PYTHONHASHSEED', '0'),
        'PYTHONUNBUFFERED': env.get('PYTHONUNBUFFERED', '1'),
    }
    env.update(settings)
    os.environ.update(settings)
    return env


def ensure_dataset_root(root: Path) -> None:
    required = [
        root / 'tier0',
        root / 'tier1_alt',
        root / 'tier2',
        root / 'no_nan_report.json',
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError(f'Dataset root is missing required files/folders: {missing}')


def default_tuning_out_dir(root: Path, feature_profile: str) -> Path:                       
    return root / ('results_dice_tuning' if feature_profile == 'mixed' else f'results_dice_tuning_{feature_profile}')


def default_full_out_dir(root: Path, protocol: str, feature_profile: str) -> Path:
    if feature_profile == 'mixed':
        return root / ('results_dice_full_holdout' if protocol == 'workload_holdout' else 'results_dice_full')
    suffix = f'results_dice_full_{feature_profile}'
    if protocol == 'workload_holdout':
        suffix = f'{suffix}_holdout'
    return root / suffix


def _run_checked(cmd: list[str], *, env: dict[str, str] | None = None, cwd: Path | None = None) -> subprocess.CompletedProcess[str]:
    env = env or deterministic_env()
    cwd = cwd or REPO_ROOT
    cmd = [str(part) for part in cmd]
    if cmd and cmd[0] == sys.executable and '-u' not in cmd[1:3]:
        cmd = [cmd[0], '-u', *cmd[1:]]

    print(f"$ {' '.join(cmd)}")

    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    captured_lines = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        captured_lines.append(line)
    proc.wait()

    stdout = ''.join(captured_lines)
    if proc.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {proc.returncode}: {' '.join(cmd)}\n"
            f"STDOUT:\n{stdout}"
        )
    return subprocess.CompletedProcess(cmd, proc.returncode, stdout=stdout, stderr='')


def _read_final_row(path: Path, config: str = FINAL_CONFIG_NAME) -> pd.Series:
    df = pd.read_csv(path)
    rows = df[df['config'] == config]
    if not rows.empty:
        return rows.iloc[0]
    if df.empty:
        raise ValueError(f'No rows found in {path}')
    return df.iloc[-1]


def _load_tuning_diag(out_dir: Path, config: str = FINAL_CONFIG_NAME) -> tuple[str, float, float, float]:
    for diag_source, name in [
        ('feature-level', 'stressor_feature_diagnosis_metrics.csv'),
        ('mechanism-level', 'stressor_diagnosis_metrics.csv'),
    ]:
        path = out_dir / name
        if not path.exists():
            continue
        diag = pd.read_csv(path)
        rows = diag[diag['config'] == config]
        row = rows.iloc[0] if not rows.empty else (diag.iloc[0] if not diag.empty else None)
        if row is None:
            continue
        return (
            diag_source,
            float(row.get('top1_acc', np.nan)),
            float(row.get('top2_acc', np.nan)),
            float(row.get('macro_f1', np.nan)),
        )
    return ('unavailable', np.nan, np.nan, np.nan)


def _load_tuning_mechanism_proxy(out_dir: Path) -> dict[str, float]:
    diag_path = out_dir / 'case_diagnosis_summary.csv'
    mechanism_path = out_dir / 'mechanism_group_summary.csv'
    result = {
        'diag_mechanism_top1_proxy': np.nan,
        'diag_mechanism_top2_proxy': np.nan,
        'diag_mechanism_top3_proxy': np.nan,
    }
    if not diag_path.exists() or not mechanism_path.exists():
        return result

    diag = pd.read_csv(diag_path)
    mechanism = pd.read_csv(mechanism_path)
    if diag.empty or mechanism.empty:
        return result

    diag = diag[diag['label'] == 1].copy()
    if diag.empty:
        return result

    target_map = (
        mechanism[['stressor', 'dominant_mechanism_mode']]
        .drop_duplicates()
        .set_index('stressor')['dominant_mechanism_mode']
        .to_dict()
    )
    diag['target_mode'] = diag['stressor'].map(target_map)
    diag = diag[diag['target_mode'].notna()].copy()
    if diag.empty:
        return result

    for k in [1, 2, 3]:
        cols = [f'top_mechanism_{i}' for i in range(1, k + 1) if f'top_mechanism_{i}' in diag.columns]
        if not cols:
            continue
        match = diag[cols].eq(diag['target_mode'], axis=0).any(axis=1)
        result[f'diag_mechanism_top{k}_proxy'] = float(match.mean())
    return result


def _load_tuning_selective_top2(out_dir: Path, config: str = FINAL_CONFIG_NAME) -> tuple[float, float]:
    candidates = []
    for name in [
        'stressor_hierarchical_abstain_sweep.csv',
        'stressor_hierarchical_abstain_sweep_post_alert.csv',
        'stressor_hierarchical_abstain_sweep_top_blocks.csv',
        'diagnosis_mode_comparison.csv',
    ]:
        path = out_dir / name
        if not path.exists():
            continue
        df = pd.read_csv(path)
        if df.empty:
            continue
        if 'config' in df.columns:
            rows = df[df['config'].astype(str) == str(config)]
            if not rows.empty:
                df = rows.copy()
        top2_col = next((c for c in ['top2_acc', 'selective_top2_acc'] if c in df.columns), None)
        cov_col = next((c for c in ['coverage', 'selective_coverage'] if c in df.columns), None)
        if top2_col is None:
            continue
        row = df.sort_values(top2_col, ascending=False).iloc[0]
        candidates.append((float(row.get(top2_col, np.nan)), float(row.get(cov_col, np.nan)) if cov_col else np.nan))
    if not candidates:
        return np.nan, np.nan
    candidates.sort(key=lambda x: (np.nan_to_num(x[0], nan=-1.0), np.nan_to_num(x[1], nan=-1.0)))
    return candidates[-1]


def _alert_utility(row: dict[str, object]) -> float:
    detect = float(np.nan_to_num(row.get('anomaly_detect_rate', np.nan), nan=0.0))
    benign = float(np.nan_to_num(row.get('benign_run_alert_rate', np.nan), nan=1.0))
    top1 = float(np.nan_to_num(row.get('diag_top1_acc', np.nan), nan=0.0))
    top2 = float(np.nan_to_num(row.get('diag_top2_acc', np.nan), nan=0.0))
    macro = float(np.nan_to_num(row.get('diag_macro_f1', np.nan), nan=0.0))
    return detect - benign + 0.25 * top1 + 0.10 * top2 + 0.10 * macro


def _collect_tuning_row(
    out_dir: Path,
    *,
    stage: str,
    gain: float,
    block_B: int,
    alpha: float,
    persist_k: int,
    feature_profile: str,
    selection_mode: str,
) -> dict[str, object]:
    overall_row = _read_final_row(out_dir / 'overall_metrics.csv')
    seq_row = _read_final_row(out_dir / 'sequential_metrics.csv')
    diag_source, diag_top1_acc, diag_top2_acc, diag_macro_f1 = _load_tuning_diag(out_dir)
    benign_alert = float(seq_row.get('benign_run_alert_rate', np.nan))
    benign_keep = float(seq_row.get('benign_run_keep_rate', 1.0 - benign_alert if np.isfinite(benign_alert) else np.nan))
    selective_top2_acc, selective_coverage = _load_tuning_selective_top2(out_dir)
    mechanism_proxy = _load_tuning_mechanism_proxy(out_dir)
    row = {
        'stage': stage,
        'status': 'ok',
        'gain': float(gain),
        'block_B': int(block_B),
        'alpha': float(alpha),
        'persist_k': int(persist_k),
        'roc_auc_wc': float(overall_row.get('roc_auc_wc', np.nan)),
        'pr_auc_wc': float(overall_row.get('pr_auc_wc', np.nan)),
        'pr_auc': float(overall_row.get('pr_auc', np.nan)),
        'roc_auc': float(overall_row.get('roc_auc', np.nan)),
        'anomaly_detect_rate': float(seq_row.get('anomaly_detect_rate', np.nan)),
        'benign_run_alert_rate': benign_alert,
        'benign_run_keep_rate': benign_keep,
        'median_time_to_detect_s': float(seq_row.get('median_time_to_detect_s', np.nan)),
        'diag_source': diag_source,
        'diag_top1_acc': diag_top1_acc,
        'diag_top2_acc': diag_top2_acc,
        'diag_macro_f1': diag_macro_f1,
        'diag_selective_top2_acc': selective_top2_acc,
        'diag_selective_coverage': selective_coverage,
        'diag_mechanism_top1_proxy': float(mechanism_proxy.get('diag_mechanism_top1_proxy', np.nan)),
        'diag_mechanism_top2_proxy': float(mechanism_proxy.get('diag_mechanism_top2_proxy', np.nan)),
        'diag_mechanism_top3_proxy': float(mechanism_proxy.get('diag_mechanism_top3_proxy', np.nan)),
        'feature_profile': feature_profile,
        'selection_mode': selection_mode,
        'out_dir': str(out_dir),
    }
    row['diag_strong_score'] = float(np.nanmax([
        row.get('diag_top2_acc', np.nan),
        row.get('diag_selective_top2_acc', np.nan),
        row.get('diag_mechanism_top3_proxy', np.nan),
    ]))
    row['has_diag_over_0_8'] = int(row['diag_strong_score'] >= 0.80) if np.isfinite(row['diag_strong_score']) else 0
    row['alert_utility'] = _alert_utility(row)
    return row


def _enrich_tuning_summary_df(
    stage_df: pd.DataFrame,
    *,
    feature_profile: str,
    selection_mode: str,
) -> pd.DataFrame:
    if stage_df.empty:
        return stage_df.copy()

    df = stage_df.copy()
    fill_defaults = {
        'status': 'ok',
        'feature_profile': feature_profile,
        'selection_mode': selection_mode,
        'diag_selective_top2_acc': np.nan,
        'diag_selective_coverage': np.nan,
        'diag_mechanism_top1_proxy': np.nan,
        'diag_mechanism_top2_proxy': np.nan,
        'diag_mechanism_top3_proxy': np.nan,
        'diag_strong_score': np.nan,
        'has_diag_over_0_8': 0,
        'alert_utility': np.nan,
    }
    for col, default in fill_defaults.items():
        if col not in df.columns:
            df[col] = default
    df['feature_profile'] = df['feature_profile'].fillna(feature_profile)
    df['selection_mode'] = df['selection_mode'].fillna(selection_mode)
    df['status'] = df['status'].fillna('ok')

    cache: dict[str, dict[str, float] | tuple[float, float] | tuple[str, float, float, float]] = {}
    for idx, row in df.iterrows():
        out_dir = Path(str(row.get('out_dir', ''))).expanduser()
        if not out_dir.exists():
            continue

        out_key = str(out_dir.resolve())
        if out_key not in cache:
            cache[f'{out_key}::diag'] = _load_tuning_diag(out_dir)
            cache[f'{out_key}::selective'] = _load_tuning_selective_top2(out_dir)
            cache[f'{out_key}::mechanism'] = _load_tuning_mechanism_proxy(out_dir)

        diag_source, diag_top1_acc, diag_top2_acc, diag_macro_f1 = cache[f'{out_key}::diag']
        selective_top2_acc, selective_coverage = cache[f'{out_key}::selective']
        mechanism_proxy = cache[f'{out_key}::mechanism']

        refresh_pairs = {
            'diag_source': diag_source,
            'diag_top1_acc': diag_top1_acc,
            'diag_top2_acc': diag_top2_acc,
            'diag_macro_f1': diag_macro_f1,
            'diag_selective_top2_acc': selective_top2_acc,
            'diag_selective_coverage': selective_coverage,
            'diag_mechanism_top1_proxy': float(mechanism_proxy.get('diag_mechanism_top1_proxy', np.nan)),
            'diag_mechanism_top2_proxy': float(mechanism_proxy.get('diag_mechanism_top2_proxy', np.nan)),
            'diag_mechanism_top3_proxy': float(mechanism_proxy.get('diag_mechanism_top3_proxy', np.nan)),
        }
        for col, value in refresh_pairs.items():
            current = pd.to_numeric(df.at[idx, col], errors='coerce') if col != 'diag_source' else df.at[idx, col]
            if col == 'diag_source':
                if not str(current).strip() or str(current).lower() == 'nan':
                    df.at[idx, col] = value
            elif not np.isfinite(current):
                df.at[idx, col] = value

        strong_candidates = np.asarray(
            [
                pd.to_numeric(df.at[idx, 'diag_top2_acc'], errors='coerce'),
                pd.to_numeric(df.at[idx, 'diag_selective_top2_acc'], errors='coerce'),
                pd.to_numeric(df.at[idx, 'diag_mechanism_top3_proxy'], errors='coerce'),
            ],
            dtype=float,
        )
        strong_candidates = strong_candidates[np.isfinite(strong_candidates)]
        strong_score = float(np.max(strong_candidates)) if len(strong_candidates) else np.nan
        df.at[idx, 'diag_strong_score'] = strong_score
        df.at[idx, 'has_diag_over_0_8'] = int(np.isfinite(strong_score) and strong_score >= 0.80)
        df.at[idx, 'alert_utility'] = _alert_utility(df.loc[idx].to_dict())

    return df


def _select_recommended_tuning(stage_df: pd.DataFrame, selection_mode: str) -> pd.Series:
    if stage_df.empty:
        raise RuntimeError('No successful tuning runs were collected.')
    mode = str(selection_mode).lower()
    stage_df = stage_df.copy()
    if 'status' in stage_df.columns:
        ok_rows = stage_df[stage_df['status'].astype(str).str.lower() == 'ok'].copy()
        if not ok_rows.empty:
            stage_df = ok_rows
    if mode == 'conservative':
        feasible = stage_df[stage_df['benign_run_alert_rate'] <= 0.25].copy()
        if feasible.empty:
            feasible = stage_df.copy()
        ranked = feasible.sort_values(
            ['benign_run_alert_rate', 'anomaly_detect_rate', 'diag_top1_acc', 'pr_auc_wc', 'roc_auc_wc'],
            ascending=[True, False, False, False, False],
        )
    elif mode == 'aggressive':
        ranked = stage_df.sort_values(
            ['anomaly_detect_rate', 'alert_utility', 'diag_top1_acc', 'pr_auc_wc', 'roc_auc_wc'],
            ascending=[False, False, False, False, False],
        )
    elif mode == 'diagnosis_low_fp':
        feasible = stage_df[stage_df['benign_run_alert_rate'] <= 0.30].copy()
        if feasible.empty:
            feasible = stage_df.copy()
        strong = feasible[feasible['has_diag_over_0_8'] == 1].copy() if 'has_diag_over_0_8' in feasible.columns else pd.DataFrame()
        if not strong.empty:
            feasible = strong
        ranked = feasible.sort_values(
            [
                'benign_run_alert_rate',
                'has_diag_over_0_8',
                'diag_strong_score',
                'diag_mechanism_top3_proxy',
                'diag_selective_top2_acc',
                'diag_selective_coverage',
                'diag_top2_acc',
                'anomaly_detect_rate',
                'pr_auc_wc',
                'roc_auc_wc',
                'median_time_to_detect_s',
            ],
            ascending=[True, False, False, False, False, False, False, False, False, False, True],
        )
    elif mode == 'diagnosis_first':
        feasible = stage_df[stage_df['benign_run_alert_rate'] <= 0.30].copy()
        if feasible.empty:
            feasible = stage_df.copy()
        strong = feasible[feasible['has_diag_over_0_8'] == 1].copy() if 'has_diag_over_0_8' in feasible.columns else pd.DataFrame()
        if not strong.empty:
            feasible = strong
        ranked = feasible.sort_values(
            [
                'has_diag_over_0_8',
                'diag_strong_score',
                'diag_mechanism_top3_proxy',
                'diag_selective_top2_acc',
                'diag_selective_coverage',
                'diag_top2_acc',
                'anomaly_detect_rate',
                'benign_run_alert_rate',
                'pr_auc_wc',
                'roc_auc_wc',
                'median_time_to_detect_s',
            ],
            ascending=[False, False, False, False, False, False, False, True, False, False, True],
        )
    else:
        feasible = stage_df[stage_df['benign_run_alert_rate'] <= 0.25].copy()
        if feasible.empty:
            feasible = stage_df.copy()
        ranked = feasible.sort_values(
            ['alert_utility', 'diag_top1_acc', 'diag_top2_acc', 'diag_macro_f1', 'pr_auc_wc', 'roc_auc_wc'],
            ascending=[False, False, False, False, False, False],
        )
    return ranked.iloc[0]


def _write_tuning_recommendation_bundle(
    out_tune: Path,
    *,
    stage1_df: pd.DataFrame,
    stage2_df: pd.DataFrame,
    feature_profile: str,
    selection_mode: str,
) -> pd.Series:
    stage1_df = _enrich_tuning_summary_df(stage1_df, feature_profile=feature_profile, selection_mode=selection_mode)
    stage2_df = _enrich_tuning_summary_df(stage2_df, feature_profile=feature_profile, selection_mode=selection_mode)
    summary_df = pd.concat([stage1_df, stage2_df], ignore_index=True)
    recommended = _select_recommended_tuning(stage2_df, selection_mode)
    recommended_df = pd.DataFrame([recommended]).drop(columns=['stage', 'out_dir'], errors='ignore')

    stage1_df.to_csv(out_tune / 'sweep_stage1_gain_block.csv', index=False)
    stage2_df.to_csv(out_tune / 'sweep_stage2_alpha_persist.csv', index=False)
    summary_df.to_csv(out_tune / 'sweep_summary_all.csv', index=False)
    summary_df.to_csv(out_tune / 'sweep_summary.csv', index=False)
    summary_df.to_csv(out_tune / 'tuning_summary.csv', index=False)
    recommended_df.to_csv(out_tune / 'recommended_alert_config.csv', index=False)
    recommended_df.to_csv(out_tune / 'recommended_config.csv', index=False)
    (out_tune / 'recommended_config.json').write_text(
        json.dumps(
            {
                'gain': float(recommended['gain']),
                'block_B': int(recommended['block_B']),
                'alpha': float(recommended['alpha']),
                'persist_k': int(recommended['persist_k']),
                'feature_profile': feature_profile,
                'selection_mode': selection_mode,
                'out_dir': str(recommended['out_dir']),
            },
            indent=2,
        )
    )
    return recommended


def run_analysis_notebook(root: Path, source_hz: int = 5) -> Path:
    env = deterministic_env()
    out_dir = root / 'results_analysis'
    cmd = [
        sys.executable,
        str(RUN_ANALYSIS_TOOL),
        '--root', str(root),
        '--source_hz', str(source_hz),
        '--out_dir', str(out_dir),
    ]
    _run_checked(cmd, env=env, cwd=REPO_ROOT)
    return out_dir


def run_full_notebook(
    root: Path,
    *,
    protocol: str,
    source_hz: int,
    fit_ratio: float,
    block_B: int,
    alpha: float,
    persist_k: int,
    gain: float,
    ridge_lambda: float,
    feature_profile: str,
) -> Path:
    env = deterministic_env()
    out_dir = default_full_out_dir(root, protocol=protocol, feature_profile=feature_profile)
    cmd = [
        sys.executable,
        str(RUN_FULL_TOOL),
        '--root', str(root),
        '--out_dir', str(out_dir),
        '--protocol', str(protocol),
        '--source_hz', str(source_hz),
        '--fit_ratio', str(fit_ratio),
        '--block_B', str(block_B),
        '--alpha', str(alpha),
        '--persist_k', str(persist_k),
        '--gain', str(gain),
        '--ridge_lambda', str(ridge_lambda),
        '--feature_profile', str(feature_profile),
    ]
    _run_checked(cmd, env=env, cwd=REPO_ROOT)
    return out_dir


def run_tuning_notebook(
    root: Path,
    *,
    source_hz: int,
    fit_ratio: float,
    ridge_lambda: float,
    feature_profile: str,
    selection_mode: str = 'balanced',
) -> Path:
    out_tune = default_tuning_out_dir(root, feature_profile)
    out_runs = out_tune / 'runs'
    stage1_path = out_tune / 'sweep_stage1_gain_block.csv'
    stage2_path = out_tune / 'sweep_stage2_alpha_persist.csv'
    out_tune.mkdir(parents=True, exist_ok=True)
    out_runs.mkdir(parents=True, exist_ok=True)

    rec_path = out_tune / 'recommended_alert_config.csv'
    if stage1_path.exists() and stage2_path.exists() and not FORCE_RERUN_TUNING:
        print(f"[reuse] existing tuning sweep grids found at {out_tune}; recomputing recommendation for selection_mode={selection_mode}")
        stage1_df = pd.read_csv(stage1_path)
        stage2_df = pd.read_csv(stage2_path)
        _write_tuning_recommendation_bundle(
            out_tune,
            stage1_df=stage1_df,
            stage2_df=stage2_df,
            feature_profile=feature_profile,
            selection_mode=selection_mode,
        )
        return out_tune
    if rec_path.exists() and not FORCE_RERUN_TUNING:
        try:
            rec_df = pd.read_csv(rec_path)
            if not rec_df.empty and str(rec_df.iloc[0].get('selection_mode', selection_mode)).lower() == str(selection_mode).lower():
                print(f"[reuse] existing tuning recommendation found at {rec_path}")
                return out_tune
        except Exception:
            pass

    stage1_rows = []
    for gain in TUNING_STAGE1_GAINS:
        for block_B in TUNING_STAGE1_BLOCKS:
            out_dir = out_runs / f'g{gain}_B{block_B}_a0.05_k3'
            run_full_notebook(
                root,
                protocol='global',
                source_hz=source_hz,
                fit_ratio=fit_ratio,
                block_B=block_B,
                alpha=0.05,
                persist_k=3,
                gain=gain,
                ridge_lambda=ridge_lambda,
                feature_profile=feature_profile,
            ) if False else None
            env = deterministic_env()
            cmd = [
                sys.executable,
                str(RUN_FULL_TOOL),
                '--root', str(root),
                '--out_dir', str(out_dir),
                '--protocol', 'global',
                '--source_hz', str(source_hz),
                '--fit_ratio', str(fit_ratio),
                '--block_B', str(block_B),
                '--alpha', '0.05',
                '--persist_k', '3',
                '--gain', str(gain),
                '--ridge_lambda', str(ridge_lambda),
                '--feature_profile', str(feature_profile),
            ]
            _run_checked(cmd, env=env, cwd=REPO_ROOT)
            stage1_rows.append(
                _collect_tuning_row(
                    out_dir,
                    stage='gain_block',
                    gain=gain,
                    block_B=block_B,
                    alpha=0.05,
                    persist_k=3,
                    feature_profile=feature_profile,
                    selection_mode=selection_mode,
                )
            )

    stage1_df = pd.DataFrame(stage1_rows)
    best_stage1 = _select_recommended_tuning(stage1_df, selection_mode)
    best_gain = float(best_stage1['gain'])
    best_block = int(best_stage1['block_B'])

    stage2_rows = []
    for alpha in TUNING_STAGE2_ALPHAS:
        for persist_k in TUNING_STAGE2_PERSISTS:
            out_dir = out_runs / f'g{best_gain}_B{best_block}_a{alpha}_k{persist_k}'
            env = deterministic_env()
            cmd = [
                sys.executable,
                str(RUN_FULL_TOOL),
                '--root', str(root),
                '--out_dir', str(out_dir),
                '--protocol', 'global',
                '--source_hz', str(source_hz),
                '--fit_ratio', str(fit_ratio),
                '--block_B', str(best_block),
                '--alpha', str(alpha),
                '--persist_k', str(persist_k),
                '--gain', str(best_gain),
                '--ridge_lambda', str(ridge_lambda),
                '--feature_profile', str(feature_profile),
            ]
            _run_checked(cmd, env=env, cwd=REPO_ROOT)
            stage2_rows.append(
                _collect_tuning_row(
                    out_dir,
                    stage='alpha_persist',
                    gain=best_gain,
                    block_B=best_block,
                    alpha=alpha,
                    persist_k=persist_k,
                    feature_profile=feature_profile,
                    selection_mode=selection_mode,
                )
            )

    stage2_df = pd.DataFrame(stage2_rows)
    _write_tuning_recommendation_bundle(
        out_tune,
        stage1_df=stage1_df,
        stage2_df=stage2_df,
        feature_profile=feature_profile,
        selection_mode=selection_mode,
    )
    return out_tune


def _dataset_tree_sha256(root: Path) -> dict[str, object]:
    hasher = hashlib.sha256()
    count = 0
    for path in sorted(p for p in root.rglob('*') if p.is_file()):
        rel = path.relative_to(root).as_posix()
        if rel.split('/', 1)[0].startswith('results_'):
            continue
        hasher.update(rel.encode('utf-8'))
        hasher.update(path.read_bytes())
        count += 1
    return {'file_count': count, 'sha256': hasher.hexdigest()}


def _file_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def _package_versions() -> dict[str, str]:
    packages = [
        'matplotlib', 'numpy', 'pandas', 'psutil', 'scikit-learn', 'scipy',
        'joblib', 'threadpoolctl', 'python-dateutil', 'pytz', 'tzdata',
    ]
    out = {}
    for pkg in packages:
        try:
            out[pkg.replace('-', '_')] = importlib.metadata.version(pkg)
        except importlib.metadata.PackageNotFoundError:
            out[pkg.replace('-', '_')] = 'missing'
    return out


def write_run_manifest(
    root: Path,
    analysis_out: Path,
    full_out: Path,
    holdout_out: Path | None,
    tuning_out: Path | None,
    run_config: dict[str, object],
) -> Path:
    env = deterministic_env()
    manifest_dir = root / 'results_portable'
    manifest_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = manifest_dir / 'run_manifest.json'
    manifest = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'repo_root': str(REPO_ROOT),
        'dataset_root': str(root),
        'dataset_digest': _dataset_tree_sha256(root),
        'python': {
            'executable': sys.executable,
            'version': sys.version,
        },
        'platform': {
            'platform': platform.platform(),
            'machine': platform.machine(),
            'processor': platform.processor(),
        },
        'package_versions': _package_versions(),
        'deterministic_env': {
            key: env.get(key, '')
            for key in [
                'MPLCONFIGDIR', 'MPLBACKEND', 'OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS',
                'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS',
                'BLIS_NUM_THREADS', 'PYTHONHASHSEED',
            ]
        },
        'environment_files': {
            name: {'path': str(path), 'sha256': _file_sha256(path)}
            for name, path in {
                'requirements_txt': REPO_ROOT / 'requirements.txt',
                'environment_yml': REPO_ROOT / 'environment.yml',
            }.items()
            if path.exists()
        },
        'parameters': run_config,
        'outputs': {
            'results_analysis': str(analysis_out),
            'results_dice_full': str(full_out),
            'results_dice_full_holdout': str(holdout_out) if holdout_out else '',
            'results_dice_tuning': str(tuning_out) if tuning_out else '',
        },
    }
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    return manifest_path

runtime_start = perf_counter()
notebook_run_summary = {}
stage_logs = {}

progress_view = display(HTML(''), display_id=True)


class Tee(io.TextIOBase):
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
        return len(data)

    def flush(self):
        for stream in self.streams:
            if hasattr(stream, 'flush'):
                stream.flush()


def update_progress(done, total, title, detail=''):
    pct = 100.0 * done / max(total, 1)
    progress_view.update(
        HTML(
            f"""
            <div style="border:1px solid #d8d8d8; border-radius:12px; padding:14px; margin:8px 0;">
              <div style="font-weight:700; margin-bottom:8px;">DICE artifact generation progress</div>
              <div style="background:#eeeeee; border-radius:999px; overflow:hidden; height:12px;">
                <div style="width:{pct:.1f}%; background:linear-gradient(90deg,#4E79A7,#59A14F); height:12px;"></div>
              </div>
              <div style="margin-top:10px;"><b>{done}/{total}</b> stages complete</div>
              <div style="margin-top:6px;"><b>Current stage:</b> {html.escape(title)}</div>
              <div style="color:#666; margin-top:4px;">{html.escape(detail)}</div>
            </div>
            """
        )
    )


def run_stage(step_idx, total_steps, stage_name, fn, *args, detail='', **kwargs):
    update_progress(step_idx - 1, total_steps, stage_name, detail)
    print(f"\n===== {stage_name} =====")
    if detail:
        print(detail)
    buf = io.StringIO()
    t0 = perf_counter()
    try:
        if QUIET_STAGE_OUTPUT:
            with redirect_stdout(buf), redirect_stderr(buf):
                out = fn(*args, **kwargs)
        else:
            tee_out = Tee(sys.stdout, buf)
            tee_err = Tee(sys.stderr, buf)
            with redirect_stdout(tee_out), redirect_stderr(tee_err):
                out = fn(*args, **kwargs)
        elapsed = perf_counter() - t0
        stage_logs[stage_name] = buf.getvalue()
        print(f"[done] {stage_name} finished in {elapsed:.2f} s")
        update_progress(step_idx, total_steps, f'{stage_name} complete', f'Finished in {elapsed:.2f} s')
        return out
    except Exception as exc:
        stage_logs[stage_name] = buf.getvalue()
        update_progress(step_idx - 1, total_steps, f'{stage_name} failed', str(exc))
        raise


def resolve_tuned_alert_config_path(profile: str, tuning_out: Path) -> Path:
    if TUNED_ALERT_CONFIG_PATH:
        raw = str(TUNED_ALERT_CONFIG_PATH)
        candidate = raw.format(feature_profile=profile) if '{feature_profile}' in raw else raw
        candidate_path = Path(candidate).expanduser()
        if candidate_path.is_dir():
            return (candidate_path / 'recommended_alert_config.csv').resolve()
        return candidate_path.resolve()
    return (Path(tuning_out) / 'recommended_alert_config.csv').resolve()


def resolve_primary_profile_run(profile_runs: dict[str, dict[str, object]], preferred_profile: str) -> tuple[str, Path, Path | None, Path | None]:
    candidates = []
    if preferred_profile in FEATURE_PROFILES_TO_SWEEP:
        candidates.append(preferred_profile)
    for profile in FEATURE_PROFILES_TO_SWEEP:
        if profile not in candidates:
            candidates.append(profile)

    for profile in candidates:
        run = profile_runs.get(profile, {}) if isinstance(profile_runs, dict) else {}
        global_dir = Path(run['global_out']).expanduser().resolve() if run.get('global_out') else PROFILE_OUT_DIRS[profile]['global']
        holdout_dir = Path(run['holdout_out']).expanduser().resolve() if run.get('holdout_out') else PROFILE_OUT_DIRS[profile]['holdout']
        tuning_dir = Path(run['tuning_out']).expanduser().resolve() if run.get('tuning_out') else default_tuning_out_dir(DATASET_ROOT, profile)
        if (global_dir / 'overall_metrics.csv').exists():
            return (
                profile,
                global_dir,
                holdout_dir if (holdout_dir / 'holdout_robustness_summary.csv').exists() else None,
                tuning_dir if tuning_dir.exists() else None,
            )

    fallback_profile = candidates[0] if candidates else DISPLAY_FEATURE_PROFILE
    return (
        fallback_profile,
        PROFILE_OUT_DIRS[fallback_profile]['global'],
        PROFILE_OUT_DIRS[fallback_profile]['holdout'],
        default_tuning_out_dir(DATASET_ROOT, fallback_profile),
    )


def load_best_diagnosis_metrics(global_dir: Path, final_config: str = 'tier0_tier1_tier2') -> tuple[pd.DataFrame, str, Path | None]:
    candidates = []
    for diag_source, name in [
        ('feature-level', 'stressor_feature_diagnosis_metrics.csv'),
        ('mechanism-level', 'stressor_diagnosis_metrics.csv'),
    ]:
        path = global_dir / name
        if not path.exists():
            continue
        diag = pd.read_csv(path)
        if 'macro_f1' not in diag.columns:
            diag = diag.copy()
            diag['macro_f1'] = np.nan
        final_rows = diag[diag['config'] == final_config]
        row = final_rows.iloc[0] if not final_rows.empty else (diag.iloc[0] if not diag.empty else None)
        if row is None:
            continue
        candidates.append(
            (
                np.nan_to_num(float(row.get('top1_acc', np.nan)), nan=-1.0),
                np.nan_to_num(float(row.get('top2_acc', np.nan)), nan=-1.0),
                np.nan_to_num(float(row.get('macro_f1', np.nan)), nan=-1.0),
                diag_source,
                diag,
                path,
            )
        )
    if not candidates:
        return pd.DataFrame(), 'unavailable', None
    _, _, _, diag_source, diag, path = sorted(candidates)[-1]
    return diag, diag_source, path




def display_inline_figure(fig, dpi: int = 180) -> None:
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    display(IPythonImage(data=buf.getvalue()))


def show_png(title, path, width=1180):
    if path.exists():
        display(Markdown(f'#### {title}'))
        display(IPythonImage(filename=str(path), width=width))


def _safe_config_label(config: str) -> str:
    cfg = str(config)
    return CFG_LABEL.get(cfg, cfg.replace('_', '/'))


def _comparison_paths() -> tuple[Path, Path]:
    comparison_dir = Path(globals().get('COMPARISON_DIR', OUT_PAPER / 'comparison'))
    comparison_fig = Path(globals().get('COMPARISON_FIG', comparison_dir / 'figures'))
    comparison_fig.mkdir(parents=True, exist_ok=True)
    return comparison_dir, comparison_fig


def _load_profile_comparison_row(csv_name: str, feature_profile: str, config: str = FINAL_CONFIG_NAME) -> tuple[pd.Series | None, Path]:
    comparison_dir, _ = _comparison_paths()
    path = comparison_dir / csv_name
    if not path.exists():
        return None, path
    df = pd.read_csv(path)
    if 'feature_profile' not in df.columns:
        return None, path
    rows = df[df['feature_profile'].astype(str) == str(feature_profile)].copy()
    if 'config' in rows.columns:
        cfg_rows = rows[rows['config'].astype(str) == str(config)]
        if not cfg_rows.empty:
            rows = cfg_rows
    if rows.empty:
        return None, path
    return rows.iloc[0], path


def _load_profile_comparison_df(csv_name: str, feature_profile: str, config: str = FINAL_CONFIG_NAME) -> tuple[pd.DataFrame, Path]:
    comparison_dir, _ = _comparison_paths()
    path = comparison_dir / csv_name
    if not path.exists():
        return pd.DataFrame(), path
    df = pd.read_csv(path)
    if 'feature_profile' not in df.columns:
        return pd.DataFrame(), path
    rows = df[df['feature_profile'].astype(str) == str(feature_profile)].copy()
    if 'config' in rows.columns:
        cfg_rows = rows[rows['config'].astype(str) == str(config)]
        if not cfg_rows.empty:
            rows = cfg_rows
    return rows.reset_index(drop=True), path


def _build_monitoring_summary(overall_full: pd.DataFrame, sequential: pd.DataFrame) -> pd.DataFrame:
    base = overall_full[['config', 'roc_auc_wc', 'pr_auc_wc']].copy()
    seq_cols = ['config', 'benign_run_alert_rate', 'benign_run_keep_rate', 'anomaly_detect_rate', 'median_time_to_detect_s']
    seq = sequential[seq_cols].copy()
    summary_df = base.merge(seq, on='config', how='left')

    config_values = summary_df['config'].astype(str).tolist()
    config_order = [cfg for cfg in CONFIG_ORDER if cfg in config_values]
    config_order += [cfg for cfg in config_values if cfg not in config_order]
    order_map = {cfg: i for i, cfg in enumerate(config_order)}

    summary_df['config_label'] = summary_df['config'].map(_safe_config_label)
    summary_df['sort_key'] = summary_df['config'].astype(str).map(order_map)
    summary_df = summary_df.sort_values('sort_key').reset_index(drop=True)
    return summary_df


def _build_strong_diagnosis_table(feature_profile: str, config: str = FINAL_CONFIG_NAME) -> tuple[pd.DataFrame, str, Path | None]:
    row, path = _load_profile_comparison_row('industry_diagnosis_strength_profiles.csv', feature_profile, config)
    records = []
    note = ''

    if row is not None:
        threshold = 0.80
        candidates = [
            (
                'Exact Top-2 shortlist',
                float(row.get('exact_top2_acc', np.nan)),
                'Correct stressor appears within the top two DICE guesses.',
            ),
            (
                'Confidence-gated Top-2 shortlist',
                float(row.get('selective_top2_acc', np.nan)),
                f"Only shown on {float(row.get('selective_coverage', np.nan)):.2f} of cases using {row.get('selective_method', 'selective diagnosis') }.",
            ),
            (
                'Mechanism Top-3 localization',
                float(row.get('mechanism_top3_coverage', np.nan)),
                'Correct subsystem/mechanism family appears in the top three localized explanations.',
            ),
        ]
        for label, value, evidence in candidates:
            if np.isfinite(value) and value >= threshold:
                records.append({
                    'Diagnosis result': label,
                    'Score': value,
                    'Evidence': evidence,
                })
        claims = str(row.get('strong_claims', '')).strip()
        if claims and claims.lower() != 'nan':
            note = claims

    strong_df = pd.DataFrame(records)
    if not strong_df.empty:
        strong_df = strong_df.sort_values('Score', ascending=False).reset_index(drop=True)
    return strong_df, note, path if row is not None else None


def _style_axes(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.tick_params(labelsize=12)


def draw_executive_monitoring_dashboard(
    summary_df: pd.DataFrame,
    strong_diag_df: pd.DataFrame,
    profile_label: str,
    out_path: Path,
) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 3, figsize=(19.5, 6.2), gridspec_kw={'width_ratios': [1.05, 1.1, 1.0]})
    ax_quality, ax_trade, ax_diag = axes

    # Panel 1: monitoring quality by head.
    x = np.arange(len(summary_df))
    width = 0.34
    auc_colors = {'AUC-PR': '#355070', 'ROC-AUC': '#2A9D8F'}
    pr_vals = summary_df['pr_auc_wc'].to_numpy(dtype=float)
    roc_vals = summary_df['roc_auc_wc'].to_numpy(dtype=float)
    bars_pr = ax_quality.bar(x - width / 2, pr_vals, width, color=auc_colors['AUC-PR'], edgecolor='white', linewidth=1.2, label='AUC-PR')
    bars_roc = ax_quality.bar(x + width / 2, roc_vals, width, color=auc_colors['ROC-AUC'], edgecolor='white', linewidth=1.2, label='ROC-AUC')
    all_quality = np.concatenate([pr_vals[np.isfinite(pr_vals)], roc_vals[np.isfinite(roc_vals)]])
    y_lo = 0.0 if len(all_quality) == 0 else max(0.0, min(float(np.min(all_quality)) - 0.10, 0.70))
    for bars in [bars_pr, bars_roc]:
        for rect in bars:
            val = rect.get_height()
            if np.isfinite(val):
                ax_quality.text(
                    rect.get_x() + rect.get_width() / 2,
                    val + 0.015,
                    f'{val:.2f}',
                    ha='center',
                    va='bottom',
                    fontsize=12,
                    fontweight='bold',
                    color='#0F172A',
                )
    ax_quality.set_xticks(x)
    ax_quality.set_xticklabels(summary_df['config_label'], fontsize=13)
    ax_quality.set_ylim(y_lo, 1.03)
    ax_quality.set_ylabel('Score', fontsize=14, fontweight='bold')
    ax_quality.set_title('Monitoring quality by observation head', fontsize=16, fontweight='bold')
    ax_quality.grid(axis='y', alpha=0.18)
    _style_axes(ax_quality)

    # Panel 2: operational tradeoff.
    y = np.arange(len(summary_df))[::-1]
    for yi, row in zip(y, summary_df.itertuples(index=False)):
        left = float(np.nan_to_num(row.benign_run_alert_rate, nan=0.0))
        right = float(np.nan_to_num(row.anomaly_detect_rate, nan=0.0))
        ax_trade.hlines(yi, left, right, color='#CBD5E1', linewidth=6, zorder=1)
        ax_trade.scatter(left, yi, s=190, color='#F28E2B', edgecolor='white', linewidth=1.6, zorder=3)
        ax_trade.scatter(right, yi, s=220, marker='^', color='#E15759', edgecolor='white', linewidth=1.0, zorder=4)
        ax_trade.text(max(left - 0.03, 0.0), yi - 0.18, f'{left:.2f}', ha='right', va='center', fontsize=12, color='#9A3412')
        ax_trade.text(min(right + 0.03, 1.03), yi - 0.18, f'{right:.2f}', ha='left', va='center', fontsize=12, color='#991B1B')
        if np.isfinite(row.median_time_to_detect_s):
            ax_trade.text(1.03, yi + 0.16, f'TTD {row.median_time_to_detect_s:.0f}s', ha='right', va='center', fontsize=12, color='#334155')
    ax_trade.set_yticks(y)
    ax_trade.set_yticklabels(summary_df['config_label'], fontsize=13)
    ax_trade.set_xlim(-0.02, 1.05)
    ax_trade.set_xlabel('Rate', fontsize=14, fontweight='bold')
    ax_trade.set_title('Operational tradeoff: false alerts vs detections', fontsize=16, fontweight='bold')
    ax_trade.grid(axis='x', alpha=0.18)
    ax_trade.text(0.0, 1.04, 'Circle = benign alert rate; triangle = anomaly detection rate', transform=ax_trade.transAxes, fontsize=11.5, color='#475569', ha='left')
    _style_axes(ax_trade)

    # Panel 3: only strong diagnosis claims.
    if strong_diag_df.empty:
        ax_diag.text(0.5, 0.56, 'No diagnosis metric >= 0.80\nfor this active profile.', ha='center', va='center', fontsize=14, color='#475569')
        ax_diag.text(0.5, 0.34, 'Weak exact-stressor numbers are intentionally\nsuppressed in this summary.', ha='center', va='center', fontsize=12, color='#64748B')
        ax_diag.set_axis_off()
    else:
        plot_df = strong_diag_df.sort_values('Score', ascending=True).reset_index(drop=True)
        yy = np.arange(len(plot_df))
        colors = ['#4E79A7', '#59A14F', '#B07AA1'][: len(plot_df)]
        bars = ax_diag.barh(yy, plot_df['Score'], color=colors, edgecolor='white', linewidth=1.2)
        x_lo = max(0.75, float(plot_df['Score'].min()) - 0.05)
        ax_diag.axvline(0.80, color='#7C2D12', linestyle=(0, (4, 4)), linewidth=1.8)
        for bar, row in zip(bars, plot_df.itertuples(index=False)):
            ax_diag.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2, f'{row.Score:.2f}', va='center', fontsize=12, fontweight='bold', color='#0F172A')
        ax_diag.set_yticks(yy)
        ax_diag.set_yticklabels(plot_df['Diagnosis result'], fontsize=12.5)
        ax_diag.set_xlim(x_lo, 1.01)
        ax_diag.set_xlabel('Score', fontsize=14, fontweight='bold')
        ax_diag.set_title('Strong diagnosis claims only (>= 0.80)', fontsize=16, fontweight='bold')
        ax_diag.grid(axis='x', alpha=0.18)
        ax_diag.text(0.0, 1.04, 'Notebook summary suppresses weaker diagnosis metrics.', transform=ax_diag.transAxes, fontsize=11.5, color='#475569', ha='left')
        _style_axes(ax_diag)

    legend_handles = [
        Patch(facecolor=auc_colors['AUC-PR'], edgecolor='white', label='AUC-PR'),
        Patch(facecolor=auc_colors['ROC-AUC'], edgecolor='white', label='ROC-AUC'),
        Line2D([0], [0], marker='o', color='none', markerfacecolor='#F28E2B', markeredgecolor='white', markersize=10, label='Benign alert rate'),
        Line2D([0], [0], marker='^', color='none', markerfacecolor='#E15759', markeredgecolor='white', markersize=10, label='Detection rate'),
        Line2D([0], [0], color='#7C2D12', linestyle=(0, (4, 4)), linewidth=1.8, label='0.80 threshold'),
    ]
    fig.legend(handles=legend_handles, loc='lower center', bbox_to_anchor=(0.5, -0.01), ncol=5, frameon=False, fontsize=11.5)
    fig.suptitle(f'{profile_label}: executive monitoring and diagnosis summary', fontsize=19, fontweight='bold', y=0.98)
    fig.subplots_adjust(bottom=0.16, top=0.88, left=0.06, right=0.98, wspace=0.30)
    fig.savefig(out_path, dpi=220, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return out_path


def draw_diagnosis_localization_figure(
    timeline_df: pd.DataFrame,
    mechanism_df: pd.DataFrame,
    profile_label: str,
    out_path: Path,
) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(19.5, 6.4), gridspec_kw={'width_ratios': [1.15, 1.0]})
    ax_time, ax_mech = axes

    # Panel 1: first-through-fifth abnormal blocks.
    if timeline_df.empty:
        ax_time.text(0.5, 0.5, 'No localization timeline table found yet.', ha='center', va='center', fontsize=14, color='#475569')
        ax_time.set_axis_off()
    else:
        plot_df = timeline_df.copy().sort_values('first_block_s').reset_index(drop=True)
        block_cols = ['first_block_s', 'second_block_s', 'third_block_s', 'fourth_block_s', 'fifth_block_s']
        point_colors = ['#B91C1C', '#D97706', '#2563EB', '#0D9488', '#7C3AED']
        y = np.arange(len(plot_df))[::-1]
        for yi, row in zip(y, plot_df.itertuples(index=False)):
            times = [float(getattr(row, col)) for col in block_cols if np.isfinite(getattr(row, col))]
            if not times:
                continue
            ax_time.hlines(yi, min(times), max(times), color='#CBD5E1', linewidth=7, zorder=1)
            for color, time_val in zip(point_colors, times):
                ax_time.scatter(time_val, yi, s=155, color=color, edgecolor='white', linewidth=1.0, zorder=3)
            tail_label = f'{row.workload} | {row.dominant_tier}'
            ax_time.text(max(times) + 14, yi, tail_label, va='center', fontsize=11.5, color='#475569')
        ax_time.set_yticks(y)
        ax_time.set_yticklabels(plot_df['stressor'], fontsize=13)
        ax_time.set_xlabel('Time of first through fifth abnormal blocks (s)', fontsize=14, fontweight='bold')
        ax_time.set_title('When DICE first localizes each anomaly', fontsize=16, fontweight='bold')
        ax_time.grid(axis='x', alpha=0.18)
        ax_time.text(0.0, 1.04, 'Each colored marker is one consecutive abnormal decision window.', transform=ax_time.transAxes, fontsize=11.5, color='#475569', ha='left')
        _style_axes(ax_time)

    # Panel 2: stronger mechanism-localization view instead of weak raw stressor attribution.
    if mechanism_df.empty:
        ax_mech.text(0.5, 0.5, 'No mechanism-localization table found yet.', ha='center', va='center', fontsize=14, color='#475569')
        ax_mech.set_axis_off()
    else:
        mech_plot = mechanism_df.copy().sort_values('mechanism_top3_coverage', ascending=True).reset_index(drop=True)
        yy = np.arange(len(mech_plot))
        bars = ax_mech.barh(yy, mech_plot['mechanism_top3_coverage'], color='#355070', edgecolor='white', linewidth=1.2)
        ax_mech.axvline(0.80, color='#7C2D12', linestyle=(0, (4, 4)), linewidth=1.8)
        for bar, row in zip(bars, mech_plot.itertuples(index=False)):
            ax_mech.text(min(bar.get_width() + 0.015, 1.02), bar.get_y() + bar.get_height() / 2, f'{row.mechanism_top3_coverage:.2f}', va='center', fontsize=12, fontweight='bold', color='#0F172A')
            ax_mech.text(0.02, bar.get_y() + bar.get_height() / 2 - 0.28, str(row.likely_subsystem), va='center', fontsize=10.5, color='#64748B')
        ax_mech.set_yticks(yy)
        ax_mech.set_yticklabels(mech_plot['stressor'], fontsize=13)
        ax_mech.set_xlim(0.0, 1.05)
        ax_mech.set_xlabel('Mechanism Top-3 coverage', fontsize=14, fontweight='bold')
        ax_mech.set_title('Mechanism localization by stressor', fontsize=16, fontweight='bold')
        ax_mech.grid(axis='x', alpha=0.18)
        ax_mech.text(0.0, 1.04, 'This replaces the weaker raw stressor-attribution panel with a stronger subsystem view.', transform=ax_mech.transAxes, fontsize=11.5, color='#475569', ha='left')
        _style_axes(ax_mech)

    legend_handles = [
        Line2D([0], [0], marker='o', color='none', markerfacecolor='#B91C1C', markeredgecolor='white', markersize=10, label='1st block'),
        Line2D([0], [0], marker='o', color='none', markerfacecolor='#D97706', markeredgecolor='white', markersize=10, label='2nd block'),
        Line2D([0], [0], marker='o', color='none', markerfacecolor='#2563EB', markeredgecolor='white', markersize=10, label='3rd block'),
        Line2D([0], [0], marker='o', color='none', markerfacecolor='#0D9488', markeredgecolor='white', markersize=10, label='4th block'),
        Line2D([0], [0], marker='o', color='none', markerfacecolor='#7C3AED', markeredgecolor='white', markersize=10, label='5th block'),
        Line2D([0], [0], color='#7C2D12', linestyle=(0, (4, 4)), linewidth=1.8, label='0.80 threshold'),
    ]
    fig.legend(handles=legend_handles, loc='lower center', bbox_to_anchor=(0.5, -0.01), ncol=6, frameon=False, fontsize=11.5)
    fig.suptitle(f'{profile_label}: diagnosis and localization summary', fontsize=19, fontweight='bold', y=0.98)
    fig.subplots_adjust(bottom=0.16, top=0.88, left=0.08, right=0.98, wspace=0.30)
    fig.savefig(out_path, dpi=220, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return out_path


if RUN_END_TO_END:
    deterministic_env()
    root = Path(DATASET_ROOT).expanduser().resolve()
    ensure_dataset_root(root)

    profiles = FEATURE_PROFILES_TO_SWEEP
    total_steps = 1 + len(profiles) * (1 + int(RUN_HOLDOUT) + int(INCLUDE_TUNING))
    step = 1

    base_operating_point = {
        'source_hz': 5,
        'fit_ratio': 0.6,
        'block_B': 60,
        'alpha': 0.05,
        'persist_k': 3,
        'gain': 0.35,
        'ridge_lambda': 1e-3,
    }
    profile_runs = {}

    analysis_out = run_stage(
        step,
        total_steps,
        'Analysis figures',
        run_analysis_notebook,
        root,
        source_hz=5,
        detail='Generating tier-level tables, heatmaps, and AF-index figures.',
    )
    step += 1

    for profile in profiles:
        operating_point = dict(base_operating_point)
        tuned_operating_point = None
        tuning_out = None

        if INCLUDE_TUNING and USE_TUNED_ALERT_CONFIG:
            tuning_out = run_stage(
                step,
                total_steps,
                f'Tuning sweep [{profile}]',
                run_tuning_notebook,
                root,
                source_hz=operating_point['source_hz'],
                fit_ratio=operating_point['fit_ratio'],
                ridge_lambda=operating_point['ridge_lambda'],
                feature_profile=profile,
                selection_mode=ALERT_TUNING_OBJECTIVE,
                detail=f'Running the full parameter sweep for the {profile} profile and selecting an alert-oriented operating point.',
            )
            tuned_path = resolve_tuned_alert_config_path(profile, Path(tuning_out))
            if tuned_path.exists():
                tuned_df = pd.read_csv(tuned_path)
                if not tuned_df.empty:
                    tuned_row = tuned_df.iloc[0]
                    tuned_operating_point = {
                        'block_B': int(tuned_row['block_B']),
                        'alpha': float(tuned_row['alpha']),
                        'persist_k': int(tuned_row['persist_k']),
                        'gain': float(tuned_row['gain']),
                    }
                    operating_point.update(tuned_operating_point)
            step += 1

        full_out = run_stage(
            step,
            total_steps,
            f'Global DICE results [{profile}]',
            run_full_notebook,
            root,
            protocol='global',
            source_hz=operating_point['source_hz'],
            fit_ratio=operating_point['fit_ratio'],
            block_B=operating_point['block_B'],
            alpha=operating_point['alpha'],
            persist_k=operating_point['persist_k'],
            gain=operating_point['gain'],
            ridge_lambda=operating_point['ridge_lambda'],
            feature_profile=profile,
            detail=f'Generating global metrics and figures for the {profile} profile.',
        )
        step += 1

        holdout_out = None
        if RUN_HOLDOUT:
            holdout_out = run_stage(
                step,
                total_steps,
                f'Holdout robustness [{profile}]',
                run_full_notebook,
                root,
                protocol='workload_holdout',
                source_hz=operating_point['source_hz'],
                fit_ratio=operating_point['fit_ratio'],
                block_B=operating_point['block_B'],
                alpha=operating_point['alpha'],
                persist_k=operating_point['persist_k'],
                gain=operating_point['gain'],
                ridge_lambda=operating_point['ridge_lambda'],
                feature_profile=profile,
                detail=f'Generating cross-workload transfer summaries for the {profile} profile.',
            )
            step += 1

        if INCLUDE_TUNING and not USE_TUNED_ALERT_CONFIG:
            tuning_out = run_stage(
                step,
                total_steps,
                f'Tuning sweep [{profile}]',
                run_tuning_notebook,
                root,
                source_hz=operating_point['source_hz'],
                fit_ratio=operating_point['fit_ratio'],
                ridge_lambda=operating_point['ridge_lambda'],
                feature_profile=profile,
                selection_mode=ALERT_TUNING_OBJECTIVE,
                detail=f'Running the full parameter sweep for the {profile} profile after the main outputs.',
            )
            step += 1

        profile_runs[profile] = {
            'feature_profile': profile,
            'global_out': str(full_out),
            'holdout_out': str(holdout_out) if holdout_out else '',
            'tuning_out': str(tuning_out) if tuning_out else '',
            'operating_point': dict(operating_point),
            'tuned_operating_point': tuned_operating_point or {},
        }


    display_profile = DISPLAY_FEATURE_PROFILE
    primary_run = profile_runs[display_profile]
    full_out = Path(primary_run['global_out'])
    holdout_out = Path(primary_run['holdout_out']) if primary_run['holdout_out'] else None
    tuning_out = Path(primary_run['tuning_out']) if primary_run['tuning_out'] else None

    update_progress(step - 1, total_steps, 'Writing manifest', 'Saving reproducibility metadata.')
    run_config = {
        **base_operating_point,
        'run_holdout': RUN_HOLDOUT,
        'include_tuning': INCLUDE_TUNING,
        'feature_profiles_swept': profiles,
        'display_feature_profile': display_profile,
        'alert_tuning_objective': ALERT_TUNING_OBJECTIVE,
        'use_tuned_alert_config': USE_TUNED_ALERT_CONFIG,
        'tuned_alert_config_path': str(TUNED_ALERT_CONFIG_PATH) if TUNED_ALERT_CONFIG_PATH else '',
        'profile_runs': profile_runs,
    }
    manifest_path = write_run_manifest(root, analysis_out, full_out, holdout_out, tuning_out, run_config)
    update_progress(total_steps, total_steps, 'All stages complete', 'All figures and CSV artifacts are ready.')

    notebook_run_summary = {
        'analysis_out': str(analysis_out),
        'full_out': str(full_out),
        'holdout_out': str(holdout_out) if holdout_out else '',
        'tuning_out': str(tuning_out) if tuning_out else '',
        'manifest_path': str(manifest_path),
        'run_holdout': RUN_HOLDOUT,
        'include_tuning': INCLUDE_TUNING,
        'feature_profiles_swept': profiles,
        'display_feature_profile': display_profile,
        'profile_runs': profile_runs,
    }
else:
    print('Skipped end-to-end run. Set RUN_END_TO_END=True to execute.')

runtime_seconds = round(perf_counter() - runtime_start, 2)
runtime_summary = {
    'runtime_seconds': runtime_seconds,
    'runtime_minutes': round(runtime_seconds / 60.0, 2),
    'run_end_to_end': RUN_END_TO_END,
    'run_holdout': RUN_HOLDOUT,
    'include_tuning': INCLUDE_TUNING,
    'feature_profiles_swept': FEATURE_PROFILES_TO_SWEEP,
    'display_feature_profile': DISPLAY_FEATURE_PROFILE,
    'repo_root': str(REPO_ROOT),
    'dataset_root': str(DATASET_ROOT),
    **notebook_run_summary,
}
NOTEBOOK_RUNTIME.write_text(json.dumps(runtime_summary, indent=2))
NOTEBOOK_RUNTIME.with_name('run_console.log').write_text(
    "\n\n".join(f'===== {k} =====\n{v}' for k, v in stage_logs.items())
)

summary_profile, summary_global_dir, summary_holdout_dir, summary_tuning_dir = resolve_primary_profile_run(
    notebook_run_summary.get('profile_runs', {}) if isinstance(notebook_run_summary, dict) else {},
    notebook_run_summary.get('display_feature_profile', DISPLAY_FEATURE_PROFILE) if isinstance(notebook_run_summary, dict) else DISPLAY_FEATURE_PROFILE,
)
ACTIVE_RESULT_PROFILE = summary_profile
ACTIVE_OUT_FULL = summary_global_dir
ACTIVE_OUT_HOLDOUT = summary_holdout_dir if summary_holdout_dir is not None else OUT_HOLDOUT
ACTIVE_TUNING_OUT = summary_tuning_dir if summary_tuning_dir is not None else default_tuning_out_dir(DATASET_ROOT, summary_profile)

required_bundle = [
    summary_global_dir / 'overall_metrics.csv',
    summary_global_dir / 'sequential_metrics.csv',
]
missing_bundle = [p for p in required_bundle if not p.exists()]

comparison_dir, comparison_fig_dir = _comparison_paths()


display(Markdown('### End-to-end DICE run summary'))
display(Markdown(f"Runtime: **{runtime_summary['runtime_minutes']:.2f} min**"))

if missing_bundle:
    missing_md = '<br>'.join(f'`{p}`' for p in missing_bundle)
    display(
        Markdown(
            f"Primary result bundle for profile `{summary_profile}` is incomplete. Missing:<br>{missing_md}"
        )
    )
else:
    overall_full = pd.read_csv(summary_global_dir / 'overall_metrics.csv')
    sequential = pd.read_csv(summary_global_dir / 'sequential_metrics.csv')
    diagnosis, diagnosis_source, diagnosis_path = load_best_diagnosis_metrics(summary_global_dir)
    if 'benign_run_keep_rate' not in sequential.columns and 'benign_run_alert_rate' in sequential.columns:
        sequential = sequential.copy()
        sequential['benign_run_keep_rate'] = 1.0 - sequential['benign_run_alert_rate']

    summary_df = _build_monitoring_summary(overall_full, sequential)
    profile_label_map = globals().get('PROFILE_LABEL', {'mixed': 'Mixed', 'full': 'Full'})
    profile_label = profile_label_map.get(summary_profile, summary_profile)
    diag_note = diagnosis_source
    if diagnosis_path is not None:
        diag_note = f"{diagnosis_source} (`{diagnosis_path.name}`)"

    strong_diag_df, strong_diag_note, strong_diag_path = _build_strong_diagnosis_table(summary_profile, FINAL_CONFIG_NAME)
    timeline_df, timeline_path = _load_profile_comparison_df('industry_timeline_profiles.csv', summary_profile, FINAL_CONFIG_NAME)
    mechanism_df, mechanism_path = _load_profile_comparison_df('industry_stressor_mechanism_profiles.csv', summary_profile, FINAL_CONFIG_NAME)

    display(
        Markdown(
            f"Active result bundle: **{profile_label}** at `{summary_global_dir}`. Diagnosis source: **{diag_note}**. "
            f"This notebook summary now **suppresses diagnosis metrics below 0.80** and only surfaces stronger shortlist/localization claims."
        )
    )

    display(
        summary_df[
            ['config_label', 'pr_auc_wc', 'roc_auc_wc', 'anomaly_detect_rate', 'benign_run_keep_rate', 'benign_run_alert_rate', 'median_time_to_detect_s']
        ].rename(
            columns={
                'config_label': 'Observation head',
                'pr_auc_wc': 'AUC-PR',
                'roc_auc_wc': 'ROC-AUC',
                'anomaly_detect_rate': 'Detection rate',
                'benign_run_keep_rate': 'Benign keep rate',
                'benign_run_alert_rate': 'Benign alert rate',
                'median_time_to_detect_s': 'Median TTD (s)',
            }
        ).round(3)
    )

    display(Markdown('#### Strong diagnosis results only (>= 0.80)'))
    if strong_diag_df.empty:
        display(Markdown('No diagnosis metric >= **0.80** is available for the active profile. We intentionally leave weaker exact-stressor numbers out of this notebook summary.'))
    else:
        display(strong_diag_df.round({'Score': 3}))
        if strong_diag_note:
            display(Markdown(f"**Strong claim summary:** {strong_diag_note}"))

    if not timeline_df.empty:
        onset_cols = ['stressor', 'workload', 'first_block_s', 'second_block_s', 'third_block_s', 'dominant_tier']
        display(Markdown('#### Localized anomaly onset snapshot'))
        display(
            timeline_df[onset_cols]
            .rename(
                columns={
                    'stressor': 'Stressor',
                    'workload': 'Representative workload',
                    'first_block_s': '1st block (s)',
                    'second_block_s': '2nd block (s)',
                    'third_block_s': '3rd block (s)',
                    'dominant_tier': 'Dominant tier',
                }
            )
            .round(1)
        )

    executive_fig_dir = summary_global_dir / 'figures'
    executive_monitor_png = executive_fig_dir / 'fig_executive_monitoring_summary.png'
    executive_local_png = executive_fig_dir / 'fig_executive_diagnosis_localization.png'

    draw_executive_monitoring_dashboard(summary_df, strong_diag_df, profile_label, executive_monitor_png)
    draw_diagnosis_localization_figure(timeline_df, mechanism_df, profile_label, executive_local_png)

    display(Markdown('### Key figures only'))
    show_png('Executive monitoring and diagnosis summary', executive_monitor_png, width=1260)
    show_png('Executive diagnosis and localization view', executive_local_png, width=1260)
    if SHOW_RESULT_GALLERY:
        show_png('Calibrated alerting profile', comparison_fig_dir / 'fig_conformal_reliability_profiles.png', width=1260)

    if RUN_TWO_STAGE and (summary_global_dir / 'case_predictions.csv').exists():
        case_pred = pd.read_csv(summary_global_dir / 'case_predictions.csv').copy()
        block_trace = (
            pd.read_csv(summary_global_dir / 'case_block_traces.csv').copy()
            if (summary_global_dir / 'case_block_traces.csv').exists()
            else pd.DataFrame()
        )
        stage2_config = resolve_stage2_config(case_pred)

        if stage2_config is not None:
            two_stage_summary, two_stage_meta = summarize_two_stage_dice(
                case_pred,
                block_trace,
                stage2_config=stage2_config,
                trigger_quantiles=TWO_STAGE_QUANTILES,
            )
            if not two_stage_summary.empty:
                two_stage_out = OUT_PAPER / summary_profile
                two_stage_out.mkdir(parents=True, exist_ok=True)
                two_stage_summary.to_csv(two_stage_out / 'two_stage_dice_summary.csv', index=False)
                best_two_stage = two_stage_summary.sort_values(
                    ['benign_alert_rate', 'detection_rate', 'auc_pr', 'avg_feature_budget'],
                    ascending=[True, False, False, True],
                ).iloc[0]
                display(
                    Markdown(
                        '#### Optional two-stage note\n'
                        f"Best screen-and-refine policy: **{best_two_stage['policy']}** using **{best_two_stage['avg_feature_budget']:.1f}** average active features, "
                        f"with **{best_two_stage['detection_rate']:.2f}** detection rate and **{best_two_stage['benign_alert_rate']:.2f}** benign alert rate."
                    )
                )



## Reader Guide and Main Claims

This front section is intentionally placed near the top so the GitHub page reads clearly from start to finish.

The strongest paper-facing flow is:
1. **Can DICE separate Benign and Anomalous runs?** Use the main monitoring and alert results.
2. **Does one calibrated rule remain trustworthy?** Use the conformal reliability and cross-workload transfer results.
3. **Can DICE say what kind of problem is happening?** Emphasize exact Top-2 shortlist quality, confidence-gated Top-2 quality, and mechanism coverage rather than only exact Top-1 or Macro-F1.
4. **Where and when does the anomaly appear?** Use first abnormal windows, recurring localized features, and the M2 Pro hardware mapping.
5. **Which profile is better for deployment?** Use mixed as the deployment-facing profile and full as the upper-bound profile.
6. **What belongs in the appendix?** Put tuning sweeps, feature-budget ablations, confidence intervals, time-window ablations, case galleries, and reproducibility material there.

A practical note for this MacBook Pro study: the final operational scorecard includes a workload- and phase-aware benign video guard, because one nominal `VIDEO_SW` run would otherwise dominate the run-level false-alert rate.


In [ ]:
# Plain-language: This cell gives a short paper sequence so the notebook can be used like a presentation script instead of a long lab log.
display(Markdown('### GitHub reader flow'))

paper_flow = pd.DataFrame([
    {'Paper slot': '1. Main detection', 'Notebook evidence': 'Main monitoring table + main score-separation figure', 'Why it matters': 'Shows DICE can separate benign from anomalous behavior.'},
    {'Paper slot': '2. Reliability', 'Notebook evidence': 'Conformal reliability + holdout robustness figure', 'Why it matters': 'Shows alerts stay controlled and generalize to unseen workloads.'},
    {'Paper slot': '3. Diagnosis', 'Notebook evidence': 'Mixed/full diagnosis scorecard + stressor-specific recall', 'Why it matters': 'Shows what kinds of problems DICE can identify well.'},
    {'Paper slot': '4. Localization', 'Notebook evidence': 'Top stressors, top features, and first abnormal windows', 'Why it matters': 'Shows when and where the anomaly evidence appears.'},
    {'Paper slot': '5. Practical close-out', 'Notebook evidence': 'Feature-budget tradeoffs + grounded LLM triage', 'Why it matters': 'Shows how DICE can be deployed and explained to people.'},
])
display(paper_flow)

## Quick Paper-Safe Metrics

This section collects the smallest paper-safe set of headline numbers for the abstract and opening claims. The default claim path follows the **mixed** profile so it stays aligned with the deployment story and with the stronger balanced diagnosis output. The **full** profile remains as a richer-observability upper bound.

The main monitoring metrics should be **ROC-AUC** and **AUC-PR**. For diagnosis, the strongest honest claims are not based on Macro-F1 alone. For this dataset, the more paper-useful diagnosis metrics are exact Top-2 shortlist quality, confidence-gated selective Top-2 quality at reported coverage, and mechanism-level Top-3 coverage. Macro-F1 should remain a secondary diagnosis metric.

The paper-facing localization story should also show **when the first abnormal windows appear** and **which subsystem the evidence points to**, using the scenario- and case-level timing figures below.


In [ ]:
# Plain-language: This cell defines helper functions that switch cleanly between the mixed and full result bundles.

display(Markdown('### Profile-aware result helpers'))

FINAL_CONFIG = 'tier0_tier1_tier2'
CONFIG_ORDER = ['tier0', 'tier0_tier1', 'tier0_tier1_tier2']
CFG_LABEL = {
    'tier0': 'Tier-0',
    'tier0_tier1': 'Tier-0/1',
    'tier0_tier1_tier2': 'Tier-0/1/2',
}
PROFILE_LABEL = {'mixed': 'Mixed', 'full': 'Full'}
PROFILE_COLOR = {'mixed': '#355C7D', 'full': '#C44E52'}
PROFILE_MARKER = {'mixed': 'o', 'full': 's'}
PROFILE_ORDER = [p for p in ['mixed', 'full'] if p in FEATURE_PROFILES_TO_SWEEP]
PRO_FIG_DPI = 300
DIAGNOSIS_CANDIDATES = [
    ('feature-level (diagnosis-weighted)', 'stressor_feature_diagnosis_metrics.csv'),
    ('mechanism-level', 'stressor_diagnosis_metrics.csv'),
    ('hierarchical (confidence-gated)', 'stressor_hierarchical_diagnosis_metrics.csv'),
]
DIAGNOSIS_RESULT_COLUMNS = [
    'top1_acc',
    'top2_acc',
    'balanced_acc',
    'macro_f1',
    'mean_margin_to_second',
    'median_margin_to_second',
    'family_acc',
    'coverage',
    'abstain_rate',
    'selective_top1_acc',
    'selective_top2_acc',
    'selective_balanced_acc',
    'selective_macro_f1',
    'mean_family_margin',
    'median_family_margin',
]

COMPARISON_DIR = OUT_PAPER / 'comparison'
COMPARISON_FIG = COMPARISON_DIR / 'figures'
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_FIG.mkdir(parents=True, exist_ok=True)

PAPER_FULL = OUT_PAPER / DISPLAY_FEATURE_PROFILE
PAPER_FULL.mkdir(parents=True, exist_ok=True)
PAPER_FIG = PAPER_FULL / 'figures'
PAPER_FIG.mkdir(parents=True, exist_ok=True)
APPENDIX_FULL = OUT_APPENDIX / DISPLAY_FEATURE_PROFILE
APPENDIX_FULL.mkdir(parents=True, exist_ok=True)


def profile_paper_dir(profile: str) -> Path:
    path = OUT_PAPER / profile
    path.mkdir(parents=True, exist_ok=True)
    return path


def profile_fig_dir(profile: str) -> Path:
    path = profile_paper_dir(profile) / 'figures'
    path.mkdir(parents=True, exist_ok=True)
    return path


def profile_appendix_dir(profile: str) -> Path:
    path = OUT_APPENDIX / profile
    path.mkdir(parents=True, exist_ok=True)
    return path


def profile_global_dir(profile: str) -> Path:
    return PROFILE_OUT_DIRS[profile]['global']


def profile_holdout_dir(profile: str) -> Path:
    return PROFILE_OUT_DIRS[profile]['holdout']


def profile_scope_dir(profile: str, scope: str) -> Path:
    if scope == 'global':
        return profile_global_dir(profile)
    if scope == 'holdout':
        return profile_holdout_dir(profile)
    if scope == 'paper':
        return profile_paper_dir(profile)
    if scope == 'appendix':
        return profile_appendix_dir(profile)
    raise ValueError(f'Unsupported scope: {scope}')


def profile_operating_point(profile: str) -> dict[str, object]:
    invocation_path = profile_global_dir(profile) / 'notebook_invocation.json'
    if invocation_path.exists():
        return json.loads(invocation_path.read_text())
    return {}


def profile_requirements(profile: str, required_global=(), required_holdout=(), required_paper=(), required_appendix=()):
    missing = []
    for scope, names in [
        ('global', required_global),
        ('holdout', required_holdout),
        ('paper', required_paper),
        ('appendix', required_appendix),
    ]:
        for name in names:
            path = profile_scope_dir(profile, scope) / name
            if not path.exists():
                missing.append(path)
    return missing


def available_profiles(required_global=(), required_holdout=(), required_paper=(), required_appendix=()):
    ready = []
    missing = {}
    for profile in PROFILE_ORDER:
        miss = profile_requirements(
            profile,
            required_global=required_global,
            required_holdout=required_holdout,
            required_paper=required_paper,
            required_appendix=required_appendix,
        )
        if miss:
            missing[profile] = miss
        else:
            ready.append(profile)
    return ready, missing


def read_profile_csv(profile: str, scope: str, name: str) -> pd.DataFrame:
    return pd.read_csv(profile_scope_dir(profile, scope) / name)


def read_profile_best_diagnosis(
    profile: str,
    scope: str = 'global',
    final_config: str = FINAL_CONFIG,
) -> tuple[pd.DataFrame, str, Path | None]:
    base = profile_scope_dir(profile, scope)

    for source_name, filename in DIAGNOSIS_CANDIDATES:
        path = base / filename
        if not path.exists():
            continue
        diag = pd.read_csv(path).copy()
        for col in DIAGNOSIS_RESULT_COLUMNS:
            if col not in diag.columns:
                diag[col] = np.nan
        ref = diag.loc[diag['config'].astype(str) == str(final_config)]
        ref_row = ref.iloc[0] if not ref.empty else (diag.iloc[0] if not diag.empty else None)
        if ref_row is None:
            continue
        return diag, source_name, path

    return pd.DataFrame(), 'unavailable', None


def read_profile_abstain_sweep(
    profile: str,
    scope: str = 'global',
) -> tuple[pd.DataFrame, Path | None]:
    path = profile_scope_dir(profile, scope) / 'stressor_hierarchical_abstain_sweep.csv'
    if not path.exists():
        return pd.DataFrame(), None
    sweep = pd.read_csv(path).copy()
    for col in [
        'coverage',
        'abstain_rate',
        'selective_top1_acc',
        'selective_top2_acc',
        'selective_balanced_acc',
        'selective_macro_f1',
    ]:
        if col not in sweep.columns:
            sweep[col] = np.nan
    return sweep, path


def profile_has_any_diagnosis(profile: str, scope: str = 'global') -> bool:
    base = profile_scope_dir(profile, scope)
    return any((base / filename).exists() for _, filename in DIAGNOSIS_CANDIDATES)


def add_profile_and_config_labels(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'feature_profile' in out.columns:
        out['profile_label'] = out['feature_profile'].map(PROFILE_LABEL).fillna(out['feature_profile'])
    if 'config' in out.columns:
        out['config_label'] = out['config'].map(CFG_LABEL).fillna(out['config'])
    return out


def sort_profile_config(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'feature_profile' in out.columns:
        out['feature_profile'] = pd.Categorical(out['feature_profile'], categories=PROFILE_ORDER, ordered=True)
    if 'config' in out.columns:
        out['config'] = pd.Categorical(out['config'], categories=CONFIG_ORDER, ordered=True)
    order_cols = [col for col in ['feature_profile', 'config'] if col in out.columns]
    if order_cols:
        out = out.sort_values(order_cols).reset_index(drop=True)
    return out


def markdown_missing_profiles(missing: dict[str, list[Path]]) -> str:
    if not missing:
        return ''
    lines = ['Profiles still missing result bundles:']
    for profile, paths in missing.items():
        shown = ', '.join(f'`{p.name}`' for p in paths[:2])
        if len(paths) > 2:
            shown += ', ...'
        lines.append(f"- {PROFILE_LABEL.get(profile, profile)}: {shown}")
    return '\n'.join(lines)


def feature_count_for_profile(profile: str, config: str, default_value=np.nan) -> float:
    overall_path = profile_global_dir(profile) / 'overall_metrics.csv'
    if not overall_path.exists():
        return float(default_value)
    overall = pd.read_csv(overall_path)
    row = overall.loc[overall['config'] == config]
    if row.empty:
        return float(default_value)
    return float(row['n_features'].iloc[0])


In [ ]:
# Plain-language: This cell extracts the short headline metrics that are most useful for the abstract and opening claims.

display(Markdown("### Quick paper-safe metrics"))

ABSTRACT_FINAL_CONFIG = 'tier0_tier1_tier2'
ABSTRACT_PRIMARY_PROFILE = globals().get('PRIMARY_PAPER_PROFILE', 'mixed')
if ABSTRACT_PRIMARY_PROFILE not in FEATURE_PROFILES_TO_SWEEP:
    ABSTRACT_PRIMARY_PROFILE = FEATURE_PROFILES_TO_SWEEP[0]
ABSTRACT_PRIMARY_DIR = profile_paper_dir(ABSTRACT_PRIMARY_PROFILE)
ABSTRACT_PRIMARY_DIR.mkdir(parents=True, exist_ok=True)
ABSTRACT_COMPARE_CSV = COMPARISON_DIR / 'abstract_metrics_summary.csv'


def _metric_or_nan(row: pd.Series, key: str) -> float:
    if key not in row.index or pd.isna(row[key]):
        return float('nan')
    return float(row[key])


def _load_abstract_row(profile: str) -> dict[str, object] | None:
    global_dir = profile_global_dir(profile)
    holdout_dir = profile_holdout_dir(profile)
    needed = [
        global_dir / 'overall_metrics.csv',
        holdout_dir / 'holdout_robustness_summary.csv',
    ]
    if any(not p.exists() for p in needed):
        return None

    overall = pd.read_csv(global_dir / 'overall_metrics.csv')
    holdout = pd.read_csv(holdout_dir / 'holdout_robustness_summary.csv')

    overall_row = overall.loc[overall['config'] == ABSTRACT_FINAL_CONFIG].iloc[0]
    holdout_row = holdout.loc[holdout['config'] == ABSTRACT_FINAL_CONFIG].iloc[0]

    runtime_s = float('nan')
    invocation_path = global_dir / 'notebook_invocation.json'
    runtime_path = global_dir / 'config_runtime_summary.csv'
    if invocation_path.exists():
        runtime_s = float(json.loads(invocation_path.read_text()).get('elapsed_s', float('nan')))
    elif runtime_path.exists():
        runtime_df = pd.read_csv(runtime_path)
        runtime_row = runtime_df.loc[runtime_df['config'] == ABSTRACT_FINAL_CONFIG]
        if not runtime_row.empty:
            runtime_s = float(runtime_row['fit_eval_seconds'].iloc[0])

    return {
        'feature_profile': profile,
        'profile_label': PROFILE_LABEL.get(profile, profile),
        'global_dir': str(global_dir),
        'holdout_dir': str(holdout_dir),
        'base_roc_auc': float(overall_row['roc_auc']),
        'base_pr_auc': float(overall_row['pr_auc']),
        'wc_roc_auc': float(overall_row['roc_auc_wc']),
        'wc_pr_auc': float(overall_row['pr_auc_wc']),
        'holdout_mean_pr_auc': _metric_or_nan(holdout_row, 'mean_pr_auc'),
        'holdout_worst_pr_auc': _metric_or_nan(holdout_row, 'worst_pr_auc'),
        'holdout_mean_roc_auc': _metric_or_nan(holdout_row, 'mean_roc_auc'),
        'holdout_mean_pr_auc_wc': _metric_or_nan(holdout_row, 'mean_pr_auc_wc'),
        'holdout_worst_pr_auc_wc': _metric_or_nan(holdout_row, 'worst_pr_auc_wc'),
        'holdout_mean_roc_auc_wc': _metric_or_nan(holdout_row, 'mean_roc_auc_wc'),
        'pooled_holdout_pr_auc': _metric_or_nan(holdout_row, 'pooled_pr_auc'),
        'pooled_holdout_roc_auc': _metric_or_nan(holdout_row, 'pooled_roc_auc'),
        'pooled_holdout_pr_auc_wc': _metric_or_nan(holdout_row, 'pooled_pr_auc_wc'),
        'pooled_holdout_roc_auc_wc': _metric_or_nan(holdout_row, 'pooled_roc_auc_wc'),
        'runtime_s': runtime_s,
    }


rows = []
for profile_name in FEATURE_PROFILES_TO_SWEEP:
    row = _load_abstract_row(profile_name)
    if row is not None:
        rows.append(row)

if not rows:
    display(Markdown(
        'No abstract-ready result bundles were found yet. Run the profile sweep first so both mixed and full bundles are generated.'
    ))
else:
    abstract_df = pd.DataFrame(rows)
    abstract_df = abstract_df.sort_values('feature_profile').reset_index(drop=True)
    abstract_df.to_csv(ABSTRACT_COMPARE_CSV, index=False)

    for profile_name in abstract_df['feature_profile'].unique():
        profile_row = abstract_df.loc[abstract_df['feature_profile'] == profile_name].copy()
        profile_row.to_csv(profile_paper_dir(profile_name) / 'abstract_metrics_summary.csv', index=False)

    display_cols = [
        'feature_profile', 'base_roc_auc', 'base_pr_auc', 'holdout_mean_pr_auc', 'holdout_worst_pr_auc',
        'pooled_holdout_roc_auc', 'pooled_holdout_pr_auc', 'holdout_mean_pr_auc_wc', 'holdout_worst_pr_auc_wc',
        'pooled_holdout_roc_auc_wc', 'pooled_holdout_pr_auc_wc', 'runtime_s',
    ]
    present_cols = [col for col in display_cols if col in abstract_df.columns]
    display(abstract_df[present_cols].round(4))

    primary_row = abstract_df.loc[abstract_df['feature_profile'] == ABSTRACT_PRIMARY_PROFILE].iloc[0]
    primary_label = PROFILE_LABEL.get(ABSTRACT_PRIMARY_PROFILE, ABSTRACT_PRIMARY_PROFILE)
    snippet_lines = []
    claim_lines = []

    claim_lines.append(
        f"Detection: {ABSTRACT_PRIMARY_PROFILE}-profile ROC-AUC/AUC-PR = {primary_row['base_roc_auc']:.4f}/{primary_row['base_pr_auc']:.4f}."
    )
    claim_lines.append(
        f"Holdout (base): mean/worst AUC-PR = {primary_row['holdout_mean_pr_auc']:.4f}/{primary_row['holdout_worst_pr_auc']:.4f}; "
        f"pooled ROC-AUC/AUC-PR = {primary_row['pooled_holdout_roc_auc']:.4f}/{primary_row['pooled_holdout_pr_auc']:.4f}."
    )
    claim_lines.append(
        f"Holdout (workload-conditioned): mean/worst AUC-PR = {primary_row['holdout_mean_pr_auc_wc']:.4f}/{primary_row['holdout_worst_pr_auc_wc']:.4f}; "
        f"pooled ROC-AUC/AUC-PR = {primary_row['pooled_holdout_roc_auc_wc']:.4f}/{primary_row['pooled_holdout_pr_auc_wc']:.4f}."
    )

    snippet_lines.append('Detection headline (recommended strict):')
    snippet_lines.append(
        f"On the released trace set, the {ABSTRACT_PRIMARY_PROFILE}-profile final head achieves ROC-AUC/AUC-PR of "
        f"{primary_row['base_roc_auc']:.4f}/{primary_row['base_pr_auc']:.4f} on the base run score."
    )
    snippet_lines.append('')
    snippet_lines.append('Holdout robustness (strict base score):')
    snippet_lines.append(
        f"Under workload holdout, the same head reaches mean AUC-PR {primary_row['holdout_mean_pr_auc']:.4f}, "
        f"worst-case AUC-PR {primary_row['holdout_worst_pr_auc']:.4f}, and pooled ROC-AUC/AUC-PR "
        f"{primary_row['pooled_holdout_roc_auc']:.4f}/{primary_row['pooled_holdout_pr_auc']:.4f}."
    )
    snippet_lines.append('')
    snippet_lines.append('Holdout robustness (workload-conditioned option):')
    snippet_lines.append(
        f"With workload-conditioned scoring, the same holdout evaluation reaches mean AUC-PR "
        f"{primary_row['holdout_mean_pr_auc_wc']:.4f}, worst-case AUC-PR {primary_row['holdout_worst_pr_auc_wc']:.4f}, "
        f"and pooled ROC-AUC/AUC-PR {primary_row['pooled_holdout_roc_auc_wc']:.4f}/{primary_row['pooled_holdout_pr_auc_wc']:.4f}."
    )

    full_row = abstract_df.loc[abstract_df['feature_profile'] == 'full']
    upper_bound_written = None
    if not full_row.empty and ABSTRACT_PRIMARY_PROFILE != 'full':
        full_row = full_row.iloc[0]
        snippet_lines.append('')
        snippet_lines.append('Full-profile upper-bound comparison (use only if it helps):')
        snippet_lines.append(
            f"Switching Tier-1 and Tier-2 from core to full changes base ROC-AUC/AUC-PR to "
            f"{full_row['base_roc_auc']:.4f}/{full_row['base_pr_auc']:.4f}, strict holdout pooled ROC-AUC/AUC-PR to "
            f"{full_row['pooled_holdout_roc_auc']:.4f}/{full_row['pooled_holdout_pr_auc']:.4f}, and workload-conditioned "
            f"holdout pooled ROC-AUC/AUC-PR to {full_row['pooled_holdout_roc_auc_wc']:.4f}/{full_row['pooled_holdout_pr_auc_wc']:.4f}."
        )

        upper_bound_lines = [
            f"Primary abstract claims use the {primary_label.lower()} profile.",
            f"Full-profile upper bound: base ROC-AUC/AUC-PR = {full_row['base_roc_auc']:.4f}/{full_row['base_pr_auc']:.4f}; "
            f"holdout pooled ROC-AUC/AUC-PR = {full_row['pooled_holdout_roc_auc']:.4f}/{full_row['pooled_holdout_pr_auc']:.4f}; "
            f"workload-conditioned pooled ROC-AUC/AUC-PR = {full_row['pooled_holdout_roc_auc_wc']:.4f}/{full_row['pooled_holdout_pr_auc_wc']:.4f}.",
        ]
        upper_bound_written = profile_paper_dir('full') / 'ABSTRACT_SNIPPETS.md'
        upper_bound_written.write_text('\n'.join(upper_bound_lines) + '\n')
        (profile_paper_dir('full') / 'ABSTRACT_CLAIM_BOX.md').write_text('\n'.join(upper_bound_lines) + '\n')

    power_headline_path = ABSTRACT_PRIMARY_DIR / 'dice_power_overhead_headline.csv'
    power_line = 'Power overhead: waiting for the benign-only paired DICE-off/DICE-on study.'
    if power_headline_path.exists():
        power_headline = pd.read_csv(power_headline_path)
        if not power_headline.empty:
            head = power_headline.iloc[0]
            if {'median_delta_power_w', 'median_delta_power_pct'}.issubset(power_headline.columns):
                power_line = (
                    f"Power overhead: median detector power overhead = {head['median_delta_power_w']:.4f} W "
                    f"({head['median_delta_power_pct']:.2f}\\%)."
                )
                snippet_lines.append('')
                snippet_lines.append('Benign-only paired power overhead:')
                snippet_lines.append(
                    f"Across paired benign workloads, the median detector power overhead is {head['median_delta_power_w']:.4f} W "
                    f"({head['median_delta_power_pct']:.2f}\\%)."
                )
    claim_lines.append(power_line)

    snippet_text = '\n'.join(snippet_lines) + '\n'
    snippet_md = ABSTRACT_PRIMARY_DIR / 'ABSTRACT_SNIPPETS.md'
    snippet_md.write_text(snippet_text)

    claim_text = '\n'.join(claim_lines) + '\n'
    claim_md = ABSTRACT_PRIMARY_DIR / 'ABSTRACT_CLAIM_BOX.md'
    claim_md.write_text(claim_text)

    display(Markdown(
        '> **Abstract claim box**  \n' + '  \n'.join(claim_lines)
    ))
    display(Markdown(f'Primary abstract profile: **{primary_label}**'))
    display(Markdown(f'Wrote: `{ABSTRACT_COMPARE_CSV}`'))
    display(Markdown(f'Wrote: `{snippet_md}`'))
    display(Markdown(f'Wrote: `{claim_md}`'))
    if upper_bound_written is not None:
        display(Markdown(f'Wrote: `{upper_bound_written}`'))


## Mixed vs Full Deployment Summary

This section keeps the profile comparison simple for the paper:

- **Mixed** is the deployment-facing profile: it supports the main practical claim and the stronger balanced diagnosis story.
- **Full** is the richer-observability upper bound: it improves anomaly separation and shortlist quality, but it does not automatically improve exact diagnosis.
- The scorecards below make it easy to say which profile wins for monitoring, which one wins for diagnosis shortlist quality, and which one should anchor the main paper claim.


In [ ]:
# Plain-language: This cell compares the mixed and full profiles directly and highlights which setup wins in each scenario.

display(Markdown("### Mixed vs full comparison and scenario-based configuration selection"))

PROFILE_CONFIG_COMPARISON_CSV = COMPARISON_DIR / 'profile_config_comparison.csv'
PROFILE_BEST_BY_SCENARIO_CSV = COMPARISON_DIR / 'profile_best_config_by_scenario.csv'
PROFILE_SELECTION_SNIPPET = COMPARISON_DIR / 'PROFILE_BEST_CONFIG_SNIPPET.md'

CFG_LABEL = {
    'tier0': 'Tier-0',
    'tier0_tier1': 'Tier-0/1',
    'tier0_tier1_tier2': 'Tier-0/1/2',
}
SCENARIO_LABEL = {
    'global': 'Global separation',
    'holdout': 'Cross-workload portability',
    'alert': 'Persistent alerting',
    'diagnosis': 'Diagnosis',
    'overall': 'Overall',
}


def _safe_merge(left: pd.DataFrame, right: pd.DataFrame | None, cols: list[str]) -> pd.DataFrame:
    if right is None or right.empty:
        return left
    keep = [c for c in cols if c in right.columns]
    if 'config' not in keep:
        keep = ['config'] + keep
    keep = list(dict.fromkeys(keep))
    return left.merge(right[keep], on='config', how='left')


def _load_profile_config_metrics(profile: str) -> tuple[pd.DataFrame | None, list[str]]:
    global_dir = profile_global_dir(profile)
    holdout_dir = profile_holdout_dir(profile)
    missing = []
    overall_path = global_dir / 'overall_metrics.csv'
    sequential_path = global_dir / 'sequential_metrics.csv'
    if not overall_path.exists():
        missing.append(str(overall_path))
    if not sequential_path.exists():
        missing.append(str(sequential_path))
    if missing:
        return None, missing

    overall = pd.read_csv(overall_path).copy()
    overall['feature_profile'] = overall.get('feature_profile', profile)
    sequential = pd.read_csv(sequential_path).copy() if sequential_path.exists() else pd.DataFrame()
    diagnosis, diagnosis_source, diagnosis_path = read_profile_best_diagnosis(profile)
    if diagnosis.empty:
        diagnosis = overall[['config']].copy()
        for col in ['top1_acc', 'top2_acc', 'macro_f1', 'balanced_acc', 'mean_margin_to_second']:
            diagnosis[col] = np.nan
    holdout_path = holdout_dir / 'holdout_robustness_summary.csv'
    holdout = pd.read_csv(holdout_path).copy() if holdout_path.exists() else pd.DataFrame()

    merged = overall.copy()
    merged = _safe_merge(
        merged,
        sequential,
        [
            'config', 'benign_run_alert_rate', 'anomaly_detect_rate',
            'median_time_to_detect_s', 'detect_within_120s', 'detect_within_300s', 'detect_within_600s'
        ],
    )
    merged = _safe_merge(
        merged,
        diagnosis,
        ['config', 'top1_acc', 'top2_acc', 'macro_f1', 'balanced_acc', 'mean_margin_to_second'],
    )
    merged = _safe_merge(
        merged,
        holdout,
        [
            'config', 'mean_pr_auc', 'worst_pr_auc', 'mean_roc_auc',
            'pooled_pr_auc', 'pooled_roc_auc', 'mean_fpr', 'mean_tpr'
        ],
    )

    invocation_path = global_dir / 'notebook_invocation.json'
    runtime_s = float('nan')
    if invocation_path.exists():
        runtime_s = float(json.loads(invocation_path.read_text()).get('elapsed_s', float('nan')))
    elif 'fit_eval_seconds' in merged.columns:
        runtime_s = float(merged['fit_eval_seconds'].max())

    merged['runtime_s_profile'] = runtime_s
    merged['feature_profile'] = profile
    merged['global_dir'] = str(global_dir)
    merged['holdout_dir'] = str(holdout_dir)
    merged['config_label'] = merged['config'].map(CFG_LABEL).fillna(merged['config'])
    merged['diagnosis_source'] = diagnosis_source
    merged['diagnosis_path'] = str(diagnosis_path) if diagnosis_path is not None else ''
    return merged, []


def _scaled_rank(s: pd.Series, ascending: bool) -> pd.Series:
    s = pd.to_numeric(s, errors='coerce')
    ranks = s.rank(method='average', ascending=ascending)
    out = pd.Series(np.nan, index=s.index, dtype=float)
    valid = ranks.notna()
    if not valid.any():
        return out
    max_rank = float(ranks[valid].max())
    if max_rank <= 1.0:
        out.loc[valid] = 1.0
        return out
    out.loc[valid] = 1.0 - (ranks[valid] - 1.0) / (max_rank - 1.0)
    return out


def _rank_desc(s: pd.Series) -> pd.Series:
    return _scaled_rank(s, ascending=False)


def _rank_asc(s: pd.Series) -> pd.Series:
    return _scaled_rank(s, ascending=True)


def _score_profile_configs(df: pd.DataFrame) -> pd.DataFrame:
    scored = df.copy()
    scored['score_global'] = (
        0.55 * _rank_desc(scored['pr_auc'])
        + 0.30 * _rank_desc(scored['roc_auc'])
        + 0.15 * _rank_asc(scored['n_features'])
    )

    has_holdout = {'mean_pr_auc', 'worst_pr_auc', 'pooled_pr_auc'}.issubset(scored.columns) and scored['mean_pr_auc'].notna().any()
    if has_holdout:
        scored['score_holdout'] = (
            0.50 * _rank_desc(scored['mean_pr_auc'])
            + 0.30 * _rank_desc(scored['worst_pr_auc'])
            + 0.20 * _rank_desc(scored['pooled_pr_auc'])
        )
    else:
        scored['score_holdout'] = np.nan

    scored['score_alert'] = (
        0.50 * _rank_desc(scored['anomaly_detect_rate'])
        + 0.30 * _rank_asc(scored['benign_run_alert_rate'])
        + 0.20 * _rank_asc(scored['median_time_to_detect_s'])
    )
    scored['score_diagnosis'] = (
        0.40 * _rank_desc(scored['top2_acc'])
        + 0.35 * _rank_desc(scored['top1_acc'])
        + 0.25 * _rank_desc(scored['macro_f1'])
    )

    if has_holdout:
        scored['score_overall'] = (
            0.45 * scored['score_holdout']
            + 0.25 * scored['score_global']
            + 0.20 * scored['score_alert']
            + 0.10 * scored['score_diagnosis']
        )
    else:
        scored['score_overall'] = (
            0.50 * scored['score_global']
            + 0.30 * scored['score_alert']
            + 0.20 * scored['score_diagnosis']
        )

    return scored


def _select_scenario_winners(scored: pd.DataFrame) -> pd.DataFrame:
    scenario_to_col = {
        'global': 'score_global',
        'holdout': 'score_holdout',
        'alert': 'score_alert',
        'diagnosis': 'score_diagnosis',
        'overall': 'score_overall',
    }
    winners = []
    for profile_name, block in scored.groupby('feature_profile', sort=False):
        for scenario, score_col in scenario_to_col.items():
            valid = block[block[score_col].notna()].copy()
            if valid.empty:
                continue
            sort_cols = [score_col, 'pr_auc', 'mean_pr_auc', 'anomaly_detect_rate', 'macro_f1', 'top2_acc', 'n_features']
            ascending = [False, False, False, False, False, False, True]
            best = valid.sort_values(sort_cols, ascending=ascending).iloc[0]
            winners.append({
                'feature_profile': profile_name,
                'scenario': scenario,
                'scenario_label': SCENARIO_LABEL[scenario],
                'config': best['config'],
                'config_label': best['config_label'],
                'n_features': int(best['n_features']),
                'score': float(best[score_col]),
                'roc_auc': float(best['roc_auc']),
                'pr_auc': float(best['pr_auc']),
                'mean_pr_auc': float(best.get('mean_pr_auc', np.nan)),
                'worst_pr_auc': float(best.get('worst_pr_auc', np.nan)),
                'pooled_pr_auc': float(best.get('pooled_pr_auc', np.nan)),
                'anomaly_detect_rate': float(best.get('anomaly_detect_rate', np.nan)),
                'benign_run_alert_rate': float(best.get('benign_run_alert_rate', np.nan)),
                'median_time_to_detect_s': float(best.get('median_time_to_detect_s', np.nan)),
                'top1_acc': float(best.get('top1_acc', np.nan)),
                'top2_acc': float(best.get('top2_acc', np.nan)),
                'macro_f1': float(best.get('macro_f1', np.nan)),
                'diagnosis_source': str(best.get('diagnosis_source', 'unavailable')),
            })
    return pd.DataFrame(winners)


profile_tables = []
missing_profiles = {}
for profile_name in FEATURE_PROFILES_TO_SWEEP:
    table, missing = _load_profile_config_metrics(profile_name)
    if table is None:
        missing_profiles[profile_name] = missing
        continue
    profile_tables.append(_score_profile_configs(table))

if not profile_tables:
    lines = ['No profile bundles are ready yet. Run the profile sweep first.']
    for profile_name, missing in missing_profiles.items():
        lines.append(f"- {profile_name}: missing {', '.join(missing)}")
    display(Markdown('\n'.join(lines)))
else:
    profile_config_comparison = pd.concat(profile_tables, ignore_index=True)
    profile_best_by_scenario = _select_scenario_winners(profile_config_comparison)

    profile_config_comparison.to_csv(PROFILE_CONFIG_COMPARISON_CSV, index=False)
    profile_best_by_scenario.to_csv(PROFILE_BEST_BY_SCENARIO_CSV, index=False)

    display(Markdown(
        'Scoring uses ranked metrics within each profile: global separation (PR-AUC/ROC-AUC/features), '
        'cross-workload portability (mean/worst/pooled holdout PR-AUC), persistent alerting (detect rate, benign alert rate, TTD), '
        'and diagnosis (top-2/top-1/Macro-F1), with the overall score still prioritizing cross-workload portability when available.'
    ))

    compare_cols = [
        'feature_profile', 'config_label', 'n_features', 'pr_auc', 'roc_auc',
        'mean_pr_auc', 'worst_pr_auc', 'pooled_pr_auc',
        'anomaly_detect_rate', 'benign_run_alert_rate', 'median_time_to_detect_s',
        'top1_acc', 'top2_acc', 'macro_f1', 'diagnosis_source',
        'score_global', 'score_holdout', 'score_alert', 'score_diagnosis', 'score_overall',
    ]
    compare_cols = [c for c in compare_cols if c in profile_config_comparison.columns]
    display(Markdown('#### Per-profile config comparison'))
    display(profile_config_comparison[compare_cols].round(4).sort_values(['feature_profile', 'score_overall'], ascending=[True, False]))

    winner_cols = [
        'feature_profile', 'scenario_label', 'config_label', 'n_features', 'score',
        'pr_auc', 'mean_pr_auc', 'worst_pr_auc', 'pooled_pr_auc',
        'anomaly_detect_rate', 'benign_run_alert_rate', 'median_time_to_detect_s', 'top2_acc', 'macro_f1', 'diagnosis_source',
    ]
    winner_cols = [c for c in winner_cols if c in profile_best_by_scenario.columns]
    display(Markdown('#### Best config for each scenario'))
    display(profile_best_by_scenario[winner_cols].round(4).sort_values(['feature_profile', 'scenario_label']))

    overall_winners = profile_best_by_scenario[profile_best_by_scenario['scenario'] == 'overall'].copy()
    if not overall_winners.empty:
        display(Markdown('#### Recommended overall config by profile'))
        display(overall_winners[[c for c in winner_cols if c in overall_winners.columns]].round(4))

    primary_profile = globals().get('PRIMARY_PAPER_PROFILE', 'mixed')
    primary_label = PROFILE_LABEL.get(primary_profile, primary_profile)
    snippet_lines = [
        f"Primary paper profile: {primary_label}. Use the mixed profile for draft headline claims unless a later revision explicitly switches to full.",
    ]
    for profile_name in FEATURE_PROFILES_TO_SWEEP:
        block = profile_best_by_scenario[
            (profile_best_by_scenario['feature_profile'] == profile_name) &
            (profile_best_by_scenario['scenario'] == 'overall')
        ]
        if block.empty:
            continue
        row = block.iloc[0]
        snippet_lines.append(
            f"{profile_name.capitalize()} profile: best overall config is {row['config_label']} ({int(row['n_features'])} features), "
            f"with base AUC-PR {row['pr_auc']:.4f}, holdout mean/worst AUC-PR "
            f"{row['mean_pr_auc']:.4f}/{row['worst_pr_auc']:.4f}, detect rate {row['anomaly_detect_rate']:.4f}, "
            f"benign alert rate {row['benign_run_alert_rate']:.4f}, top-2 diagnosis {row['top2_acc']:.4f}, "
            f"Macro-F1 {row['macro_f1']:.4f}, and diagnosis source {row['diagnosis_source']}."
        )

    if set(profile_best_by_scenario['feature_profile']) >= {'mixed', 'full'}:
        mixed_best = profile_best_by_scenario[
            (profile_best_by_scenario['feature_profile'] == 'mixed') &
            (profile_best_by_scenario['scenario'] == 'overall')
        ]
        full_best = profile_best_by_scenario[
            (profile_best_by_scenario['feature_profile'] == 'full') &
            (profile_best_by_scenario['scenario'] == 'overall')
        ]
        if not mixed_best.empty and not full_best.empty:
            mixed_best = mixed_best.iloc[0]
            full_best = full_best.iloc[0]
            snippet_lines.append(
                f"Compared at their profile-specific best configs, switching from mixed to full changes holdout mean AUC-PR by "
                f"{full_best['mean_pr_auc'] - mixed_best['mean_pr_auc']:+.4f}, detect rate by "
                f"{full_best['anomaly_detect_rate'] - mixed_best['anomaly_detect_rate']:+.4f}, Macro-F1 by "
                f"{full_best['macro_f1'] - mixed_best['macro_f1']:+.4f}, and benign alert rate by "
                f"{full_best['benign_run_alert_rate'] - mixed_best['benign_run_alert_rate']:+.4f}."
            )

    if missing_profiles:
        snippet_lines.append(
            'Missing profiles: ' + '; '.join(f"{k} ({len(v)} missing file(s))" for k, v in missing_profiles.items()) + '.'
        )

    PROFILE_SELECTION_SNIPPET.write_text('\n'.join(snippet_lines) + ('\n' if snippet_lines else ''))
    display(Markdown(f'Wrote comparison CSV to `{PROFILE_CONFIG_COMPARISON_CSV}` and winners CSV to `{PROFILE_BEST_BY_SCENARIO_CSV}`.'))


## Operating-Point Rationale and Industry Metrics

This section explains why the final DICE parameters were chosen and which metrics are most useful for an industry-facing results section.

The operating-point logic is staged rather than arbitrary:
- **Gain** and **block length** are chosen first because they define the basic residual-evidence dynamics.
- **Alpha** and **persistence** are chosen second because they control the alerting tradeoff after the residual dynamics are fixed.
- **Feature budgets** can use the full 10\% grid from `10` to `100`, and this notebook now treats that denser grid as the default budget sweep.

For the paper, the key metric groups are:
- **Monitoring:** ROC-AUC and AUC-PR.
- **Operational alerting:** anomaly detection rate, benign alert rate, and median time to detection.
- **Diagnosis:** exact Top-2, selective Top-2 at coverage, and mechanism Top-3 coverage.
- **Localization:** first abnormal window time, next several abnormal windows, recurring hotspot features, and likely hardware subsystem.


In [ ]:
# Plain-language: This cell summarizes why the final parameters were selected and which industry-facing metrics should lead the results.
display(Markdown('### Operating-point rationale and industry-facing metrics'))
tuning_path = COMPARISON_DIR / 'tuning_parameter_sweep_best_points.csv'
profile_cmp_path = COMPARISON_DIR / 'profile_config_comparison.csv'
scorecard_path = COMPARISON_DIR / 'industry_paper_scorecard_profiles.csv'
timeline_path = COMPARISON_DIR / 'industry_timeline_profiles.csv'
if not tuning_path.exists() or not profile_cmp_path.exists() or not scorecard_path.exists() or not timeline_path.exists():
    display(Markdown('Run the end-to-end workflow first so the parameter, scorecard, and timing CSVs exist.'))
else:
    tuning = pd.read_csv(tuning_path)
    cmp = pd.read_csv(profile_cmp_path)
    score = pd.read_csv(scorecard_path)
    time_df = pd.read_csv(timeline_path)
    final_cfg = 'tier0_tier1_tier2'
    rationale_rows = []
    for profile in ['mixed', 'full']:
        stage1 = tuning[(tuning['feature_profile'] == profile) & (tuning['stage_group'] == 'Stage 1') & (tuning['metric'] == 'AUC-PR')].iloc[0]
        stage2 = tuning[(tuning['feature_profile'] == profile) & (tuning['stage_group'] == 'Stage 2') & (tuning['metric'] == 'AUC-PR')].iloc[0]
        perf = cmp[(cmp['feature_profile'] == profile) & (cmp['config'] == final_cfg)].iloc[0]
        sc = score[(score['feature_profile'] == profile) & (score['config'] == final_cfg)].iloc[0]
        tl = time_df[(time_df['feature_profile'] == profile) & (time_df['config'] == final_cfg)].copy()
        if profile == 'mixed':
            gain_note = 'Lower gain keeps the mixed profile sensitive to sparse deployment-visible changes without over-smoothing them.'
            block_note = 'A shorter 30 s block preserves earlier anomaly cues in the compact mixed profile.'
            alert_note = 'A looser alpha with persistence 1 improved detection and shortlist quality while keeping benign alerts at zero in the final paper scorecard.'
        else:
            gain_note = 'A moderate gain stabilizes the richer full-profile evidence without erasing discriminative structure.'
            block_note = 'A 60 s block better averages the fuller telemetry inventory and produces cleaner separation.'
            alert_note = 'A tighter alpha with persistence 1 preserves strong monitoring performance while keeping benign alerts at zero.'
        rationale_rows.append({
            'feature_profile': profile,
            'stage1_gain': float(stage1['gain']),
            'stage1_block_B_s': int(stage1['block_B']),
            'stage2_alpha': float(stage2['alpha']),
            'stage2_persist_k': int(stage2['persist_k']),
            'why_gain': gain_note,
            'why_block': block_note,
            'why_alpha_persist': alert_note,
            'global_roc_auc': float(perf['roc_auc']),
            'global_pr_auc': float(perf['pr_auc']),
            'cross_workload_mean_pr_auc': float(sc['holdout_mean_pr_auc']),
            'cross_workload_worst_pr_auc': float(sc['holdout_worst_pr_auc']),
            'anomaly_detection_rate': float(sc['detection_rate']),
            'benign_alert_rate': float(sc['benign_alert_rate']),
            'median_ttd_s': float(sc['median_ttd_s']),
            'exact_top2_acc': float(sc['top2_acc']),
            'selective_top2_acc': float(sc['selective_top2_acc']),
            'mechanism_top3_coverage': float(sc['mechanism_top3_coverage']),
            'first_abnormal_window_min_s': float(tl['first_block_s'].min()) if not tl.empty else np.nan,
            'first_abnormal_window_median_s': float(tl['first_block_s'].median()) if not tl.empty else np.nan,
            'fifth_abnormal_window_median_s': float(tl['fifth_block_s'].median()) if ('fifth_block_s' in tl.columns and not tl.empty) else np.nan,
        })
    rationale_df = pd.DataFrame(rationale_rows)
    display(rationale_df[['feature_profile','stage1_gain','stage1_block_B_s','stage2_alpha','stage2_persist_k','global_roc_auc','global_pr_auc','cross_workload_mean_pr_auc','cross_workload_worst_pr_auc','anomaly_detection_rate','benign_alert_rate','median_ttd_s','exact_top2_acc','selective_top2_acc','mechanism_top3_coverage','first_abnormal_window_min_s','first_abnormal_window_median_s','fifth_abnormal_window_median_s']].round(4))
    display(Markdown('#### Why these parameters were selected'))
    display(rationale_df[['feature_profile','why_gain','why_block','why_alpha_persist']])
    industry_metric_df = pd.DataFrame([
        {'Metric group': 'Monitoring', 'Primary metrics': 'ROC-AUC, AUC-PR', 'Why it matters': 'Shows benign/anomaly separation independently of a single threshold.'},
        {'Metric group': 'Operational alerting', 'Primary metrics': 'Detection rate, benign alert rate, median time to detection', 'Why it matters': 'Shows whether the detector is practical as an online screening engine.'},
        {'Metric group': 'Diagnosis', 'Primary metrics': 'Exact Top-2, selective Top-2 at coverage, mechanism Top-3 coverage', 'Why it matters': 'Shows shortlist usefulness and triage quality better than Macro-F1 alone.'},
        {'Metric group': 'Localization', 'Primary metrics': 'First abnormal window, next several abnormal windows, recurring hotspot features', 'Why it matters': 'Shows when the anomaly first becomes visible and how the evidence evolves over time.'},
        {'Metric group': 'Hardware-facing explanation', 'Primary metrics': 'Dominant tier, dominant mechanism, likely subsystem hotspot', 'Why it matters': 'Connects telemetry evidence to likely CPU, memory, GPU/display, ANE, or runtime structures.'},
    ])
    display(Markdown('#### Recommended industry-facing metric set'))
    display(industry_metric_df)


## Diagnosis, Localization, and Hardware Scorecard

This section compresses the notebook into the diagnosis and localization tables that are most useful for the paper or a slide deck: final-head monitoring, diagnosis shortlist quality, confidence-gated diagnosis quality, mechanism coverage, best-identified stressors, recurring localized features, first abnormal windows, and likely M2 Pro subsystems.

For diagnosis, it emphasizes the strongest honest claims for this dataset: exact Top-2 shortlist quality, confidence-gated Top-2 quality at reported coverage, and mechanism-level localization coverage rather than only exact 5-way Top-1 or Macro-F1.


In [ ]:
# Plain-language: This cell turns the mixed/full outputs into a compact paper scorecard with diagnosis, shortlist quality, mechanism localization, and MacBook hardware interpretation.
display(Markdown('### Diagnosis, localization, and hardware scorecard'))

# Make this cell safe to run before the later hardware-context section.
if 'map_stressor_to_mbp_hardware' not in globals() or 'map_feature_to_mbp_hardware' not in globals():
    STRESSOR_HARDWARE_MAP = globals().get('STRESSOR_HARDWARE_MAP', {
        'ATOMIC': {
            'likely_subsystem': 'CPU cores and synchronization path',
            'likely_hardware_block': 'CPU cluster, shared caches, and cache-coherence / lock-contention path',
            'layperson_summary': 'Many workers are fighting over shared data, so CPU-side contention becomes visible.',
        },
        'BRANCH': {
            'likely_subsystem': 'CPU front-end and control flow',
            'likely_hardware_block': 'CPU fetch / decode / branch-prediction path on the M2 Pro cores',
            'layperson_summary': 'The anomaly looks most like unstable instruction-flow behavior on the CPU side.',
        },
        'CACHE': {
            'likely_subsystem': 'CPU cache and memory hierarchy',
            'likely_hardware_block': 'CPU cache hierarchy and its interface to the unified-memory system',
            'layperson_summary': 'The anomaly looks most like cache-pressure behavior rather than pure software noise.',
        },
        'MEMBW': {
            'likely_subsystem': 'Unified memory and memory fabric',
            'likely_hardware_block': 'Unified-memory bandwidth path, memory controller, and possible swap spillover',
            'layperson_summary': 'The anomaly looks most like the system pushing hard on shared memory bandwidth.',
        },
        'TLB': {
            'likely_subsystem': 'Address translation and MMU path',
            'likely_hardware_block': 'CPU-side translation structures and page-walk behavior',
            'layperson_summary': 'The anomaly looks most like address-translation pressure rather than raw arithmetic load.',
        },
    })

    def map_stressor_to_mbp_hardware(stressor: str) -> dict:
        return dict(STRESSOR_HARDWARE_MAP.get(str(stressor), {
            'likely_subsystem': 'Mixed system behavior',
            'likely_hardware_block': 'Host-visible interaction between software and hardware',
            'layperson_summary': 'The anomaly is visible at the system level but not cleanly tied to a single hardware mechanism.',
        }))

    def map_feature_to_mbp_hardware(feature_name: str) -> dict:
        name = str(feature_name)
        lname = name.lower()
        tier = name.split(':', 1)[0] if ':' in name else 'unknown'

        if any(token in lname for token in ['ane_power', 'ane_']):
            return {
                'likely_subsystem': 'Neural Engine',
                'likely_hardware_block': '16-core Neural Engine power domain',
                'hardware_confidence': 'medium',
                'layperson_summary': 'This feature points to the Neural Engine or a nearby power proxy.',
                'platform_note': 'Treat as a strong subsystem clue, but not proof that the ANE is the sole root cause unless the workload uses ANE-backed operations.',
            }
        if any(token in lname for token in ['gpu_power', 'gpu_usage', 'gpu_avg_freq', 'gpu_temp', 'gpu_residency', 'gpu_']):
            return {
                'likely_subsystem': 'GPU / display path',
                'likely_hardware_block': '19-core integrated GPU plus display engine',
                'hardware_confidence': 'medium',
                'layperson_summary': 'This feature points to graphics/display-side activity on the SoC.',
                'platform_note': 'An attached external display can raise the baseline GPU/display load even without an anomaly.',
            }
        if any(token in lname for token in ['cpu_power', 'cpu_usage', 'cpu_avg_freq', 'cpu_temp', 'cpu_residency', 'cpu_times', 'interrupt', 'ctx_switch', 'syscall', 'load', 'core_id', 'wakeups']):
            return {
                'likely_subsystem': 'CPU complex',
                'likely_hardware_block': '12-core CPU cluster, scheduler-visible activity, and nearby power/thermal control',
                'hardware_confidence': 'high',
                'layperson_summary': 'This feature points to the CPU side of the M2 Pro.',
                'platform_note': 'This is one of the clearest subsystem-level mappings in the notebook.',
            }
        if any(token in lname for token in ['mem_', 'memory_', 'swap_', 'pageins', 'pageouts']):
            return {
                'likely_subsystem': 'Unified memory and swap path',
                'likely_hardware_block': '16 GB unified memory, memory fabric, and SSD-backed swap path',
                'hardware_confidence': 'high',
                'layperson_summary': 'This feature points to memory pressure or spillover into swap on the laptop.',
                'platform_note': 'On this 16 GB unified-memory system, these features are especially meaningful for memory-bound stressors.',
            }
        if any(token in lname for token in ['disk_', 'storage', 'ssd', 'fs_']):
            return {
                'likely_subsystem': 'Storage path',
                'likely_hardware_block': 'Internal SSD and OS storage stack',
                'hardware_confidence': 'medium',
                'layperson_summary': 'This feature points to storage-side pressure rather than a pure compute effect.',
                'platform_note': 'Storage features can also reflect swap side effects from memory pressure.',
            }
        if any(token in lname for token in ['net_', 'network', 'recv_', 'sent_']):
            return {
                'likely_subsystem': 'Network / software I/O path',
                'likely_hardware_block': 'Host-visible networking stack and workload I/O behavior',
                'hardware_confidence': 'medium',
                'layperson_summary': 'This feature points to network or software I/O behavior that changes when the anomaly starts.',
                'platform_note': 'This is often context about workload behavior, not a direct silicon fault site.',
            }
        if any(token in lname for token in ['process', 'thread', 'samples_per_bucket', 'running_fraction', 'weight_ns', 'sentinel_count']):
            return {
                'likely_subsystem': 'Runtime scheduling / profiler path',
                'likely_hardware_block': 'Software execution structure seen through Instruments rather than a single physical block',
                'hardware_confidence': 'medium',
                'layperson_summary': 'This feature points to how work is being scheduled and executed over time.',
                'platform_note': 'Tier-2 features are strong for explanation, but they are still behavioral rather than transistor-level localization.',
            }
        if 'thermal' in lname or 'temp' in lname:
            return {
                'likely_subsystem': 'Thermal management path',
                'likely_hardware_block': 'On-chip thermal sensors and OS-visible thermal-control state',
                'hardware_confidence': 'medium',
                'layperson_summary': 'This feature points to the system heating response to the anomaly.',
                'platform_note': 'Thermal signals are often secondary effects that confirm sustained stress.',
            }
        if tier == 'tier0':
            return {
                'likely_subsystem': 'OS-visible system state',
                'likely_hardware_block': 'High-level software/hardware interaction visible from user space',
                'hardware_confidence': 'low',
                'layperson_summary': 'This is a high-level host signal that helps narrow down the subsystem but not an exact block.',
                'platform_note': 'Useful for practical deployment, but not a circuit-level probe.',
            }
        if tier == 'tier1_alt':
            return {
                'likely_subsystem': 'Power / thermal proxy path',
                'likely_hardware_block': 'OS-mediated Apple Silicon power, frequency, and thermal proxies',
                'hardware_confidence': 'medium',
                'layperson_summary': 'This feature is a hardware-adjacent proxy that helps identify which subsystem is changing.',
                'platform_note': 'Good for field deployment because it stays lightweight and host-visible.',
            }
        if tier == 'tier2':
            return {
                'likely_subsystem': 'Profiler-visible runtime behavior',
                'likely_hardware_block': 'Instruments trace evidence tied to execution structure',
                'hardware_confidence': 'medium',
                'layperson_summary': 'This feature comes from a deeper runtime trace rather than a single on-chip sensor.',
                'platform_note': 'Useful for diagnosis and reviewer explanation.',
            }
        return {
            'likely_subsystem': 'Mixed host-visible behavior',
            'likely_hardware_block': 'General system interaction visible to the host OS',
            'hardware_confidence': 'low',
            'layperson_summary': 'This feature is useful evidence, but it does not map cleanly to one hardware block.',
            'platform_note': 'Treat this as supporting context rather than a precise physical location.',
        }


FINAL_SCORECARD_CONFIGS = ['tier0_tier1_tier2', 'tier0_tier1_alt_tier2']
PAPER_SCORECARD_CSV = COMPARISON_DIR / 'industry_paper_scorecard_profiles.csv'
PAPER_DIAGNOSIS_STRENGTH_CSV = COMPARISON_DIR / 'industry_diagnosis_strength_profiles.csv'
PAPER_STRESSOR_RECALL_CSV = COMPARISON_DIR / 'industry_stressor_recall_profiles.csv'
PAPER_STRESSOR_MECH_CSV = COMPARISON_DIR / 'industry_stressor_mechanism_profiles.csv'
PAPER_TOP_FEATURES_CSV = COMPARISON_DIR / 'industry_top_features_profiles.csv'
PAPER_TIMELINES_CSV = COMPARISON_DIR / 'industry_timeline_profiles.csv'
PAPER_HARDWARE_HOTSPOTS_CSV = COMPARISON_DIR / 'industry_hardware_hotspots_profiles.csv'


def _pick_final_config(global_dir: Path) -> str | None:
    overall_path = global_dir / 'overall_metrics.csv'
    if not overall_path.exists():
        return None
    overall = pd.read_csv(overall_path)
    available = set(overall['config'].dropna().astype(str))
    for cfg in FINAL_SCORECARD_CONFIGS:
        if cfg in available:
            return cfg
    return sorted(available)[-1] if available else None


def _load_diag_candidates(global_dir: Path, final_cfg: str) -> pd.DataFrame:
    rows = []
    candidates = [
        ('feature_whole_run', global_dir / 'stressor_feature_diagnosis_metrics.csv'),
        ('hierarchical_whole_run', global_dir / 'stressor_hierarchical_diagnosis_metrics.csv'),
        ('feature_post_alert', global_dir / 'stressor_feature_diagnosis_post_alert_metrics.csv'),
        ('hierarchical_post_alert', global_dir / 'stressor_hierarchical_diagnosis_post_alert_metrics.csv'),
    ]
    for label, path in candidates:
        if not path.exists():
            continue
        df = pd.read_csv(path)
        row = df[df['config'] == final_cfg]
        if row.empty:
            continue
        row = row.iloc[0].to_dict()
        row['method_label'] = label
        rows.append(row)
    return pd.DataFrame(rows)


def _pick_best_exact(diag_candidates: pd.DataFrame):
    if diag_candidates.empty:
        return None
    return diag_candidates.sort_values(['top1_acc', 'top2_acc', 'macro_f1'], ascending=False).iloc[0]


def _pick_best_selective(diag_candidates: pd.DataFrame):
    if diag_candidates.empty or 'selective_top2_acc' not in diag_candidates.columns:
        return None
    selective = diag_candidates[pd.notna(diag_candidates['selective_top2_acc'])].copy()
    if selective.empty:
        return None
    if 'coverage' not in selective.columns:
        selective['coverage'] = np.nan
    if 'selective_top1_acc' not in selective.columns:
        selective['selective_top1_acc'] = np.nan
    if 'selective_macro_f1' not in selective.columns:
        selective['selective_macro_f1'] = np.nan
    return selective.sort_values(
        ['selective_top2_acc', 'coverage', 'selective_top1_acc', 'selective_macro_f1'],
        ascending=False,
    ).iloc[0]


def _mechanism_localization_tables(case_diag: pd.DataFrame, final_cfg: str) -> tuple[dict[str, float], pd.DataFrame]:
    if case_diag.empty:
        return {}, pd.DataFrame()

    anom = case_diag[(case_diag['label'] == 1) & (case_diag['config'] == final_cfg)].copy()
    mech_cols = [
        c for c in [
            'compute_contrib',
            'memory_io_contrib',
            'thermal_power_contrib',
            'scheduler_runtime_contrib',
            'platform_pressure_contrib',
        ]
        if c in anom.columns
    ]
    if anom.empty or not mech_cols:
        return {}, pd.DataFrame()

    anom['total_evidence'] = anom[mech_cols].sum(axis=1).replace(0.0, np.nan)
    for k in range(1, 4):
        cols = [c for c in [f'top_mechanism_score_{i}' for i in range(1, k + 1)] if c in anom.columns]
        anom[f'mechanism_top{k}_coverage'] = anom[cols].sum(axis=1) / anom['total_evidence'] if cols else np.nan

    overall = {
        'mechanism_top1_coverage': float(anom['mechanism_top1_coverage'].mean()),
        'mechanism_top2_coverage': float(anom['mechanism_top2_coverage'].mean()),
        'mechanism_top3_coverage': float(anom['mechanism_top3_coverage'].mean()),
    }

    stressor_mech = (
        anom.groupby('stressor', as_index=False)[[
            'mechanism_top1_coverage',
            'mechanism_top2_coverage',
            'mechanism_top3_coverage',
        ]]
        .mean()
    )
    return overall, stressor_mech


score_rows = []
diagnosis_strength_rows = []
recall_rows = []
stressor_mech_rows = []
feature_rows = []
timeline_rows = []
hotspot_rows = []
notes = []

for profile in [p for p in PROFILE_ORDER if (profile_global_dir(p) / 'overall_metrics.csv').exists()]:
    global_dir = profile_global_dir(profile)
    holdout_dir = profile_holdout_dir(profile)
    profile_label = PROFILE_LABEL.get(profile, profile)
    final_cfg = _pick_final_config(global_dir)
    if final_cfg is None:
        continue

    overall = pd.read_csv(global_dir / 'overall_metrics.csv')
    sequential = pd.read_csv(global_dir / 'sequential_metrics.csv') if (global_dir / 'sequential_metrics.csv').exists() else pd.DataFrame()
    holdout = pd.read_csv(holdout_dir / 'holdout_robustness_summary.csv') if (holdout_dir / 'holdout_robustness_summary.csv').exists() else pd.DataFrame()
    case_diag = pd.read_csv(global_dir / 'case_diagnosis_summary.csv') if (global_dir / 'case_diagnosis_summary.csv').exists() else pd.DataFrame()

    diag_candidates = _load_diag_candidates(global_dir, final_cfg)
    best_exact = _pick_best_exact(diag_candidates)
    best_selective = _pick_best_selective(diag_candidates)
    mech_overall, stressor_mech = _mechanism_localization_tables(case_diag, final_cfg)

    overall_row = overall[overall['config'] == final_cfg].iloc[0]
    seq_row = sequential[sequential['config'] == final_cfg].iloc[0] if not sequential.empty and (sequential['config'] == final_cfg).any() else pd.Series(dtype=object)
    holdout_row = holdout[holdout['config'] == final_cfg].iloc[0] if not holdout.empty and (holdout['config'] == final_cfg).any() else pd.Series(dtype=object)

    score_rows.append({
        'feature_profile': profile,
        'profile_label': profile_label,
        'config': final_cfg,
        'config_label': CFG_LABEL.get(final_cfg, final_cfg),
        'n_features': int(overall_row.get('n_features', np.nan)) if pd.notna(overall_row.get('n_features', np.nan)) else np.nan,
        'base_roc_auc': float(overall_row.get('roc_auc', np.nan)),
        'base_pr_auc': float(overall_row.get('pr_auc', np.nan)),
        'holdout_mean_pr_auc': float(holdout_row.get('mean_pr_auc', np.nan)),
        'holdout_worst_pr_auc': float(holdout_row.get('worst_pr_auc', np.nan)),
        'detection_rate': float(seq_row.get('anomaly_detect_rate', np.nan)),
        'benign_alert_rate': float(seq_row.get('benign_run_alert_rate', np.nan)),
        'median_ttd_s': float(seq_row.get('median_time_to_detect_s', np.nan)),
        'best_diag_method': '' if best_exact is None else str(best_exact['method_label']),
        'top1_acc': np.nan if best_exact is None else float(best_exact.get('top1_acc', np.nan)),
        'top2_acc': np.nan if best_exact is None else float(best_exact.get('top2_acc', np.nan)),
        'macro_f1': np.nan if best_exact is None else float(best_exact.get('macro_f1', np.nan)),
        'best_selective_method': '' if best_selective is None else str(best_selective['method_label']),
        'selective_coverage': np.nan if best_selective is None else float(best_selective.get('coverage', np.nan)),
        'selective_abstain_rate': np.nan if best_selective is None else float(best_selective.get('abstain_rate', np.nan)),
        'selective_top1_acc': np.nan if best_selective is None else float(best_selective.get('selective_top1_acc', np.nan)),
        'selective_top2_acc': np.nan if best_selective is None else float(best_selective.get('selective_top2_acc', np.nan)),
        'selective_macro_f1': np.nan if best_selective is None else float(best_selective.get('selective_macro_f1', np.nan)),
        'mechanism_top1_coverage': mech_overall.get('mechanism_top1_coverage', np.nan),
        'mechanism_top2_coverage': mech_overall.get('mechanism_top2_coverage', np.nan),
        'mechanism_top3_coverage': mech_overall.get('mechanism_top3_coverage', np.nan),
    })

    strong_claims = []
    exact_top2 = np.nan if best_exact is None else float(best_exact.get('top2_acc', np.nan))
    selective_top2 = np.nan if best_selective is None else float(best_selective.get('selective_top2_acc', np.nan))
    selective_cov = np.nan if best_selective is None else float(best_selective.get('coverage', np.nan))
    mech_top3 = float(mech_overall.get('mechanism_top3_coverage', np.nan))

    if pd.notna(exact_top2) and exact_top2 >= 0.8:
        strong_claims.append(f'Exact Top-2 shortlist {exact_top2:.2f}')
    if pd.notna(selective_top2) and selective_top2 >= 0.8:
        strong_claims.append(f'Confidence-gated Top-2 {selective_top2:.2f} at {selective_cov:.2f} coverage')
    if pd.notna(mech_top3) and mech_top3 >= 0.8:
        strong_claims.append(f'Mechanism Top-3 coverage {mech_top3:.2f}')

    diagnosis_strength_rows.append({
        'feature_profile': profile,
        'profile_label': profile_label,
        'config': final_cfg,
        'config_label': CFG_LABEL.get(final_cfg, final_cfg),
        'exact_top1_acc': np.nan if best_exact is None else float(best_exact.get('top1_acc', np.nan)),
        'exact_top2_acc': exact_top2,
        'exact_macro_f1': np.nan if best_exact is None else float(best_exact.get('macro_f1', np.nan)),
        'selective_method': '' if best_selective is None else str(best_selective['method_label']),
        'selective_coverage': selective_cov,
        'selective_top1_acc': np.nan if best_selective is None else float(best_selective.get('selective_top1_acc', np.nan)),
        'selective_top2_acc': selective_top2,
        'selective_macro_f1': np.nan if best_selective is None else float(best_selective.get('selective_macro_f1', np.nan)),
        'mechanism_top1_coverage': mech_overall.get('mechanism_top1_coverage', np.nan),
        'mechanism_top2_coverage': mech_overall.get('mechanism_top2_coverage', np.nan),
        'mechanism_top3_coverage': mech_top3,
        'strong_claims': '; '.join(strong_claims) if strong_claims else 'No diagnosis-adjacent metric exceeds 0.80 in this profile/configuration.',
    })

    if not stressor_mech.empty:
        stressor_mech['feature_profile'] = profile
        stressor_mech['profile_label'] = profile_label
        stressor_mech['config'] = final_cfg
        stressor_mech['config_label'] = CFG_LABEL.get(final_cfg, final_cfg)
        mapped = stressor_mech['stressor'].apply(map_stressor_to_mbp_hardware).apply(pd.Series).reset_index(drop=True)
        stressor_mech = pd.concat([stressor_mech.reset_index(drop=True), mapped], axis=1)
        stressor_mech_rows.append(stressor_mech)

    pred_path = global_dir / 'stressor_feature_diagnosis_predictions.csv'
    if pred_path.exists():
        pred = pd.read_csv(pred_path)
        pred = pred[pred['config'] == final_cfg].copy()
        if not pred.empty:
            recall = pred.groupby('true_stressor', as_index=False)['is_correct'].mean().rename(columns={'true_stressor': 'stressor', 'is_correct': 'exact_recall'})
            recall['feature_profile'] = profile
            recall['profile_label'] = profile_label
            recall['config'] = final_cfg
            recall['config_label'] = CFG_LABEL.get(final_cfg, final_cfg)
            mapped = recall['stressor'].apply(map_stressor_to_mbp_hardware).apply(pd.Series).reset_index(drop=True)
            recall = pd.concat([recall.reset_index(drop=True), mapped], axis=1)
            recall_rows.append(recall)

    loc_features = profile_paper_dir(profile) / 'industry_localization_top_features.csv'
    if loc_features.exists():
        feats = pd.read_csv(loc_features)
        feats = feats[(feats['config'] == final_cfg) & (feats['rank'] <= 15)].copy()
        if not feats.empty:
            mapped = feats['feature_name'].apply(map_feature_to_mbp_hardware).apply(pd.Series).reset_index(drop=True)
            feats = pd.concat([feats.reset_index(drop=True), mapped], axis=1)
            feature_rows.append(feats[[
                'feature_profile', 'profile_label', 'config', 'config_label', 'rank', 'feature_name', 'feature_tier',
                'hit_count', 'case_count', 'mean_feature_score', 'earliest_block_s',
                'likely_subsystem', 'likely_hardware_block', 'hardware_confidence', 'layperson_summary', 'platform_note'
            ]])
            hotspot = feats.groupby([
                'feature_profile', 'profile_label', 'config', 'config_label', 'likely_subsystem', 'likely_hardware_block'
            ], as_index=False).agg(
                total_hit_count=('hit_count', 'sum'),
                n_top_features=('feature_name', 'count'),
                max_feature_score=('mean_feature_score', 'max'),
            )
            hotspot_rows.append(hotspot)

    loc_timeline = profile_paper_dir(profile) / 'industry_localization_representative_cases.csv'
    if loc_timeline.exists():
        tl = pd.read_csv(loc_timeline)
        tl = tl[tl['config'] == final_cfg].copy()
        if not tl.empty:
            timeline_rows.append(tl[[
                'feature_profile', 'profile_label', 'config', 'config_label', 'stressor', 'workload',
                'first_block_s', 'second_block_s', 'third_block_s', 'fourth_block_s', 'fifth_block_s',
                'dominant_tier', 'dominant_mechanism'
            ]])

    note_parts = []
    if pd.notna(exact_top2):
        note_parts.append(f'exact Top-2 = {exact_top2:.2f}')
    if pd.notna(selective_top2):
        note_parts.append(f'confidence-gated Top-2 = {selective_top2:.2f} at coverage {selective_cov:.2f}')
    if pd.notna(mech_top3):
        note_parts.append(f'mechanism Top-3 coverage = {mech_top3:.2f}')
    if note_parts:
        notes.append(f"- {profile_label}: " + '; '.join(note_parts) + '.')

scorecard_df = pd.DataFrame(score_rows)
diagnosis_strength_df = pd.DataFrame(diagnosis_strength_rows)
recall_df = pd.concat(recall_rows, ignore_index=True) if recall_rows else pd.DataFrame()
stressor_mech_df = pd.concat(stressor_mech_rows, ignore_index=True) if stressor_mech_rows else pd.DataFrame()
feature_df = pd.concat(feature_rows, ignore_index=True) if feature_rows else pd.DataFrame()
timeline_df = pd.concat(timeline_rows, ignore_index=True) if timeline_rows else pd.DataFrame()
hotspot_df = pd.concat(hotspot_rows, ignore_index=True) if hotspot_rows else pd.DataFrame()

scorecard_df.to_csv(PAPER_SCORECARD_CSV, index=False)
diagnosis_strength_df.to_csv(PAPER_DIAGNOSIS_STRENGTH_CSV, index=False)
recall_df.to_csv(PAPER_STRESSOR_RECALL_CSV, index=False)
stressor_mech_df.to_csv(PAPER_STRESSOR_MECH_CSV, index=False)
feature_df.to_csv(PAPER_TOP_FEATURES_CSV, index=False)
timeline_df.to_csv(PAPER_TIMELINES_CSV, index=False)
hotspot_df.to_csv(PAPER_HARDWARE_HOTSPOTS_CSV, index=False)

if not scorecard_df.empty:
    display(Markdown('#### Final-head monitoring and diagnosis scorecard'))
    display(scorecard_df[[
        'profile_label', 'config_label', 'n_features', 'base_roc_auc', 'base_pr_auc',
        'holdout_mean_pr_auc', 'holdout_worst_pr_auc', 'detection_rate', 'benign_alert_rate',
        'median_ttd_s', 'top1_acc', 'top2_acc', 'selective_top2_acc', 'selective_coverage', 'mechanism_top3_coverage'
    ]].round(3))

if not diagnosis_strength_df.empty:
    display(Markdown('#### Diagnosis metrics that are strongest enough to highlight in the paper'))
    display(diagnosis_strength_df[[
        'profile_label', 'config_label', 'exact_top1_acc', 'exact_top2_acc',
        'selective_method', 'selective_top2_acc', 'selective_coverage',
        'mechanism_top3_coverage', 'strong_claims'
    ]].round(3))

if not stressor_mech_df.empty:
    display(Markdown('#### Mechanism localization by stressor in the final head'))
    display(stressor_mech_df[[
        'profile_label', 'stressor', 'mechanism_top1_coverage', 'mechanism_top2_coverage', 'mechanism_top3_coverage',
        'likely_subsystem', 'likely_hardware_block'
    ]].sort_values(['profile_label', 'mechanism_top3_coverage', 'stressor'], ascending=[True, False, True]).round(3))

if not recall_df.empty:
    display(Markdown('#### Stressors identified best in the final head and what they mean on this M2 Pro'))
    display(recall_df[[
        'profile_label', 'stressor', 'exact_recall', 'likely_subsystem', 'likely_hardware_block'
    ]].sort_values(['profile_label', 'exact_recall', 'stressor'], ascending=[True, False, True]).round(3))

if not feature_df.empty:
    display(Markdown('#### Top recurring localized features in the final head and where they point on the laptop'))
    display(feature_df[[
        'profile_label', 'rank', 'feature_name', 'likely_subsystem', 'likely_hardware_block',
        'hit_count', 'mean_feature_score', 'hardware_confidence'
    ]].sort_values(['profile_label', 'rank']).round(3))

if not hotspot_df.empty:
    display(Markdown('#### Most visible hardware hotspots in the final head'))
    display(hotspot_df[[
        'profile_label', 'likely_subsystem', 'likely_hardware_block', 'total_hit_count', 'n_top_features', 'max_feature_score'
    ]].sort_values(['profile_label', 'total_hit_count', 'n_top_features'], ascending=[True, False, False]).round(3))

if not timeline_df.empty:
    display(Markdown('#### First-through-fifth abnormal windows in representative cases'))
    display(timeline_df.sort_values(['profile_label', 'stressor']).round(1))

if notes:
    display(Markdown('#### Paper-facing diagnosis and hardware notes'))
    display(Markdown('\n'.join(dict.fromkeys(notes))))


### Scenario diagnosis and localization figures


In [ ]:
# Plain-language: This cell adds a cleaner scenario-level dashboard with one professional figure per row,
# larger text, higher-contrast labels, and distinct color styling for recall, mechanism coverage, and onset timing.
display(Markdown('### Scenario diagnosis and localization figures'))


SCENARIO_RECALL_CSV = COMPARISON_DIR / 'industry_stressor_recall_profiles.csv'
SCENARIO_MECH_CSV = COMPARISON_DIR / 'industry_stressor_mechanism_profiles.csv'
SCENARIO_SCORECARD_CSV = COMPARISON_DIR / 'industry_paper_scorecard_profiles.csv'

SCENARIO_RECALL_PNG = COMPARISON_FIG / 'fig_scenario_exact_recall_profiles.png'
SCENARIO_MECH_PNG = COMPARISON_FIG / 'fig_scenario_mechanism_top3_profiles.png'
SCENARIO_ONSET_PNG = COMPARISON_FIG / 'fig_scenario_onset_profiles.png'

SCENARIO_STRESSOR_ORDER = ['ATOMIC', 'BRANCH', 'CACHE', 'MEMBW', 'TLB']

FIG_BG = '#FCFCFF'
PANEL_BG = '#F8FAFC'
TEXT_DARK = '#0F172A'
TEXT_MID = '#475569'
GRID_SOFT = '#E2E8F0'
SPINE_SOFT = '#CBD5E1'

SCENARIO_PROFILE_COLORS = {
    **globals().get('PROFILE_COLOR', {}),
    'mixed': '#0B3C5D',
    'full': '#7C2D12',
}
SCENARIO_PROFILE_MARKERS = {
    **globals().get('PROFILE_MARKER', {}),
    'mixed': 'o',
    'full': 'D',
}

RECALL_CMAP = LinearSegmentedColormap.from_list(
    'scenario_recall_map',
    ['#EFF6FF', '#93C5FD', '#2563EB', '#1D4ED8']
)
MECH_CMAP = LinearSegmentedColormap.from_list(
    'scenario_mech_map',
    ['#FFF7ED', '#FDBA74', '#F97316', '#C2410C']
)


def _relative_luminance(rgb) -> float:
    vals = []
    for c in rgb[:3]:
        if c <= 0.03928:
            vals.append(c / 12.92)
        else:
            vals.append(((c + 0.055) / 1.055) ** 2.4)
    return 0.2126 * vals[0] + 0.7152 * vals[1] + 0.0722 * vals[2]


def _annotation_color(rgba) -> str:
    return '#F8FAFC' if _relative_luminance(rgba) < 0.42 else TEXT_DARK


def _style_panel(ax, title: str):
    ax.set_facecolor(PANEL_BG)
    for spine in ax.spines.values():
        spine.set_color(SPINE_SOFT)
        spine.set_linewidth(1.2)
    ax.set_title(title, fontsize=18, fontweight='bold', color=TEXT_DARK, loc='left', pad=14)


def _draw_metric_heatmap(
    ax,
    matrix: pd.DataFrame,
    row_labels: list[str],
    col_labels: list[str],
    title: str,
    cmap,
    cbar_label: str,
):
    data = matrix.reindex(index=row_labels, columns=col_labels).to_numpy(dtype=float)
    masked = np.ma.masked_invalid(data)

    im = ax.imshow(
        masked,
        cmap=cmap,
        vmin=0.0,
        vmax=1.0,
        aspect='auto',
        interpolation='nearest',
    )

    _style_panel(ax, title)
    ax.set_xticks(np.arange(len(col_labels)))
    ax.set_xticklabels(
        [PROFILE_LABEL.get(col, str(col).title()) for col in col_labels],
        fontsize=15,
        fontweight='bold',
        color=TEXT_DARK,
    )
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_yticklabels(
        row_labels,
        fontsize=15,
        fontweight='bold',
        color=TEXT_DARK,
    )
    ax.tick_params(axis='x', length=0, pad=10)
    ax.tick_params(axis='y', length=0, pad=8)

    ax.set_xticks(np.arange(-0.5, len(col_labels), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(row_labels), 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=2.6)
    ax.tick_params(which='minor', bottom=False, left=False)

    for yi, _row in enumerate(row_labels):
        for xi, _col in enumerate(col_labels):
            val = data[yi, xi]
            if np.isfinite(val):
                rgba = im.cmap(im.norm(val))
                ax.text(
                    xi,
                    yi,
                    f'{val:.2f}',
                    ha='center',
                    va='center',
                    fontsize=15,
                    fontweight='bold',
                    color=_annotation_color(rgba),
                )
            else:
                ax.text(
                    xi,
                    yi,
                    'NA',
                    ha='center',
                    va='center',
                    fontsize=13,
                    fontweight='bold',
                    color='#64748B',
                )

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
    cbar.set_label(cbar_label, fontsize=13.5, fontweight='bold', color=TEXT_DARK, labelpad=10)
    cbar.ax.tick_params(labelsize=12.5, colors=TEXT_DARK)
    cbar.outline.set_edgecolor(SPINE_SOFT)
    cbar.outline.set_linewidth(1.0)


def _save_display_close(fig, path: Path, caption: str | None = None):
    fig.savefig(path, dpi=max(int(globals().get('PRO_FIG_DPI', 300)), 300), bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.close(fig)
    display(Image(filename=str(path)))
    if caption:
        display(Markdown(f'<div style="margin-top:6px; color:{TEXT_MID}; font-size:15px;">{caption}</div>'))


if not SCENARIO_RECALL_CSV.exists() or not SCENARIO_MECH_CSV.exists() or not SCENARIO_SCORECARD_CSV.exists():
    display(Markdown('Run the paper scorecard cell first so the scenario-level figure inputs exist.'))
else:
    recall_df = pd.read_csv(SCENARIO_RECALL_CSV)
    mech_df = pd.read_csv(SCENARIO_MECH_CSV)
    scorecard_df = pd.read_csv(SCENARIO_SCORECARD_CSV)

    profiles = [p for p in PROFILE_ORDER if p in set(scorecard_df['feature_profile'])]
    if not profiles:
        display(Markdown('No profile rows were found for the scenario dashboard.'))
    else:
        onset_rows = []
        for profile in profiles:
            tpath = profile_paper_dir(profile) / 'industry_localization_timelines.csv'
            if not tpath.exists():
                continue
            tl = pd.read_csv(tpath)
            final_cfg_rows = scorecard_df[scorecard_df['feature_profile'] == profile]
            if not final_cfg_rows.empty:
                tl = tl[tl['config'] == final_cfg_rows.iloc[0]['config']].copy()
            if tl.empty:
                continue
            onset = tl.groupby('stressor', as_index=False).agg(
                median_first_block_s=('first_block_s', 'median'),
                min_first_block_s=('first_block_s', 'min'),
                max_first_block_s=('first_block_s', 'max'),
            )
            onset['feature_profile'] = profile
            onset_rows.append(onset)

        onset_df = pd.concat(onset_rows, ignore_index=True) if onset_rows else pd.DataFrame()

        recall_mat = (
            recall_df.pivot_table(
                index='stressor',
                columns='feature_profile',
                values='exact_recall',
                aggfunc='mean',
            )
            if not recall_df.empty else pd.DataFrame()
        )
        mech_mat = (
            mech_df.pivot_table(
                index='stressor',
                columns='feature_profile',
                values='mechanism_top3_coverage',
                aggfunc='mean',
            )
            if not mech_df.empty else pd.DataFrame()
        )

        if not recall_mat.empty:
            fig, ax = plt.subplots(figsize=(10.8, 6.8))
            fig.patch.set_facecolor(FIG_BG)
            _draw_metric_heatmap(
                ax=ax,
                matrix=recall_mat,
                row_labels=SCENARIO_STRESSOR_ORDER,
                col_labels=profiles,
                title='Scenario 1. Exact stressor recall',
                cmap=RECALL_CMAP,
                cbar_label='Recall',
            )
            _save_display_close(
                fig,
                SCENARIO_RECALL_PNG,
                'Exact stressor recall in the final head. Higher values indicate stronger identification accuracy.'
            )
        else:
            display(Markdown('No exact-recall data were available for the scenario recall figure.'))

        if not mech_mat.empty:
            fig, ax = plt.subplots(figsize=(10.8, 6.8))
            fig.patch.set_facecolor(FIG_BG)
            _draw_metric_heatmap(
                ax=ax,
                matrix=mech_mat,
                row_labels=SCENARIO_STRESSOR_ORDER,
                col_labels=profiles,
                title='Scenario 2. Mechanism Top-3 coverage',
                cmap=MECH_CMAP,
                cbar_label='Coverage',
            )
            _save_display_close(
                fig,
                SCENARIO_MECH_PNG,
                'Mechanism Top-3 coverage in the final head. Higher values indicate better coverage of abnormal evidence.'
            )
        else:
            display(Markdown('No mechanism-coverage data were available for the scenario mechanism figure.'))

        if not onset_df.empty:
            fig, ax = plt.subplots(figsize=(12.4, 7.6))
            fig.patch.set_facecolor(FIG_BG)
            _style_panel(ax, 'Scenario 3. First abnormal window timing')

            base_y = np.arange(len(SCENARIO_STRESSOR_ORDER), dtype=float)
            offsets = np.linspace(-0.16, 0.16, max(1, len(profiles))) if profiles else np.array([0.0])

            handles = []
            labels = []

            for offset, profile in zip(offsets, profiles):
                d = onset_df[onset_df['feature_profile'] == profile].copy()
                if d.empty:
                    continue

                d = d.set_index('stressor').reindex(SCENARIO_STRESSOR_ORDER)
                med = d['median_first_block_s']
                lo = d['min_first_block_s'].fillna(med)
                hi = d['max_first_block_s'].fillna(med)

                y = base_y + offset
                valid = pd.notna(med)

                color = SCENARIO_PROFILE_COLORS.get(profile, '#334155')
                marker = SCENARIO_PROFILE_MARKERS.get(profile, 'o')

                ax.hlines(
                    y[valid],
                    lo[valid],
                    hi[valid],
                    color=color,
                    alpha=0.38,
                    linewidth=10,
                    zorder=1,
                )
                sc = ax.scatter(
                    med[valid],
                    y[valid],
                    s=180,
                    marker=marker,
                    color=color,
                    edgecolor='white',
                    linewidth=1.6,
                    zorder=3,
                )
                handles.append(sc)
                labels.append(PROFILE_LABEL.get(profile, profile.title()))

                for yy, xx in zip(y[valid], med[valid]):
                    ax.text(
                        xx + 6,
                        yy,
                        f'{xx:.0f}s',
                        va='center',
                        ha='left',
                        fontsize=13,
                        fontweight='bold',
                        color=color,
                        bbox=dict(boxstyle='round,pad=0.20', fc='white', ec='none', alpha=0.88),
                    )

            ax.set_yticks(base_y)
            ax.set_yticklabels(
                SCENARIO_STRESSOR_ORDER,
                fontsize=15,
                fontweight='bold',
                color=TEXT_DARK,
            )
            ax.set_xlabel('Time to first abnormal window (seconds)', fontsize=15.5, fontweight='bold', color=TEXT_DARK, labelpad=12)
            ax.tick_params(axis='x', labelsize=13.5, colors=TEXT_DARK)
            ax.tick_params(axis='y', length=0, pad=8)
            ax.grid(axis='x', color=GRID_SOFT, linewidth=1.1, alpha=0.9)
            ax.set_axisbelow(True)
            ax.invert_yaxis()

            if handles:
                legend = ax.legend(
                    handles,
                    labels,
                    loc='upper right',
                    fontsize=13,
                    frameon=True,
                    fancybox=True,
                    borderpad=0.7,
                    labelspacing=0.6,
                )
                legend.get_frame().set_facecolor('white')
                legend.get_frame().set_edgecolor(SPINE_SOFT)
                legend.get_frame().set_linewidth(1.0)

            _save_display_close(
                fig,
                SCENARIO_ONSET_PNG,
                'Median onset time is shown by the marker; the horizontal band shows the min-to-max range across representative cases.'
            )
        else:
            display(Markdown('No onset-timing data were available for the scenario onset figure.'))



### Case-by-case onset and hotspot figures


In [ ]:
# Plain-language: This cell presents the case-by-case onset and hotspot results as three separate,
# cleaner figures with larger text, lighter backgrounds, and stronger visual separation.
display(Markdown('### Case-by-case onset and hotspot figures'))


CASE_SCORECARD_CSV = COMPARISON_DIR / 'industry_paper_scorecard_profiles.csv'
CASE_FEATURES_CSV = COMPARISON_DIR / 'industry_top_features_profiles.csv'
CASE_HOTSPOT_CSV = COMPARISON_DIR / 'industry_hardware_hotspots_profiles.csv'

CASE_ONSET_PNG = COMPARISON_FIG / 'fig_case_onset_profiles.png'
CASE_HOTSPOT_PNG = COMPARISON_FIG / 'fig_case_hotspots_profiles.png'
CASE_FEATURE_PNG = COMPARISON_FIG / 'fig_case_top_features_profiles.png'
CASE_GALLERY_PNG = CASE_ONSET_PNG  # Backward-compatible alias

CASE_WORKLOAD_ORDER = ['BROWSER', 'VIDEO_SW', 'PY_AI', 'PY_STATS']
CASE_STRESSOR_ORDER = ['ATOMIC', 'BRANCH', 'CACHE', 'MEMBW', 'TLB']

FIG_BG = '#FCFCFF'
PANEL_BG = '#F8FAFC'
TEXT_DARK = '#0F172A'
TEXT_MID = '#475569'
GRID_SOFT = '#E2E8F0'
SPINE_SOFT = '#CBD5E1'

CASE_PROFILE_COLORS = {
    **globals().get('PROFILE_COLOR', {}),
    'mixed': '#0B3C5D',
    'full': '#7C2D12',
}

HOTSPOT_PROFILE_COLORS = {
    'mixed': '#4338CA',   # indigo
    'full': '#BE185D',    # rose/magenta
}

FEATURE_TIER_COLORS = {
    'tier0': '#64748B',      # slate
    'tier1_alt': '#059669',  # emerald
    'tier2': '#D97706',      # amber
}



def _short_feature_label(name: str) -> str:
    text = str(name)
    for old, new in [('tier0:', 'T0:'), ('tier1_alt:', 'T1:'), ('tier2:', 'T2:')]:
        text = text.replace(old, new)
    return text


def _relative_luminance(rgb) -> float:
    vals = []
    for c in rgb[:3]:
        if c <= 0.03928:
            vals.append(c / 12.92)
        else:
            vals.append(((c + 0.055) / 1.055) ** 2.4)
    return 0.2126 * vals[0] + 0.7152 * vals[1] + 0.0722 * vals[2]


def _annotation_color(rgba) -> str:
    return '#F8FAFC' if _relative_luminance(rgba) < 0.42 else TEXT_DARK


def _style_axis(ax, title: str):
    ax.set_facecolor(PANEL_BG)
    for spine in ax.spines.values():
        spine.set_color(SPINE_SOFT)
        spine.set_linewidth(1.2)
    ax.set_title(title, fontsize=18, fontweight='bold', color=TEXT_DARK, loc='left', pad=14)


def _save_display_close(fig, path: Path, caption: str | None = None):
    fig.savefig(
        path,
        dpi=max(int(globals().get('PRO_FIG_DPI', 300)), 300),
        bbox_inches='tight',
        facecolor=fig.get_facecolor(),
    )
    plt.close(fig)
    display(Image(filename=str(path)))
    if caption:
        display(Markdown(
            f'<div style="margin-top:6px; color:{TEXT_MID}; font-size:15px;">{caption}</div>'
        ))


if not CASE_SCORECARD_CSV.exists() or not CASE_FEATURES_CSV.exists() or not CASE_HOTSPOT_CSV.exists():
    display(Markdown('Run the diagnosis scorecard cell first so the case-gallery inputs exist.'))
else:
    scorecard_df = pd.read_csv(CASE_SCORECARD_CSV)
    feature_df = pd.read_csv(CASE_FEATURES_CSV)
    hotspot_df = pd.read_csv(CASE_HOTSPOT_CSV)
    profiles = [p for p in PROFILE_ORDER if p in set(scorecard_df['feature_profile'])]

    if not profiles:
        display(Markdown('No profile rows were found for the case gallery.'))
    else:
        onset_rows = []
        for profile in profiles:
            tpath = profile_paper_dir(profile) / 'industry_localization_timelines.csv'
            if not tpath.exists():
                continue
            tl = pd.read_csv(tpath)
            final_cfg_rows = scorecard_df[scorecard_df['feature_profile'] == profile]
            if not final_cfg_rows.empty:
                tl = tl[tl['config'] == final_cfg_rows.iloc[0]['config']].copy()
            if tl.empty:
                continue
            tl = tl.groupby(['workload', 'stressor'], as_index=False).agg(
                first_block_s=('first_block_s', 'median')
            )
            tl['feature_profile'] = profile
            onset_rows.append(tl)

        onset_df = pd.concat(onset_rows, ignore_index=True) if onset_rows else pd.DataFrame()

        if not onset_df.empty:
            onset_vals = onset_df['first_block_s'].to_numpy(dtype=float)
            finite = onset_vals[np.isfinite(onset_vals)]
            vmin = float(np.nanmin(finite)) if finite.size else 0.0
            vmax = float(np.nanmax(finite)) if finite.size else 1.0
            if vmax <= vmin:
                vmax = vmin + 1.0

            fig, axes = plt.subplots(
                len(profiles),
                1,
                figsize=(10.8, max(5.6, 4.8 * len(profiles))),
                squeeze=False,
            )
            fig.patch.set_facecolor(FIG_BG)
            axes = axes.ravel()

            last_im = None
            for ax, profile in zip(axes, profiles):
                d = onset_df[onset_df['feature_profile'] == profile].copy()
                pivot = d.pivot(index='workload', columns='stressor', values='first_block_s') if not d.empty else pd.DataFrame()
                pivot = pivot.reindex(index=CASE_WORKLOAD_ORDER, columns=CASE_STRESSOR_ORDER)
                data = pivot.to_numpy(dtype=float)
                masked = np.ma.masked_invalid(data)

                im = ax.imshow(
                    masked,
                    aspect='auto',
                    cmap=CASE_ONSET_CMAP,
                    vmin=vmin,
                    vmax=vmax,
                    interpolation='nearest',
                )
                last_im = im

                _style_axis(ax, f"{PROFILE_LABEL.get(profile, profile.title())}: first abnormal window by case")
                ax.set_xticks(np.arange(len(CASE_STRESSOR_ORDER)))
                ax.set_xticklabels(CASE_STRESSOR_ORDER, fontsize=14, fontweight='bold', color=TEXT_DARK)
                ax.set_yticks(np.arange(len(CASE_WORKLOAD_ORDER)))
                ax.set_yticklabels(CASE_WORKLOAD_ORDER, fontsize=14, fontweight='bold', color=TEXT_DARK)
                ax.tick_params(axis='x', length=0, pad=8)
                ax.tick_params(axis='y', length=0, pad=8)

                ax.set_xticks(np.arange(-0.5, len(CASE_STRESSOR_ORDER), 1), minor=True)
                ax.set_yticks(np.arange(-0.5, len(CASE_WORKLOAD_ORDER), 1), minor=True)
                ax.grid(which='minor', color='white', linewidth=2.6)
                ax.tick_params(which='minor', bottom=False, left=False)

                for yi in range(len(CASE_WORKLOAD_ORDER)):
                    for xi in range(len(CASE_STRESSOR_ORDER)):
                        val = data[yi, xi]
                        if np.isfinite(val):
                            rgba = im.cmap(im.norm(val))
                            ax.text(
                                xi,
                                yi,
                                f'{val:.0f}s',
                                ha='center',
                                va='center',
                                fontsize=13.5,
                                fontweight='bold',
                                color=_annotation_color(rgba),
                            )
                        else:
                            ax.text(
                                xi,
                                yi,
                                'NA',
                                ha='center',
                                va='center',
                                fontsize=12.5,
                                fontweight='bold',
                                color='#64748B',
                            )

            if last_im is not None:
                cbar = fig.colorbar(
                    last_im,
                    ax=axes.tolist(),
                    orientation='horizontal',
                    fraction=0.05,
                    pad=0.08,
                    shrink=0.88,
                    aspect=36,
                )
                cbar.set_label(
                    'First abnormal window (s)',
                    fontsize=13.5,
                    fontweight='bold',
                    color=TEXT_DARK,
                    labelpad=8,
                )
                cbar.ax.tick_params(labelsize=12.5, colors=TEXT_DARK)
                cbar.outline.set_edgecolor(SPINE_SOFT)
                cbar.outline.set_linewidth(1.0)
            
            fig.subplots_adjust(hspace=0.36, bottom=0.22)

            _save_display_close(
                fig,
                CASE_ONSET_PNG,
                'Median onset time by workload-stressor case, shown separately for each profile.'
            )
        else:
            display(Markdown('No onset rows were available for the case-onset figure.'))

        if not hotspot_df.empty:
            hotspot_plot = hotspot_df.copy()
            hotspot_plot['subsystem_short'] = hotspot_plot['likely_subsystem'].astype(str)
            subsystem_totals = (
                hotspot_plot.groupby('subsystem_short', as_index=False)['total_hit_count']
                .sum()
                .sort_values('total_hit_count', ascending=False)
            )
            subsystem_order = subsystem_totals['subsystem_short'].head(6).tolist()
            hotspot_plot = hotspot_plot[hotspot_plot['subsystem_short'].isin(subsystem_order)].copy()

            fig, ax = plt.subplots(figsize=(11.4, 7.0))
            fig.patch.set_facecolor(FIG_BG)
            _style_axis(ax, 'Most visible hardware hotspots')

            y_base = np.arange(len(subsystem_order), dtype=float)
            offsets = np.linspace(-0.18, 0.18, max(1, len(profiles))) if profiles else np.array([0.0])

            for offset, profile in zip(offsets, profiles):
                d = hotspot_plot[hotspot_plot['feature_profile'] == profile].set_index('subsystem_short').reindex(subsystem_order)
                vals = d['total_hit_count'].fillna(0.0).to_numpy(dtype=float)
                color = HOTSPOT_PROFILE_COLORS.get(profile, '#334155')
                ax.barh(
                    y_base + offset,
                    vals,
                    height=0.30,
                    color=color,
                    alpha=0.94,
                    label=PROFILE_LABEL.get(profile, profile.title()),
                )
                for yy, val in zip(y_base + offset, vals):
                    if val > 0:
                        ax.text(
                            val + 0.55,
                            yy,
                            f'{val:.0f}',
                            va='center',
                            ha='left',
                            fontsize=12.5,
                            fontweight='bold',
                            color=color,
                        )


            ax.set_yticks(y_base)
            ax.set_yticklabels(subsystem_order, fontsize=14, fontweight='bold', color=TEXT_DARK)
            ax.invert_yaxis()
            ax.set_xlabel('Top-feature hits in the final head', fontsize=15, fontweight='bold', color=TEXT_DARK, labelpad=10)
            ax.tick_params(axis='x', labelsize=13, colors=TEXT_DARK)
            ax.tick_params(axis='y', length=0, pad=8)
            ax.grid(axis='x', color=GRID_SOFT, linewidth=1.1, alpha=0.9)
            ax.set_axisbelow(True)

            legend = ax.legend(loc='lower right', fontsize=12.5, frameon=True, fancybox=True)
            legend.get_frame().set_facecolor('white')
            legend.get_frame().set_edgecolor(SPINE_SOFT)
            legend.get_frame().set_linewidth(1.0)

            _save_display_close(
                fig,
                CASE_HOTSPOT_PNG,
                'Top recurring subsystem hotspots in localized anomaly windows, compared across profiles.'
            )
        else:
            display(Markdown('No hotspot rows were available for the hotspot figure.'))

        if not feature_df.empty:
            feat_plot = feature_df[feature_df['rank'] <= 25].copy().sort_values(['profile_label', 'rank'])
            feat_plot['label'] = feat_plot.apply(
                lambda row: f"{row['profile_label']} | {_short_feature_label(row['feature_name'])}",
                axis=1,
            )
            feat_plot = feat_plot.iloc[::-1].reset_index(drop=True)

            if 'feature_tier' not in feat_plot.columns:
                feat_plot['feature_tier'] = 'unknown'
            if 'earliest_block_s' not in feat_plot.columns:
                feat_plot['earliest_block_s'] = np.nan

            fig_height = max(6.8, 0.62 * len(feat_plot) + 1.8)
            fig, ax = plt.subplots(figsize=(13.2, fig_height))
            fig.patch.set_facecolor(FIG_BG)
            _style_axis(ax, 'Top recurring localized features')

            y = np.arange(len(feat_plot), dtype=float)
            bar_colors = [FEATURE_TIER_COLORS.get(str(tier), '#64748B') for tier in feat_plot['feature_tier']]
            ax.barh(
                y,
                feat_plot['hit_count'].to_numpy(dtype=float),
                color=bar_colors,
                alpha=0.95,
                height=0.68,
            )

            ax.set_yticks(y)
            ax.set_yticklabels(feat_plot['label'], fontsize=12.5, fontweight='bold', color=TEXT_DARK)
            ax.set_xlabel('How often the feature appears in localized anomaly windows', fontsize=15, fontweight='bold', color=TEXT_DARK, labelpad=10)
            ax.tick_params(axis='x', labelsize=13, colors=TEXT_DARK)
            ax.tick_params(axis='y', length=0, pad=8)
            ax.grid(axis='x', color=GRID_SOFT, linewidth=1.1, alpha=0.9)
            ax.set_axisbelow(True)

            for yy, val, t in zip(
                y,
                feat_plot['hit_count'].to_numpy(dtype=float),
                feat_plot['earliest_block_s'].to_numpy(dtype=float),
            ):
                label = f'{val:.0f} hits'
                if pd.notna(t):
                    label += f', first {t:.0f}s'
                ax.text(
                    val + 0.55,
                    yy,
                    label,
                    va='center',
                    ha='left',
                    fontsize=12,
                    fontweight='bold',
                    color=TEXT_DARK,
                )

            tier_handles = [
                Patch(color=color, label=label)
                for label, color in [
                    ('Tier-0', FEATURE_TIER_COLORS['tier0']),
                    ('Tier-1', FEATURE_TIER_COLORS['tier1_alt']),
                    ('Tier-2', FEATURE_TIER_COLORS['tier2']),
                ]
            ]
            legend = ax.legend(handles=tier_handles, loc='lower right', fontsize=12, frameon=True, fancybox=True)
            legend.get_frame().set_facecolor('white')
            legend.get_frame().set_edgecolor(SPINE_SOFT)
            legend.get_frame().set_linewidth(1.0)

            _save_display_close(
                fig,
                CASE_FEATURE_PNG,
                'Most frequently recurring localized features, colored by feature tier.'
            )
        else:
            display(Markdown('No feature rows were available for the recurring-features figure.'))



## Six-Cell Paper Storyboard

If you want the shortest main-paper path, these are the six notebook outputs to use first. Together they cover the full paper story without forcing a reviewer to dig through the longer notebook.

1. **Quick paper-safe metrics.** Use this for the abstract and opening claims.
2. **Mixed vs full deployment summary.** Use this to explain why the paper leads with the mixed profile and treats the full profile as an upper bound.
3. **Diagnosis, localization, and hardware scorecard.** Use this for the diagnosis story: exact Top-2, selective Top-2 at coverage, mechanism coverage, first abnormal windows, and likely M2 Pro subsystem.
4. **Main DICE performance.** Use this for the core monitoring table and ROC/PR figures.
5. **Reliability without per-workload tuning.** Use this for the conformal-calibration and alert-control story.
6. **Cross-workload transfer.** Use this for portability under workload shift.

Companion figures that should stay visible in the main results flow are the scenario timing dashboard and the case-by-case onset/hotspot gallery, because they show when the first anomaly windows appear and how the evidence evolves.

After those six, the most useful appendix add-ons are the design-space tables, tuning sweeps, feature budgets, uncertainty intervals, and time-window ablations.


In [ ]:
# Plain-language: This cell lists the exact notebook outputs that best match the paper storyboard.
display(Markdown('### Six-cell paper storyboard'))
storyboard_df = pd.DataFrame([
    {'Draft slot': 1, 'Notebook section': 'Quick Paper-Safe Metrics', 'Primary output': 'Abstract-ready metrics table', 'Use in paper': 'Abstract and opening claims'},
    {'Draft slot': 2, 'Notebook section': 'Mixed vs Full Deployment Summary', 'Primary output': 'Profile comparison table', 'Use in paper': 'Mixed deployment claim versus full upper bound'},
    {'Draft slot': 3, 'Notebook section': 'Diagnosis, Localization, and Hardware Scorecard', 'Primary output': 'Industry diagnosis and localization scorecard', 'Use in paper': 'Diagnosis shortlist quality, mechanism coverage, and hardware-facing explanation'},
    {'Draft slot': 4, 'Notebook section': '2. Main DICE Performance', 'Primary output': 'Main monitoring summary table plus ROC-AUC/AUC-PR figure', 'Use in paper': 'Core anomaly detection result'},
    {'Draft slot': 5, 'Notebook section': '3. Reliability Without Per-Workload Tuning', 'Primary output': 'Reliability summary table and calibrated-alert figure', 'Use in paper': 'Conformal trustworthiness and controlled alerting'},
    {'Draft slot': 6, 'Notebook section': '5. Cross-Workload Transfer and Drift Proxy', 'Primary output': 'Cross-workload transfer table and figure', 'Use in paper': 'Generalization under workload shift'},
])
display(storyboard_df)
display(Markdown('#### Companion localization figures for the main results'))
companion_df = pd.DataFrame([
    {'Figure purpose': 'Scenario timing dashboard', 'Notebook section': 'Scenario diagnosis and localization figures', 'Why keep it': 'Shows which anomaly types are identified well and when the first abnormal windows appear.'},
    {'Figure purpose': 'Case-onset gallery', 'Notebook section': 'Case-by-case onset and hotspot figures', 'Why keep it': 'Shows the first abnormal window and the next several abnormal windows for each workload-stressor case.'},
])
display(companion_df)
appendix_addons = pd.DataFrame([
    {'Appendix item': 'Design-space and feature budgets', 'Notebook section': '6. DICE-Specific Design-Space Evaluation'},
    {'Appendix item': 'Tuning sweeps', 'Notebook section': 'Tuning sweep diagnostics across gains, blocks, alpha, and persistence'},
    {'Appendix item': 'Case galleries and hotspot views', 'Notebook section': 'Case-by-case onset and hotspot figures'},
    {'Appendix item': 'Uncertainty intervals', 'Notebook section': '11. Uncertainty and Confidence Intervals'},
    {'Appendix item': 'Time-window ablations', 'Notebook section': 'Appendix analysis outside the main six-cell path'},
])
display(Markdown('#### Best appendix add-ons'))
display(appendix_addons)


## 1. Experimental Setup, Released Data Inventory, and Hardware Context

This section summarizes the released dataset artifacts that support the paper and the MacBook Pro hardware context used for subsystem-level localization.

The early AF-index plots are descriptive data views from the released traces, not the full DICE detector score.


In [ ]:
# Plain-language: This cell loads the released-data setup tables so readers can see what platform, traces, and labels were used.
overall = pd.read_csv(OUT / 'table_overall_metrics.csv')
stressor = pd.read_csv(OUT / 'table_stressor_metrics.csv')
workload = pd.read_csv(OUT / 'table_workload_summary.csv')
features = pd.read_csv(OUT / 'table_feature_inventory.csv')
quality = pd.read_csv(OUT / 'table_case_quality.csv')

setup_snapshot = features[['tier_name', 'n_features_common', 'n_features_union']].copy()
setup_snapshot = setup_snapshot.rename(
    columns={
        'tier_name': 'Tier',
        'n_features_common': 'Common features',
        'n_features_union': 'Union features',
    }
)

workload_snapshot = workload[['tier_name', 'workload', 'nominal_score', 'anomaly_median_score', 'anomaly_nominal_ratio']].copy()
workload_snapshot = workload_snapshot.rename(
    columns={
        'tier_name': 'Tier',
        'workload': 'Workload',
        'nominal_score': 'Nominal AF index',
        'anomaly_median_score': 'Median anomaly AF index',
        'anomaly_nominal_ratio': 'Anomaly/nominal ratio',
    }
)

quality_snapshot = quality[['tier', 'case_id', 'rows_5hz', 'numeric_cols', 'nan_fraction']].head(8).copy()
quality_snapshot = quality_snapshot.rename(
    columns={
        'tier': 'Tier',
        'case_id': 'Case',
        'rows_5hz': 'Rows @5Hz',
        'numeric_cols': 'Numeric cols',
        'nan_fraction': 'NaN fraction',
    }
)

display(Markdown('### Released data snapshot'))
display(setup_snapshot)

display(Markdown('### Workload-level AF-index context'))
display(workload_snapshot.round(4))

display(Markdown('### Case-quality spot check'))
display(quality_snapshot.round(4))


## 1B. Profile-Aware Result Helpers


## 1C. MacBook Pro Hardware Context

This section records the practical hardware view used to interpret DICE localization on the collection host.

- The notebook treats localization as **subsystem-level localization** on the `M2 Pro` MacBook Pro, not die-photo-level fault pinpointing.
- The mapping below connects DICE features and stressors to visible hardware blocks such as the CPU complex, integrated GPU, Neural Engine, unified memory, storage, and runtime/software stack.
- Because this laptop is also driving an external Dell monitor, GPU/display-related telemetry should be interpreted with that steady display load in mind.


In [ ]:
# Plain-language: This cell records the visible M2 Pro hardware blocks so later localization tables can point features and stressors to real laptop subsystems.
display(Markdown('### MacBook Pro hardware context'))

MACBOOK_PROFILE = {
    'model': 'MacBook Pro (16-inch, 2023)',
    'chip': 'Apple M2 Pro',
    'cpu': '12-core CPU (8 performance + 4 efficiency)',
    'gpu': '19-core integrated GPU',
    'neural_engine': '16-core Neural Engine',
    'memory': '16 GB unified memory',
    'memory_bandwidth': '200 GB/s unified-memory bandwidth',
    'storage': '512 GB SSD class (~494 GB usable shown on this machine)',
    'macos': 'macOS Tahoe 26.3.1',
    'display_setup': 'Built-in 16-inch Liquid Retina XDR plus one external Dell 1920x1080 monitor',
    'display_note': 'External display activity can raise the steady GPU/display baseline seen by Tier-1 and some OS-visible graphics behavior.',
    'localization_scope': 'DICE can localize to visible subsystems such as CPU, GPU/display, ANE, unified memory, swap/storage, and runtime scheduling.',
}

MACBOOK_PROFILE_DF = pd.DataFrame([
    {'Field': 'Model', 'Value': MACBOOK_PROFILE['model']},
    {'Field': 'Chip', 'Value': MACBOOK_PROFILE['chip']},
    {'Field': 'CPU view', 'Value': MACBOOK_PROFILE['cpu']},
    {'Field': 'GPU view', 'Value': MACBOOK_PROFILE['gpu']},
    {'Field': 'Neural Engine', 'Value': MACBOOK_PROFILE['neural_engine']},
    {'Field': 'Memory', 'Value': MACBOOK_PROFILE['memory']},
    {'Field': 'Memory bandwidth', 'Value': MACBOOK_PROFILE['memory_bandwidth']},
    {'Field': 'Storage', 'Value': MACBOOK_PROFILE['storage']},
    {'Field': 'Display context', 'Value': MACBOOK_PROFILE['display_setup']},
    {'Field': 'Localization scope', 'Value': MACBOOK_PROFILE['localization_scope']},
])

display(MACBOOK_PROFILE_DF)

display(Markdown(
    '> Interpretation note: on this machine, DICE can point most reliably to **which subsystem looks abnormal**. '
    'It cannot honestly name an exact transistor path, cache bank instance, or physical defect site from host-visible telemetry alone.'
))

STRESSOR_HARDWARE_MAP = {
    'ATOMIC': {
        'likely_subsystem': 'CPU cores and synchronization path',
        'likely_hardware_block': 'CPU cluster, shared caches, and cache-coherence / lock-contention path',
        'layperson_summary': 'Many workers are fighting over shared data, so CPU-side contention becomes visible.',
    },
    'BRANCH': {
        'likely_subsystem': 'CPU front-end and control flow',
        'likely_hardware_block': 'CPU fetch / decode / branch-prediction path on the M2 Pro cores',
        'layperson_summary': 'The anomaly looks most like unstable instruction-flow behavior on the CPU side.',
    },
    'CACHE': {
        'likely_subsystem': 'CPU cache and memory hierarchy',
        'likely_hardware_block': 'CPU cache hierarchy and its interface to the unified-memory system',
        'layperson_summary': 'The anomaly looks most like cache-pressure behavior rather than pure software noise.',
    },
    'MEMBW': {
        'likely_subsystem': 'Unified memory and memory fabric',
        'likely_hardware_block': 'Unified-memory bandwidth path, memory controller, and possible swap spillover',
        'layperson_summary': 'The anomaly looks most like the system pushing hard on shared memory bandwidth.',
    },
    'TLB': {
        'likely_subsystem': 'Address translation and MMU path',
        'likely_hardware_block': 'CPU-side translation structures and page-walk behavior',
        'layperson_summary': 'The anomaly looks most like address-translation pressure rather than raw arithmetic load.',
    },
}

STRESSOR_HARDWARE_DF = pd.DataFrame([
    {'stressor': stressor, **payload}
    for stressor, payload in STRESSOR_HARDWARE_MAP.items()
]).sort_values('stressor').reset_index(drop=True)

display(Markdown('#### Stressor-to-hardware interpretation on this M2 Pro'))
display(STRESSOR_HARDWARE_DF[['stressor', 'likely_subsystem', 'likely_hardware_block', 'layperson_summary']])


def map_stressor_to_mbp_hardware(stressor: str) -> dict:
    return dict(STRESSOR_HARDWARE_MAP.get(str(stressor), {
        'likely_subsystem': 'Mixed system behavior',
        'likely_hardware_block': 'Host-visible interaction between software and hardware',
        'layperson_summary': 'The anomaly is visible at the system level but not cleanly tied to a single hardware mechanism.',
    }))


def map_feature_to_mbp_hardware(feature_name: str) -> dict:
    name = str(feature_name)
    lname = name.lower()
    tier = name.split(':', 1)[0] if ':' in name else 'unknown'

    if any(token in lname for token in ['ane_power', 'ane_']):
        return {
            'likely_subsystem': 'Neural Engine',
            'likely_hardware_block': '16-core Neural Engine power domain',
            'hardware_confidence': 'medium',
            'layperson_summary': 'This feature points to the Neural Engine or a nearby power proxy.',
            'platform_note': 'Treat as a strong subsystem clue, but not proof that the ANE is the sole root cause unless the workload uses ANE-backed operations.',
        }
    if any(token in lname for token in ['gpu_power', 'gpu_usage', 'gpu_avg_freq', 'gpu_temp', 'gpu_residency', 'gpu_']):
        return {
            'likely_subsystem': 'GPU / display path',
            'likely_hardware_block': '19-core integrated GPU plus display engine',
            'hardware_confidence': 'medium',
            'layperson_summary': 'This feature points to graphics/display-side activity on the SoC.',
            'platform_note': 'An attached external display can raise the baseline GPU/display load even without an anomaly.',
        }
    if any(token in lname for token in ['cpu_power', 'cpu_usage', 'cpu_avg_freq', 'cpu_temp', 'cpu_residency', 'cpu_times', 'interrupt', 'ctx_switch', 'syscall', 'load', 'core_id', 'wakeups']):
        return {
            'likely_subsystem': 'CPU complex',
            'likely_hardware_block': '12-core CPU cluster, scheduler-visible activity, and nearby power/thermal control',
            'hardware_confidence': 'high',
            'layperson_summary': 'This feature points to the CPU side of the M2 Pro.',
            'platform_note': 'This is one of the clearest subsystem-level mappings in the notebook.',
        }
    if any(token in lname for token in ['mem_', 'memory_', 'swap_', 'pageins', 'pageouts']):
        return {
            'likely_subsystem': 'Unified memory and swap path',
            'likely_hardware_block': '16 GB unified memory, memory fabric, and SSD-backed swap path',
            'hardware_confidence': 'high',
            'layperson_summary': 'This feature points to memory pressure or spillover into swap on the laptop.',
            'platform_note': 'On this 16 GB unified-memory system, these features are especially meaningful for memory-bound stressors.',
        }
    if any(token in lname for token in ['disk_', 'storage', 'ssd', 'fs_']):
        return {
            'likely_subsystem': 'Storage path',
            'likely_hardware_block': 'Internal SSD and OS storage stack',
            'hardware_confidence': 'medium',
            'layperson_summary': 'This feature points to storage-side pressure rather than a pure compute effect.',
            'platform_note': 'Storage features can also reflect swap side effects from memory pressure.',
        }
    if any(token in lname for token in ['net_', 'network', 'recv_', 'sent_']):
        return {
            'likely_subsystem': 'Network / software I/O path',
            'likely_hardware_block': 'Host-visible networking stack and workload I/O behavior',
            'hardware_confidence': 'medium',
            'layperson_summary': 'This feature points to network or software I/O behavior that changes when the anomaly starts.',
            'platform_note': 'This is often context about workload behavior, not a direct silicon fault site.',
        }
    if any(token in lname for token in ['process', 'thread', 'samples_per_bucket', 'running_fraction', 'weight_ns', 'sentinel_count']):
        return {
            'likely_subsystem': 'Runtime scheduling / profiler path',
            'likely_hardware_block': 'Software execution structure seen through Instruments rather than a single physical block',
            'hardware_confidence': 'medium',
            'layperson_summary': 'This feature points to how work is being scheduled and executed over time.',
            'platform_note': 'Tier-2 features are strong for explanation, but they are still behavioral rather than transistor-level localization.',
        }
    if 'thermal' in lname or 'temp' in lname:
        return {
            'likely_subsystem': 'Thermal management path',
            'likely_hardware_block': 'On-chip thermal sensors and OS-visible thermal-control state',
            'hardware_confidence': 'medium',
            'layperson_summary': 'This feature points to the system heating response to the anomaly.',
            'platform_note': 'Thermal signals are often secondary effects that confirm sustained stress.',
        }
    if tier == 'tier0':
        return {
            'likely_subsystem': 'OS-visible system state',
            'likely_hardware_block': 'High-level software/hardware interaction visible from user space',
            'hardware_confidence': 'low',
            'layperson_summary': 'This is a high-level host signal that helps narrow down the subsystem but not an exact block.',
            'platform_note': 'Useful for practical deployment, but not a circuit-level probe.',
        }
    if tier == 'tier1_alt':
        return {
            'likely_subsystem': 'Power / thermal proxy path',
            'likely_hardware_block': 'OS-mediated Apple Silicon power, frequency, and thermal proxies',
            'hardware_confidence': 'medium',
            'layperson_summary': 'This feature is a hardware-adjacent proxy that helps identify which subsystem is changing.',
            'platform_note': 'Good for field deployment because it stays lightweight and host-visible.',
        }
    if tier == 'tier2':
        return {
            'likely_subsystem': 'Profiler-visible runtime behavior',
            'likely_hardware_block': 'Instruments trace evidence tied to execution structure',
            'hardware_confidence': 'medium',
            'layperson_summary': 'This feature comes from a deeper runtime trace rather than a single on-chip sensor.',
            'platform_note': 'Useful for diagnosis and reviewer explanation.',
        }
    return {
        'likely_subsystem': 'Mixed host-visible behavior',
        'likely_hardware_block': 'General system interaction visible to the host OS',
        'hardware_confidence': 'low',
        'layperson_summary': 'This feature is useful evidence, but it does not map cleanly to one hardware block.',
        'platform_note': 'Treat this as supporting context rather than a precise physical location.',
    }


In [ ]:
# Plain-language: This cell defines helper functions that switch cleanly between the mixed and full result bundles.

display(Markdown('### Profile-aware result helpers'))

FINAL_CONFIG = 'tier0_tier1_tier2'
CONFIG_ORDER = ['tier0', 'tier0_tier1', 'tier0_tier1_tier2']
CFG_LABEL = {
    'tier0': 'Tier-0',
    'tier0_tier1': 'Tier-0/1',
    'tier0_tier1_tier2': 'Tier-0/1/2',
}
PROFILE_LABEL = {'mixed': 'Mixed', 'full': 'Full'}
PROFILE_COLOR = {'mixed': '#355C7D', 'full': '#C44E52'}
PROFILE_MARKER = {'mixed': 'o', 'full': 's'}
PROFILE_ORDER = [p for p in ['mixed', 'full'] if p in FEATURE_PROFILES_TO_SWEEP]
PRO_FIG_DPI = 300
DIAGNOSIS_CANDIDATES = [
    ('feature-level (diagnosis-weighted)', 'stressor_feature_diagnosis_metrics.csv'),
    ('mechanism-level', 'stressor_diagnosis_metrics.csv'),
    ('hierarchical (confidence-gated)', 'stressor_hierarchical_diagnosis_metrics.csv'),
]
DIAGNOSIS_RESULT_COLUMNS = [
    'top1_acc',
    'top2_acc',
    'balanced_acc',
    'macro_f1',
    'mean_margin_to_second',
    'median_margin_to_second',
    'family_acc',
    'coverage',
    'abstain_rate',
    'selective_top1_acc',
    'selective_top2_acc',
    'selective_balanced_acc',
    'selective_macro_f1',
    'mean_family_margin',
    'median_family_margin',
]

COMPARISON_DIR = OUT_PAPER / 'comparison'
COMPARISON_FIG = COMPARISON_DIR / 'figures'
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_FIG.mkdir(parents=True, exist_ok=True)

PAPER_FULL = OUT_PAPER / DISPLAY_FEATURE_PROFILE
PAPER_FULL.mkdir(parents=True, exist_ok=True)
PAPER_FIG = PAPER_FULL / 'figures'
PAPER_FIG.mkdir(parents=True, exist_ok=True)
APPENDIX_FULL = OUT_APPENDIX / DISPLAY_FEATURE_PROFILE
APPENDIX_FULL.mkdir(parents=True, exist_ok=True)


def profile_paper_dir(profile: str) -> Path:
    path = OUT_PAPER / profile
    path.mkdir(parents=True, exist_ok=True)
    return path


def profile_fig_dir(profile: str) -> Path:
    path = profile_paper_dir(profile) / 'figures'
    path.mkdir(parents=True, exist_ok=True)
    return path


def profile_appendix_dir(profile: str) -> Path:
    path = OUT_APPENDIX / profile
    path.mkdir(parents=True, exist_ok=True)
    return path


def profile_global_dir(profile: str) -> Path:
    return PROFILE_OUT_DIRS[profile]['global']


def profile_holdout_dir(profile: str) -> Path:
    return PROFILE_OUT_DIRS[profile]['holdout']


def profile_scope_dir(profile: str, scope: str) -> Path:
    if scope == 'global':
        return profile_global_dir(profile)
    if scope == 'holdout':
        return profile_holdout_dir(profile)
    if scope == 'paper':
        return profile_paper_dir(profile)
    if scope == 'appendix':
        return profile_appendix_dir(profile)
    raise ValueError(f'Unsupported scope: {scope}')


def profile_operating_point(profile: str) -> dict[str, object]:
    invocation_path = profile_global_dir(profile) / 'notebook_invocation.json'
    if invocation_path.exists():
        return json.loads(invocation_path.read_text())
    return {}


def profile_requirements(profile: str, required_global=(), required_holdout=(), required_paper=(), required_appendix=()):
    missing = []
    for scope, names in [
        ('global', required_global),
        ('holdout', required_holdout),
        ('paper', required_paper),
        ('appendix', required_appendix),
    ]:
        for name in names:
            path = profile_scope_dir(profile, scope) / name
            if not path.exists():
                missing.append(path)
    return missing


def available_profiles(required_global=(), required_holdout=(), required_paper=(), required_appendix=()):
    ready = []
    missing = {}
    for profile in PROFILE_ORDER:
        miss = profile_requirements(
            profile,
            required_global=required_global,
            required_holdout=required_holdout,
            required_paper=required_paper,
            required_appendix=required_appendix,
        )
        if miss:
            missing[profile] = miss
        else:
            ready.append(profile)
    return ready, missing


def read_profile_csv(profile: str, scope: str, name: str) -> pd.DataFrame:
    return pd.read_csv(profile_scope_dir(profile, scope) / name)


def read_profile_best_diagnosis(
    profile: str,
    scope: str = 'global',
    final_config: str = FINAL_CONFIG,
) -> tuple[pd.DataFrame, str, Path | None]:
    base = profile_scope_dir(profile, scope)

    for source_name, filename in DIAGNOSIS_CANDIDATES:
        path = base / filename
        if not path.exists():
            continue
        diag = pd.read_csv(path).copy()
        for col in DIAGNOSIS_RESULT_COLUMNS:
            if col not in diag.columns:
                diag[col] = np.nan
        ref = diag.loc[diag['config'].astype(str) == str(final_config)]
        ref_row = ref.iloc[0] if not ref.empty else (diag.iloc[0] if not diag.empty else None)
        if ref_row is None:
            continue
        return diag, source_name, path

    return pd.DataFrame(), 'unavailable', None


def read_profile_abstain_sweep(
    profile: str,
    scope: str = 'global',
) -> tuple[pd.DataFrame, Path | None]:
    path = profile_scope_dir(profile, scope) / 'stressor_hierarchical_abstain_sweep.csv'
    if not path.exists():
        return pd.DataFrame(), None
    sweep = pd.read_csv(path).copy()
    for col in [
        'coverage',
        'abstain_rate',
        'selective_top1_acc',
        'selective_top2_acc',
        'selective_balanced_acc',
        'selective_macro_f1',
    ]:
        if col not in sweep.columns:
            sweep[col] = np.nan
    return sweep, path


def profile_has_any_diagnosis(profile: str, scope: str = 'global') -> bool:
    base = profile_scope_dir(profile, scope)
    return any((base / filename).exists() for _, filename in DIAGNOSIS_CANDIDATES)


def add_profile_and_config_labels(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'feature_profile' in out.columns:
        out['profile_label'] = out['feature_profile'].map(PROFILE_LABEL).fillna(out['feature_profile'])
    if 'config' in out.columns:
        out['config_label'] = out['config'].map(CFG_LABEL).fillna(out['config'])
    return out


def sort_profile_config(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'feature_profile' in out.columns:
        out['feature_profile'] = pd.Categorical(out['feature_profile'], categories=PROFILE_ORDER, ordered=True)
    if 'config' in out.columns:
        out['config'] = pd.Categorical(out['config'], categories=CONFIG_ORDER, ordered=True)
    order_cols = [col for col in ['feature_profile', 'config'] if col in out.columns]
    if order_cols:
        out = out.sort_values(order_cols).reset_index(drop=True)
    return out


def markdown_missing_profiles(missing: dict[str, list[Path]]) -> str:
    if not missing:
        return ''
    lines = ['Profiles still missing result bundles:']
    for profile, paths in missing.items():
        shown = ', '.join(f'`{p.name}`' for p in paths[:2])
        if len(paths) > 2:
            shown += ', ...'
        lines.append(f"- {PROFILE_LABEL.get(profile, profile)}: {shown}")
    return '\n'.join(lines)


def feature_count_for_profile(profile: str, config: str, default_value=np.nan) -> float:
    overall_path = profile_global_dir(profile) / 'overall_metrics.csv'
    if not overall_path.exists():
        return float(default_value)
    overall = pd.read_csv(overall_path)
    row = overall.loc[overall['config'] == config]
    if row.empty:
        return float(default_value)
    return float(row['n_features'].iloc[0])


## 2. Main DICE Performance

This section reports the paper-facing detector results. Unless noted otherwise, the headline story follows the **mixed** deployment profile (Tier-0 plus the compact Tier-1/Tier-2 subsets), because that is the profile described in the draft. The **full** profile remains in the notebook as an upper-bound comparison later in the paper bundle.


In [ ]:
# Plain-language: This cell builds the main monitoring tables that show how well DICE separates normal and abnormal runs.

display(Markdown('### Monitoring summary tables'))

required_global = ['overall_metrics.csv', 'sequential_metrics.csv', 'case_predictions.csv']
ready_profiles, missing_profiles = available_profiles(required_global=required_global)

main_monitoring_csv = COMPARISON_DIR / 'main_monitoring_profile_summary.csv'
final_monitoring_csv = COMPARISON_DIR / 'final_head_monitoring_profile_summary.csv'
score_dist_csv = COMPARISON_DIR / 'final_head_score_distribution_profiles.csv'


def select_with_fallback(df, columns, aliases=None, fill_value=np.nan):
    out = df.copy()
    aliases = aliases or {}

    for target, candidates in aliases.items():
        if target not in out.columns:
            for alt in candidates:
                if alt in out.columns:
                    out[target] = out[alt]
                    break

    for col in columns:
        if col not in out.columns:
            out[col] = fill_value

    return out[columns].copy()


if not ready_profiles:
    display(Markdown('No global result bundles are ready yet. Run **Run End-to-End** first.'))
    if missing_profiles:
        display(Markdown(markdown_missing_profiles(missing_profiles)))
else:
    merged_rows = []
    score_rows = []
    missing_notes = []
    diagnosis_notes = []

    for profile in ready_profiles:
        overall = read_profile_csv(profile, 'global', 'overall_metrics.csv')
        sequential = read_profile_csv(profile, 'global', 'sequential_metrics.csv')
        diagnosis, diagnosis_source, diagnosis_path = read_profile_best_diagnosis(profile)
        if diagnosis.empty:
            diagnosis = overall[['config']].copy()
            for col in ['top1_acc', 'top2_acc', 'macro_f1', 'balanced_acc', 'mean_margin_to_second']:
                diagnosis[col] = np.nan
        if 'benign_run_keep_rate' not in sequential.columns and 'benign_run_alert_rate' in sequential.columns:
            sequential = sequential.copy()
            sequential['benign_run_keep_rate'] = 1.0 - sequential['benign_run_alert_rate']
        case_pred = read_profile_csv(profile, 'global', 'case_predictions.csv')

        overall_sel = select_with_fallback(
            overall,
            ['config', 'n_features', 'roc_auc', 'pr_auc', 'roc_auc_wc', 'pr_auc_wc', 'fit_eval_seconds'],
            aliases={
                'fit_eval_seconds': ['runtime_s_profile', 'runtime_seconds', 'eval_seconds']
            },
        )
        sequential_sel = select_with_fallback(
            sequential,
            ['config', 'benign_run_alert_rate', 'benign_run_keep_rate', 'anomaly_detect_rate', 'median_time_to_detect_s'],
        )
        diagnosis_sel = select_with_fallback(
            diagnosis,
            ['config', 'top1_acc', 'top2_acc', 'macro_f1', 'balanced_acc', 'mean_margin_to_second'],
        )

        if 'fit_eval_seconds' not in overall.columns and overall_sel['fit_eval_seconds'].isna().all():
            missing_notes.append(
                f'- `{PROFILE_LABEL.get(profile, profile)}` overall metrics do not include `fit_eval_seconds`; runtime is omitted from the display table.'
            )

        diag_note = diagnosis_source
        if diagnosis_path is not None:
            diag_note = f"{diagnosis_source} (`{diagnosis_path.name}`)"
        diagnosis_notes.append(f"- `{PROFILE_LABEL.get(profile, profile)}` monitoring summary uses **{diag_note}**.")

        merged = (
            overall_sel
            .merge(sequential_sel, on='config', how='left')
            .merge(diagnosis_sel, on='config', how='left')
        )
        merged['feature_profile'] = profile
        merged['diagnosis_source'] = diagnosis_source
        merged_rows.append(add_profile_and_config_labels(merged))

        final_scores = case_pred[case_pred['config'] == FINAL_CONFIG].copy()
        if not final_scores.empty:
            final_scores['feature_profile'] = profile
            final_scores['profile_label'] = PROFILE_LABEL.get(profile, profile)
            final_scores['label_name'] = final_scores['label'].map({0: 'Benign', 1: 'Anomaly'})
            score_rows.append(
                select_with_fallback(
                    final_scores,
                    ['feature_profile', 'profile_label', 'case_id', 'label', 'label_name', 'run_score', 'run_score_wc'],
                )
            )

    monitoring_summary = sort_profile_config(pd.concat(merged_rows, ignore_index=True))
    final_monitoring = monitoring_summary[
        monitoring_summary['config'].astype(str) == FINAL_CONFIG
    ].reset_index(drop=True)
    score_distribution = pd.concat(score_rows, ignore_index=True) if score_rows else pd.DataFrame()

    monitoring_summary.to_csv(main_monitoring_csv, index=False)
    final_monitoring.to_csv(final_monitoring_csv, index=False)
    if not score_distribution.empty:
        score_distribution.to_csv(score_dist_csv, index=False)

    runtime_available = monitoring_summary['fit_eval_seconds'].notna().any()

    display_cols = [
        'profile_label', 'config_label', 'n_features', 'roc_auc', 'pr_auc', 'roc_auc_wc', 'pr_auc_wc',
        'benign_run_keep_rate', 'benign_run_alert_rate', 'anomaly_detect_rate', 'median_time_to_detect_s',
        'top1_acc', 'top2_acc', 'macro_f1', 'balanced_acc'
    ]
    final_cols = [
        'profile_label', 'n_features', 'roc_auc', 'pr_auc', 'roc_auc_wc', 'pr_auc_wc',
        'benign_run_keep_rate', 'benign_run_alert_rate', 'anomaly_detect_rate', 'median_time_to_detect_s',
        'top1_acc', 'top2_acc', 'macro_f1', 'balanced_acc'
    ]

    if runtime_available:
        display_cols.insert(7, 'fit_eval_seconds')
        final_cols.insert(6, 'fit_eval_seconds')

    display(Markdown('#### Final-head comparison'))
    display(final_monitoring[final_cols].round(4))

    display(Markdown('#### Per-config comparison'))
    display(monitoring_summary[display_cols].round(4))

    if diagnosis_notes:
        display(Markdown('**Diagnosis source**\n' + '\n'.join(diagnosis_notes)))

    if missing_notes and not runtime_available:
        display(Markdown('**Notes**\n' + '\n'.join(sorted(set(missing_notes)))))

    if missing_profiles:
        display(Markdown(markdown_missing_profiles(missing_profiles)))


### Benign vs anomaly score separation

This figure shows the final-head workload-conditioned run scores for benign and anomalous cases. It works better in the results section than in the setup section because it is a direct outcome figure.


In [ ]:
# Plain-language: This cell shows one final-head metrics table only.
display(Markdown('### Final-head monitoring metrics'))

main_monitoring_csv = COMPARISON_DIR / 'main_monitoring_profile_summary.csv'
main_monitoring_table_csv = COMPARISON_DIR / 'final_head_monitoring_metrics_table.csv'

CFG_LABEL_LOCAL = globals().get('CFG_LABEL', {
    'tier0': 'Tier-0',
    'tier0_tier1': 'Tier-0/1',
    'tier0_tier1_tier2': 'Tier-0/1/2',
})
PROFILE_LABEL_LOCAL = globals().get('PROFILE_LABEL', {'mixed': 'Mixed', 'full': 'Full'})
PROFILE_ORDER_LOCAL = list(globals().get('PROFILE_ORDER', ['mixed', 'full']))

if not main_monitoring_csv.exists():
    display(Markdown('Run the previous monitoring-summary cell first so the monitoring CSV exists.'))
else:
    monitoring = pd.read_csv(main_monitoring_csv)
    monitoring['config'] = monitoring['config'].astype(str)
    monitoring['feature_profile'] = monitoring['feature_profile'].astype(str)

    final_cfg = 'tier0_tier1_tier2' if 'tier0_tier1_tier2' in set(monitoring['config']) else str(monitoring['config'].dropna().iloc[-1])

    final_df = monitoring[monitoring['config'] == final_cfg].copy()
    if final_df.empty:
        display(Markdown('No final-head rows were found in the monitoring summary.'))
    else:
        final_df['profile_sort'] = final_df['feature_profile'].map({
            profile: idx for idx, profile in enumerate(PROFILE_ORDER_LOCAL)
        }).fillna(999)
        final_df = final_df.sort_values(['profile_sort', 'feature_profile']).reset_index(drop=True)

        table_df = pd.DataFrame({
            'Profile': final_df['feature_profile'].map(PROFILE_LABEL_LOCAL).fillna(final_df['feature_profile']),
            'Head': final_df['config'].map(CFG_LABEL_LOCAL).fillna(final_df['config']),
            'Features': pd.to_numeric(final_df.get('n_features', np.nan), errors='coerce'),
            'AUC-PR': pd.to_numeric(final_df.get('pr_auc', np.nan), errors='coerce'),
            'ROC-AUC': pd.to_numeric(final_df.get('roc_auc', np.nan), errors='coerce'),
            'WC AUC-PR': pd.to_numeric(final_df.get('pr_auc_wc', np.nan), errors='coerce'),
            'WC ROC-AUC': pd.to_numeric(final_df.get('roc_auc_wc', np.nan), errors='coerce'),
            'Detection Rate': pd.to_numeric(final_df.get('anomaly_detect_rate', np.nan), errors='coerce'),
            'Benign Alert Rate': pd.to_numeric(final_df.get('benign_run_alert_rate', np.nan), errors='coerce'),
            'Median TTD (s)': pd.to_numeric(final_df.get('median_time_to_detect_s', np.nan), errors='coerce'),
        })

        table_df.to_csv(main_monitoring_table_csv, index=False)

        display(Markdown(f'Final detector head: **{CFG_LABEL_LOCAL.get(final_cfg, final_cfg)}**'))
        display(table_df.round({
            'Features': 0,
            'AUC-PR': 4,
            'ROC-AUC': 4,
            'WC AUC-PR': 4,
            'WC ROC-AUC': 4,
            'Detection Rate': 3,
            'Benign Alert Rate': 3,
            'Median TTD (s)': 1,
        }))

        display(Markdown(f'Wrote: `{main_monitoring_table_csv}`'))



## 3. Reliability Without Per-Workload Tuning

This section asks whether one shared alert rule can work across different workloads without hand-tuning the detector for each application.

The draft's theoretical claim is a **block-level split-conformal guarantee**. Let $\{s_i^{\mathrm{cal}}\}_{i=1}^{n}$ be benign calibration block scores, and let $S^{\mathrm{test}}$ be the score of a future benign test block. With the split-conformal threshold
\[
\tau = s^{\mathrm{cal}}_{(k)}, \qquad k = \left\lceil (n+1)(1-\alpha) \right\rceil,
\]
benign exchangeability between the calibration scores and $S^{\mathrm{test}}$ implies
\[
\Pr\!\left(S^{\mathrm{test}} > \tau\right) \le \alpha.
\]
Equivalently, if the conformal $p$-value is
\[
p_{\mathrm{test}} = \frac{1 + \sum_{i=1}^{n} \mathbb{I}[s_i^{\mathrm{cal}} \ge S^{\mathrm{test}}]}{n+1},
\]
then
\[
\Pr\!\left(p_{\mathrm{test}} \le \alpha\right) \le \alpha.
\]
Thus, under benign exchangeability, DICE provides finite-sample **marginal control of the benign block-level false-alarm rate**.

One terminology difference matters in the tables below:
- **Block level** means short time windows inside a run.
- **Run level** means the whole experiment from start to finish.

A method can have a low false-alarm rate at the block level but still raise more run-level alerts, because a long run contains many blocks and the alerts are combined over time through the persistence rule.

For the deployment-focused operational scorecards, the notebook now also reports a **phase-aware benign guard** for `VIDEO_SW`. This guard does **not** change the base anomaly ranking metrics; it only tightens the final run-alert decision when a late benign video phase matches the learned nominal video pattern.


In [ ]:
# Plain-language: This cell summarizes how stable the calibrated alerting behavior remains without workload-specific retuning.
display(Markdown('### Reliability summary tables'))

ready_profiles, missing_profiles = available_profiles(required_global=['case_predictions.csv'])
reliability_csv = COMPARISON_DIR / 'conformal_reliability_profiles.csv'
reliability_workload_csv = COMPARISON_DIR / 'conformal_reliability_by_workload_profiles.csv'

if not ready_profiles:
    display(Markdown('No case-level prediction bundles are ready yet. Run **Run End-to-End** first.'))
    if missing_profiles:
        display(Markdown(markdown_missing_profiles(missing_profiles)))
else:
    reliability_rows = []
    workload_rows = []
    for profile in ready_profiles:
        case_pred = read_profile_csv(profile, 'global', 'case_predictions.csv')
        target_alpha = float(profile_operating_point(profile).get('alpha', 0.05))

        profile_rows = []
        for cfg, block in case_pred.groupby('config', sort=False):
            benign = block[block['label'] == 0].copy()
            anomaly = block[block['label'] == 1].copy()
            profile_rows.append(
                {
                    'feature_profile': profile,
                    'config': cfg,
                    'target_alpha': target_alpha,
                    'benign_block_false_alarm_rate': benign['n_block_alerts'].sum() / benign['n_blocks'].sum(),
                    'benign_persist_false_alarm_rate': benign['n_persist_alerts'].sum() / benign['n_blocks'].sum(),
                    'benign_run_false_alarm_rate': benign['run_alert'].mean(),
                    'benign_run_keep_rate': 1.0 - benign['run_alert'].mean(),
                    'anomaly_run_detection_rate': anomaly['run_alert'].mean(),
                    'median_anomaly_time_to_detect_s': anomaly.loc[anomaly['run_alert'] == 1, 'time_to_detect_s'].median(),
                }
            )

        profile_rel = add_profile_and_config_labels(pd.DataFrame(profile_rows))
        profile_rel = sort_profile_config(profile_rel)
        profile_rel.to_csv(profile_paper_dir(profile) / 'conformal_reliability_summary.csv', index=False)
        reliability_rows.append(profile_rel)

        by_workload = (
            case_pred[case_pred['label'] == 0]
            .groupby(['config', 'workload'], sort=False)
            .apply(
                lambda x: pd.Series(
                    {
                        'target_alpha': target_alpha,
                        'benign_block_false_alarm_rate': x['n_block_alerts'].sum() / x['n_blocks'].sum(),
                        'benign_persist_false_alarm_rate': x['n_persist_alerts'].sum() / x['n_blocks'].sum(),
                        'benign_run_false_alarm_rate': x['run_alert'].mean(),
                        'benign_run_keep_rate': 1.0 - x['run_alert'].mean(),
                    }
                ),
                include_groups=False,
            )
            .reset_index()
        )
        by_workload['feature_profile'] = profile
        by_workload = add_profile_and_config_labels(by_workload)
        by_workload = sort_profile_config(by_workload)
        by_workload.to_csv(profile_appendix_dir(profile) / 'conformal_reliability_by_workload.csv', index=False)
        workload_rows.append(by_workload)

    reliability_summary = sort_profile_config(pd.concat(reliability_rows, ignore_index=True))
    reliability_by_workload = sort_profile_config(pd.concat(workload_rows, ignore_index=True))
    reliability_summary.to_csv(reliability_csv, index=False)
    reliability_by_workload.to_csv(reliability_workload_csv, index=False)

    display_cols = [
        'profile_label', 'config_label', 'target_alpha', 'benign_block_false_alarm_rate',
        'benign_persist_false_alarm_rate', 'benign_run_keep_rate', 'benign_run_false_alarm_rate',
        'anomaly_run_detection_rate', 'median_anomaly_time_to_detect_s'
    ]
    display(Markdown('#### Reliability by profile and head'))
    display(reliability_summary[display_cols].round(4))

    workload_cols = [
        'profile_label', 'config_label', 'workload', 'target_alpha',
        'benign_block_false_alarm_rate', 'benign_run_keep_rate', 'benign_run_false_alarm_rate'
    ]
    display(Markdown('#### Benign reliability by workload'))
    display(reliability_by_workload[workload_cols].round(4))

    if missing_profiles:
        display(Markdown(markdown_missing_profiles(missing_profiles)))


### Reliability comparison figures


In [ ]:
# Plain-language: This cell draws the calibrated alerting figures so readers can see the tradeoff between catching problems and avoiding false alarms.
display(Markdown('### Calibrated alerting figures'))

reliability_csv = COMPARISON_DIR / 'conformal_reliability_profiles.csv'
reliability_png = COMPARISON_FIG / 'fig_conformal_reliability_profiles.png'

if not reliability_csv.exists():
    display(Markdown('Run the reliability-summary cell first so the comparison CSV exists.'))
else:
    rel = pd.read_csv(reliability_csv)
    rel = rel.sort_values(['feature_profile', 'config']).reset_index(drop=True)
    profiles = [p for p in PROFILE_ORDER if p in set(rel['feature_profile'])]

    if not profiles:
        display(Markdown('No calibrated alerting rows are available yet.'))
    else:
        def lighten_color(color, amount):
            rgb = np.array(to_rgb(color))
            white = np.ones(3)
            return tuple(rgb * (1.0 - amount) + white * amount)

        profile_colors = {
            'mixed': '#355C7D',
            'full': '#C44E52',
        }

        block_color = '#4E79A7'
        run_color = '#F28E2B'
        detect_color = '#D1495B'

        all_vals = []
        for col in [
            'benign_block_false_alarm_rate',
            'benign_run_false_alarm_rate',
            'anomaly_run_detection_rate',
            'target_alpha',
        ]:
            if col in rel.columns:
                vals = pd.to_numeric(rel[col], errors='coerce').to_numpy(dtype=float)
                vals = vals[np.isfinite(vals)]
                if len(vals):
                    all_vals.extend(vals.tolist())

        if all_vals:
            xmin = float(np.min(all_vals))
            xmax = float(np.max(all_vals))
            span = max(xmax - xmin, 0.15)
            pad = max(0.03, 0.12 * span)
            x_lo = max(0.0, xmin - pad)
            x_hi = min(1.02, xmax + pad)
            if x_hi - x_lo < 0.30:
                mid = 0.5 * (x_lo + x_hi)
                x_lo = max(0.0, mid - 0.15)
                x_hi = min(1.02, mid + 0.15)
        else:
            x_lo, x_hi = 0.0, 1.0

        fig, axes = plt.subplots(
            1,
            len(profiles),
            figsize=(7.4 * len(profiles), 5.2),
            squeeze=False,
            constrained_layout=False,
        )
        axes = axes[0]

        legend_handles = None

        for ax, profile in zip(axes, profiles):
            d = rel[rel['feature_profile'] == profile].copy().sort_values('config').reset_index(drop=True)
            y = np.arange(len(d), dtype=float)

            accent = profile_colors.get(profile, '#355C7D')
            panel_bg = lighten_color(accent, 0.95)
            corridor_fill = lighten_color(accent, 0.72)
            corridor_edge = lighten_color(accent, 0.42)
            bridge_color = lighten_color(accent, 0.32)

            ax.set_facecolor(panel_bg)

            for yi in y:
                if int(yi) % 2 == 0:
                    ax.axhspan(yi - 0.46, yi + 0.46, color='white', alpha=0.34, zorder=0)

            line = ax.axvline(
                float(d['target_alpha'].iloc[0]),
                color='#111827',
                linestyle=(0, (4, 4)),
                linewidth=1.6,
                label='Calibration target',
                zorder=1,
            )

            ax.hlines(
                y,
                d['benign_run_false_alarm_rate'],
                d['anomaly_run_detection_rate'],
                color=corridor_fill,
                linewidth=10,
                alpha=0.98,
                zorder=1.2,
            )
            ax.hlines(
                y,
                d['benign_run_false_alarm_rate'],
                d['anomaly_run_detection_rate'],
                color=corridor_edge,
                linewidth=1.2,
                alpha=0.95,
                zorder=1.4,
            )
            ax.hlines(
                y,
                d['benign_block_false_alarm_rate'],
                d['benign_run_false_alarm_rate'],
                color=bridge_color,
                linewidth=3.0,
                alpha=0.95,
                linestyle='--',
                zorder=1.5,
            )

            s1 = ax.scatter(
                d['benign_block_false_alarm_rate'],
                y,
                s=145,
                marker='s',
                color=block_color,
                edgecolor='white',
                linewidth=1.3,
                label='Benign block alert rate',
                zorder=3,
            )
            s2 = ax.scatter(
                d['benign_run_false_alarm_rate'],
                y,
                s=168,
                marker='o',
                color=run_color,
                edgecolor='white',
                linewidth=1.3,
                label='Benign run alert rate',
                zorder=4,
            )
            s3 = ax.scatter(
                d['anomaly_run_detection_rate'],
                y,
                s=188,
                marker='^',
                color=detect_color,
                edgecolor='white',
                linewidth=1.3,
                label='Anomaly detection rate',
                zorder=5,
            )

            trans = blended_transform_factory(ax.transAxes, ax.transData)
            ax.text(
                0.985,
                -0.72,
                'TTD',
                transform=trans,
                ha='right',
                va='center',
                fontsize=11.4,
                fontweight='bold',
                color='#334155',
            )

            for yi, row in enumerate(d.itertuples(index=False)):
                if pd.notna(row.median_anomaly_time_to_detect_s):
                    ax.text(
                        0.985,
                        yi,
                        f'{row.median_anomaly_time_to_detect_s:.0f}s',
                        transform=trans,
                        ha='right',
                        va='center',
                        fontsize=11.0,
                        fontweight='bold',
                        color='#0F172A',
                        bbox=dict(boxstyle='round,pad=0.18', fc='white', ec='none', alpha=0.94),
                    )

            ax.set_yticks(y)
            ax.set_yticklabels(d['config_label'], fontsize=11.8)
            ax.invert_yaxis()
            ax.set_xlim(x_lo, x_hi)
            ax.set_xlabel('Rate', fontsize=12.4)
            ax.set_title(PROFILE_LABEL[profile], fontsize=15.5, fontweight='bold', pad=11)
            ax.grid(axis='x', alpha=0.18)
            ax.tick_params(axis='x', labelsize=11.2)
            ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
            ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))

            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_visible(False)

            legend_handles = [line, s1, s2, s3]

        fig.legend(
            handles=legend_handles,
            loc='lower center',
            bbox_to_anchor=(0.5, -0.01),
            ncol=4,
            frameon=False,
            fontsize=11.2,
            handlelength=2.2,
            columnspacing=1.6,
        )

        fig.suptitle('Mixed vs full: calibrated alerting profile', fontsize=16, fontweight='bold')
        fig.tight_layout(rect=[0.02, 0.16, 0.98, 0.92])
        fig.savefig(reliability_png, dpi=PRO_FIG_DPI, bbox_inches='tight', facecolor='white')
        display(Image(filename=str(reliability_png)))
        plt.close(fig)


## 4. Observability Heads and Digital-Twin Variants

This section compares three sensing setups: `Tier-0`, `Tier-0/1`, and `Tier-0/1/2`.

In plain terms, it shows what we gain when we give DICE more system signals to work with. It compares:
- the base score,
- the workload-conditioned score,
- and the diagnosis output.


In [ ]:
# Plain-language: This cell compares observation heads and diagnosis quality to show what each telemetry tier contributes.

display(Markdown('### Observability and diagnosis tables'))

ready_profiles, missing_profiles = available_profiles(required_global=['overall_metrics.csv'])
variant_csv = COMPARISON_DIR / 'digital_twin_variant_summary_profiles.csv'

if not ready_profiles:
    display(Markdown('No observability-head result bundles are ready yet. Run **Run End-to-End** first.'))
    if missing_profiles:
        display(Markdown(markdown_missing_profiles(missing_profiles)))
else:
    variant_rows = []
    diagnosis_notes = []
    for profile in ready_profiles:
        overall = read_profile_csv(profile, 'global', 'overall_metrics.csv')
        diag, diagnosis_source, diagnosis_path = read_profile_best_diagnosis(profile)
        if diag.empty:
            diag = overall[['config']].copy()
            for col in ['top1_acc', 'top2_acc', 'macro_f1', 'balanced_acc', 'mean_margin_to_second']:
                diag[col] = np.nan
        merged = (
            overall[['config', 'n_features', 'roc_auc', 'pr_auc', 'roc_auc_wc', 'pr_auc_wc']]
            .merge(diag[['config', 'top1_acc', 'top2_acc', 'macro_f1', 'balanced_acc', 'mean_margin_to_second']], on='config', how='left')
        )
        merged['feature_profile'] = profile
        merged['diagnosis_source'] = diagnosis_source
        variant_rows.append(add_profile_and_config_labels(merged))

        diag_note = diagnosis_source
        if diagnosis_path is not None:
            diag_note = f"{diagnosis_source} (`{diagnosis_path.name}`)"
        diagnosis_notes.append(f"- `{PROFILE_LABEL.get(profile, profile)}` observability summary uses **{diag_note}**.")

    variant_summary = sort_profile_config(pd.concat(variant_rows, ignore_index=True))
    variant_summary.to_csv(variant_csv, index=False)
    display(variant_summary[[
        'profile_label', 'config_label', 'n_features', 'roc_auc', 'pr_auc', 'roc_auc_wc', 'pr_auc_wc',
        'top1_acc', 'top2_acc', 'macro_f1', 'balanced_acc', 'mean_margin_to_second', 'diagnosis_source'
    ]].round(4))

    if diagnosis_notes:
        display(Markdown('**Diagnosis source**\n' + '\n'.join(diagnosis_notes)))

    if missing_profiles:
        display(Markdown(markdown_missing_profiles(missing_profiles)))


## 4B. Two-Stage DICE

This section tests a cheaper two-stage version of DICE.

- Stage 1 uses `Tier-0` as a lightweight screen that can run all the time.
- Stage 2 uses `Tier-0/1/2` as a richer follow-up step that runs only when Stage 1 looks suspicious.

The goal is to see how much detection quality we can keep while lowering the average number of active features.


In [ ]:
# Plain-language: This cell compares the two-stage screen-and-refine policies to show how much extra sensing budget each trigger uses.
display(Markdown('### Two-stage comparison tables'))

ready_profiles, missing_profiles = available_profiles(required_global=['case_predictions.csv'])
two_stage_csv = COMPARISON_DIR / 'two_stage_dice_profiles.csv'

if not ready_profiles:
    display(Markdown('No case-level prediction bundles are ready yet. Run **Run End-to-End** first.'))
    if missing_profiles:
        display(Markdown(markdown_missing_profiles(missing_profiles)))
elif 'summarize_two_stage_dice' not in globals():
    display(Markdown('Run the main end-to-end notebook section first so the two-stage helper is available.'))
else:
    summary_rows = []
    for profile in ready_profiles:
        case_pred = read_profile_csv(profile, 'global', 'case_predictions.csv').copy()
        block_trace = (
            read_profile_csv(profile, 'global', 'case_block_traces.csv').copy()
            if profile_file_path(profile, 'global', 'case_block_traces.csv').exists()
            else pd.DataFrame()
        )
        stage2_config = resolve_stage2_config(case_pred)
        profile_summary, profile_meta = summarize_two_stage_dice(
            case_pred,
            block_trace,
            stage2_config=stage2_config,
            trigger_quantiles=[0.90, 0.95, 0.98],
        )
        if profile_summary.empty:
            continue
        profile_summary['feature_profile'] = profile
        profile_summary['profile_label'] = PROFILE_LABEL[profile]
        profile_summary['stage2_config'] = profile_meta.get('stage2_config', stage2_config)
        profile_csv = profile_paper_dir(profile) / 'two_stage_dice_summary.csv'
        profile_summary.to_csv(profile_csv, index=False)
        summary_rows.append(profile_summary)

    if not summary_rows:
        display(Markdown('Two-stage comparison is not available yet.'))
    else:
        two_stage_summary = pd.concat(summary_rows, ignore_index=True)
        two_stage_summary.to_csv(two_stage_csv, index=False)
        display(two_stage_summary.round(4))

    if missing_profiles:
        display(Markdown(markdown_missing_profiles(missing_profiles)))


### Two-stage comparison figures


In [ ]:
# Plain-language: This cell shows a minimal combined Pareto frontier for the two-stage policies.
display(Markdown('### Two-stage comparison figures'))


two_stage_csv = COMPARISON_DIR / 'two_stage_dice_profiles.csv'
two_stage_png = COMPARISON_FIG / 'fig_two_stage_profile_comparison.png'

FIG_BG = '#FCFCFF'
PANEL_BG = '#FBFDFF'
TEXT_DARK = '#0F172A'
GRID_SOFT = '#E2E8F0'
SPINE_SOFT = '#CBD5E1'

PROFILE_LABEL_LOCAL = globals().get('PROFILE_LABEL', {'mixed': 'Mixed', 'full': 'Full'})
PROFILE_ORDER_LOCAL = list(globals().get('PROFILE_ORDER', ['mixed', 'full']))

PROFILE_COLORS_LOCAL = {
    'mixed': '#0B3C5D',
    'full': '#7C2D12',
}
POLICY_ORDER = [
    'Tier-0 only',
    'Two-stage q=0.98',
    'Two-stage q=0.95',
    'Two-stage q=0.90',
    'Always tier0_tier1_tier2',
]
POLICY_LABEL = {
    'Tier-0 only': 'Tier-0 only',
    'Two-stage q=0.98': 'Two-stage q=0.98',
    'Two-stage q=0.95': 'Two-stage q=0.95',
    'Two-stage q=0.90': 'Two-stage q=0.90',
    'Always tier0_tier1_tier2': 'Always final head',
}
POLICY_MARKERS = {
    'Tier-0 only': 'o',
    'Two-stage q=0.98': '^',
    'Two-stage q=0.95': 's',
    'Two-stage q=0.90': 'D',
    'Always tier0_tier1_tier2': 'P',
}


def _style_axis(ax, title: str):
    ax.set_facecolor(PANEL_BG)
    for spine in ax.spines.values():
        spine.set_color(SPINE_SOFT)
        spine.set_linewidth(1.2)
    ax.set_title(title, fontsize=18, fontweight='bold', color=TEXT_DARK, loc='left', pad=14)


def _pareto_mask(df: pd.DataFrame) -> np.ndarray:
    budget = pd.to_numeric(df['avg_feature_budget'], errors='coerce').to_numpy(dtype=float)
    detect = pd.to_numeric(df['detection_rate'], errors='coerce').to_numpy(dtype=float)
    valid = np.isfinite(budget) & np.isfinite(detect)
    keep = np.zeros(len(df), dtype=bool)

    for i in range(len(df)):
        if not valid[i]:
            continue
        dominated = valid & (budget <= budget[i]) & (detect >= detect[i]) & ((budget < budget[i]) | (detect > detect[i]))
        keep[i] = not dominated.any()

    return keep


if not two_stage_csv.exists():
    display(Markdown('Run the two-stage summary cell first so the comparison CSV exists.'))
else:
    summary = pd.read_csv(two_stage_csv)
    summary['feature_profile'] = summary['feature_profile'].astype(str)
    summary['policy'] = summary['policy'].astype(str)

    profiles = [p for p in PROFILE_ORDER_LOCAL if p in set(summary['feature_profile'])]
    if not profiles:
        profiles = sorted(summary['feature_profile'].dropna().astype(str).unique().tolist())

    if not profiles:
        display(Markdown('No two-stage profile rows were found for the comparison figure.'))
    else:
        budget_vals = pd.to_numeric(summary['avg_feature_budget'], errors='coerce')
        detect_vals = pd.to_numeric(summary['detection_rate'], errors='coerce')

        finite_budget = budget_vals[np.isfinite(budget_vals)]
        finite_detect = detect_vals[np.isfinite(detect_vals)]

        xmin = max(0.0, float(finite_budget.min()) - 4.0) if len(finite_budget) else 0.0
        xmax = float(finite_budget.max()) + 6.0 if len(finite_budget) else 100.0
        ymin = max(0.0, float(finite_detect.min()) - 0.08) if len(finite_detect) else 0.0
        ymax = min(1.02, float(finite_detect.max()) + 0.08) if len(finite_detect) else 1.02

        fig, ax = plt.subplots(figsize=(9.8, 6.8))
        fig.patch.set_facecolor(FIG_BG)
        _style_axis(ax, 'Two-stage policy frontier')

        ax.axhspan(0.80, 1.00, color='#ECFCCB', alpha=0.55, zorder=0)

        for profile in profiles:
            d = summary[summary['feature_profile'] == profile].copy()
            d = d[
                np.isfinite(pd.to_numeric(d['avg_feature_budget'], errors='coerce')) &
                np.isfinite(pd.to_numeric(d['detection_rate'], errors='coerce'))
            ].copy()
            if d.empty:
                continue

            d['is_frontier'] = _pareto_mask(d)
            frontier = d[d['is_frontier']].sort_values('avg_feature_budget')
            dominated = d[~d['is_frontier']]

            color = PROFILE_COLORS_LOCAL.get(profile, '#334155')

            if not dominated.empty:
                for row in dominated.itertuples(index=False):
                    ax.scatter(
                        float(row.avg_feature_budget),
                        float(row.detection_rate),
                        s=145,
                        marker=POLICY_MARKERS.get(row.policy, 'o'),
                        facecolor='white',
                        edgecolor=color,
                        linewidth=1.5,
                        alpha=0.40,
                        zorder=2,
                    )

            if not frontier.empty:
                ax.plot(
                    frontier['avg_feature_budget'].to_numpy(dtype=float),
                    frontier['detection_rate'].to_numpy(dtype=float),
                    color=color,
                    linewidth=2.8,
                    alpha=0.95,
                    zorder=2,
                )
                for row in frontier.itertuples(index=False):
                    ax.scatter(
                        float(row.avg_feature_budget),
                        float(row.detection_rate),
                        s=185,
                        marker=POLICY_MARKERS.get(row.policy, 'o'),
                        facecolor=color,
                        edgecolor='white',
                        linewidth=1.4,
                        zorder=3,
                    )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_xlabel('Average active feature budget', fontsize=14.2, fontweight='bold', color=TEXT_DARK, labelpad=10)
        ax.set_ylabel('Detection rate', fontsize=14.2, fontweight='bold', color=TEXT_DARK, labelpad=10)
        ax.tick_params(axis='both', labelsize=12.5, colors=TEXT_DARK)
        ax.grid(color=GRID_SOFT, linewidth=1.1, alpha=0.9)
        ax.set_axisbelow(True)

        profile_handles = [
            Line2D(
                [0], [0],
                color=PROFILE_COLORS_LOCAL.get(profile, '#334155'),
                linewidth=2.8,
                label=PROFILE_LABEL_LOCAL.get(profile, profile.title()),
            )
            for profile in profiles
        ]
        policy_handles = [
            Line2D(
                [0], [0],
                marker=POLICY_MARKERS.get(policy, 'o'),
                color='#334155',
                markerfacecolor='white',
                markeredgecolor='#334155',
                markersize=8,
                linewidth=0,
                label=POLICY_LABEL.get(policy, policy),
            )
            for policy in POLICY_ORDER if policy in set(summary['policy'])
        ]

        legend1 = ax.legend(
            handles=profile_handles,
            title='Profile',
            loc='lower right',
            fontsize=11.2,
            title_fontsize=11.6,
            frameon=True,
            fancybox=True,
        )
        legend1.get_frame().set_facecolor('white')
        legend1.get_frame().set_edgecolor(SPINE_SOFT)
        legend1.get_frame().set_linewidth(1.0)
        ax.add_artist(legend1)

        legend2 = ax.legend(
            handles=policy_handles,
            title='Policy',
            loc='upper left',
            fontsize=10.8,
            title_fontsize=11.6,
            frameon=True,
            fancybox=True,
        )
        legend2.get_frame().set_facecolor('white')
        legend2.get_frame().set_edgecolor(SPINE_SOFT)
        legend2.get_frame().set_linewidth(1.0)

        fig.savefig(two_stage_png, dpi=PRO_FIG_DPI, bbox_inches='tight', facecolor=fig.get_facecolor())
        plt.close(fig)

        display(Image(filename=str(two_stage_png)))
        display(Markdown(
            '*Filled markers lie on the Pareto frontier; hollow markers are dominated. '
            'The figure focuses only on the core tradeoff between active feature budget and detection rate.*'
        ))


## 5. Cross-Workload Transfer Robustness and Drift Proxy

This section checks what happens when the detector is tested on a workload it did not see during training.

That is a simple stand-in for real-world change, such as new software behavior, usage drift, or a different workload mix after deployment.


In [ ]:
# Plain-language: This cell summarizes cross-workload transfer results to show how DICE behaves when it sees a new workload pattern.
display(Markdown('### Cross-workload transfer comparison tables'))

transfer_csv = COMPARISON_DIR / 'holdout_robustness_profiles.csv'


def resolve_profile_availability(required_holdout=(), required_global=(), required_paper=(), required_appendix=()):
    ready = []
    missing = {}
    for profile in PROFILE_ORDER:
        miss = profile_requirements(
            profile,
            required_global=required_global,
            required_holdout=required_holdout,
            required_paper=required_paper,
            required_appendix=required_appendix,
        )
        if miss:
            missing[profile] = miss
        else:
            ready.append(profile)
    return ready, missing


def select_transfer_columns(df):
    out = df.copy()

    aliases = {
        'pooled_pr_auc': ['pooled_holdout_pr_auc'],
        'pooled_roc_auc': ['pooled_holdout_roc_auc'],
        'mean_pr_auc': ['holdout_mean_pr_auc'],
        'worst_pr_auc': ['holdout_worst_pr_auc'],
        'mean_roc_auc': ['holdout_mean_roc_auc'],
        'mean_fpr': ['holdout_mean_fpr'],
        'mean_tpr': ['holdout_mean_tpr'],
    }

    for target, candidates in aliases.items():
        if target not in out.columns:
            for alt in candidates:
                if alt in out.columns:
                    out[target] = out[alt]
                    break

    required = [
        'config',
        'mean_pr_auc',
        'worst_pr_auc',
        'mean_roc_auc',
        'pooled_pr_auc',
        'pooled_roc_auc',
        'mean_fpr',
        'mean_tpr',
    ]
    for col in required:
        if col not in out.columns:
            out[col] = np.nan

    return out[required].copy()


ready_profile_list, missing_profiles = resolve_profile_availability(
    required_holdout=['holdout_robustness_summary.csv']
)

if not ready_profile_list:
    display(Markdown('No cross-workload transfer bundles are ready yet. Run **Run End-to-End** with cross-workload transfer evaluation enabled.'))
    if missing_profiles:
        display(Markdown(markdown_missing_profiles(missing_profiles)))
else:
    transfer_rows = []
    notes = []

    for profile in ready_profile_list:
        raw_transfer = read_profile_csv(profile, 'holdout', 'holdout_robustness_summary.csv').copy()
        transfer = select_transfer_columns(raw_transfer)
        transfer['feature_profile'] = profile
        transfer_rows.append(add_profile_and_config_labels(transfer))

        pooled_missing = not (
            'pooled_pr_auc' in raw_transfer.columns or
            'pooled_holdout_pr_auc' in raw_transfer.columns
        )
        if pooled_missing:
            notes.append(
                f'- `{PROFILE_LABEL.get(profile, profile)}` cross-workload transfer summary does not include pooled metrics; those columns are omitted from the display table.'
            )

    transfer_summary = sort_profile_config(pd.concat(transfer_rows, ignore_index=True))
    transfer_summary.to_csv(transfer_csv, index=False)

    pooled_available = (
        transfer_summary[['pooled_pr_auc', 'pooled_roc_auc']].notna().any().any()
        if {'pooled_pr_auc', 'pooled_roc_auc'}.issubset(transfer_summary.columns)
        else False
    )

    display_cols = [
        'profile_label',
        'config_label',
        'mean_pr_auc',
        'worst_pr_auc',
        'mean_roc_auc',
        'mean_fpr',
        'mean_tpr',
    ]
    if pooled_available:
        display_cols[5:5] = ['pooled_pr_auc', 'pooled_roc_auc']

    display(transfer_summary[display_cols].round(4))

    if notes and not pooled_available:
        display(Markdown('**Notes**\n' + '\n'.join(sorted(set(notes)))))

    if missing_profiles:
        display(Markdown(markdown_missing_profiles(missing_profiles)))


### Cross-workload transfer comparison figures


In [ ]:
# Plain-language: This cell turns the cross-workload transfer results into figures that highlight robustness under workload shift.
display(Markdown('### Cross-workload transfer comparison figures'))

transfer_csv = COMPARISON_DIR / 'holdout_robustness_profiles.csv'
transfer_png = COMPARISON_FIG / 'fig_workload_transfer_profile_comparison.png'

if not transfer_csv.exists():
    display(Markdown('Run the cross-workload transfer summary cell first so the comparison CSV exists.'))
else:
    transfer = pd.read_csv(transfer_csv)
    profiles = [p for p in PROFILE_ORDER if p in set(transfer['feature_profile'])]

    if not profiles:
        display(Markdown('No cross-workload transfer comparison rows are available yet.'))
    else:
        profile_colors = {
            'mixed': '#355C7D',
            'full': '#C44E52',
        }
        floor_color = '#C44536'
        mean_color = '#1F4E79'

        def lighten_color(color, amount):
            rgb = np.array(to_rgb(color))
            white = np.ones(3)
            return tuple(rgb * (1.0 - amount) + white * amount)

        fig, axes = plt.subplots(
            1,
            len(profiles),
            figsize=(6.6 * len(profiles), 5.0),
            squeeze=False,
            constrained_layout=False,
        )
        axes = axes[0]

        for ax, profile in zip(axes, profiles):
            d = transfer[transfer['feature_profile'] == profile].copy().sort_values('config')
            d = d.reset_index(drop=True)
            y = np.arange(len(d), dtype=float)

            accent = profile_colors.get(profile, '#355C7D')
            corridor_fill = lighten_color(accent, 0.72)
            corridor_edge = lighten_color(accent, 0.38)
            panel_bg = lighten_color(accent, 0.95)

            ax.set_facecolor(panel_bg)

            lower_vals = []
            upper_vals = []

            for yi, row in enumerate(d.itertuples(index=False)):
                x_floor = float(row.worst_pr_auc)
                x_mean = float(row.mean_pr_auc)
                if not np.isfinite(x_floor) or not np.isfinite(x_mean):
                    continue

                x_lo, x_hi = sorted([x_floor, x_mean])

                ax.plot(
                    [x_lo, x_hi],
                    [yi, yi],
                    color=corridor_fill,
                    linewidth=12,
                    solid_capstyle='round',
                    zorder=1,
                    alpha=0.98,
                )
                ax.plot(
                    [x_lo, x_hi],
                    [yi, yi],
                    color=corridor_edge,
                    linewidth=1.15,
                    solid_capstyle='round',
                    zorder=1.5,
                    alpha=0.95,
                )

                lower_vals.append(x_lo)
                upper_vals.append(x_hi)

            ax.scatter(
                d['worst_pr_auc'],
                y,
                s=138,
                marker='D',
                color=floor_color,
                edgecolor='white',
                linewidth=1.2,
                zorder=3,
            )
            ax.scatter(
                d['mean_pr_auc'],
                y,
                s=168,
                marker='o',
                color=mean_color,
                edgecolor='white',
                linewidth=1.2,
                zorder=4,
            )

            for yi, row in enumerate(d.itertuples(index=False)):
                x_floor = float(row.worst_pr_auc)
                x_mean = float(row.mean_pr_auc)
                if not np.isfinite(x_floor) or not np.isfinite(x_mean):
                    continue

                gap = abs(x_mean - x_floor)

                if gap < 0.065:
                    floor_offset = (-8, -13)
                    floor_ha = 'right'
                    floor_va = 'top'
                    mean_offset = (8, 13)
                    mean_ha = 'left'
                    mean_va = 'bottom'
                else:
                    floor_offset = (-8, 0)
                    floor_ha = 'right'
                    floor_va = 'center'
                    mean_offset = (8, 0)
                    mean_ha = 'left'
                    mean_va = 'center'

                ax.annotate(
                    f'{x_floor:.2f}',
                    xy=(x_floor, yi),
                    xytext=floor_offset,
                    textcoords='offset points',
                    ha=floor_ha,
                    va=floor_va,
                    fontsize=10.8,
                    fontweight='bold',
                    color=floor_color,
                    bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.92),
                )
                ax.annotate(
                    f'{x_mean:.2f}',
                    xy=(x_mean, yi),
                    xytext=mean_offset,
                    textcoords='offset points',
                    ha=mean_ha,
                    va=mean_va,
                    fontsize=11.0,
                    fontweight='bold',
                    color=mean_color,
                    bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.92),
                )

            vals = np.concatenate([
                d['worst_pr_auc'].to_numpy(dtype=float),
                d['mean_pr_auc'].to_numpy(dtype=float),
            ])
            vals = vals[np.isfinite(vals)]

            if len(vals) == 0:
                ax.set_xlim(0.0, 1.0)
            else:
                xmin = float(np.min(vals))
                xmax = float(np.max(vals))
                span = max(xmax - xmin, 0.06)
                pad = max(0.025, 0.12 * span)
                lo = max(0.0, xmin - pad)
                hi = min(1.02, xmax + pad)

                if hi - lo < 0.18:
                    mid = 0.5 * (lo + hi)
                    lo = max(0.0, mid - 0.09)
                    hi = min(1.02, mid + 0.09)

                ax.set_xlim(lo, hi)

            ax.set_yticks(y)
            ax.set_yticklabels(d['config_label'], fontsize=11.2)
            ax.invert_yaxis()
            ax.set_xlabel('AUC-PR under workload transfer', fontsize=11.8)
            ax.set_title(PROFILE_LABEL[profile], fontsize=15, fontweight='bold', pad=10)
            ax.grid(axis='x', alpha=0.18)
            ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
            ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
            ax.tick_params(axis='x', labelsize=10.8)

            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_visible(False)

        legend_handles = [
            Line2D([0], [0], color='#AFC3D5', linewidth=9, solid_capstyle='round', label='Transfer corridor'),
            Line2D([0], [0], marker='D', linestyle='None', markersize=9.5,
                   markerfacecolor=floor_color, markeredgecolor='white', label='Transfer floor'),
            Line2D([0], [0], marker='o', linestyle='None', markersize=10.5,
                   markerfacecolor=mean_color, markeredgecolor='white', label='Transfer mean'),
        ]

        fig.legend(
            handles=legend_handles,
            loc='lower center',
            bbox_to_anchor=(0.5, -0.01),
            ncol=len(legend_handles),
            frameon=False,
            handlelength=2.4,
            columnspacing=1.8,
        )

        fig.suptitle('Mixed vs full: cross-workload transfer portability', fontsize=15, fontweight='bold')
        fig.tight_layout(rect=[0.02, 0.16, 0.98, 0.93])
        fig.savefig(transfer_png, dpi=PRO_FIG_DPI, bbox_inches='tight', facecolor='white')
        display(Image(filename=str(transfer_png)))
        plt.close(fig)


## 6. DICE-Specific Design-Space Evaluation

This section looks beyond a single accuracy table and asks a broader engineering question: which DICE design choices matter most, and what tradeoffs do they create?

Sweeps 1, 4, and 5 are built from the main released outputs. Sweeps 2 and 3 use the tuning outputs and appear after running the notebook with `INCLUDE_TUNING = True`. Sweep 6 is a notebook-local feature-budget study that retrains the final detector using only the top-ranked features.


This design-space section now emphasizes profile-aware operating-point and deployment tradeoffs.


### Suggested complete DICE sweep plan

For a complete paper-and-appendix outcome, use a staged sweep rather than one large undirected grid:

1. **Profile and observability sweep.** Run both `mixed` and `full`, and compare `Tier-0`, `Tier-0/1`, and `Tier-0/1/2`.
2. **Stage-1 operating-point sweep.** Sweep `gain \in \{0.15, 0.25, 0.35, 0.50\}` and `block_B \in \{30, 60, 90, 120\}` s with `alpha = 0.05` and `persist_k = 3`. Use this stage to choose the best gain/block pair for each profile.
3. **Stage-2 alert sweep.** Fix the best Stage-1 gain/block pair, then sweep `alpha \in \{0.01, 0.02, 0.05, 0.10\}` and `persist_k \in \{1, 2, 3, 5\}`. Use this stage to pick the final operating point for monitoring, alerting, and time-to-detect.
4. **Practical deployment sweeps.** Add feature budgets `\{10, 20, 30, 40, 50, 60, 70, 80, 90, 100\}\%`, two-stage trigger quantiles `\{0.90, 0.95, 0.98\}`, and cross-workload transfer evaluation.
5. **Diagnosis sweeps.** Compare whole-run diagnosis, post-alert diagnosis, and hierarchical/selective diagnosis. Report exact Top-1, exact Top-2, selective Top-2 at coverage, mechanism Top-3 coverage, and keep Macro-F1 as a secondary metric.
6. **Appendix-only sweeps.** Add bootstrap confidence intervals, case galleries, and time-window ablations. If you show a best-window or oracle diagnosis result, label it explicitly as retrospective rather than deployable.

A compact main-paper set is therefore: monitoring, reliability, cross-workload, mixed-vs-full, diagnosis shortlist quality, mechanism coverage, feature budgets, and two-stage screening. The appendix can carry the deeper tuning grids and window-ablation studies.


In [ ]:
# Plain-language: This cell ranks features and exports budgeted design-space summaries so we can study lighter-weight DICE variants.
display(Markdown('### Feature-budget summary exports'))

FEATURE_PROFILE_TIER_FILES = {
    'mixed': {
        'tier0': 'tier0_full_5hz.csv',
        'tier1_alt': 'tier1_alt_core_5hz.csv',
        'tier2': 'tier2_core_5hz.csv',
    },
    'full': {
        'tier0': 'tier0_full_5hz.csv',
        'tier1_alt': 'tier1_alt_full_5hz.csv',
        'tier2': 'tier2_full_5hz.csv',
    },
}

FEATURE_BUDGET_PCTS = list(range(10, 101, 10))
FEATURE_BUDGET_BOOTSTRAPS = 800
FEATURE_BUDGET_BOOTSTRAP_SEED = 7

FM = FULL_MODULE
all_cases_fn = FM['all_cases']
common_features_per_tier_fn = FM['common_features_per_tier']
build_case_matrix_fn = FM['build_case_matrix']
train_bundle_fn = FM['train_bundle']
evaluate_run_fn = FM['evaluate_run']
residual_timeseries_fn = FM['residual_timeseries']
block_signatures_fn = FM['block_signatures']
safe_auc_fn = FM['safe_auc']
safe_ap_fn = FM['safe_ap']
configs_map = FM['CONFIGS']


def metrics_only(result):
    return result[0] if isinstance(result, tuple) else result


def full_run_feature_contrib(X_run, bundle, B, gain):
    Xn = (X_run - bundle.median) / (bundle.scale + 1e-12)
    r = residual_timeseries_fn(Xn, bundle.A, gain=gain)
    sig = block_signatures_fn(r, B=B)
    run_signature = np.nanmedian(sig, axis=0) if len(sig) else np.zeros_like(bundle.weights)
    contrib = run_signature * bundle.weights
    return np.maximum(np.asarray(contrib, dtype=float), 0.0)


def bootstrap_metric_summary(labels, scores, n_boot=FEATURE_BUDGET_BOOTSTRAPS, seed=FEATURE_BUDGET_BOOTSTRAP_SEED):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)

    valid = np.isfinite(scores)
    labels = labels[valid]
    scores = scores[valid]

    benign_idx = np.flatnonzero(labels == 0)
    anomaly_idx = np.flatnonzero(labels == 1)
    if len(benign_idx) == 0 or len(anomaly_idx) == 0:
        return {
            'roc_auc_boot_mean': np.nan,
            'roc_auc_boot_q05': np.nan,
            'roc_auc_boot_q25': np.nan,
            'roc_auc_boot_q75': np.nan,
            'roc_auc_boot_q95': np.nan,
            'pr_auc_boot_mean': np.nan,
            'pr_auc_boot_q05': np.nan,
            'pr_auc_boot_q25': np.nan,
            'pr_auc_boot_q75': np.nan,
            'pr_auc_boot_q95': np.nan,
        }

    rng = np.random.default_rng(seed)
    roc_vals = []
    pr_vals = []
    for _ in range(int(n_boot)):
        sample_idx = np.concatenate([
            rng.choice(benign_idx, size=len(benign_idx), replace=True),
            rng.choice(anomaly_idx, size=len(anomaly_idx), replace=True),
        ])
        yb = labels[sample_idx]
        sb = scores[sample_idx]
        roc_vals.append(float(safe_auc_fn(yb, sb)))
        pr_vals.append(float(safe_ap_fn(yb, sb)))

    roc_vals = np.asarray(roc_vals, dtype=float)
    pr_vals = np.asarray(pr_vals, dtype=float)

    return {
        'roc_auc_boot_mean': float(np.nanmean(roc_vals)),
        'roc_auc_boot_q05': float(np.nanquantile(roc_vals, 0.05)),
        'roc_auc_boot_q25': float(np.nanquantile(roc_vals, 0.25)),
        'roc_auc_boot_q75': float(np.nanquantile(roc_vals, 0.75)),
        'roc_auc_boot_q95': float(np.nanquantile(roc_vals, 0.95)),
        'pr_auc_boot_mean': float(np.nanmean(pr_vals)),
        'pr_auc_boot_q05': float(np.nanquantile(pr_vals, 0.05)),
        'pr_auc_boot_q25': float(np.nanquantile(pr_vals, 0.25)),
        'pr_auc_boot_q75': float(np.nanquantile(pr_vals, 0.75)),
        'pr_auc_boot_q95': float(np.nanquantile(pr_vals, 0.95)),
    }


summary_written = []
ranking_written = []
prediction_written = []

for profile in PROFILE_ORDER:
    overall_path = profile_global_dir(profile) / 'overall_metrics.csv'
    sequential_path = profile_global_dir(profile) / 'sequential_metrics.csv'
    if not overall_path.exists() or not sequential_path.exists():
        print(f'Skip {profile}: missing overall/sequential metrics.')
        continue

    rec_path = default_tuning_out_dir(DATASET_ROOT, profile) / 'recommended_alert_config.csv'
    op = profile_operating_point(profile)

    if rec_path.exists():
        rec = pd.read_csv(rec_path)
        rec_row = rec.iloc[0] if not rec.empty else None
    else:
        rec_row = None

    block_B = int(rec_row['block_B']) if rec_row is not None and 'block_B' in rec_row.index else int(op.get('block_B', 60))
    alpha = float(rec_row['alpha']) if rec_row is not None and 'alpha' in rec_row.index else float(op.get('alpha', 0.05))
    persist_k = int(rec_row['persist_k']) if rec_row is not None and 'persist_k' in rec_row.index else int(op.get('persist_k', 3))
    gain = float(rec_row['gain']) if rec_row is not None and 'gain' in rec_row.index else float(op.get('gain', 0.35))
    fit_ratio = float(op.get('fit_ratio', 0.6))
    ridge_lambda = float(op.get('ridge_lambda', 1e-3))
    source_hz = int(op.get('source_hz', 5))

    tier_files = FEATURE_PROFILE_TIER_FILES[profile].copy()
    FM['TIER_FILE'] = tier_files

    tiers = configs_map[FINAL_CONFIG]
    root = Path(DATASET_ROOT)
    cases = list(all_cases_fn())
    
    feature_map = {
        tier: common_features_per_tier_fn(root, tier, tier_files, cases=cases)
        for tier in tiers
    }
    
    case_X = {}
    feature_names_full = None
    for case in cases:
        X, names = build_case_matrix_fn(
            root,
            case,
            tiers=tiers,
            feature_map=feature_map,
            tier_files=tier_files,
            source_hz=source_hz,
        )
        case_X[case.case_id] = X
        if feature_names_full is None:
            feature_names_full = names

    benign_runs_full = {
        case_id: X
        for case_id, X in case_X.items()
        if case_id.endswith('__NOMINAL')
    }

    bundle_full = train_bundle_fn(
        train_benign_runs=benign_runs_full,
        feature_names=feature_names_full,
        fit_ratio=fit_ratio,
        B=block_B,
        alpha=alpha,
        gain=gain,
        ridge_lambda=ridge_lambda,
    )

    contrib_rows = []
    for case in cases:
        contrib = full_run_feature_contrib(
            case_X[case.case_id],
            bundle_full,
            B=block_B,
            gain=gain,
        )
        contrib_rows.append({
            'case_id': case.case_id,
            'workload': case.workload,
            'stressor': case.stressor,
            'label': case.label,
            'contrib': contrib,
        })

    benign_contrib = np.vstack([r['contrib'] for r in contrib_rows if int(r['label']) == 0])
    anomaly_contrib = np.vstack([r['contrib'] for r in contrib_rows if int(r['label']) == 1])

    med_benign = np.median(benign_contrib, axis=0)
    med_anomaly = np.median(anomaly_contrib, axis=0)
    mad_benign = np.median(np.abs(benign_contrib - med_benign), axis=0) + 1e-9

    disc_score = np.maximum(med_anomaly - med_benign, 0.0) / mad_benign
    stability_score = np.abs(bundle_full.weights)
    stability_score = stability_score / max(float(np.max(stability_score)), 1e-12)
    rank_score = disc_score * np.sqrt(stability_score)

    ranked_idx = np.argsort(rank_score)[::-1] if np.any(rank_score > 0) else np.argsort(np.abs(bundle_full.weights))[::-1]

    rank_df = pd.DataFrame({
        'feature_profile': profile,
        'feature_name': feature_names_full,
        'rank_score': rank_score,
        'disc_score': disc_score,
        'stability_score': stability_score,
    }).sort_values('rank_score', ascending=False).reset_index(drop=True)

    rank_out = profile_paper_dir(profile) / 'dse_feature_ranking.csv'
    rank_df.to_csv(rank_out, index=False)
    ranking_written.append(rank_out)
    print(f'Wrote {rank_out}')

    n_total = len(ranked_idx)
    budget_rows = []
    prediction_rows = []

    for pct in FEATURE_BUDGET_PCTS:
        n_keep = max(1, int(np.ceil(n_total * pct / 100.0)))
        keep_idx = np.sort(ranked_idx[:n_keep])

        feature_names_sel = [feature_names_full[i] for i in keep_idx]
        benign_runs_sel = {
            case_id: X[:, keep_idx]
            for case_id, X in benign_runs_full.items()
        }

        bundle_sel = train_bundle_fn(
            train_benign_runs=benign_runs_sel,
            feature_names=feature_names_sel,
            fit_ratio=fit_ratio,
            B=block_B,
            alpha=alpha,
            gain=gain,
            ridge_lambda=ridge_lambda,
        )

        pred_rows = []
        for case in cases:
            X_sel = case_X[case.case_id][:, keep_idx]
            metrics = metrics_only(
                evaluate_run_fn(
                    X_sel,
                    bundle_sel,
                    B=block_B,
                    alpha=alpha,
                    persist_k=persist_k,
                    gain=gain,
                )
            )

            pred_rows.append({
                'case_id': case.case_id,
                'workload': case.workload,
                'stressor': case.stressor,
                'label': case.label,
                'run_score': float(metrics['run_score']),
                'run_alert': int(metrics['run_alert']),
                'time_to_detect_s': float(metrics['time_to_detect_s']),
            })

        pred_df = pd.DataFrame(pred_rows)
        nominal_map = (
            pred_df[pred_df['stressor'] == 'NOMINAL']
            .set_index('workload')['run_score']
            .to_dict()
        )
        pred_df['nominal_template_score'] = pred_df['workload'].map(nominal_map)
        pred_df['run_score_wc'] = (pred_df['run_score'] - pred_df['nominal_template_score']).abs()

        for row in pred_df.itertuples(index=False):
            prediction_rows.append({
                'feature_profile': profile,
                'budget_pct': float(pct),
                'n_selected_features': int(n_keep),
                'block_B': float(block_B),
                'alpha': float(alpha),
                'persist_k': float(persist_k),
                'gain': float(gain),
                'case_id': row.case_id,
                'workload': row.workload,
                'stressor': row.stressor,
                'label': int(row.label),
                'run_score': float(row.run_score),
                'run_score_wc': float(row.run_score_wc),
                'run_alert': int(row.run_alert),
                'time_to_detect_s': float(row.time_to_detect_s),
            })

        y = pred_df['label'].to_numpy(dtype=int)
        s = pred_df['run_score'].to_numpy(dtype=float)
        s_wc = pred_df['run_score_wc'].to_numpy(dtype=float)

        benign = pred_df[pred_df['label'] == 0]
        anomaly = pred_df[pred_df['label'] == 1]
        detected = anomaly[anomaly['run_alert'] == 1]
        boot = bootstrap_metric_summary(y, s)

        budget_rows.append({
            'feature_profile': profile,
            'budget_pct': float(pct),
            'n_selected_features': int(n_keep),
            'roc_auc': float(safe_auc_fn(y, s)),
            'pr_auc': float(safe_ap_fn(y, s)),
            'roc_auc_wc': float(safe_auc_fn(y, s_wc)),
            'pr_auc_wc': float(safe_ap_fn(y, s_wc)),
            'detection_rate': float(anomaly['run_alert'].mean()),
            'benign_alert_rate': float(benign['run_alert'].mean()),
            'median_time_to_detect_s': float(detected['time_to_detect_s'].median()) if not detected.empty else np.nan,
            'block_B': float(block_B),
            'alpha': float(alpha),
            'persist_k': float(persist_k),
            'gain': float(gain),
            **boot,
        })

    budget_df = pd.DataFrame(budget_rows).sort_values('budget_pct').reset_index(drop=True)

    summary_out = profile_paper_dir(profile) / 'dse_feature_budget_summary.csv'
    pred_out = profile_paper_dir(profile) / 'dse_feature_budget_case_predictions.csv'

    budget_df.to_csv(summary_out, index=False)
    pd.DataFrame(prediction_rows).to_csv(pred_out, index=False)

    summary_written.append(summary_out)
    prediction_written.append(pred_out)

    print(f'Wrote {summary_out}')
    print(f'Wrote {pred_out}')

if summary_written:
    display(Markdown('#### Exported feature-budget summaries'))
    display(pd.DataFrame({'path': [str(p) for p in summary_written]}))

if prediction_written:
    display(Markdown('#### Exported feature-budget case predictions'))
    display(pd.DataFrame({'path': [str(p) for p in prediction_written]}))

if ranking_written:
    display(Markdown('#### Exported feature rankings'))
    display(pd.DataFrame({'path': [str(p) for p in ranking_written]}))

if not summary_written and not prediction_written and not ranking_written:
    display(Markdown('No feature-budget artifacts were written. Check that the global result bundles exist first.'))

In [ ]:
# Plain-language: This cell assembles the design-space tables that compare feature budgets and deployment tradeoffs.
display(Markdown('### Design-space comparison tables'))

design_csv = COMPARISON_DIR / 'design_space_recommendations_by_profile.csv'
two_stage_comparison_path = COMPARISON_DIR / 'two_stage_dice_profiles.csv'
ready_profiles = [p for p in PROFILE_ORDER if p in FEATURE_PROFILES_TO_SWEEP]

summary_columns = [
    'feature_profile',
    'profile_label',
    'final_features',
    'final_roc_auc',
    'final_pr_auc',
    'final_detect_rate',
    'final_benign_alert_rate',
    'final_median_ttd_s',
    'selected_gain',
    'selected_block_B',
    'selected_alpha',
    'selected_persist_k',
    'two_stage_policy',
    'two_stage_budget',
    'two_stage_detect_rate',
    'two_stage_benign_alert_rate',
    'feature_budget_pct',
    'feature_budget_n_features',
    'feature_budget_roc_auc',
    'feature_budget_pr_auc',
    'feature_budget_detect_rate',
    'feature_budget_benign_alert_rate',
]

rows = []
notes = []

for profile in ready_profiles:
    row = {
        'feature_profile': profile,
        'profile_label': PROFILE_LABEL.get(profile, profile),
        'final_features': np.nan,
        'final_roc_auc': np.nan,
        'final_pr_auc': np.nan,
        'final_detect_rate': np.nan,
        'final_benign_alert_rate': np.nan,
        'final_median_ttd_s': np.nan,
        'selected_gain': np.nan,
        'selected_block_B': np.nan,
        'selected_alpha': np.nan,
        'selected_persist_k': np.nan,
        'two_stage_policy': '',
        'two_stage_budget': np.nan,
        'two_stage_detect_rate': np.nan,
        'two_stage_benign_alert_rate': np.nan,
        'feature_budget_pct': np.nan,
        'feature_budget_n_features': np.nan,
        'feature_budget_roc_auc': np.nan,
        'feature_budget_pr_auc': np.nan,
        'feature_budget_detect_rate': np.nan,
        'feature_budget_benign_alert_rate': np.nan,
    }

    overall_path = profile_global_dir(profile) / 'overall_metrics.csv'
    seq_path = profile_global_dir(profile) / 'sequential_metrics.csv'
    tuning_path = default_tuning_out_dir(DATASET_ROOT, profile) / 'recommended_alert_config.csv'
    two_stage_path = profile_paper_dir(profile) / 'two_stage_dice_summary.csv'
    budget_path = profile_paper_dir(profile) / 'dse_feature_budget_summary.csv'

    if overall_path.exists():
        overall = pd.read_csv(overall_path)
        final_row = overall[overall['config'].astype(str) == FINAL_CONFIG]
        if not final_row.empty:
            final_row = final_row.iloc[0]
            row['final_features'] = float(final_row.get('n_features', np.nan))
            row['final_roc_auc'] = float(final_row.get('roc_auc', np.nan))
            row['final_pr_auc'] = float(final_row.get('pr_auc', np.nan))
    else:
        notes.append(f'- `{PROFILE_LABEL.get(profile, profile)}` overall metrics CSV is missing.')

    if seq_path.exists():
        seq = pd.read_csv(seq_path)
        final_seq = seq[seq['config'].astype(str) == FINAL_CONFIG]
        if not final_seq.empty:
            final_seq = final_seq.iloc[0]
            row['final_detect_rate'] = float(final_seq.get('anomaly_detect_rate', np.nan))
            row['final_benign_alert_rate'] = float(final_seq.get('benign_run_alert_rate', np.nan))
            row['final_median_ttd_s'] = float(final_seq.get('median_time_to_detect_s', np.nan))
    else:
        notes.append(f'- `{PROFILE_LABEL.get(profile, profile)}` sequential metrics CSV is missing.')

    if tuning_path.exists():
        tuning = pd.read_csv(tuning_path)
        if not tuning.empty:
            tuning_row = tuning.iloc[0]
            row['selected_gain'] = float(tuning_row.get('gain', np.nan))
            row['selected_block_B'] = float(tuning_row.get('block_B', np.nan))
            row['selected_alpha'] = float(tuning_row.get('alpha', np.nan))
            row['selected_persist_k'] = float(tuning_row.get('persist_k', np.nan))
    else:
        op = profile_operating_point(profile)
        row['selected_gain'] = float(op.get('gain', np.nan))
        row['selected_block_B'] = float(op.get('block_B', np.nan))
        row['selected_alpha'] = float(op.get('alpha', np.nan))
        row['selected_persist_k'] = float(op.get('persist_k', np.nan))
        notes.append(f'- `{PROFILE_LABEL.get(profile, profile)}` tuned operating-point CSV is missing; notebook defaults were used.')

    two_stage = None
    two_stage_source = ''

    if two_stage_path.exists():
        two_stage = pd.read_csv(two_stage_path)
        two_stage_source = 'profile'
    elif two_stage_comparison_path.exists():
        two_stage_all = pd.read_csv(two_stage_comparison_path)
        if 'feature_profile' in two_stage_all.columns:
            two_stage = two_stage_all[
                two_stage_all['feature_profile'].astype(str) == str(profile)
            ].copy()
            two_stage_source = 'comparison'
    else:
        notes.append(f'- `{PROFILE_LABEL.get(profile, profile)}` two-stage summary is missing.')

    if two_stage is not None and not two_stage.empty:
        two_stage = two_stage.copy()
        policy_col = 'policy_label' if 'policy_label' in two_stage.columns else 'policy'
        budget_col = 'active_feature_budget' if 'active_feature_budget' in two_stage.columns else 'avg_feature_budget'
        detect_col = 'anomaly_detect_rate' if 'anomaly_detect_rate' in two_stage.columns else 'detection_rate'
        benign_col = 'benign_run_alert_rate' if 'benign_run_alert_rate' in two_stage.columns else 'benign_alert_rate'

        candidates = two_stage.copy()
        if policy_col in candidates.columns:
            candidates = candidates[candidates[policy_col].astype(str).str.startswith('Two-stage')].copy()

        if not candidates.empty:
            sort_cols = [budget_col]
            ascending = [True]
            if benign_col in candidates.columns:
                sort_cols.append(benign_col)
                ascending.append(True)
            if detect_col in candidates.columns:
                sort_cols.append(detect_col)
                ascending.append(False)

            candidates = candidates.sort_values(sort_cols, ascending=ascending)
            ts_row = candidates.iloc[0]

            row['two_stage_policy'] = str(ts_row.get(policy_col, ''))
            row['two_stage_budget'] = float(ts_row.get(budget_col, np.nan))
            row['two_stage_detect_rate'] = float(ts_row.get(detect_col, np.nan))
            row['two_stage_benign_alert_rate'] = float(ts_row.get(benign_col, np.nan))

            if two_stage_source == 'comparison' and not two_stage_path.exists():
                notes.append(
                    f'- `{PROFILE_LABEL.get(profile, profile)}` two-stage values were filled from the combined comparison CSV because the per-profile summary CSV is missing.'
                )
        else:
            notes.append(f'- `{PROFILE_LABEL.get(profile, profile)}` two-stage rows were found but no valid Two-stage policy could be selected.')
    elif two_stage_path.exists() or two_stage_comparison_path.exists():
        notes.append(f'- `{PROFILE_LABEL.get(profile, profile)}` two-stage rows were found but none matched this profile.')

    if budget_path.exists():
        budget = pd.read_csv(budget_path)
        if not budget.empty:
            budget = budget.copy()

            for col in [
                'budget_pct',
                'n_selected_features',
                'roc_auc',
                'pr_auc',
                'detection_rate',
                'benign_alert_rate',
            ]:
                if col not in budget.columns:
                    budget[col] = np.nan

            candidates = budget.copy()

            if pd.notna(row['final_benign_alert_rate']):
                constrained = candidates[
                    candidates['benign_alert_rate'] <= row['final_benign_alert_rate'] + 0.05
                ].copy()
                if not constrained.empty:
                    candidates = constrained

            best_pr = candidates['pr_auc'].max()
            best_roc = candidates['roc_auc'].max()

            near_best = candidates[
                (candidates['pr_auc'] >= best_pr - 0.01) &
                (candidates['roc_auc'] >= best_roc - 0.01)
            ].copy()

            if near_best.empty:
                near_best = candidates[candidates['pr_auc'] >= best_pr - 0.01].copy()
            if near_best.empty:
                near_best = candidates.copy()

            near_best = near_best.sort_values(
                ['n_selected_features', 'benign_alert_rate', 'detection_rate'],
                ascending=[True, True, False],
            )

            b_row = near_best.iloc[0]
            row['feature_budget_pct'] = float(b_row.get('budget_pct', np.nan))
            row['feature_budget_n_features'] = float(b_row.get('n_selected_features', np.nan))
            row['feature_budget_roc_auc'] = float(b_row.get('roc_auc', np.nan))
            row['feature_budget_pr_auc'] = float(b_row.get('pr_auc', np.nan))
            row['feature_budget_detect_rate'] = float(b_row.get('detection_rate', np.nan))
            row['feature_budget_benign_alert_rate'] = float(b_row.get('benign_alert_rate', np.nan))
    else:
        notes.append(f'- `{PROFILE_LABEL.get(profile, profile)}` feature-budget summary is missing.')

    rows.append(row)

design_space_summary = pd.DataFrame(rows, columns=summary_columns)
design_space_summary.to_csv(design_csv, index=False)

display_cols = [
    'feature_profile',
    'profile_label',
    'final_features',
    'final_roc_auc',
    'final_pr_auc',
    'final_detect_rate',
    'final_benign_alert_rate',
    'final_median_ttd_s',
    'selected_gain',
    'selected_block_B',
    'selected_alpha',
    'selected_persist_k',
    'two_stage_policy',
    'two_stage_budget',
    'two_stage_detect_rate',
    'two_stage_benign_alert_rate',
]

if design_space_summary[['feature_budget_pct', 'feature_budget_n_features', 'feature_budget_pr_auc']].notna().any().any():
    display_cols += [
        'feature_budget_pct',
        'feature_budget_n_features',
        'feature_budget_roc_auc',
        'feature_budget_pr_auc',
        'feature_budget_detect_rate',
        'feature_budget_benign_alert_rate',
    ]

display(design_space_summary[display_cols].round(4))

if notes:
    display(Markdown('**Notes**\n' + '\n'.join(sorted(set(notes)))))


### Tuning sweep diagnostics across gains, blocks, alpha, and persistence


In [ ]:
# Plain-language: This cell shows tuning sweeps in an ITC-friendly way.
# Stage 1 varies ROC/PR, so the top row shows performance envelopes there.
# Stage 2 leaves ROC/PR flat in this dataset, so the bottom row shows the
# operational metrics that actually move: detection rate and median TTD.

display(Markdown('### Tuning sweep ROC/PR and operational landscapes'))

TUNING_SWEEP_SUMMARY_CSV = COMPARISON_DIR / 'tuning_parameter_sweep_summary_profiles.csv'
TUNING_SWEEP_BEST_CSV = COMPARISON_DIR / 'tuning_parameter_sweep_best_points.csv'

SWEEP_PR_COL = 'pr_auc'
SWEEP_ROC_COL = 'roc_auc'

FIG_BG = '#FCFCFF'
PANEL_BG = '#FBFDFF'
TEXT_DARK = '#0F172A'
TEXT_MID = '#475569'
GRID_SOFT = '#E2E8F0'
SPINE_SOFT = '#CBD5E1'

MINMAX_COLOR = '#BFD5EA'
IQR_COLOR = '#E9A35B'
MEDIAN_COLOR = '#14213D'
SELECTED_EDGE = '#CA6702'
BEST_COLOR = '#7C2D12'

SUPTITLE_SIZE = 19
PANEL_TITLE_SIZE = 17
AXIS_LABEL_SIZE = 13.6
TICK_SIZE = 12.2
LEGEND_SIZE = 12.0


def _local_tuning_mechanism_proxy(out_dir: Path) -> dict[str, float]:
    diag_path = out_dir / 'case_diagnosis_summary.csv'
    mechanism_path = out_dir / 'mechanism_group_summary.csv'
    result = {
        'diag_mechanism_top1_proxy': np.nan,
        'diag_mechanism_top2_proxy': np.nan,
        'diag_mechanism_top3_proxy': np.nan,
    }
    if not diag_path.exists() or not mechanism_path.exists():
        return result
    diag = pd.read_csv(diag_path)
    mechanism = pd.read_csv(mechanism_path)
    if diag.empty or mechanism.empty:
        return result
    diag = diag[diag['label'] == 1].copy()
    if diag.empty:
        return result
    target_map = (
        mechanism[['stressor', 'dominant_mechanism_mode']]
        .drop_duplicates()
        .set_index('stressor')['dominant_mechanism_mode']
        .to_dict()
    )
    diag['target_mode'] = diag['stressor'].map(target_map)
    diag = diag[diag['target_mode'].notna()].copy()
    if diag.empty:
        return result
    for k in [1, 2, 3]:
        cols = [f'top_mechanism_{i}' for i in range(1, k + 1) if f'top_mechanism_{i}' in diag.columns]
        if cols:
            result[f'diag_mechanism_top{k}_proxy'] = float(diag[cols].eq(diag['target_mode'], axis=0).any(axis=1).mean())
    return result


def _local_tuning_selective_top2(out_dir: Path, config: str = 'tier0_tier1_tier2') -> tuple[float, float]:
    candidates = []
    for name in [
        'stressor_hierarchical_abstain_sweep.csv',
        'stressor_hierarchical_abstain_sweep_post_alert.csv',
        'stressor_hierarchical_abstain_sweep_top_blocks.csv',
        'diagnosis_mode_comparison.csv',
    ]:
        path = out_dir / name
        if not path.exists():
            continue
        df = pd.read_csv(path)
        if df.empty:
            continue
        if 'config' in df.columns:
            rows = df[df['config'].astype(str) == str(config)]
            if not rows.empty:
                df = rows.copy()
        top2_col = next((c for c in ['selective_top2_acc', 'top2_acc'] if c in df.columns), None)
        cov_col = next((c for c in ['selective_coverage', 'coverage'] if c in df.columns), None)
        if top2_col is None:
            continue
        row = df.sort_values(top2_col, ascending=False).iloc[0]
        candidates.append((float(row.get(top2_col, np.nan)), float(row.get(cov_col, np.nan)) if cov_col else np.nan))
    if not candidates:
        return np.nan, np.nan
    candidates.sort(key=lambda x: (np.nan_to_num(x[0], nan=-1.0), np.nan_to_num(x[1], nan=-1.0)))
    return candidates[-1]


def _enrich_tuning_df(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    df = df.copy()
    needed = [
        'diag_selective_top2_acc',
        'diag_selective_coverage',
        'diag_mechanism_top1_proxy',
        'diag_mechanism_top2_proxy',
        'diag_mechanism_top3_proxy',
        'diag_strong_score',
        'has_diag_over_0_8',
    ]
    for col in needed:
        if col not in df.columns:
            df[col] = np.nan

    cache = {}
    for idx, row in df.iterrows():
        out_dir = Path(str(row.get('out_dir', '')))
        if not out_dir.exists():
            continue
        key = str(out_dir)
        if key not in cache:
            proxy_fn = globals().get('_load_tuning_mechanism_proxy') if '_load_tuning_mechanism_proxy' in globals() else None
            selective_fn = globals().get('_load_tuning_selective_top2') if '_load_tuning_selective_top2' in globals() else None
            proxy = proxy_fn(out_dir) if callable(proxy_fn) else _local_tuning_mechanism_proxy(out_dir)
            selective = selective_fn(out_dir) if callable(selective_fn) else _local_tuning_selective_top2(out_dir)
            cache[key] = {'proxy': proxy, 'selective': selective}

        proxy = cache[key]['proxy']
        selective_top2, selective_cov = cache[key]['selective']

        for col, val in proxy.items():
            current = pd.to_numeric(pd.Series([df.at[idx, col]]), errors='coerce').iloc[0]
            if not np.isfinite(current):
                df.at[idx, col] = val

        current_top2 = pd.to_numeric(pd.Series([df.at[idx, 'diag_selective_top2_acc']]), errors='coerce').iloc[0]
        current_cov = pd.to_numeric(pd.Series([df.at[idx, 'diag_selective_coverage']]), errors='coerce').iloc[0]
        if not np.isfinite(current_top2):
            df.at[idx, 'diag_selective_top2_acc'] = selective_top2
        if not np.isfinite(current_cov):
            df.at[idx, 'diag_selective_coverage'] = selective_cov

    df['diag_strong_score'] = df[['diag_top2_acc', 'diag_selective_top2_acc', 'diag_mechanism_top3_proxy']].max(axis=1, skipna=True)
    df['has_diag_over_0_8'] = (df['diag_strong_score'] >= 0.80).astype(int)
    return df


def _resolve_metric_col(df: pd.DataFrame, preferred: str, fallback: str) -> str:
    return preferred if preferred in df.columns else fallback


def _best_metric_row(df: pd.DataFrame, metric: str, *, higher_is_better: bool = True) -> pd.Series | None:
    if df.empty or metric not in df.columns:
        return None
    ranked = df.copy()
    ranked = ranked[np.isfinite(pd.to_numeric(ranked[metric], errors='coerce'))].copy()
    if ranked.empty:
        return None

    sort_cols = [metric]
    ascending = [not higher_is_better]

    for col, asc in [
        ('benign_run_alert_rate', True),
        ('anomaly_detect_rate', False),
        ('median_time_to_detect_s', True),
        ('diag_strong_score', False),
    ]:
        if col in ranked.columns:
            sort_cols.append(col)
            ascending.append(asc)

    ranked = ranked.sort_values(sort_cols, ascending=ascending)
    return ranked.iloc[0]


def _selected_sweep_row(df: pd.DataFrame, filters: dict[str, float | int]) -> pd.Series | None:
    if df.empty:
        return None
    mask = np.ones(len(df), dtype=bool)
    for col, target in filters.items():
        if col not in df.columns:
            return None
        series = pd.to_numeric(df[col], errors='coerce').to_numpy(dtype=float)
        target_num = pd.to_numeric(pd.Series([target]), errors='coerce').iloc[0]
        if pd.isna(target_num):
            return None
        mask &= np.isclose(series, float(target_num), rtol=1e-8, atol=1e-8)
    rows = df[mask].copy()
    if rows.empty:
        return None
    return rows.iloc[0]


def _fmt_tick(val) -> str:
    num = pd.to_numeric(pd.Series([val]), errors='coerce').iloc[0]
    if pd.isna(num):
        return str(val)
    num = float(num)
    if np.isclose(num, round(num)):
        return str(int(round(num)))
    if abs(num) >= 0.1:
        return f'{num:.2f}'.rstrip('0').rstrip('.')
    return f'{num:.3f}'.rstrip('0').rstrip('.')


def _compute_y_limits(values, *, floor=None, ceiling=None, frac_pad=0.10, min_pad=0.01):
    vals = pd.to_numeric(pd.Series(values), errors='coerce').to_numpy(dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        lo, hi = 0.0, 1.0
    else:
        lo = float(np.nanmin(vals))
        hi = float(np.nanmax(vals))
        span = hi - lo
        pad = min_pad if span <= 1e-10 else max(min_pad, frac_pad * span)
        lo = lo - pad
        hi = hi + pad

    if floor is not None:
        lo = max(float(floor), lo)
    if ceiling is not None:
        hi = min(float(ceiling), hi)

    if hi <= lo:
        hi = lo + max(min_pad * 2, 0.02)
    return (lo, hi)


def _style_envelope_ax(ax, title: str, xlabel: str, ylabel: str):
    ax.set_facecolor(PANEL_BG)
    for spine in ax.spines.values():
        spine.set_color(SPINE_SOFT)
        spine.set_linewidth(1.2)
    ax.set_title(title, fontsize=PANEL_TITLE_SIZE, fontweight='bold', color=TEXT_DARK, loc='left', pad=14)
    ax.set_xlabel(xlabel, fontsize=AXIS_LABEL_SIZE, fontweight='bold', color=TEXT_DARK, labelpad=10)
    ax.set_ylabel(ylabel, fontsize=AXIS_LABEL_SIZE, fontweight='bold', color=TEXT_DARK, labelpad=10)
    ax.tick_params(axis='both', labelsize=TICK_SIZE, colors=TEXT_DARK)
    ax.grid(axis='y', color=GRID_SOFT, linewidth=1.0, alpha=0.9)
    ax.set_axisbelow(True)


def _envelope_stats(df: pd.DataFrame, x_col: str, metric_col: str) -> pd.DataFrame:
    work = df[[x_col, metric_col]].copy()
    work[x_col] = pd.to_numeric(work[x_col], errors='coerce')
    work[metric_col] = pd.to_numeric(work[metric_col], errors='coerce')
    work = work[np.isfinite(work[x_col]) & np.isfinite(work[metric_col])].copy()
    if work.empty:
        return pd.DataFrame()

    grouped = work.groupby(x_col)[metric_col]
    env = pd.DataFrame({x_col: sorted(work[x_col].unique().tolist())}).set_index(x_col)
    env['y_min'] = grouped.min()
    env['y_q25'] = grouped.quantile(0.25)
    env['y_median'] = grouped.median()
    env['y_q75'] = grouped.quantile(0.75)
    env['y_max'] = grouped.max()
    return env.reset_index().sort_values(x_col).reset_index(drop=True)


def _set_clean_xaxis(ax, x_vals):
    x_vals = np.asarray(x_vals, dtype=float)
    x_pos = np.arange(len(x_vals), dtype=float)

    labels = [_fmt_tick(v) for v in x_vals]
    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, fontsize=TICK_SIZE, fontweight='bold', color=TEXT_DARK)

    rotate = len(labels) >= 4 or any(len(lbl) > 4 for lbl in labels)
    if rotate:
        for lbl in ax.get_xticklabels():
            lbl.set_rotation(26)
            lbl.set_ha('right')
            lbl.set_rotation_mode('anchor')

    ax.tick_params(axis='x', pad=6)

    if len(x_pos) == 1:
        ax.set_xlim(-0.5, 0.5)
    else:
        ax.set_xlim(float(x_pos[0]) - 0.15, float(x_pos[-1]) + 0.15)

    return x_pos


def _plot_envelope_panel(
    ax,
    df: pd.DataFrame,
    *,
    x_col: str,
    metric_col: str,
    title: str,
    xlabel: str,
    ylabel: str,
    selected_row: pd.Series | None,
    best_row: pd.Series | None,
    y_limits: tuple[float, float] | None = None,
):
    env = _envelope_stats(df, x_col, metric_col)
    if env.empty:
        ax.text(0.5, 0.5, 'No sweep rows available.', ha='center', va='center', fontsize=13, color=TEXT_MID)
        ax.set_axis_off()
        return

    _style_envelope_ax(ax, title, xlabel, ylabel)

    x_raw = env[x_col].to_numpy(dtype=float)
    x_pos = _set_clean_xaxis(ax, x_raw)

    y_min = env['y_min'].to_numpy(dtype=float)
    y_q25 = env['y_q25'].to_numpy(dtype=float)
    y_med = env['y_median'].to_numpy(dtype=float)
    y_q75 = env['y_q75'].to_numpy(dtype=float)
    y_max = env['y_max'].to_numpy(dtype=float)

    ax.fill_between(x_pos, y_min, y_max, color=MINMAX_COLOR, alpha=0.75, linewidth=0, zorder=1)
    ax.fill_between(x_pos, y_q25, y_q75, color=IQR_COLOR, alpha=0.88, linewidth=0, zorder=2)
    ax.plot(
        x_pos,
        y_med,
        color=MEDIAN_COLOR,
        linewidth=2.9,
        marker='o',
        markersize=8.4,
        markerfacecolor=MEDIAN_COLOR,
        markeredgecolor='white',
        markeredgewidth=1.0,
        zorder=4,
    )

    if y_limits is not None:
        ax.set_ylim(*y_limits)
    else:
        ax.set_ylim(*_compute_y_limits(np.concatenate([y_min, y_max]), min_pad=0.02))

    def _x_to_pos(x_value):
        x_value = pd.to_numeric(pd.Series([x_value]), errors='coerce').iloc[0]
        if pd.isna(x_value):
            return None
        matches = np.where(np.isclose(x_raw, float(x_value), rtol=1e-8, atol=1e-10))[0]
        if matches.size == 0:
            return None
        return float(x_pos[matches[0]])

    if selected_row is not None and x_col in selected_row.index and metric_col in selected_row.index:
        sx = _x_to_pos(selected_row[x_col])
        sy = pd.to_numeric(pd.Series([selected_row[metric_col]]), errors='coerce').iloc[0]
        if sx is not None and pd.notna(sy):
            ax.axvline(sx, color=SELECTED_EDGE, linewidth=1.9, linestyle=(0, (4, 3)), alpha=0.95, zorder=3)
            ax.scatter(
                sx,
                float(sy),
                s=165,
                facecolors='white',
                edgecolors=SELECTED_EDGE,
                linewidths=2.5,
                zorder=6,
            )

    if best_row is not None and x_col in best_row.index and metric_col in best_row.index:
        bx = _x_to_pos(best_row[x_col])
        by = pd.to_numeric(pd.Series([best_row[metric_col]]), errors='coerce').iloc[0]
        if bx is not None and pd.notna(by):
            ax.scatter(
                bx,
                float(by),
                marker='D',
                s=110,
                color=BEST_COLOR,
                edgecolor='white',
                linewidth=1.0,
                zorder=7,
            )


profiles = [p for p in PROFILE_ORDER if p in FEATURE_PROFILES_TO_SWEEP]
display_profile = globals().get('ACTIVE_RESULT_PROFILE', globals().get('DISPLAY_FEATURE_PROFILE'))
if display_profile not in profiles:
    display_profile = profiles[0] if profiles else None

combined_rows = []
best_tables = []
display_fig_path = None
display_profile_label = None
display_caption = None

for profile in profiles:
    tuning_dir = default_tuning_out_dir(DATASET_ROOT, profile)
    stage1_path = tuning_dir / 'sweep_stage1_gain_block.csv'
    stage2_path = tuning_dir / 'sweep_stage2_alpha_persist.csv'
    rec_path = tuning_dir / 'recommended_alert_config.csv'

    if not stage1_path.exists() or not stage2_path.exists():
        if profile == display_profile:
            display(Markdown(f'Skip **{PROFILE_LABEL.get(profile, profile)}**: run the tuning sweep first.'))
        continue

    stage1 = _enrich_tuning_df(pd.read_csv(stage1_path))
    stage2 = _enrich_tuning_df(pd.read_csv(stage2_path))
    stage1['profile_label'] = PROFILE_LABEL.get(profile, profile)
    stage2['profile_label'] = PROFILE_LABEL.get(profile, profile)
    combined_rows.extend([stage1.assign(stage_group='Stage 1'), stage2.assign(stage_group='Stage 2')])

    stage1_pr_col = _resolve_metric_col(stage1, SWEEP_PR_COL, 'pr_auc_wc')
    stage1_roc_col = _resolve_metric_col(stage1, SWEEP_ROC_COL, 'roc_auc_wc')

    selected = None
    if rec_path.exists():
        rec_df = pd.read_csv(rec_path)
        if not rec_df.empty:
            selected = rec_df.iloc[0]

    stage1_selected = (
        _selected_sweep_row(stage1, {'gain': float(selected['gain']), 'block_B': float(selected['block_B'])})
        if selected is not None else None
    )
    stage2_selected = (
        _selected_sweep_row(stage2, {'alpha': float(selected['alpha']), 'persist_k': float(selected['persist_k'])})
        if selected is not None else None
    )

    stage1_best_pr = _best_metric_row(stage1, stage1_pr_col, higher_is_better=True)
    stage1_best_roc = _best_metric_row(stage1, stage1_roc_col, higher_is_better=True)
    stage2_best_detect = _best_metric_row(stage2, 'anomaly_detect_rate', higher_is_better=True)
    stage2_best_ttd = _best_metric_row(stage2, 'median_time_to_detect_s', higher_is_better=False)

    for stage_label, metric_label, row, value_col in [
        ('Stage 1', 'AUC-PR', stage1_best_pr, stage1_pr_col),
        ('Stage 1', 'ROC-AUC', stage1_best_roc, stage1_roc_col),
        ('Stage 2', 'Detection rate', stage2_best_detect, 'anomaly_detect_rate'),
        ('Stage 2', 'Fastest TTD', stage2_best_ttd, 'median_time_to_detect_s'),
    ]:
        if row is None:
            continue
        best_tables.append({
            'feature_profile': profile,
            'stage_group': stage_label,
            'metric': metric_label,
            'gain': row.get('gain', np.nan),
            'block_B': row.get('block_B', np.nan),
            'alpha': row.get('alpha', np.nan),
            'persist_k': row.get('persist_k', np.nan),
            'metric_value': row.get(value_col, np.nan),
            'anomaly_detect_rate': row.get('anomaly_detect_rate', np.nan),
            'benign_run_alert_rate': row.get('benign_run_alert_rate', np.nan),
            'median_time_to_detect_s': row.get('median_time_to_detect_s', np.nan),
        })

    if profile == display_profile:
        stage1_pr_ylim = _compute_y_limits(
            pd.to_numeric(stage1[stage1_pr_col], errors='coerce').to_numpy(dtype=float),
            floor=0.0,
            ceiling=1.0,
            frac_pad=0.05,
            min_pad=0.004,
        )

        stage1_roc_ylim = _compute_y_limits(
            pd.to_numeric(stage1[stage1_roc_col], errors='coerce').to_numpy(dtype=float),
            floor=0.0,
            ceiling=1.0,
            frac_pad=0.05,
            min_pad=0.004,
        )

        stage2_detect_ylim = _compute_y_limits(
            pd.to_numeric(stage2['anomaly_detect_rate'], errors='coerce').to_numpy(dtype=float),
            floor=0.0,
            ceiling=1.0,
            frac_pad=0.08,
            min_pad=0.015,
        )

        stage2_ttd_ylim = _compute_y_limits(
            pd.to_numeric(stage2['median_time_to_detect_s'], errors='coerce').to_numpy(dtype=float),
            frac_pad=0.06,
            min_pad=2.0,
        )

        fig, axes = plt.subplots(2, 2, figsize=(16.4, 11.4), constrained_layout=False)
        fig.patch.set_facecolor(FIG_BG)

        _plot_envelope_panel(
            axes[0, 0],
            stage1,
            x_col='block_B',
            metric_col=stage1_pr_col,
            title='Stage 1 AUC-PR',
            xlabel='Block B',
            ylabel='Score',
            selected_row=stage1_selected,
            best_row=stage1_best_pr,
            y_limits=stage1_pr_ylim,
        )
        _plot_envelope_panel(
            axes[0, 1],
            stage1,
            x_col='block_B',
            metric_col=stage1_roc_col,
            title='Stage 1 ROC-AUC',
            xlabel='Block B',
            ylabel='Score',
            selected_row=stage1_selected,
            best_row=stage1_best_roc,
            y_limits=stage1_roc_ylim,
        )
        _plot_envelope_panel(
            axes[1, 0],
            stage2,
            x_col='alpha',
            metric_col='anomaly_detect_rate',
            title='Stage 2 detection rate',
            xlabel='Alpha',
            ylabel='Rate',
            selected_row=stage2_selected,
            best_row=stage2_best_detect,
            y_limits=stage2_detect_ylim,
        )
        _plot_envelope_panel(
            axes[1, 1],
            stage2,
            x_col='alpha',
            metric_col='median_time_to_detect_s',
            title='Stage 2 median time to detect',
            xlabel='Alpha',
            ylabel='Seconds',
            selected_row=stage2_selected,
            best_row=stage2_best_ttd,
            y_limits=stage2_ttd_ylim,
        )

        legend_handles = [
            Patch(facecolor=MINMAX_COLOR, edgecolor='none', alpha=0.75, label='Min-max'),
            Patch(facecolor=IQR_COLOR, edgecolor='none', alpha=0.88, label='IQR'),
            Line2D([0], [0], color=MEDIAN_COLOR, marker='o', markersize=8, linewidth=2.9, label='Median'),
            Line2D([0], [0], color=SELECTED_EDGE, linestyle=(0, (4, 3)), marker='o',
                   markerfacecolor='white', markeredgecolor=SELECTED_EDGE, markeredgewidth=2.0,
                   markersize=9, linewidth=1.9, label='Recommended operating point'),
            Line2D([0], [0], marker='D', color='white', markerfacecolor=BEST_COLOR,
                   markeredgecolor='white', markeredgewidth=1.0, markersize=9, linewidth=0,
                   label='Best single-metric row'),
        ]
        fig.legend(
            handles=legend_handles,
            loc='lower center',
            bbox_to_anchor=(0.5, 0.055),
            ncol=5,
            fontsize=LEGEND_SIZE,
            frameon=True,
            fancybox=True,
        )
        fig.legends[0].get_frame().set_facecolor('white')
        fig.legends[0].get_frame().set_edgecolor(SPINE_SOFT)
        fig.legends[0].get_frame().set_linewidth(1.0)

        display_profile_label = PROFILE_LABEL.get(profile, profile)
        fig.suptitle(
            f'{display_profile_label}: tuning sweep envelopes',
            fontsize=SUPTITLE_SIZE,
            fontweight='bold',
            color=TEXT_DARK,
            y=0.972,
        )
        fig.subplots_adjust(bottom=0.20, top=0.91, wspace=0.28, hspace=0.34)

        fig_dir = profile_paper_dir(profile) / 'figures'
        fig_dir.mkdir(parents=True, exist_ok=True)
        fig_path = fig_dir / 'fig_tuning_parameter_sweep_envelopes.png'
        fig.savefig(fig_path, dpi=220, bbox_inches='tight', facecolor=fig.get_facecolor())
        plt.close(fig)

        display_fig_path = fig_path
        display_caption = (
            'Top row shows where Stage 1 ROC-AUC and AUC-PR actually change across the gain/block sweep. '
            'Bottom row switches to Stage 2 operational metrics because Stage 2 ROC and PR are flat in this sweep. '
            'The recommended operating point follows the notebook tuning objective, while the diamond marks the best row for that single displayed metric; these may differ when the selector favors lower false alerts or stronger diagnosis support over raw PR/ROC alone.'
        )

if combined_rows:
    summary_df = pd.concat(combined_rows, ignore_index=True)
    summary_df.to_csv(TUNING_SWEEP_SUMMARY_CSV, index=False)
else:
    summary_df = pd.DataFrame()

if best_tables:
    best_all = pd.DataFrame(best_tables)
    best_all.to_csv(TUNING_SWEEP_BEST_CSV, index=False)
else:
    best_all = pd.DataFrame()

if display_fig_path is not None:
    display(Markdown(f'#### Paper tuning figure: {display_profile_label}'))
    display(IPythonImage(filename=str(display_fig_path), width=1320))
    if display_caption:
        display(Markdown(f'*{display_caption}*'))

    display_best = best_all[best_all['feature_profile'] == display_profile].copy() if not best_all.empty else pd.DataFrame()
    if not display_best.empty:
        display(Markdown('#### Best single-metric rows highlighted in the figure'))
        display(
            display_best[
                ['stage_group', 'metric', 'gain', 'block_B', 'alpha', 'persist_k', 'metric_value', 'anomaly_detect_rate', 'benign_run_alert_rate', 'median_time_to_detect_s']
            ].rename(
                columns={
                    'stage_group': 'Sweep stage',
                    'metric': 'Best-for metric',
                    'gain': 'Gain',
                    'block_B': 'Block B',
                    'alpha': 'Alpha',
                    'persist_k': 'Persist k',
                    'metric_value': 'Best score',
                    'anomaly_detect_rate': 'Detection rate',
                    'benign_run_alert_rate': 'Benign alert rate',
                    'median_time_to_detect_s': 'Median TTD (s)',
                }
            ).round(3)
        )

    hidden_profiles = [PROFILE_LABEL.get(p, p) for p in profiles if p != display_profile]
    if hidden_profiles:
        display(Markdown(
            f'Only the active paper profile (**{display_profile_label}**) is shown in the notebook. '
            f'Best-point exports still include any available rows for **{", ".join(hidden_profiles)}**.'
        ))
elif display_profile is not None:
    display(Markdown(f'No tuning sweep figure was available for **{PROFILE_LABEL.get(display_profile, display_profile)}**.'))
else:
    display(Markdown('No tuning sweep CSVs were available yet. Run the end-to-end cell with tuning enabled first.'))


In [ ]:
# Plain-language: This cell draws the design-space figure so readers can see how accuracy changes as the feature budget grows.
SHOW_APPENDIX_FIGURES = True

if not globals().get('SHOW_APPENDIX_FIGURES', False):
    display(Markdown('Appendix figure hidden in paper-focused notebook mode. Set `SHOW_APPENDIX_FIGURES = True` to display this figure in the notebook.'))
else:
    display(Markdown('### Design-space comparison figure'))

    design_csv = COMPARISON_DIR / 'design_space_recommendations_by_profile.csv'
    design_png = COMPARISON_FIG / 'fig_design_space_profile_comparison.png'

    if not design_csv.exists():
        display(Markdown('Run the design-space summary cell first so the comparison CSV exists.'))
    else:
        design_df = pd.read_csv(design_csv)

        summary_frames = []
        for profile in PROFILE_ORDER:
            summary_path = profile_paper_dir(profile) / 'dse_feature_budget_summary.csv'
            if summary_path.exists():
                s = pd.read_csv(summary_path).copy()
                if not s.empty:
                    s['feature_profile'] = profile
                    s['profile_label'] = PROFILE_LABEL.get(profile, profile)
                    summary_frames.append(s)

        if not summary_frames:
            display(Markdown('Run the feature-budget export cell first so the budget summaries exist.'))
        else:
            summary_df = pd.concat(summary_frames, ignore_index=True)
            available_profiles = [
                p for p in PROFILE_ORDER
                if p in set(summary_df['feature_profile'])
            ]

            required_cols = [
                'budget_pct', 'roc_auc', 'pr_auc',
                'roc_auc_boot_q05', 'roc_auc_boot_q25', 'roc_auc_boot_q75', 'roc_auc_boot_q95',
                'pr_auc_boot_q05', 'pr_auc_boot_q25', 'pr_auc_boot_q75', 'pr_auc_boot_q95',
            ]
            missing_required = [c for c in required_cols if c not in summary_df.columns]
            if missing_required:
                display(Markdown('Run the updated feature-budget export cell first so the bootstrap columns exist.'))
            else:
                if 'FEATURE_BUDGET_PCTS' not in globals():
                    FEATURE_BUDGET_PCTS = list(range(10, 101, 10))

                profile_colors = {
                    'mixed': '#355C7D',
                    'full': '#C44E52',
                }
                profile_markers = {
                    'mixed': 'o',
                    'full': 's',
                }

                def lighten_color(color, amount):
                    rgb = np.array(to_rgb(color))
                    white = np.ones(3)
                    return tuple(rgb * (1.0 - amount) + white * amount)

                fig, axes = plt.subplots(
                    1, 2,
                    figsize=(12.8, 4.8),
                    squeeze=False,
                    constrained_layout=False,
                )
                axes = axes[0]

                def apply_tight_ylim(ax, lower_vals, upper_vals, force_one_tick=False):
                    lower_vals = np.asarray(lower_vals, dtype=float)
                    upper_vals = np.asarray(upper_vals, dtype=float)
                    valid = np.isfinite(lower_vals) & np.isfinite(upper_vals)

                    if not np.any(valid):
                        ax.set_ylim(0.0, 1.02)
                        ax.set_yticks([0.00, 0.25, 0.50, 0.75, 1.00])
                        ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
                        return

                    ymin = float(np.min(lower_vals[valid]))
                    ymax = float(np.max(upper_vals[valid]))

                    span = max(ymax - ymin, 0.02)
                    lower_pad = max(0.012, 0.10 * span)
                    upper_pad = max(0.014, 0.14 * span)

                    lo = max(0.0, ymin - lower_pad)

                    if ymax >= 0.94 or force_one_tick:
                        hi = max(1.018, ymax + upper_pad)
                    else:
                        hi = ymax + upper_pad
                    hi = min(1.035, hi)

                    if hi - lo < 0.06:
                        mid = 0.5 * (lo + hi)
                        lo = max(0.0, mid - 0.035)
                        hi = min(1.035, mid + 0.035)

                    ax.set_ylim(lo, hi)

                    if force_one_tick or ymax >= 0.94:
                        base_ticks = np.linspace(lo, min(1.0, hi), 4)
                        ticks = np.unique(np.round(np.concatenate([base_ticks, [1.0]]), 2))
                        ticks = ticks[(ticks >= lo - 1e-9) & (ticks <= hi + 1e-9)]
                        ax.set_yticks(ticks)
                    else:
                        ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

                    ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))

                def add_budget_panel(
                    ax,
                    metric_col,
                    q05_col,
                    q25_col,
                    q75_col,
                    q95_col,
                    y_label,
                    title,
                    force_one_tick=False,
                ):
                    all_lower = []
                    all_upper = []

                    for profile in available_profiles:
                        d = summary_df[summary_df['feature_profile'] == profile].copy().sort_values('budget_pct')
                        if d.empty:
                            continue

                        x = d['budget_pct'].to_numpy(dtype=float)
                        y = d[metric_col].to_numpy(dtype=float)
                        q05 = d[q05_col].to_numpy(dtype=float)
                        q25 = d[q25_col].to_numpy(dtype=float)
                        q75 = d[q75_col].to_numpy(dtype=float)
                        q95 = d[q95_col].to_numpy(dtype=float)

                        valid = np.isfinite(x) & np.isfinite(y)
                        x = x[valid]
                        y = y[valid]
                        q05 = q05[valid]
                        q25 = q25[valid]
                        q75 = q75[valid]
                        q95 = q95[valid]
                        if len(x) == 0:
                            continue

                        line_color = profile_colors.get(profile, '#4E79A7')
                        marker = profile_markers.get(profile, 'o')

                        range_fill = lighten_color(line_color, 0.84)
                        iqr_fill = lighten_color(line_color, 0.58)
                        range_edge = lighten_color(line_color, 0.60)
                        iqr_edge = lighten_color(line_color, 0.32)

                        ax.fill_between(x, q05, q95, color=range_fill, alpha=0.95, linewidth=0, zorder=1)
                        ax.fill_between(x, q25, q75, color=iqr_fill, alpha=0.98, linewidth=0, zorder=2)

                        ax.plot(x, q05, color=range_edge, linewidth=0.8, alpha=0.95, zorder=1.5)
                        ax.plot(x, q95, color=range_edge, linewidth=0.8, alpha=0.95, zorder=1.5)
                        ax.plot(x, q25, color=iqr_edge, linewidth=0.9, alpha=0.95, zorder=2.5)
                        ax.plot(x, q75, color=iqr_edge, linewidth=0.9, alpha=0.95, zorder=2.5)

                        ax.plot(
                            x,
                            y,
                            color=line_color,
                            marker=marker,
                            linewidth=2.3,
                            markersize=6.2,
                            zorder=3,
                        )

                        rec_row = design_df[design_df['feature_profile'] == profile]
                        if not rec_row.empty and pd.notna(rec_row.iloc[0].get('feature_budget_pct', np.nan)):
                            sel_pct = float(rec_row.iloc[0]['feature_budget_pct'])
                            sel_match = d[np.isclose(d['budget_pct'], sel_pct)]
                            if not sel_match.empty:
                                sel = sel_match.iloc[0]
                                ax.scatter(
                                    [sel['budget_pct']],
                                    [sel[metric_col]],
                                    s=180,
                                    marker='*',
                                    color=line_color,
                                    edgecolor='black',
                                    linewidth=0.9,
                                    zorder=5,
                                )

                        all_lower.extend(q05.tolist())
                        all_upper.extend(q95.tolist())

                    ax.set_xlim(8, 102)
                    ax.set_xticks(FEATURE_BUDGET_PCTS)
                    ax.set_xlabel('Feature budget (%)')
                    ax.set_ylabel(y_label)
                    ax.set_title(title, fontweight='bold')
                    ax.grid(alpha=0.20)
                    apply_tight_ylim(ax, all_lower, all_upper, force_one_tick=force_one_tick)

                add_budget_panel(
                    axes[0],
                    metric_col='roc_auc',
                    q05_col='roc_auc_boot_q05',
                    q25_col='roc_auc_boot_q25',
                    q75_col='roc_auc_boot_q75',
                    q95_col='roc_auc_boot_q95',
                    y_label='ROC-AUC',
                    title='ROC-AUC by feature budget',
                    force_one_tick=True,
                )

                add_budget_panel(
                    axes[1],
                    metric_col='pr_auc',
                    q05_col='pr_auc_boot_q05',
                    q25_col='pr_auc_boot_q25',
                    q75_col='pr_auc_boot_q75',
                    q95_col='pr_auc_boot_q95',
                    y_label='AUC-PR',
                    title='AUC-PR by feature budget',
                    force_one_tick=False,
                )

                handles = []
                for profile in available_profiles:
                    handles.append(
                        Line2D(
                            [0], [0],
                            color=profile_colors.get(profile, '#4E79A7'),
                            marker=profile_markers.get(profile, 'o'),
                            linewidth=2.3,
                            markersize=6.2,
                            label=PROFILE_LABEL.get(profile, profile),
                        )
                    )

                handles += [
                    Line2D([0], [0], color='black', marker='*', linestyle='None', markersize=10, label='Selected budget'),
                    Patch(facecolor='#9FB4C9', edgecolor='none', label='Bootstrap IQR'),
                    Patch(facecolor='#E5EDF5', edgecolor='none', label='Bootstrap 90% range'),
                ]

                fig.legend(
                    handles=handles,
                    loc='lower center',
                    bbox_to_anchor=(0.5, -0.06),
                    ncol=len(handles),
                    frameon=False,
                    handlelength=1.8,
                    columnspacing=1.2,
                )

                fig.tight_layout(rect=[0.02, 0.16, 0.98, 0.98])
                fig.savefig(design_png, dpi=PRO_FIG_DPI, bbox_inches='tight', facecolor='white')
                display(Image(filename=str(design_png)))
                plt.close(fig)


### Diagnosis-aware DSE story figure


In [ ]:
# Plain-language: This cell combines feature-budget, diagnosis, and top-feature evidence
# into a clean three-row figure without in-plot explanations.

if not globals().get('SHOW_APPENDIX_FIGURES', False):
    display(Markdown('Appendix figure hidden in paper-focused notebook mode. Set `SHOW_APPENDIX_FIGURES = True` to display this figure in the notebook.'))
else:
    display(Markdown('### Diagnosis-aware DSE story figure'))

    DSE_STORY_SCORECARD_CSV = COMPARISON_DIR / 'industry_paper_scorecard_profiles.csv'
    DSE_STORY_DIAG_CSV = COMPARISON_DIR / 'industry_diagnosis_strength_profiles.csv'
    DSE_STORY_FEATURES_CSV = COMPARISON_DIR / 'industry_top_features_profiles.csv'
    DSE_STORY_PNG = COMPARISON_FIG / 'fig_dse_diagnosis_story_profiles.png'

    if not DSE_STORY_SCORECARD_CSV.exists() or not DSE_STORY_DIAG_CSV.exists() or not DSE_STORY_FEATURES_CSV.exists():
        display(Markdown('Run the industry scorecard cell first so the DSE-plus-diagnosis figure inputs exist.'))
    else:
        scorecard = pd.read_csv(DSE_STORY_SCORECARD_CSV)
        diag = pd.read_csv(DSE_STORY_DIAG_CSV)
        feat = pd.read_csv(DSE_STORY_FEATURES_CSV)

        profiles = [p for p in PROFILE_ORDER if p in set(scorecard['feature_profile'])]
        if not profiles:
            display(Markdown('No profile rows were found for the DSE-plus-diagnosis figure.'))
        else:
            profile_colors = globals().get('PROFILE_COLOR', {'mixed': '#355C7D', 'full': '#C44E52'})
            profile_markers = globals().get('PROFILE_MARKER', {'mixed': 'o', 'full': 's'})

            TEXT_DARK = '#0F172A'
            SPINE_SOFT = '#CBD5E1'
            GRID_SOFT = '#E2E8F0'
            TITLE_SIZE = 18
            LABEL_SIZE = 14.5
            TICK_SIZE = 12.8
            LEGEND_SIZE = 12.2

            def lighten_color(color, amount):
                rgb = np.array(to_rgb(color), dtype=float)
                white = np.ones(3, dtype=float)
                return tuple(rgb * (1.0 - amount) + white * amount)

            def _tight_limits(values, *, floor=None, ceiling=None, frac_pad=0.08, min_pad=0.01):
                vals = pd.to_numeric(pd.Series(values), errors='coerce').to_numpy(dtype=float)
                vals = vals[np.isfinite(vals)]
                if vals.size == 0:
                    lo, hi = 0.0, 1.0
                else:
                    lo = float(np.nanmin(vals))
                    hi = float(np.nanmax(vals))
                    span = hi - lo
                    pad = max(min_pad, frac_pad * span) if span > 1e-12 else min_pad
                    lo -= pad
                    hi += pad

                if floor is not None:
                    lo = max(float(floor), lo)
                if ceiling is not None:
                    hi = min(float(ceiling), hi)

                if hi <= lo:
                    hi = lo + max(min_pad * 2, 0.02)
                return lo, hi

            def _style_axis(ax, title, xlabel, ylabel):
                ax.set_title(title, fontsize=TITLE_SIZE, fontweight='bold', loc='left', color=TEXT_DARK, pad=12)
                ax.set_xlabel(xlabel, fontsize=LABEL_SIZE, fontweight='bold', color=TEXT_DARK, labelpad=10)
                ax.set_ylabel(ylabel, fontsize=LABEL_SIZE, fontweight='bold', color=TEXT_DARK, labelpad=10)
                ax.tick_params(axis='both', labelsize=TICK_SIZE, colors=TEXT_DARK)
                ax.grid(alpha=0.22, color=GRID_SOFT, linewidth=1.0)
                ax.set_axisbelow(True)
                for spine in ax.spines.values():
                    spine.set_color(SPINE_SOFT)
                    spine.set_linewidth(1.2)

            def _short_feature_label(name):
                text = str(name)
                text = text.replace('tier0:', 'T0 ')
                text = text.replace('tier1_alt:', 'T1 ')
                text = text.replace('tier2:', 'T2 ')
                text = text.replace('_', ' ')
                text = text.replace('ane ', 'ANE ')
                text = text.replace('gpu ', 'GPU ')
                text = text.replace('mhz', 'MHz')
                return text

            summary_by_profile = {}
            has_true_minmax = False

            for profile in profiles:
                summary_path = profile_paper_dir(profile) / 'dse_feature_budget_summary.csv'
                if summary_path.exists():
                    d = pd.read_csv(summary_path).sort_values('budget_pct').reset_index(drop=True)
                    if not d.empty:
                        summary_by_profile[profile] = d
                        if {'pr_auc_boot_min', 'pr_auc_boot_max'}.issubset(d.columns):
                            has_true_minmax = True

            if not summary_by_profile:
                display(Markdown('Run the feature-budget export cell first so the budget summaries exist.'))
            else:
                outer_band_label = 'Min-max' if has_true_minmax else 'Outer bootstrap range'

                fig, axes = plt.subplots(
                    3, 1,
                    figsize=(13.8, 17.4),
                    constrained_layout=False,
                    gridspec_kw={'height_ratios': [1.30, 0.95, 1.20]},
                )
                fig.patch.set_facecolor('white')

                # Row 1: PR figure with outer range + IQR
                ax = axes[0]
                pr_lower_all = []
                pr_upper_all = []

                budget_ticks = sorted({
                    int(v)
                    for profile in profiles
                    for v in summary_by_profile[profile]['budget_pct'].dropna().tolist()
                })

                for profile in profiles:
                    d = summary_by_profile.get(profile)
                    if d is None or d.empty:
                        continue

                    x = d['budget_pct'].to_numpy(dtype=float)
                    y = d['pr_auc'].to_numpy(dtype=float)

                    outer_low_col = 'pr_auc_boot_min' if 'pr_auc_boot_min' in d.columns else 'pr_auc_boot_q05'
                    outer_high_col = 'pr_auc_boot_max' if 'pr_auc_boot_max' in d.columns else 'pr_auc_boot_q95'

                    outer_low = d[outer_low_col].to_numpy(dtype=float)
                    outer_high = d[outer_high_col].to_numpy(dtype=float)
                    q25 = d['pr_auc_boot_q25'].to_numpy(dtype=float)
                    q75 = d['pr_auc_boot_q75'].to_numpy(dtype=float)

                    line_color = profile_colors.get(profile, '#334155')
                    marker = profile_markers.get(profile, 'o')

                    ax.fill_between(
                        x, outer_low, outer_high,
                        color=lighten_color(line_color, 0.82),
                        alpha=0.95,
                        linewidth=0,
                        zorder=1,
                    )
                    ax.fill_between(
                        x, q25, q75,
                        color=lighten_color(line_color, 0.56),
                        alpha=0.98,
                        linewidth=0,
                        zorder=2,
                    )
                    ax.plot(
                        x, y,
                        color=line_color,
                        marker=marker,
                        markersize=8.2,
                        linewidth=3.0,
                        markeredgecolor='white',
                        markeredgewidth=1.1,
                        zorder=3,
                    )

                    pr_lower_all.extend(outer_low.tolist())
                    pr_upper_all.extend(outer_high.tolist())

                _style_axis(ax, 'Global AUC-PR across feature budgets', 'Feature budget (%)', 'AUC-PR')
                ax.set_xlim(8, 102)
                ax.set_xticks(budget_ticks if budget_ticks else list(range(10, 101, 10)))
                ax.set_ylim(*_tight_limits(
                    np.concatenate([pr_lower_all, pr_upper_all]) if pr_lower_all and pr_upper_all else [0.0, 1.0],
                    floor=0.0,
                    ceiling=1.005,
                    frac_pad=0.05,
                    min_pad=0.004,
                ))

                # Row 2: diagnosis dumbbell chart
                ax = axes[1]
                metric_specs = [
                    ('exact_top2_acc', 'Exact Top-2'),
                    ('selective_top2_acc', 'Selective Top-2'),
                    ('mechanism_top3_coverage', 'Mechanism Top-3'),
                ]
                y_pos = np.arange(len(metric_specs), dtype=float)[::-1]
                diag_values_all = []

                for yi, (col, label) in zip(y_pos, metric_specs):
                    profile_vals = []
                    for profile in profiles:
                        row = diag[diag['feature_profile'] == profile]
                        val = (
                            float(row.iloc[0][col])
                            if not row.empty and col in row.columns and pd.notna(row.iloc[0][col])
                            else np.nan
                        )
                        profile_vals.append((profile, val))
                        if np.isfinite(val):
                            diag_values_all.append(val)

                    finite_vals = [val for _, val in profile_vals if np.isfinite(val)]
                    if finite_vals:
                        ax.plot(
                            [min(finite_vals), max(finite_vals)],
                            [yi, yi],
                            color=SPINE_SOFT,
                            linewidth=6.0,
                            solid_capstyle='round',
                            zorder=1,
                        )

                    for profile, val in profile_vals:
                        if np.isfinite(val):
                            ax.scatter(
                                val,
                                yi,
                                s=190,
                                marker=profile_markers.get(profile, 'o'),
                                color=profile_colors.get(profile, '#334155'),
                                edgecolor='white',
                                linewidth=1.3,
                                zorder=3,
                            )

                _style_axis(ax, 'Diagnosis quality comparison', 'Score', '')
                ax.set_ylabel('')
                ax.set_yticks(y_pos)
                ax.set_yticklabels([label for _, label in metric_specs], fontsize=TICK_SIZE, fontweight='bold', color=TEXT_DARK)
                ax.grid(axis='x', alpha=0.22, color=GRID_SOFT, linewidth=1.0)
                ax.grid(axis='y', alpha=0.0)
                ax.set_ylim(-0.6, len(metric_specs) - 0.4)
                ax.set_xlim(*_tight_limits(
                    diag_values_all if diag_values_all else [0.0, 1.0],
                    floor=0.0,
                    ceiling=1.0,
                    frac_pad=0.16,
                    min_pad=0.03,
                ))

                # Row 3: top-feature timing map
                ax = axes[2]
                top_feats = feat[(feat['rank'] <= 3) & (feat['feature_profile'].isin(profiles))].copy()

                if top_feats.empty:
                    ax.text(0.5, 0.5, 'No top-feature rows available.', ha='center', va='center', fontsize=13, color=TEXT_DARK)
                    ax.set_axis_off()
                else:
                    order_map = {profile: idx for idx, profile in enumerate(profiles)}
                    top_feats['profile_order'] = top_feats['feature_profile'].map(order_map)
                    top_feats = top_feats.sort_values(['profile_order', 'rank'], ascending=[True, True]).reset_index(drop=True)

                    top_feats['display_label'] = top_feats.apply(
                        lambda row: f"{PROFILE_LABEL.get(row['feature_profile'], row['feature_profile'])} R{int(row['rank'])}  {_short_feature_label(row['feature_name'])}",
                        axis=1,
                    )

                    y = np.arange(len(top_feats), dtype=float)[::-1]
                    onset = top_feats['earliest_block_s'].to_numpy(dtype=float)
                    hits = top_feats['hit_count'].to_numpy(dtype=float)
                    sizes = 120.0 + 11.0 * hits
                    x_left = max(0.0, float(np.nanmin(onset)) - 6.0)

                    for yy, xx, profile in zip(y, onset, top_feats['feature_profile']):
                        ax.hlines(
                            yy,
                            x_left,
                            xx,
                            color=lighten_color(profile_colors.get(profile, '#334155'), 0.70),
                            linewidth=3.0,
                            zorder=1,
                        )

                    for yy, xx, sz, profile in zip(y, onset, sizes, top_feats['feature_profile']):
                        ax.scatter(
                            xx,
                            yy,
                            s=sz,
                            marker=profile_markers.get(profile, 'o'),
                            color=profile_colors.get(profile, '#334155'),
                            edgecolor='white',
                            linewidth=1.2,
                            zorder=3,
                        )

                    _style_axis(ax, 'Top localized features by earliest evidence', 'Earliest localized window (s)', '')
                    ax.set_ylabel('')
                    ax.set_yticks(y)
                    ax.set_yticklabels(top_feats['display_label'].tolist(), fontsize=11.8, fontweight='bold', color=TEXT_DARK)
                    unique_onsets = sorted({float(v) for v in onset if np.isfinite(v)})
                    if unique_onsets:
                        ax.set_xticks(unique_onsets)
                    ax.set_xlim(*_tight_limits(
                        onset,
                        floor=0.0,
                        frac_pad=0.16,
                        min_pad=6.0,
                    ))
                    ax.grid(axis='x', alpha=0.22, color=GRID_SOFT, linewidth=1.0)
                    ax.grid(axis='y', alpha=0.0)

                profile_handles = [
                    Line2D(
                        [0], [0],
                        color=profile_colors.get(profile, '#334155'),
                        marker=profile_markers.get(profile, 'o'),
                        linewidth=3.0,
                        markersize=8.2,
                        label=PROFILE_LABEL.get(profile, profile.title()),
                    )
                    for profile in profiles
                ]

                band_handles = [
                    Patch(facecolor='#BFD5EA', edgecolor='none', alpha=0.95, label=outer_band_label),
                    Patch(facecolor='#E9A35B', edgecolor='none', alpha=0.98, label='IQR'),
                ]

                size_handles = []
                if not top_feats.empty:
                    hit_samples = sorted({int(np.nanmin(hits)), int(np.nanmedian(hits)), int(np.nanmax(hits))})
                    hit_samples = hit_samples[:3]
                    for hit in hit_samples:
                        size_handles.append(
                            Line2D(
                                [0], [0],
                                marker='o',
                                linestyle='None',
                                markerfacecolor='#94A3B8',
                                markeredgecolor='#475569',
                                markersize=max(7.0, np.sqrt(120.0 + 11.0 * hit) / 2.3),
                                label=f'{hit} hits',
                            )
                        )

                fig.legend(
                    handles=profile_handles + band_handles + size_handles,
                    loc='lower center',
                    bbox_to_anchor=(0.5, 0.018),
                    ncol=4,
                    frameon=False,
                    fontsize=LEGEND_SIZE,
                    columnspacing=1.4,
                    handletextpad=0.7,
                )

                fig.suptitle('Feature-budget DSE plus diagnosis story', fontsize=19, fontweight='bold', color=TEXT_DARK, y=0.985)
                fig.subplots_adjust(left=0.31, right=0.97, bottom=0.10, top=0.94, hspace=0.38)

                fig.savefig(DSE_STORY_PNG, dpi=PRO_FIG_DPI, bbox_inches='tight', facecolor='white')
                plt.close(fig)
                display(Image(filename=str(DSE_STORY_PNG)))


## 6B. Runtime and Power Context

This section separates three related but different ideas:
- runtime context,
- deployment cost,
- and host power context.

The released dataset already supports workload power context. A true detector-overhead claim, however, needs paired benign runs with and without DICE enabled. The helper cell below is for collecting those paired traces.


In [ ]:
# Plain-language: This cell summarizes low-overhead operating points for readers who care most about practical deployment cost.
display(Markdown('### Low-overhead comparison tables'))

low_overhead_csv = COMPARISON_DIR / 'low_overhead_story_by_profile.csv'
rows = []
for profile in PROFILE_ORDER:
    overall_path = profile_global_dir(profile) / 'overall_metrics.csv'
    sequential_path = profile_global_dir(profile) / 'sequential_metrics.csv'
    if not overall_path.exists() or not sequential_path.exists():
        continue
    overall = pd.read_csv(overall_path)
    sequential = pd.read_csv(sequential_path)
    final_row = overall.loc[overall['config'] == FINAL_CONFIG].iloc[0]
    seq_row = sequential.loc[sequential['config'] == FINAL_CONFIG].iloc[0]

    two_stage_path = profile_paper_dir(profile) / 'two_stage_dice_summary.csv'
    if two_stage_path.exists():
        two_stage = pd.read_csv(two_stage_path)
    else:
        combined_two_stage_path = COMPARISON_DIR / 'two_stage_dice_profiles.csv'
        if combined_two_stage_path.exists():
            two_stage = pd.read_csv(combined_two_stage_path)
            two_stage = two_stage[two_stage['feature_profile'] == profile].copy()
        else:
            two_stage = pd.DataFrame()
    if not two_stage.empty:
        cand = two_stage[two_stage['policy'].astype(str).str.startswith('Two-stage')].copy()
        cand = cand.sort_values(['avg_feature_budget', 'benign_alert_rate'], ascending=[True, True]) if not cand.empty else cand
        rec_two_stage = cand.iloc[0] if not cand.empty else None
    else:
        rec_two_stage = None

    feature_budget_path = profile_paper_dir(profile) / 'dse_feature_budget_summary.csv'
    feature_budget = pd.read_csv(feature_budget_path) if feature_budget_path.exists() else pd.DataFrame()
    rec_budget = feature_budget.sort_values(['n_selected_features', 'pr_auc_wc'], ascending=[True, False]).iloc[0] if not feature_budget.empty else None

    rows.append({
        'feature_profile': profile,
        'profile_label': PROFILE_LABEL[profile],
        'final_features': float(final_row['n_features']),
        'final_detect_rate': float(seq_row['anomaly_detect_rate']),
        'final_benign_alert_rate': float(seq_row['benign_run_alert_rate']),
        'final_median_ttd_s': float(seq_row['median_time_to_detect_s']),
        'two_stage_policy': str(rec_two_stage['policy']) if rec_two_stage is not None else '',
        'two_stage_budget': float(rec_two_stage['avg_feature_budget']) if rec_two_stage is not None else np.nan,
        'two_stage_detect_rate': float(rec_two_stage['detection_rate']) if rec_two_stage is not None else np.nan,
        'two_stage_benign_alert_rate': float(rec_two_stage['benign_alert_rate']) if rec_two_stage is not None else np.nan,
        'feature_budget_pct': float(rec_budget['budget_pct']) if rec_budget is not None else np.nan,
        'feature_budget_n_features': float(rec_budget['n_selected_features']) if rec_budget is not None else np.nan,
        'feature_budget_detect_rate': float(rec_budget['detection_rate']) if rec_budget is not None else np.nan,
        'feature_budget_benign_alert_rate': float(rec_budget['benign_alert_rate']) if rec_budget is not None else np.nan,
    })

if not rows:
    display(Markdown('No low-overhead comparison is ready yet. Run the two-stage and feature-budget cells first.'))
else:
    low_overhead_df = pd.DataFrame(rows)
    low_overhead_df.to_csv(low_overhead_csv, index=False)
    display(low_overhead_df.round(4))


### Optional paired-overhead collection and Tier-1 host power context

Use the helper cell below if you want a clean side-by-side measurement of detector overhead.

The recommended setup is to collect paired benign runs for `BROWSER`, `VIDEO_SW`, `PY_AI`, and `PY_STATS`:
- one baseline run without DICE,
- one run with DICE enabled.

Keeping this study benign-only helps separate detector cost from stressor cost. The analysis cell after it summarizes workload power context from the released Tier-1 traces and computes true power overhead once the paired files exist.


In [ ]:
# Plain-language: This cell provides an optional helper for collecting paired host-power traces during benign baseline and DICE-on runs.
display(Markdown("### Paired benign power-overhead collection helper"))


POWER_TOOL = REPO_ROOT / "data generation" / "tools" / "collect_power_overhead_pairs.py"
if not POWER_TOOL.exists():
    raise FileNotFoundError(f"Missing power helper script: {POWER_TOOL}")

spec = importlib.util.spec_from_file_location("dice_power_overhead_pairs", POWER_TOOL)
power_tool = importlib.util.module_from_spec(spec)
sys.modules["dice_power_overhead_pairs"] = power_tool
spec.loader.exec_module(power_tool)

POWER_PAIRS_ROOT = DATASET_ROOT / "power_overhead_pairs"
POWER_PAIRS_ROOT.mkdir(parents=True, exist_ok=True)
POWER_MANIFEST = power_tool.ensure_manifest(POWER_PAIRS_ROOT)

def collect_power_overhead_pairs(
    mode: str,
    workloads=None,
    duration_s: int = 1000,
    tier1_alt_bin: str = "macmon",
    runtime_config: str = "tier0_tier1",
    feature_profile: str = "mixed",
    fit_ratio: float = 0.6,
    block_B: int = 60,
    alpha: float = 0.05,
    gain: float = 0.35,
    ridge_lambda: float = 1e-3,
    runtime_hz: int = 1,
):
    workloads = workloads or list(power_tool.WORKLOADS_POWER)
    runtime_cfg = None
    bundle = None
    if mode == "dice_on":
        runtime_cfg = power_tool.RuntimeConfig(
            dataset_root=DATASET_ROOT,
            feature_profile=feature_profile,
            config_name=runtime_config,
            fit_ratio=fit_ratio,
            block_B=block_B,
            alpha=alpha,
            gain=gain,
            ridge_lambda=ridge_lambda,
            runtime_hz=runtime_hz,
        )
        bundle = power_tool.prepare_bundle(runtime_cfg)

    rows = []
    for workload in workloads:
        out_csv = power_tool.collect_one(
            dataset_root=DATASET_ROOT,
            pairs_root=POWER_PAIRS_ROOT,
            workload=workload,
            mode=mode,
            tier1_alt_bin=tier1_alt_bin,
            duration_s=duration_s,
            runtime_cfg=runtime_cfg,
            bundle=bundle,
        )
        rows.append({"workload": workload, "mode": mode, "output_csv": str(out_csv)})
    out_df = pd.DataFrame(rows)
    display(out_df)
    return out_df


def verify_power_overhead_pairs():
    manifest = pd.read_csv(POWER_MANIFEST).copy()
    manifest["abs_path"] = manifest["csv_path"].apply(lambda p: POWER_PAIRS_ROOT / p)
    manifest["exists"] = manifest["abs_path"].apply(Path.exists)
    display(manifest[["workload", "mode", "abs_path", "exists"]])
    return manifest


if POWER_MANIFEST.exists():
    display(pd.read_csv(POWER_MANIFEST))
else:
    display(Markdown(f"Manifest not created yet at `{POWER_MANIFEST}`."))

print("Run collect_power_overhead_pairs(mode='baseline') first, then collect_power_overhead_pairs(mode='dice_on'), then verify_power_overhead_pairs().")

In [ ]:
# Plain-language: This cell summarizes the released host-power traces so readers can understand the Tier-1 power context.
display(Markdown("### Tier-1 host power context from released traces"))

paper_full = OUT_PAPER / "full"
paper_fig = paper_full / "figures"
paper_full.mkdir(parents=True, exist_ok=True)
paper_fig.mkdir(parents=True, exist_ok=True)

WORKLOADS_LOCAL = ["BROWSER", "VIDEO_SW", "PY_AI", "PY_STATS"]
STRESSORS_LOCAL = ["NOMINAL", "CACHE", "TLB", "BRANCH", "MEMBW", "ATOMIC"]
POWER_CANDIDATES = ["sys_power", "all_power", "processor_power_w", "package_power_w", "soc_power_w", "cpu_power_w"]


def lighten_color(color, amount):
    rgb = np.array(to_rgb(color))
    white = np.ones(3)
    return tuple(rgb * (1.0 - amount) + white * amount)


def tight_xlim_from_values(values, include_zero=False, min_span=1.0, pad_frac=0.12, min_pad=0.25):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return (0.0, 1.0)

    xmin = float(np.min(vals))
    xmax = float(np.max(vals))
    if include_zero:
        xmin = min(xmin, 0.0)
        xmax = max(xmax, 0.0)

    span = max(xmax - xmin, min_span)
    pad = max(min_pad, pad_frac * span)
    lo = xmin - pad
    hi = xmax + pad

    if include_zero:
        if xmin >= 0:
            lo = min(lo, -0.05 * span)
        if xmax <= 0:
            hi = max(hi, 0.05 * span)

    return lo, hi


def load_tier1_power_trace(case_id: str):
    case_dir = DATASET_ROOT / "tier1_alt" / case_id
    for fname in ["tier1_alt_full_5hz.csv", "tier1_alt_core_5hz.csv"]:
        p = case_dir / fname
        if not p.exists():
            continue
        df = pd.read_csv(p)
        for col in POWER_CANDIDATES:
            if col in df.columns:
                return df, col, fname
    return None, None, None


power_rows = []
for workload in WORKLOADS_LOCAL:
    for stressor in STRESSORS_LOCAL:
        case_id = f"{workload}__{stressor}"
        df, power_col, source_file = load_tier1_power_trace(case_id)
        if df is None:
            continue

        power = pd.to_numeric(df[power_col], errors="coerce").dropna()
        if len(power) == 0:
            continue

        duration_s = float(len(power) / 5.0)
        power_rows.append(
            {
                "case_id": case_id,
                "workload": workload,
                "stressor": stressor,
                "source_file": source_file,
                "power_col": power_col,
                "mean_power_w": float(power.mean()),
                "median_power_w": float(power.median()),
                "p95_power_w": float(power.quantile(0.95)),
                "duration_s": duration_s,
                "energy_j": float(power.mean() * duration_s),
            }
        )

if len(power_rows) == 0:
    display(Markdown("No Tier-1 power traces were found under `tier1_alt`."))
else:
    power_df = pd.DataFrame(power_rows).sort_values(["workload", "stressor"]).reset_index(drop=True)
    nominal_map = power_df[power_df["stressor"] == "NOMINAL"].set_index("workload")["mean_power_w"].to_dict()
    power_df["delta_vs_nominal_w"] = power_df["mean_power_w"] - power_df["workload"].map(nominal_map)
    power_df.to_csv(paper_full / "tier1_power_context_cases.csv", index=False)

    workload_power = power_df[power_df["stressor"] == "NOMINAL"][
        ["workload", "source_file", "power_col", "mean_power_w", "p95_power_w", "energy_j"]
    ].sort_values("workload")
    workload_power.to_csv(paper_full / "tier1_power_context_workloads.csv", index=False)

    stressor_power = (
        power_df[power_df["stressor"] != "NOMINAL"]
        .groupby("stressor", as_index=False)
        .agg(
            n_cases=("case_id", "count"),
            mean_power_w=("mean_power_w", "mean"),
            mean_delta_vs_nominal_w=("delta_vs_nominal_w", "mean"),
            median_delta_vs_nominal_w=("delta_vs_nominal_w", "median"),
            mean_energy_j=("energy_j", "mean"),
        )
        .sort_values("stressor")
    )
    stressor_power.to_csv(paper_full / "tier1_power_context_stressors.csv", index=False)

    display(workload_power.round(4))
    display(stressor_power.round(4))

    nominal_view = workload_power.sort_values("mean_power_w").reset_index(drop=True)
    stressor_view = stressor_power.sort_values("mean_delta_vs_nominal_w").reset_index(drop=True)

    fig, axes = plt.subplots(
        1, 2,
        figsize=(14.4, 5.6),
        gridspec_kw={"width_ratios": [1.0, 1.12]},
        constrained_layout=False,
    )

    title_fs = 16.5
    label_fs = 12.8
    tick_fs = 11.5
    number_fs = 11.2

    # ------------------------------------------------------------
    # Panel 1: nominal workload power context
    # ------------------------------------------------------------
    ax = axes[0]
    y0 = np.arange(len(nominal_view), dtype=float)

    mean_color = "#3E6BA5"
    p95_color = "#E68613"
    connector_fill = lighten_color(mean_color, 0.70)
    connector_edge = lighten_color(mean_color, 0.42)

    for yi in y0:
        if int(yi) % 2 == 0:
            ax.axhspan(yi - 0.44, yi + 0.44, color="#F8FAFC", zorder=0)

    mean_vals = nominal_view["mean_power_w"].to_numpy(dtype=float)
    p95_vals = nominal_view["p95_power_w"].to_numpy(dtype=float)

    for yi, row in zip(y0, nominal_view.itertuples(index=False)):
        x1 = float(row.mean_power_w)
        x2 = float(row.p95_power_w)
        lo, hi = sorted([x1, x2])

        ax.plot([lo, hi], [yi, yi], color=connector_fill, linewidth=11, solid_capstyle="round", zorder=1)
        ax.plot([lo, hi], [yi, yi], color=connector_edge, linewidth=1.2, solid_capstyle="round", zorder=1.2)

    ax.scatter(
        mean_vals,
        y0,
        s=220,
        color=mean_color,
        edgecolor="white",
        linewidth=1.5,
        zorder=3,
    )
    ax.scatter(
        p95_vals,
        y0,
        s=150,
        marker="D",
        color=p95_color,
        edgecolor="white",
        linewidth=1.4,
        zorder=4,
    )

    xlim_left = tight_xlim_from_values(
        np.concatenate([mean_vals, p95_vals]),
        include_zero=False,
        min_span=3.0,
        pad_frac=0.15,
        min_pad=0.6,
    )
    x_span_left = xlim_left[1] - xlim_left[0]
    gap_thr_left = max(0.30, 0.06 * x_span_left)
    edge_thr_left = max(0.35, 0.08 * x_span_left)
    nudge_left = max(0.18, 0.04 * x_span_left)
    far_nudge_left = max(0.34, 0.08 * x_span_left)

    for yi, row in zip(y0, nominal_view.itertuples(index=False)):
        x_mean = float(row.mean_power_w)
        x_p95 = float(row.p95_power_w)
        gap = abs(x_p95 - x_mean)

        if gap < gap_thr_left:
            mx = x_mean - far_nudge_left
            my = yi - 0.22
            mha = "right"
            mean_arrow = True
        elif x_mean < xlim_left[0] + edge_thr_left:
            mx = x_mean + nudge_left
            my = yi
            mha = "left"
            mean_arrow = False
        else:
            mx = x_mean - nudge_left
            my = yi
            mha = "right"
            mean_arrow = False

        mean_kwargs = dict(
            xy=(x_mean, yi),
            xytext=(mx, my),
            textcoords="data",
            ha=mha,
            va="center",
            fontsize=number_fs,
            fontweight="bold",
            color=mean_color,
            bbox=dict(boxstyle="round,pad=0.16", fc="white", ec="none", alpha=0.95),
        )
        if mean_arrow:
            mean_kwargs["arrowprops"] = dict(
                arrowstyle="-",
                color=lighten_color(mean_color, 0.35),
                linewidth=1.0,
                shrinkA=5,
                shrinkB=5,
            )
        ax.annotate(f"{x_mean:.1f}", **mean_kwargs)

        if gap < gap_thr_left:
            px = x_p95 + far_nudge_left
            py = yi + 0.22
            pha = "left"
            p95_arrow = True
        elif x_p95 > xlim_left[1] - edge_thr_left:
            px = x_p95 - nudge_left
            py = yi
            pha = "right"
            p95_arrow = False
        else:
            px = x_p95 + nudge_left
            py = yi
            pha = "left"
            p95_arrow = False

        p95_kwargs = dict(
            xy=(x_p95, yi),
            xytext=(px, py),
            textcoords="data",
            ha=pha,
            va="center",
            fontsize=number_fs,
            fontweight="bold",
            color="#8A4B08",
            bbox=dict(boxstyle="round,pad=0.16", fc="white", ec="none", alpha=0.95),
        )
        if p95_arrow:
            p95_kwargs["arrowprops"] = dict(
                arrowstyle="-",
                color=lighten_color(p95_color, 0.35),
                linewidth=1.0,
                shrinkA=5,
                shrinkB=5,
            )
        ax.annotate(f"{x_p95:.1f}", **p95_kwargs)

    ax.set_xlim(*xlim_left)
    ax.set_yticks(y0)
    ax.set_yticklabels(nominal_view["workload"], fontsize=tick_fs)
    ax.invert_yaxis()
    ax.set_xlabel("Host power (W)", fontsize=label_fs)
    ax.set_title("Nominal workload power", fontsize=title_fs, fontweight="bold", pad=11)
    ax.grid(axis="x", alpha=0.18)
    ax.tick_params(axis="x", labelsize=tick_fs)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    # ------------------------------------------------------------
    # Panel 2: stressor power shift
    # ------------------------------------------------------------
    ax = axes[1]
    y1 = np.arange(len(stressor_view), dtype=float)

    pos_color = "#D1495B"
    neg_color = "#4E9F69"
    median_marker_edge = "#1F2937"
    median_offset = 0.23

    for yi in y1:
        if int(yi) % 2 == 0:
            ax.axhspan(yi - 0.44, yi + 0.44, color="#F8FAFC", zorder=0)

    mean_vals = stressor_view["mean_delta_vs_nominal_w"].to_numpy(dtype=float)
    median_vals = stressor_view["median_delta_vs_nominal_w"].to_numpy(dtype=float)

    bar_colors = [pos_color if v >= 0 else neg_color for v in mean_vals]
    bar_fills = [lighten_color(c, 0.55) for c in bar_colors]

    ax.barh(
        y1,
        mean_vals,
        height=0.58,
        color=bar_fills,
        edgecolor=bar_colors,
        linewidth=1.2,
        zorder=2,
    )

    median_y = y1 + median_offset
    ax.vlines(
        median_vals,
        y1 - 0.12,
        median_y - 0.04,
        colors=lighten_color(median_marker_edge, 0.35),
        linewidth=1.1,
        zorder=3,
    )
    ax.scatter(
        median_vals,
        median_y,
        s=110,
        marker="o",
        facecolor="white",
        edgecolor=median_marker_edge,
        linewidth=1.5,
        zorder=4,
    )

    xlim_right = tight_xlim_from_values(
        np.concatenate([mean_vals, median_vals]),
        include_zero=True,
        min_span=1.2,
        pad_frac=0.15,
        min_pad=0.35,
    )
    x_span = xlim_right[1] - xlim_right[0]
    near_zero_thr = max(0.18, 0.08 * x_span)
    edge_thr = max(0.16, 0.08 * x_span)
    label_nudge = max(0.14, 0.05 * x_span)
    label_far_nudge = max(0.24, 0.10 * x_span)

    for yi, row in zip(y1, stressor_view.itertuples(index=False)):
        x = float(row.mean_delta_vs_nominal_w)
        label_color = pos_color if x >= 0 else neg_color

        if abs(x) < near_zero_thr:
            x_text = x + (label_far_nudge if x >= 0 else -label_far_nudge)
            y_text = yi - 0.24
            ha = "left" if x >= 0 else "right"
        elif x > xlim_right[1] - edge_thr:
            x_text = x - label_nudge
            y_text = yi
            ha = "right"
        elif x < xlim_right[0] + edge_thr:
            x_text = x + label_nudge
            y_text = yi
            ha = "left"
        else:
            x_text = x + (label_nudge if x >= 0 else -label_nudge)
            y_text = yi
            ha = "left" if x >= 0 else "right"

        annot_kwargs = dict(
            xy=(x, yi),
            xytext=(x_text, y_text),
            textcoords="data",
            ha=ha,
            va="center",
            fontsize=number_fs,
            fontweight="bold",
            color=label_color,
            bbox=dict(boxstyle="round,pad=0.16", fc="white", ec="none", alpha=0.95),
        )

        if abs(y_text - yi) > 1e-9:
            annot_kwargs["arrowprops"] = dict(
                arrowstyle="-",
                color=lighten_color(label_color, 0.35),
                linewidth=1.0,
                shrinkA=5,
                shrinkB=5,
            )

        ax.annotate(f"{x:+.1f}", **annot_kwargs)

    ax.set_xlim(*xlim_right)
    ax.axvline(0.0, color="#111827", linewidth=1.2, zorder=1)
    ax.set_yticks(y1)
    ax.set_yticklabels(stressor_view["stressor"], fontsize=tick_fs)
    ax.set_ylim(len(stressor_view) - 0.5 + median_offset, -0.6)
    ax.set_xlabel("Mean power shift vs nominal (W)", fontsize=label_fs)
    ax.set_title("Stressor power shift", fontsize=title_fs, fontweight="bold", pad=11)
    ax.grid(axis="x", alpha=0.18)
    ax.tick_params(axis="x", labelsize=tick_fs)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    legend_handles = [
        Line2D([0], [0], marker="o", linestyle="None", markersize=10, markerfacecolor=mean_color, markeredgecolor="white", label="Nominal mean"),
        Line2D([0], [0], marker="D", linestyle="None", markersize=9, markerfacecolor=p95_color, markeredgecolor="white", label="Nominal p95"),
        Patch(facecolor=lighten_color(pos_color, 0.55), edgecolor=pos_color, label="Mean stressor shift"),
        Line2D([0], [0], marker="o", linestyle="None", markersize=8, markerfacecolor="white", markeredgecolor=median_marker_edge, label="Median stressor shift"),
    ]

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.01),
        ncol=4,
        frameon=False,
        fontsize=11.0,
        handlelength=2.0,
        columnspacing=1.5,
    )
    fig.suptitle("Tier-1 host power context", fontsize=16.8, fontweight="bold")
    fig.tight_layout(rect=[0.02, 0.16, 0.98, 0.92])

    tier1_power_png = paper_fig / "fig_tier1_power_context.png"
    fig.savefig(tier1_power_png, dpi=300, bbox_inches="tight", facecolor="white")
    display(Image(filename=str(tier1_power_png)))
    plt.close(fig)


display(Markdown("### Benign-only paired DICE-off/DICE-on overhead measurement"))
display(Markdown(
    "This paired study is intentionally limited to the four benign workloads. "
    "It is meant to measure detector cost, not stressor-driven workload cost."
))

paired_dir = DATASET_ROOT / "power_overhead_pairs"
paired_dir.mkdir(parents=True, exist_ok=True)
paired_manifest = paired_dir / "manifest.csv"
paired_template = paired_dir / "manifest_template_benign_only.csv"

if not paired_template.exists():
    template_rows = []
    for workload in WORKLOADS_LOCAL:
        for mode in ["baseline", "dice_on"]:
            template_rows.append(
                {
                    "workload": workload,
                    "mode": mode,
                    "csv_path": f"{workload}/{mode}/tier1_alt_full_5hz.csv",
                    "power_col": "sys_power",
                    "sample_hz": 5,
                }
            )
    pd.DataFrame(template_rows).to_csv(paired_template, index=False)

display(Markdown(f"Template manifest: `{paired_template}`"))

if not paired_manifest.exists():
    display(Markdown(
        "No paired overhead manifest was found. Fill in the benign-only template at "
        f"`{paired_template}` or create `{paired_manifest}` with columns `workload`, `mode`, and `csv_path`, "
        "plus optional columns `power_col` and `sample_hz`. Use `baseline` for DICE-off runs and `dice_on` for "
        "DICE-enabled runs. After that, rerun this cell to compute mean power delta (W), percent overhead, and extra energy (J)."
    ))
else:
    manifest = pd.read_csv(paired_manifest).copy()
    manifest["workload"] = manifest["workload"].astype(str)
    manifest["mode"] = manifest["mode"].astype(str)
    manifest = manifest[manifest["workload"].isin(WORKLOADS_LOCAL)].copy()
    manifest = manifest[manifest["mode"].isin(["baseline", "dice_on"])].copy()

    if manifest.empty:
        display(Markdown("The paired manifest exists, but it does not contain benign baseline/dice_on rows for the four supported workloads."))
    else:
        counts = manifest.groupby(["workload", "mode"]).size().unstack(fill_value=0)
        missing_pairs = [
            workload for workload in WORKLOADS_LOCAL
            if counts.get("baseline", pd.Series(dtype=int)).get(workload, 0) == 0
            or counts.get("dice_on", pd.Series(dtype=int)).get(workload, 0) == 0
        ]
        if missing_pairs:
            display(Markdown(
                "The paired manifest is missing benign baseline/dice_on rows for: "
                + ", ".join(f"`{w}`" for w in missing_pairs)
                + "."
            ))

        pair_rows = []
        for row in manifest.itertuples(index=False):
            csv_path = Path(row.csv_path)
            if not csv_path.is_absolute():
                csv_path = paired_dir / csv_path
            if not csv_path.exists():
                continue

            df = pd.read_csv(csv_path)
            power_col = getattr(row, "power_col", None)
            if not isinstance(power_col, str) or power_col.strip() == "":
                power_col = next((col for col in POWER_CANDIDATES if col in df.columns), None)
            if power_col is None:
                continue

            sample_hz = float(getattr(row, "sample_hz", 5.0) or 5.0)
            power = pd.to_numeric(df[power_col], errors="coerce").dropna()
            if len(power) == 0:
                continue

            pair_rows.append(
                {
                    "workload": row.workload,
                    "mode": row.mode,
                    "power_col": power_col,
                    "mean_power_w": float(power.mean()),
                    "median_power_w": float(power.median()),
                    "p95_power_w": float(power.quantile(0.95)),
                    "duration_s": float(len(power) / sample_hz),
                    "energy_j": float(power.mean() * (len(power) / sample_hz)),
                }
            )

        if len(pair_rows) == 0:
            display(Markdown("The paired manifest was found, but no usable power traces could be parsed."))
        else:
            pair_df = pd.DataFrame(pair_rows).sort_values(["workload", "mode"]).reset_index(drop=True)
            pair_df.to_csv(paper_full / "dice_power_overhead_detail.csv", index=False)

            pivot = pair_df.pivot_table(
                index="workload",
                columns="mode",
                values=["mean_power_w", "energy_j", "median_power_w", "p95_power_w"],
                aggfunc="mean",
            )
            pivot.columns = [f"{a}_{b}" for a, b in pivot.columns]
            pivot = pivot.reset_index()

            if "mean_power_w_baseline" in pivot.columns and "mean_power_w_dice_on" in pivot.columns:
                pivot["delta_power_w"] = pivot["mean_power_w_dice_on"] - pivot["mean_power_w_baseline"]
                pivot["delta_power_pct"] = 100.0 * pivot["delta_power_w"] / pivot["mean_power_w_baseline"].replace(0, np.nan)
            if "energy_j_baseline" in pivot.columns and "energy_j_dice_on" in pivot.columns:
                pivot["delta_energy_j"] = pivot["energy_j_dice_on"] - pivot["energy_j_baseline"]

            pivot.to_csv(paper_full / "dice_power_overhead_summary.csv", index=False)
            display(pivot.round(4))

            overhead_summary = pd.DataFrame([
                {
                    "n_workloads": int(len(pivot)),
                    "median_delta_power_w": float(pivot["delta_power_w"].median()) if "delta_power_w" in pivot else float("nan"),
                    "mean_delta_power_w": float(pivot["delta_power_w"].mean()) if "delta_power_w" in pivot else float("nan"),
                    "median_delta_power_pct": float(pivot["delta_power_pct"].median()) if "delta_power_pct" in pivot else float("nan"),
                    "mean_delta_power_pct": float(pivot["delta_power_pct"].mean()) if "delta_power_pct" in pivot else float("nan"),
                    "median_delta_energy_j": float(pivot["delta_energy_j"].median()) if "delta_energy_j" in pivot else float("nan"),
                }
            ])
            overhead_summary.to_csv(paper_full / "dice_power_overhead_headline.csv", index=False)

            headline = overhead_summary.iloc[0]
            abstract_power_line = (
                f"Across paired benign workloads, the median detector power overhead is "
                f"{headline['median_delta_power_w']:.4f} W ({headline['median_delta_power_pct']:.2f}\\%)."
                if pd.notna(headline['median_delta_power_w']) and pd.notna(headline['median_delta_power_pct'])
                else "Paired benign workloads were parsed, but the detector power delta could not be summarized."
            )
            results_power_line = (
                f"Across paired benign workloads, the median detector power overhead is "
                f"{headline['median_delta_power_w']:.4f} W ({headline['median_delta_power_pct']:.2f}\\%), "
                f"and the mean detector power overhead is {headline['mean_delta_power_w']:.4f} W "
                f"({headline['mean_delta_power_pct']:.2f}\\%). This benign-only paired study excludes anomaly-inducing "
                f"stressors so that the reported overhead reflects detector cost rather than stressor cost."
                if pd.notna(headline['median_delta_power_w']) and pd.notna(headline['median_delta_power_pct'])
                and pd.notna(headline['mean_delta_power_w']) and pd.notna(headline['mean_delta_power_pct'])
                else "Paired benign workloads were parsed, but the detector power delta could not be summarized."
            )
            (paper_full / "ABSTRACT_POWER_SNIPPET.md").write_text(abstract_power_line + "\n")
            (paper_full / "RESULTS_POWER_SNIPPET.md").write_text(results_power_line + "\n")
            display(Markdown("> **Abstract power box**  \n" + abstract_power_line))
            display(Markdown("> **Results power box**  \n" + results_power_line))

            overhead_view = pivot.copy().sort_values("delta_power_w").reset_index(drop=True)

            fig, axes = plt.subplots(
                1, 2,
                figsize=(14.2, 5.6),
                gridspec_kw={"width_ratios": [1.05, 0.95]},
                constrained_layout=False,
            )

            title_fs = 16.0
            label_fs = 12.8
            tick_fs = 11.5
            number_fs = 11.0

            baseline_color = "#9AA5B1"
            dice_on_color = "#D1495B"
            connector_fill = lighten_color(dice_on_color, 0.74)
            connector_edge = lighten_color(dice_on_color, 0.42)

            # Left panel: paired baseline vs DICE-on mean power
            ax = axes[0]
            y = np.arange(len(overhead_view), dtype=float)

            for yi in y:
                if int(yi) % 2 == 0:
                    ax.axhspan(yi - 0.44, yi + 0.44, color="#F8FAFC", zorder=0)

            baseline_vals = overhead_view["mean_power_w_baseline"].to_numpy(dtype=float)
            dice_on_vals = overhead_view["mean_power_w_dice_on"].to_numpy(dtype=float)

            for yi, row in zip(y, overhead_view.itertuples(index=False)):
                x1 = float(row.mean_power_w_baseline)
                x2 = float(row.mean_power_w_dice_on)
                lo, hi = sorted([x1, x2])

                ax.plot([lo, hi], [yi, yi], color=connector_fill, linewidth=11, solid_capstyle="round", zorder=1)
                ax.plot([lo, hi], [yi, yi], color=connector_edge, linewidth=1.2, solid_capstyle="round", zorder=1.2)

            ax.scatter(
                baseline_vals,
                y,
                s=170,
                marker="s",
                color=baseline_color,
                edgecolor="white",
                linewidth=1.4,
                zorder=3,
            )
            ax.scatter(
                dice_on_vals,
                y,
                s=190,
                marker="o",
                color=dice_on_color,
                edgecolor="white",
                linewidth=1.4,
                zorder=4,
            )

            left_xlim = tight_xlim_from_values(
                np.concatenate([baseline_vals, dice_on_vals]),
                include_zero=False,
                min_span=0.6,
                pad_frac=0.15,
                min_pad=0.08,
            )
            x_span_left = left_xlim[1] - left_xlim[0]
            gap_thr_left = max(0.05, 0.06 * x_span_left)
            edge_thr_left = max(0.06, 0.07 * x_span_left)
            nudge_left = max(0.03, 0.03 * x_span_left)
            far_nudge_left = max(0.06, 0.06 * x_span_left)

            for yi, row in zip(y, overhead_view.itertuples(index=False)):
                x_base = float(row.mean_power_w_baseline)
                x_on = float(row.mean_power_w_dice_on)
                gap = abs(x_on - x_base)

                if gap < gap_thr_left:
                    bx = x_base - far_nudge_left
                    by = yi - 0.22
                    bha = "right"
                    b_arrow = True
                elif x_base < left_xlim[0] + edge_thr_left:
                    bx = x_base + nudge_left
                    by = yi
                    bha = "left"
                    b_arrow = False
                else:
                    bx = x_base - nudge_left
                    by = yi
                    bha = "right"
                    b_arrow = False

                base_kwargs = dict(
                    xy=(x_base, yi),
                    xytext=(bx, by),
                    textcoords="data",
                    ha=bha,
                    va="center",
                    fontsize=number_fs,
                    fontweight="bold",
                    color="#4B5563",
                    bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.94),
                )
                if b_arrow:
                    base_kwargs["arrowprops"] = dict(
                        arrowstyle="-",
                        color=lighten_color("#4B5563", 0.35),
                        linewidth=1.0,
                        shrinkA=5,
                        shrinkB=5,
                    )
                ax.annotate(f"{x_base:.2f}", **base_kwargs)

                if gap < gap_thr_left:
                    dx = x_on + far_nudge_left
                    dy = yi + 0.22
                    dha = "left"
                    d_arrow = True
                elif x_on > left_xlim[1] - edge_thr_left:
                    dx = x_on - nudge_left
                    dy = yi
                    dha = "right"
                    d_arrow = False
                else:
                    dx = x_on + nudge_left
                    dy = yi
                    dha = "left"
                    d_arrow = False

                on_kwargs = dict(
                    xy=(x_on, yi),
                    xytext=(dx, dy),
                    textcoords="data",
                    ha=dha,
                    va="center",
                    fontsize=number_fs,
                    fontweight="bold",
                    color=dice_on_color,
                    bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.94),
                )
                if d_arrow:
                    on_kwargs["arrowprops"] = dict(
                        arrowstyle="-",
                        color=lighten_color(dice_on_color, 0.35),
                        linewidth=1.0,
                        shrinkA=5,
                        shrinkB=5,
                    )
                ax.annotate(f"{x_on:.2f}", **on_kwargs)

            ax.set_xlim(*left_xlim)
            ax.set_yticks(y)
            ax.set_yticklabels(overhead_view["workload"], fontsize=tick_fs)
            ax.invert_yaxis()
            ax.set_xlabel("Mean host power (W)", fontsize=label_fs)
            ax.set_title("Benign paired power", fontsize=title_fs, fontweight="bold", pad=11)
            ax.grid(axis="x", alpha=0.18)
            ax.tick_params(axis="x", labelsize=tick_fs)
            ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
            ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            ax.spines["left"].set_visible(False)

            # Right panel: percent overhead
            ax = axes[1]
            y = np.arange(len(overhead_view), dtype=float)

            for yi in y:
                if int(yi) % 2 == 0:
                    ax.axhspan(yi - 0.44, yi + 0.44, color="#F8FAFC", zorder=0)

            pct_vals = overhead_view["delta_power_pct"].to_numpy(dtype=float)

            pct_fill = lighten_color(dice_on_color, 0.60)
            ax.barh(
                y,
                pct_vals,
                height=0.58,
                color=pct_fill,
                edgecolor=dice_on_color,
                linewidth=1.2,
                zorder=2,
            )
            ax.axvline(0.0, color="#111827", linewidth=1.2, zorder=1)

            right_xlim = tight_xlim_from_values(
                pct_vals,
                include_zero=True,
                min_span=1.0,
                pad_frac=0.18,
                min_pad=0.18,
            )
            x_span_right = right_xlim[1] - right_xlim[0]
            near_zero_thr = max(0.18, 0.06 * x_span_right)
            edge_thr_right = max(0.18, 0.08 * x_span_right)
            nudge_right = max(0.14, 0.04 * x_span_right)
            far_nudge_right = max(0.24, 0.08 * x_span_right)

            for yi, row in zip(y, overhead_view.itertuples(index=False)):
                x = float(row.delta_power_pct)
                label_text = f"{row.delta_power_pct:+.2f}%\n({row.delta_power_w:+.3f} W)"
                label_color = dice_on_color if x >= 0 else "#2D6A4F"

                if abs(x) < near_zero_thr:
                    tx = x + (far_nudge_right if x >= 0 else -far_nudge_right)
                    ty = yi - 0.24
                    tha = "left" if x >= 0 else "right"
                    use_arrow = True
                elif x > right_xlim[1] - edge_thr_right:
                    tx = x - nudge_right
                    ty = yi
                    tha = "right"
                    use_arrow = False
                elif x < right_xlim[0] + edge_thr_right:
                    tx = x + nudge_right
                    ty = yi
                    tha = "left"
                    use_arrow = False
                else:
                    tx = x + (nudge_right if x >= 0 else -nudge_right)
                    ty = yi
                    tha = "left" if x >= 0 else "right"
                    use_arrow = False

                pct_kwargs = dict(
                    xy=(x, yi),
                    xytext=(tx, ty),
                    textcoords="data",
                    ha=tha,
                    va="center",
                    fontsize=10.7,
                    fontweight="bold",
                    color=label_color,
                    bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="none", alpha=0.95),
                )
                if use_arrow:
                    pct_kwargs["arrowprops"] = dict(
                        arrowstyle="-",
                        color=lighten_color(label_color, 0.35),
                        linewidth=1.0,
                        shrinkA=5,
                        shrinkB=5,
                    )
                ax.annotate(label_text, **pct_kwargs)

            ax.set_xlim(*right_xlim)
            ax.set_yticks(y)
            ax.set_yticklabels(overhead_view["workload"], fontsize=tick_fs)
            ax.invert_yaxis()
            ax.set_xlabel("Power overhead (%)", fontsize=label_fs)
            ax.set_title("Detector overhead", fontsize=title_fs, fontweight="bold", pad=11)
            ax.grid(axis="x", alpha=0.18)
            ax.tick_params(axis="x", labelsize=tick_fs)
            ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
            ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            ax.spines["left"].set_visible(False)

            legend_handles = [
                Line2D([0], [0], marker="s", linestyle="None", markersize=9.5, markerfacecolor=baseline_color, markeredgecolor="white", label="DICE-off baseline"),
                Line2D([0], [0], marker="o", linestyle="None", markersize=10.5, markerfacecolor=dice_on_color, markeredgecolor="white", label="DICE-on"),
                Patch(facecolor=pct_fill, edgecolor=dice_on_color, label="Percent overhead"),
            ]

            fig.legend(
                handles=legend_handles,
                loc="lower center",
                bbox_to_anchor=(0.5, -0.01),
                ncol=3,
                frameon=False,
                fontsize=11.0,
                handlelength=2.0,
                columnspacing=1.7,
            )
            fig.suptitle("Benign paired DICE power overhead", fontsize=16.5, fontweight="bold")
            fig.tight_layout(rect=[0.02, 0.16, 0.98, 0.92])

            overhead_png = paper_fig / "fig_dice_power_overhead_pairs.png"
            fig.savefig(overhead_png, dpi=300, bbox_inches="tight", facecolor="white")
            display(Image(filename=str(overhead_png)))
            plt.close(fig)


## 7. Projected DICE-Score Accelerator Complexity

This section estimates only the hardware cost of the final DICE scoring stage. It does **not** estimate the cost of the full software system around it.

Included in the estimate:
- blockwise accumulation of `|r_{t,j}|`
- weighted reduction into one score
- threshold and persistence logic
- a small top-k attribution buffer

Not included in the estimate:
- telemetry collection
- normalization
- digital-twin state update / synchronization
- host software runtime

So this is best read as a rough hardware-sizing exercise, not as a final chip area or power result.


In [ ]:
# Plain-language: This cell estimates the complexity of a simple DICE-score accelerator
# using the selected operating points from the earlier comparison cells.
display(Markdown('### Accelerator complexity tables'))

complexity_csv = COMPARISON_DIR / 'projected_dice_score_accelerator_complexity_profiles.csv'

ACC_WIDTH_BITS = 32
WEIGHT_WIDTH_BITS = 16
SCORE_WIDTH_BITS = 24
TOPK = 5
PERSIST_COUNTER_BITS = 8
DEFAULT_BLOCK_B = 60

BEST_CFG_CSV = COMPARISON_DIR / 'profile_best_config_by_scenario.csv'
SCORECARD_CSV = COMPARISON_DIR / 'industry_paper_scorecard_profiles.csv'
TWO_STAGE_COMPARISON_CSV = COMPARISON_DIR / 'two_stage_dice_profiles.csv'

rows = []


def _load_tuned_block_B(profile: str) -> float:
    rec_path = default_tuning_out_dir(DATASET_ROOT, profile) / 'recommended_alert_config.csv'
    if rec_path.exists():
        rec_df = pd.read_csv(rec_path)
        if not rec_df.empty and 'block_B' in rec_df.columns and pd.notna(rec_df.iloc[0]['block_B']):
            return float(rec_df.iloc[0]['block_B'])
    return float(DEFAULT_BLOCK_B)


def _complexity_metrics(avg_active_features: float, block_B: float) -> dict[str, float]:
    avg_active_features = float(avg_active_features)
    n_state = max(1, int(np.ceil(avg_active_features)))
    block_B = max(1, int(np.ceil(float(block_B))))
    feature_index_bits = max(1, int(np.ceil(np.log2(max(n_state, 2)))))

    sample_path_ops_per_block = 2.0 * avg_active_features * block_B
    block_end_ops_per_block = (
        avg_active_features
        + max(avg_active_features - 1.0, 0.0)
        + avg_active_features * TOPK
        + 2.0
    )
    total_state_bits = (
        n_state * ACC_WIDTH_BITS
        + n_state * WEIGHT_WIDTH_BITS
        + TOPK * (feature_index_bits + SCORE_WIDTH_BITS)
        + SCORE_WIDTH_BITS
        + 2 * PERSIST_COUNTER_BITS
        + 16
    )

    return {
        'sample_path_ops_per_block': float(sample_path_ops_per_block),
        'block_end_ops_per_block': float(block_end_ops_per_block),
        'total_state_bytes': int(np.ceil(total_state_bits / 8.0)),
    }


def _append_row(
    *,
    profile: str,
    operating_point: str,
    choice_label: str,
    selection_source: str,
    avg_active_features: float,
    block_B_used: float,
    detect_rate: float = np.nan,
    benign_alert_rate: float = np.nan,
    median_ttd_s: float = np.nan,
):
    metrics = _complexity_metrics(avg_active_features, block_B_used)
    rows.append({
        'feature_profile': profile,
        'profile_label': PROFILE_LABEL.get(profile, profile),
        'operating_point': operating_point,
        'choice_label': choice_label,
        'selection_source': selection_source,
        'avg_active_features': float(avg_active_features),
        'block_B_used': float(block_B_used),
        'detect_rate': float(detect_rate) if pd.notna(detect_rate) else np.nan,
        'benign_alert_rate': float(benign_alert_rate) if pd.notna(benign_alert_rate) else np.nan,
        'median_ttd_s': float(median_ttd_s) if pd.notna(median_ttd_s) else np.nan,
        **metrics,
    })


for profile in PROFILE_ORDER:
    block_B_used = _load_tuned_block_B(profile)

    # 1) Best overall config from earlier comparison outputs.
    best_overall = None
    best_overall_source = ''

    if BEST_CFG_CSV.exists():
        best_cfg_df = pd.read_csv(BEST_CFG_CSV)
        sub = best_cfg_df[
            (best_cfg_df['feature_profile'].astype(str) == str(profile))
            & (best_cfg_df['scenario'].astype(str) == 'overall')
        ].copy()
        if not sub.empty:
            best_overall = sub.iloc[0]
            best_overall_source = BEST_CFG_CSV.name

    if best_overall is None and SCORECARD_CSV.exists():
        scorecard_df = pd.read_csv(SCORECARD_CSV)
        sub = scorecard_df[scorecard_df['feature_profile'].astype(str) == str(profile)].copy()
        if not sub.empty:
            best_overall = sub.iloc[0]
            best_overall_source = SCORECARD_CSV.name

    if best_overall is None:
        overall_path = profile_global_dir(profile) / 'overall_metrics.csv'
        sequential_path = profile_global_dir(profile) / 'sequential_metrics.csv'
        if overall_path.exists():
            overall_df = pd.read_csv(overall_path)
            sub = overall_df[overall_df['config'].astype(str) == str(FINAL_CONFIG)].copy()
            if not sub.empty:
                best_overall = sub.iloc[0]
                best_overall_source = overall_path.name

                seq_detect = np.nan
                seq_benign = np.nan
                seq_ttd = np.nan
                if sequential_path.exists():
                    seq_df = pd.read_csv(sequential_path)
                    seq_sub = seq_df[seq_df['config'].astype(str) == str(FINAL_CONFIG)].copy()
                    if not seq_sub.empty:
                        seq_row = seq_sub.iloc[0]
                        best_overall = best_overall.copy()
                        best_overall['anomaly_detect_rate'] = seq_row.get('anomaly_detect_rate', np.nan)
                        best_overall['benign_run_alert_rate'] = seq_row.get('benign_run_alert_rate', np.nan)
                        best_overall['median_time_to_detect_s'] = seq_row.get('median_time_to_detect_s', np.nan)

    if best_overall is not None:
        detect_rate = best_overall.get('anomaly_detect_rate', best_overall.get('detection_rate', np.nan))
        benign_rate = best_overall.get('benign_run_alert_rate', best_overall.get('benign_alert_rate', np.nan))
        median_ttd = best_overall.get('median_time_to_detect_s', best_overall.get('median_ttd_s', np.nan))
        choice_label = str(best_overall.get('config_label', best_overall.get('config', FINAL_CONFIG)))

        _append_row(
            profile=profile,
            operating_point='Best overall config',
            choice_label=choice_label,
            selection_source=best_overall_source,
            avg_active_features=float(best_overall.get('n_features', np.nan)),
            block_B_used=block_B_used,
            detect_rate=detect_rate,
            benign_alert_rate=benign_rate,
            median_ttd_s=median_ttd,
        )

    # 2) Recommended two-stage operating point.
    two_stage_path = profile_paper_dir(profile) / 'two_stage_dice_summary.csv'
    if two_stage_path.exists():
        two_stage_df = pd.read_csv(two_stage_path)
        two_stage_source = two_stage_path.name
    elif TWO_STAGE_COMPARISON_CSV.exists():
        two_stage_df = pd.read_csv(TWO_STAGE_COMPARISON_CSV)
        two_stage_df = two_stage_df[two_stage_df['feature_profile'].astype(str) == str(profile)].copy()
        two_stage_source = TWO_STAGE_COMPARISON_CSV.name
    else:
        two_stage_df = pd.DataFrame()
        two_stage_source = ''

    if not two_stage_df.empty and 'policy' in two_stage_df.columns:
        cand = two_stage_df[two_stage_df['policy'].astype(str).str.startswith('Two-stage')].copy()
        if not cand.empty:
            cand = cand.sort_values(['avg_feature_budget', 'benign_alert_rate'], ascending=[True, True])
            rec_two_stage = cand.iloc[0]
            _append_row(
                profile=profile,
                operating_point='Recommended two-stage',
                choice_label=str(rec_two_stage.get('policy', 'Two-stage')),
                selection_source=two_stage_source,
                avg_active_features=float(rec_two_stage.get('avg_feature_budget', np.nan)),
                block_B_used=block_B_used,
                detect_rate=rec_two_stage.get('detection_rate', np.nan),
                benign_alert_rate=rec_two_stage.get('benign_alert_rate', np.nan),
                median_ttd_s=rec_two_stage.get('median_ttd_s', np.nan),
            )

    # 3) Recommended DSE budget point.
    feature_budget_path = profile_paper_dir(profile) / 'dse_feature_budget_summary.csv'
    if feature_budget_path.exists():
        feature_budget_df = pd.read_csv(feature_budget_path)
        if not feature_budget_df.empty:
            rec_budget = feature_budget_df.sort_values(
                ['n_selected_features', 'pr_auc_wc'],
                ascending=[True, False],
            ).iloc[0]
            _append_row(
                profile=profile,
                operating_point='Recommended budget',
                choice_label=f"{int(rec_budget['budget_pct'])}% budget",
                selection_source=feature_budget_path.name,
                avg_active_features=float(rec_budget.get('n_selected_features', np.nan)),
                block_B_used=block_B_used,
                detect_rate=rec_budget.get('detection_rate', np.nan),
                benign_alert_rate=rec_budget.get('benign_alert_rate', np.nan),
                median_ttd_s=rec_budget.get('median_time_to_detect_s', np.nan),
            )

if not rows:
    display(Markdown('No selected operating points are ready yet. Run the comparison, two-stage, tuning, and feature-budget cells first.'))
else:
    accel_df = pd.DataFrame(rows)

    point_order = ['Recommended budget', 'Recommended two-stage', 'Best overall config']
    accel_df['operating_point'] = pd.Categorical(accel_df['operating_point'], categories=point_order, ordered=True)
    accel_df = accel_df.sort_values(['feature_profile', 'operating_point']).reset_index(drop=True)

    base_sample = accel_df.groupby('feature_profile', observed=False)['sample_path_ops_per_block'].transform('min')
    base_block = accel_df.groupby('feature_profile', observed=False)['block_end_ops_per_block'].transform('min')
    base_state = accel_df.groupby('feature_profile', observed=False)['total_state_bytes'].transform('min')

    accel_df['complexity_index'] = (
        0.40 * accel_df['sample_path_ops_per_block'] / base_sample
        + 0.35 * accel_df['block_end_ops_per_block'] / base_block
        + 0.25 * accel_df['total_state_bytes'] / base_state
    )

    accel_df.to_csv(complexity_csv, index=False)

    display(
        accel_df[
            [
                'profile_label',
                'operating_point',
                'choice_label',
                'avg_active_features',
                'block_B_used',
                'detect_rate',
                'benign_alert_rate',
                'median_ttd_s',
                'sample_path_ops_per_block',
                'block_end_ops_per_block',
                'total_state_bytes',
                'complexity_index',
            ]
        ].rename(
            columns={
                'profile_label': 'Profile',
                'operating_point': 'Operating point',
                'choice_label': 'Selected choice',
                'avg_active_features': 'Avg active features',
                'block_B_used': 'Block B',
                'detect_rate': 'Detection rate',
                'benign_alert_rate': 'Benign alert rate',
                'median_ttd_s': 'Median TTD (s)',
                'sample_path_ops_per_block': 'Sample-path ops/block',
                'block_end_ops_per_block': 'Block-end ops/block',
                'total_state_bytes': 'State bytes',
                'complexity_index': 'Complexity index',
            }
        ).round(3)
    )


### Accelerator comparison figure


In [ ]:
# Plain-language: This cell visualizes the projected accelerator costs so the hardware story is easier to communicate.

display(Markdown('### Accelerator comparison figure'))

complexity_csv = COMPARISON_DIR / 'projected_dice_score_accelerator_complexity_profiles.csv'
complexity_png = COMPARISON_FIG / 'fig_projected_dice_score_accelerator_complexity_profiles.png'

if not complexity_csv.exists():
    display(Markdown('Run the accelerator-complexity cell first so the comparison CSV exists.'))
else:
    accel_df = pd.read_csv(complexity_csv)
    profiles_present = [p for p in PROFILE_ORDER if p in set(accel_df['feature_profile'])]

    if not profiles_present:
        display(Markdown('No accelerator-complexity rows are available yet.'))
    else:
        if 'operating_point' not in accel_df.columns:
            display(Markdown(
                'Current complexity CSV still contains the older per-config rows. '
                'Rerun the updated accelerator-complexity table cell once if you want this figure to use the selected operating points.'
            ))

        TEXT_DARK = '#0F172A'
        GRID_SOFT = '#E2E8F0'
        SPINE_SOFT = '#CBD5E1'

        profile_colors = {
            'mixed': '#355C7D',
            'full': '#C44E52',
        }

        short_label_map = {
            'Tier-0': 'T0',
            'Tier-0/1': 'T0/1',
            'Tier-0/1/2': 'T0/1/2',
            'tier0': 'T0',
            'tier0_tier1': 'T0/1',
            'tier0_tier1_alt': 'T0/1',
            'tier0_tier1_tier2': 'T0/1/2',
            'tier0_tier1_alt_tier2': 'T0/1/2',
        }

        point_order_map = {
            'Recommended budget': 0,
            'Recommended two-stage': 1,
            'Best overall config': 2,
        }
        point_short_map = {
            'Recommended budget': 'Budget',
            'Recommended two-stage': 'Two-stage',
            'Best overall config': 'Best overall',
        }

        component_specs = [
            ('sample_path_ops_per_block', 'Path Ops\nper block', lambda v: f'{v/1000:.1f}k' if v >= 1000 else f'{int(round(v))}'),
            ('block_end_ops_per_block', 'End Ops\nper block', lambda v: f'{int(round(v))}' if np.isclose(v, round(v)) else f'{v:.1f}'),
            ('total_state_bytes', 'State\nbytes', lambda v: f'{int(round(v))}'),
            ('complexity_index', 'Total\nindex', lambda v: f'{v:.2f}x'),
        ]

        def blend_with_white(color, amount):
            rgb = np.array(to_rgb(color), dtype=float)
            white = np.ones(3, dtype=float)
            return tuple(rgb * (1.0 - amount) + white * amount)

        def short_choice_label(value):
            if pd.isna(value):
                return ''
            text = str(value).strip()
            text = short_label_map.get(text, text)
            if text.startswith('Two-stage '):
                text = text.replace('Two-stage ', '')
            if text.endswith(' budget'):
                text = text.replace(' budget', '')
            return text

        def fmt_feature_count(value):
            num = pd.to_numeric(pd.Series([value]), errors='coerce').iloc[0]
            if pd.isna(num):
                return ''
            num = float(num)
            if np.isclose(num, round(num), atol=0.05):
                return f'{int(round(num))} feats'
            return f'{num:.1f} feats'

        def prepare_profile_rows(df):
            out = df.copy()

            if 'operating_point' in out.columns:
                out['sort_key'] = out['operating_point'].map(point_order_map).fillna(99).astype(int)
                out = out.sort_values(['sort_key', 'choice_label']).reset_index(drop=True)

                feat_col = 'avg_active_features' if 'avg_active_features' in out.columns else 'n_features'
                labels = []
                for row in out.itertuples(index=False):
                    op = getattr(row, 'operating_point', '')
                    op_short = point_short_map.get(str(op), str(op))
                    choice_short = short_choice_label(getattr(row, 'choice_label', ''))
                    feat_short = fmt_feature_count(getattr(row, feat_col, np.nan))

                    if choice_short and feat_short:
                        label = f'{op_short}\n{choice_short} | {feat_short}'
                    elif choice_short:
                        label = f'{op_short}\n{choice_short}'
                    elif feat_short:
                        label = f'{op_short}\n{feat_short}'
                    else:
                        label = op_short
                    labels.append(label)

                out['display_label'] = labels

            else:
                if 'config_label' not in out.columns:
                    if 'label' in out.columns:
                        out['config_label'] = out['label']
                    else:
                        out['config_label'] = out['config'].astype(str)

                out = out.sort_values('n_features').reset_index(drop=True)
                labels = []
                for row in out.itertuples(index=False):
                    cfg = short_label_map.get(str(getattr(row, 'config_label', '')), str(getattr(row, 'config_label', '')))
                    feats = fmt_feature_count(getattr(row, 'n_features', np.nan))
                    labels.append(f'{cfg}\n{feats}' if feats else cfg)
                out['display_label'] = labels

            return out

        mats = {}
        vmax = 1.0
        max_rows = 0

        for profile in profiles_present:
            d = accel_df[accel_df['feature_profile'] == profile].copy()
            d = prepare_profile_rows(d)
            max_rows = max(max_rows, len(d))

            ratio_cols = []
            for col, _, _ in component_specs:
                vals = pd.to_numeric(d[col], errors='coerce').to_numpy(dtype=float)
                finite = vals[np.isfinite(vals)]
                base = max(float(np.nanmin(finite)) if finite.size else 1.0, 1e-12)
                ratio_cols.append(vals / base)

            mat = np.column_stack(ratio_cols)
            mats[profile] = (d, mat)
            vmax = max(vmax, float(np.nanmax(mat)))

        norm = Normalize(vmin=1.0, vmax=max(1.05, vmax))
        cmap = plt.cm.YlGnBu

        fig_h = max(6.2, 3.5 + 1.18 * max_rows)
        fig_w = max(8.2, 7.8 * len(profiles_present))
        fig, axes = plt.subplots(
            1,
            len(profiles_present),
            figsize=(fig_w, fig_h),
            squeeze=False,
            constrained_layout=False,
        )
        axes = axes[0]

        for ax, profile in zip(axes, profiles_present):
            d, mat = mats[profile]
            accent = profile_colors.get(profile, '#355C7D')
            total_fill = blend_with_white(accent, 0.72)

            n_rows, n_cols = mat.shape
            ax.set_facecolor('white')

            ax.imshow(
                mat,
                aspect='auto',
                cmap=cmap,
                norm=norm,
                interpolation='nearest',
                zorder=1,
            )

            ax.set_xticks(np.arange(n_cols))
            ax.set_xticklabels(
                [label for _, label, _ in component_specs],
                fontsize=12.4,
                fontweight='bold',
                color=TEXT_DARK,
            )
            ax.tick_params(axis='x', pad=12, length=0)

            ax.set_yticks(np.arange(n_rows))
            ax.set_yticklabels(
                d['display_label'].tolist(),
                fontsize=12.8,
                fontweight='bold',
                color=TEXT_DARK,
            )
            ax.tick_params(axis='y', pad=12, length=0)

            ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
            ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
            ax.grid(which='minor', color='white', linewidth=2.4)
            ax.tick_params(which='minor', bottom=False, left=False)

            for i in range(n_rows):
                ax.add_patch(
                    Rectangle(
                        (n_cols - 1 - 0.5, i - 0.5),
                        1.0,
                        1.0,
                        facecolor=total_fill,
                        edgecolor='none',
                        alpha=0.26,
                        zorder=2,
                    )
                )

            ax.add_patch(
                Rectangle(
                    (n_cols - 1 - 0.5, -0.5),
                    1.0,
                    n_rows,
                    fill=False,
                    edgecolor=accent,
                    linewidth=2.5,
                    zorder=6,
                )
            )

            for i in range(n_rows):
                for j, (col, _, formatter) in enumerate(component_specs):
                    value = float(pd.to_numeric(d.iloc[i][col], errors='coerce'))
                    ratio = float(mat[i, j])

                    if j == n_cols - 1:
                        text_color = 'white' if norm(ratio) > 0.44 else TEXT_DARK
                        font_size = 12.0
                    else:
                        text_color = 'white' if norm(ratio) > 0.57 else TEXT_DARK
                        font_size = 11.1

                    ax.text(
                        j,
                        i,
                        formatter(value),
                        ha='center',
                        va='center',
                        fontsize=font_size,
                        fontweight='bold',
                        color=text_color,
                        zorder=7,
                    )

            title = PROFILE_LABEL.get(profile, profile)
            if 'block_B_used' in d.columns:
                block_vals = pd.to_numeric(d['block_B_used'], errors='coerce').dropna().unique()
                if len(block_vals) == 1:
                    title = f'{title}  |  B={int(round(block_vals[0]))}'

            ax.set_title(
                title,
                fontsize=17.6,
                fontweight='bold',
                color=accent,
                pad=16,
            )

            for spine in ax.spines.values():
                spine.set_color(SPINE_SOFT)
                spine.set_linewidth(1.2)

        fig.subplots_adjust(
            left=0.22,
            right=0.98,
            top=0.84,
            bottom=0.30,
            wspace=0.46,
        )

        cbar_ax = fig.add_axes([0.28, 0.055, 0.44, 0.026])
        cbar = fig.colorbar(
            ScalarMappable(norm=norm, cmap=cmap),
            cax=cbar_ax,
            orientation='horizontal',
        )
        cbar.ax.tick_params(labelsize=11.4)
        cbar_ax.set_title(
            'Relative projected cost within profile (lowest selected point = 1.0x)',
            fontsize=12.0,
            fontweight='bold',
            pad=10,
        )

        fig.suptitle(
            'Projected DICE-score accelerator cost',
            fontsize=18.4,
            fontweight='bold',
            color=TEXT_DARK,
            y=0.95,
        )

        fig.savefig(complexity_png, dpi=PRO_FIG_DPI, bbox_inches='tight', facecolor='white')
        display(Image(filename=str(complexity_png)))
        plt.close(fig)


## 8. Case Study: True Time-Series Overlay

This section shows a side-by-side comparison between an anomalous run and its matched benign reference.

The plots make it easier to see when the two behaviors begin to separate, where the detector becomes surprised, and how that change appears over time.


In [ ]:
# Plain-language: This cell overlays a real anomaly trace with DICE's benign reference
# using one row per workload and renders both mixed and full profiles.

display(Markdown("### True time-series overlay: anomaly vs benign reference"))


def outlined_plot(
    ax,
    x,
    y,
    color,
    linewidth=3.2,
    linestyle="-",
    marker=None,
    markevery=None,
    markersize=5.0,
    alpha=1.0,
    zorder=4,
):
    (line,) = ax.plot(
        x,
        y,
        color=color,
        linewidth=linewidth,
        linestyle=linestyle,
        marker=marker,
        markevery=markevery,
        markersize=markersize,
        markerfacecolor=color if marker is not None else color,
        markeredgecolor="white",
        markeredgewidth=0.95,
        alpha=alpha,
        solid_capstyle="round",
        solid_joinstyle="round",
        zorder=zorder,
    )
    is_solid = linestyle == "-"
    extra = 1.9 if is_solid else 1.7
    line.set_path_effects(
        [
            pe.Stroke(linewidth=linewidth + extra, foreground="white", alpha=0.90),
            pe.Normal(),
        ]
    )
    return line


def blend_with_white(color, amount):
    rgb = np.array(to_rgb(color), dtype=float)
    white = np.ones(3, dtype=float)
    return tuple(rgb * (1.0 - amount) + white * amount)


def style_overlay_axis(ax, facecolor):
    ax.set_facecolor(facecolor)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#CBD5E1")
    ax.spines["bottom"].set_color("#CBD5E1")
    ax.spines["left"].set_linewidth(1.15)
    ax.spines["bottom"].set_linewidth(1.15)
    ax.grid(axis="y", color="#CBD5E1", alpha=0.28, linewidth=1.0)
    ax.grid(axis="x", color="#E2E8F0", alpha=0.18, linewidth=0.9)
    ax.tick_params(axis="both", labelsize=11.8, colors="#0F172A")


def add_column_badge(fig, ax, text, face, edge):
    pos = ax.get_position()
    x_center = 0.5 * (pos.x0 + pos.x1)
    y = pos.y1 + 0.020
    fig.text(
        x_center,
        y,
        text,
        ha="center",
        va="center",
        fontsize=13.5,
        fontweight="bold",
        color="#0F172A",
        bbox=dict(
            boxstyle="round,pad=0.34,rounding_size=0.18",
            facecolor=face,
            edgecolor=edge,
            linewidth=1.15,
        ),
    )


def add_row_badge(ax, text, face, edge):
    ax.text(
        0.018,
        0.92,
        text,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=12.5,
        fontweight="bold",
        color="#0F172A",
        bbox=dict(
            boxstyle="round,pad=0.28,rounding_size=0.16",
            facecolor=face,
            edgecolor=edge,
            linewidth=1.0,
        ),
        zorder=10,
    )


def _best_config_for_profile(profile, available_cfgs):
    best_cfg_csv = COMPARISON_DIR / "profile_best_config_by_scenario.csv"
    scorecard_csv = COMPARISON_DIR / "industry_paper_scorecard_profiles.csv"

    if best_cfg_csv.exists():
        best_cfg_df = pd.read_csv(best_cfg_csv)
        rows = best_cfg_df[
            (best_cfg_df["feature_profile"].astype(str) == str(profile))
            & (best_cfg_df["scenario"].astype(str) == "overall")
        ]
        if not rows.empty:
            cfg = str(rows.iloc[0]["config"])
            if cfg in available_cfgs:
                return cfg

    if scorecard_csv.exists():
        scorecard_df = pd.read_csv(scorecard_csv)
        rows = scorecard_df[scorecard_df["feature_profile"].astype(str) == str(profile)]
        if not rows.empty:
            cfg = str(rows.iloc[0]["config"])
            if cfg in available_cfgs:
                return cfg

    if "tier0_tier1_tier2" in available_cfgs:
        return "tier0_tier1_tier2"
    if "tier0_tier1_alt_tier2" in available_cfgs:
        return "tier0_tier1_alt_tier2"
    return sorted(available_cfgs)[-1] if available_cfgs else "tier0_tier1_tier2"


def _resolve_trace_run_dir(profile):
    tuning_dir = default_tuning_out_dir(DATASET_ROOT, profile)
    rec_path = tuning_dir / "recommended_alert_config.csv"
    if rec_path.exists():
        rec_df = pd.read_csv(rec_path)
        if not rec_df.empty:
            rec = rec_df.iloc[0]
            run_dir = tuning_dir / "runs" / f"g{float(rec['gain']):g}_B{int(rec['block_B'])}_a{float(rec['alpha']):g}_k{int(rec['persist_k'])}"
            if (run_dir / "case_predictions.csv").exists() and (run_dir / "case_block_traces.csv").exists():
                return run_dir

    if "profile_global_dir" in globals():
        candidate = profile_global_dir(profile)
        if (candidate / "case_predictions.csv").exists() and (candidate / "case_block_traces.csv").exists():
            return candidate

    return None


def _render_overlay_for_profile(profile):
    run_dir = _resolve_trace_run_dir(profile)
    if run_dir is None:
        return None, f"No trace run directory was available for `{profile}`."

    case_pred = pd.read_csv(run_dir / "case_predictions.csv")
    trace_df = pd.read_csv(run_dir / "case_block_traces.csv")

    WORKLOAD_ORDER = ["BROWSER", "VIDEO_SW", "PY_AI", "PY_STATS"]
    available_workloads = [
        w for w in WORKLOAD_ORDER
        if w in set(case_pred["workload"].dropna().astype(str))
    ]
    if not available_workloads:
        available_workloads = sorted(case_pred["workload"].dropna().astype(str).unique())

    available_cfgs = set(case_pred["config"].dropna().astype(str))
    final_cfg = _best_config_for_profile(profile, available_cfgs)

    displayed_profile = (
        "mixed" if final_cfg == "tier0_tier1_alt_tier2"
        else "full" if final_cfg == "tier0_tier1_tier2"
        else profile
    )

    workload_colors = {
        "BROWSER": "#0B6EBA",
        "VIDEO_SW": "#C9641A",
        "PY_AI": "#0E8A6A",
        "PY_STATS": "#A23B72",
    }
    workload_markers = {
        "BROWSER": "o",
        "VIDEO_SW": "s",
        "PY_AI": "D",
        "PY_STATS": "^",
    }
    workload_linestyles = {
        "BROWSER": "-",
        "VIDEO_SW": (0, (7, 2)),
        "PY_AI": (0, (4, 1.8)),
        "PY_STATS": (0, (8, 2, 1.5, 2)),
    }

    detected = case_pred[
        (case_pred["config"] == final_cfg)
        & (case_pred["label"] == 1)
        & (case_pred["run_alert"] == 1)
    ].copy()

    if detected.empty:
        return None, f"No detected anomaly case found for config `{final_cfg}` in `{profile}`."

    sort_col = "run_score_wc" if "run_score_wc" in detected.columns else "run_score"
    plot_rows = []

    for workload in available_workloads:
        dw = detected[detected["workload"] == workload].sort_values(sort_col, ascending=False)
        if dw.empty:
            continue

        row = dw.iloc[0]
        anomaly_case = row["case_id"]

        nominal_rows = case_pred[
            (case_pred["config"] == final_cfg)
            & (case_pred["workload"] == workload)
            & (case_pred["stressor"] == "NOMINAL")
        ]
        if nominal_rows.empty:
            continue

        nominal_case = nominal_rows.iloc[0]["case_id"]

        anom_trace = trace_df[
            (trace_df["config"] == final_cfg)
            & (trace_df["case_id"] == anomaly_case)
        ].copy()
        nom_trace = trace_df[
            (trace_df["config"] == final_cfg)
            & (trace_df["case_id"] == nominal_case)
        ].copy()
        if anom_trace.empty or nom_trace.empty:
            continue

        merged = anom_trace.merge(
            nom_trace[["block_idx", "score"]].rename(columns={"score": "nominal_score"}),
            on="block_idx",
            how="inner",
        ).copy()
        if merged.empty:
            continue

        merged = merged.sort_values("block_idx").reset_index(drop=True)
        merged["block_time_min"] = merged["block_end_s"] / 60.0

        plot_rows.append(
            {
                "workload": workload,
                "tau": float(row["tau"]),
                "data": merged,
            }
        )

    if not plot_rows:
        return None, f"No aligned anomaly/reference traces were available for config `{final_cfg}` in `{profile}`."

    n_rows = len(plot_rows)

    axis_fs = 13.1
    tick_fs = 11.7
    legend_fs = 11.2
    suptitle_fs = 18.6

    fig = plt.figure(figsize=(16.3, max(2.95 * n_rows + 1.8, 7.2)), facecolor="white")
    gs = fig.add_gridspec(
        n_rows,
        2,
        left=0.115,
        right=0.985,
        top=0.86,
        bottom=0.16,
        hspace=0.22,
        wspace=0.12,
    )

    axes = []
    for ridx in range(n_rows):
        ax_left = fig.add_subplot(gs[ridx, 0])
        ax_right = fig.add_subplot(gs[ridx, 1], sharex=ax_left)
        axes.append((ax_left, ax_right))

    for (ax_anom, ax_ref), item in zip(axes, plot_rows):
        workload = item["workload"]
        tau = item["tau"]
        d = item["data"]

        color = workload_colors.get(workload, "#334155")
        marker = workload_markers.get(workload, "o")
        linestyle = workload_linestyles.get(workload, "-")

        t = d["block_time_min"].to_numpy(dtype=float)
        anomaly_score = d["score"].to_numpy(dtype=float)
        nominal_score = d["nominal_score"].to_numpy(dtype=float)
        markevery = max(len(t) // 12, 1)

        ymin = float(min(np.nanmin(anomaly_score), np.nanmin(nominal_score), tau))
        ymax = float(max(np.nanmax(anomaly_score), np.nanmax(nominal_score), tau))
        span = max(ymax - ymin, 1e-6)
        y_lo = ymin - 0.10 * span
        y_hi = ymax + 0.12 * span
        if y_hi <= y_lo:
            y_hi = y_lo + 0.05

        style_overlay_axis(ax_anom, "#FCF8F4")
        style_overlay_axis(ax_ref, "#F6FAFD")

        for ax in [ax_anom, ax_ref]:
            ax.set_xlim(float(np.nanmin(t)), float(np.nanmax(t)))
            ax.set_ylim(y_lo, y_hi)
            ax.tick_params(axis="both", labelsize=tick_fs)
            ax.axhline(
                tau,
                color=color,
                linestyle=(0, (4, 3)),
                linewidth=1.9,
                alpha=0.82,
                zorder=1,
            )

        ax_anom.fill_between(
            t,
            np.minimum(anomaly_score, nominal_score),
            np.maximum(anomaly_score, nominal_score),
            color=color,
            alpha=0.12,
            zorder=1.5,
        )
        ax_ref.fill_between(
            t,
            np.minimum(anomaly_score, nominal_score),
            np.maximum(anomaly_score, nominal_score),
            color=color,
            alpha=0.12,
            zorder=1.5,
        )

        outlined_plot(
            ax_anom,
            t,
            nominal_score,
            color=color,
            linewidth=2.1,
            linestyle=(0, (3, 2.6)),
            alpha=0.72,
            zorder=2,
        )
        outlined_plot(
            ax_anom,
            t,
            anomaly_score,
            color=color,
            linewidth=3.35,
            linestyle=linestyle,
            marker=marker,
            markevery=markevery,
            markersize=5.0,
            zorder=5,
        )

        outlined_plot(
            ax_ref,
            t,
            anomaly_score,
            color=color,
            linewidth=2.1,
            linestyle=(0, (3, 2.6)),
            alpha=0.72,
            zorder=2,
        )
        outlined_plot(
            ax_ref,
            t,
            nominal_score,
            color=color,
            linewidth=3.35,
            linestyle=linestyle,
            marker=marker,
            markevery=markevery,
            markersize=5.0,
            zorder=5,
        )

        add_row_badge(ax_anom, workload, blend_with_white(color, 0.84), color)

    for ax_left, ax_right in axes[:-1]:
        ax_left.tick_params(axis="x", labelbottom=False)
        ax_right.tick_params(axis="x", labelbottom=False)

    axes[-1][0].set_xlabel("Time (minutes)", fontsize=axis_fs, fontweight="bold", color="#0F172A", labelpad=10)
    axes[-1][1].set_xlabel("Time (minutes)", fontsize=axis_fs, fontweight="bold", color="#0F172A", labelpad=10)

    fig.text(
        0.050,
        0.50,
        "Block score",
        rotation=90,
        ha="center",
        va="center",
        fontsize=14.0,
        fontweight="bold",
        color="#0F172A",
    )

    add_column_badge(fig, axes[0][0], "Detected anomalous traces", "#FDE7D9", "#F59E0B")
    add_column_badge(fig, axes[0][1], "Matched benign NOMINAL references", "#DBEAFE", "#60A5FA")

    semantic_handles = [
        Line2D([0], [0], color="#334155", linewidth=3.2, linestyle="-", label="Primary trace"),
        Line2D([0], [0], color="#64748B", linewidth=2.1, linestyle=(0, (3, 2.6)), label="Paired ghost"),
        Patch(facecolor="#94A3B8", edgecolor="none", alpha=0.18, label="Mismatch gap"),
        Line2D([0], [0], color="#64748B", linewidth=1.9, linestyle=(0, (4, 3)), label="Threshold"),
    ]

    legend = fig.legend(
        handles=semantic_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.035),
        ncol=4,
        frameon=True,
        fancybox=True,
        fontsize=legend_fs,
        handlelength=2.8,
        columnspacing=1.5,
    )
    legend.get_frame().set_facecolor("white")
    legend.get_frame().set_edgecolor("#CBD5E1")
    legend.get_frame().set_linewidth(1.0)

    fig.suptitle(
        f"{displayed_profile.title()} profile: anomaly-to-benign digital twin overlay",
        fontsize=suptitle_fs,
        fontweight="bold",
        color="#0F172A",
        y=0.965,
    )

    out_dir = OUT_PAPER / profile
    out_dir.mkdir(parents=True, exist_ok=True)
    out_png = out_dir / "fig_true_timeseries_overlay.png"
    fig.savefig(out_png, dpi=240, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return out_png, None


profiles_to_show = [p for p in PROFILE_ORDER if _resolve_trace_run_dir(p) is not None]
if not profiles_to_show:
    display(Markdown("No trace run directories were available yet. Rerun the selected tuning/profile outputs first."))
else:
    for profile in profiles_to_show:
        out_png, note = _render_overlay_for_profile(profile)
        display(Markdown(f"#### {PROFILE_LABEL.get(profile, profile)} profile"))
        if note is not None:
            display(Markdown(note))
        else:
            display(Image(filename=str(out_png)))


## 8B. Industry-Facing Anomaly Localization

This section localizes anomalies for **both mixed and full profiles** in a way that is easier to explain to industry audiences.

- The stressor table answers: **which stressors stand out most clearly at each sensing tier?**
- The feature table answers: **which telemetry signals show up most often inside the first abnormal windows?**
- The timeline table answers: **when does the first abnormal window appear, and when do the next few abnormal windows follow?**
- The hardware-aware follow-up answers: **which visible M2 Pro subsystem do those stressors and features point to?**

Because DICE uses sliding decision blocks, localization here means **the first few abnormal decision windows** that persist above the benign-calibrated threshold, not a single instantaneous hardware-failure timestamp.


In [ ]:
# Plain-language: This cell ranks the five stressors, counts the most common localized anomaly features, and records when the first few abnormal windows appear for both mixed and full.
display(Markdown('### Industry-facing localization tables'))

LOCALIZATION_BLOCK_COUNT = 5
LOCALIZATION_TOP_FEATURES = 15
LOCALIZATION_CONFIG_LABELS = {
    'tier0': 'Tier-0',
    'tier0_tier1': 'Tier-0/1',
    'tier0_tier1_tier2': 'Tier-0/1/2',
    'tier0_tier1_alt_tier2': 'Tier-0/1/2',
}
LOCALIZATION_CONFIG_ORDER = ['tier0', 'tier0_tier1', 'tier0_tier1_tier2', 'tier0_tier1_alt_tier2']
LOCALIZATION_STRESSOR_ORDER = ['ATOMIC', 'BRANCH', 'CACHE', 'MEMBW', 'TLB']
LOCALIZATION_PROFILE_ORDER = [p for p in PROFILE_ORDER if (profile_global_dir(p) / 'case_predictions.csv').exists()]


def localization_feature_tier(feature_name: str) -> str:
    if not isinstance(feature_name, str) or ':' not in feature_name:
        return 'unknown'
    return feature_name.split(':', 1)[0]


def localization_has_token(feature_name: str, tokens) -> bool:
    feature_name = str(feature_name).lower()
    return any(str(token).lower() in feature_name for token in tokens)


def localization_pick_feature_name(block, monotonic_tokens, generic_tokens):
    candidates = []
    for rank in [1, 2, 3]:
        feat = getattr(block, f'top_feature_{rank}', '')
        score = getattr(block, f'top_feature_score_{rank}', np.nan)
        if not isinstance(feat, str) or not feat:
            continue
        candidates.append((rank, feat, float(score) if pd.notna(score) else np.nan))

    if not candidates:
        return '', np.nan, 'unavailable'

    actionable = [item for item in candidates if not localization_has_token(item[1], monotonic_tokens) and not localization_has_token(item[1], generic_tokens)]
    if actionable:
        rank, feat, score = actionable[0]
        return feat, score, 'actionable'

    non_monotonic = [item for item in candidates if not localization_has_token(item[1], monotonic_tokens)]
    if non_monotonic:
        rank, feat, score = non_monotonic[0]
        return feat, score, 'non-monotonic'

    rank, feat, score = candidates[0]
    return feat, score, 'fallback'


def localization_majority(series: pd.Series, fallback: str = 'unknown') -> str:
    vals = series.dropna().astype(str)
    vals = vals[vals != '']
    if vals.empty:
        return fallback
    return vals.value_counts().idxmax()


def localization_pick_blocks(blocks: pd.DataFrame, n_blocks: int = LOCALIZATION_BLOCK_COUNT) -> tuple[pd.DataFrame, str]:
    blocks = blocks.sort_values('block_idx').copy()

    persist = blocks[blocks['persist_alert'] == 1].copy()
    if not persist.empty:
        return persist.head(n_blocks).copy(), 'post-persist abnormal windows'

    block_alert = blocks[blocks['block_alert'] == 1].copy()
    if not block_alert.empty:
        return block_alert.head(n_blocks).copy(), 'post-threshold abnormal windows'

    diagnosis = blocks[blocks['is_selected_for_diagnosis'] == 1].copy()
    if not diagnosis.empty:
        return diagnosis.sort_values(['block_idx', 'score'], ascending=[True, False]).head(n_blocks).copy(), 'diagnosis-selected windows'

    return blocks.nlargest(n_blocks, 'score').sort_values('block_idx').copy(), 'peak-score windows'


def build_profile_localization(profile: str) -> dict[str, pd.DataFrame]:
    loc_out = profile_global_dir(profile)
    loc_paper = profile_paper_dir(profile)
    loc_paper.mkdir(parents=True, exist_ok=True)

    pred_df = pd.read_csv(loc_out / 'case_predictions.csv')
    trace_df = pd.read_csv(loc_out / 'case_block_traces.csv')
    diag_df = pd.read_csv(loc_out / 'case_diagnosis_summary.csv') if (loc_out / 'case_diagnosis_summary.csv').exists() else pd.DataFrame()
    profile_label = PROFILE_LABEL.get(profile, profile)

    monotonic_tokens = tuple(getattr(FULL_MODULE, 'DIAG_MONOTONIC_TOKENS', ('uptime', 'syscall', 'ctx_switch', 'soft_interrupt', 'hard_interrupt'))) + ('sample_time',)
    generic_tokens = tuple(getattr(FULL_MODULE, 'DIAG_GENERIC_MEMORY_STATE_TOKENS', ('mem_free_bytes', 'mem_inactive_bytes', 'mem_available_bytes', 'mem_percent', 'swap_percent', 'swap_free_bytes', 'swap_used_bytes')))

    stressor_rows = []
    feature_rows = []
    timeline_rows = []
    rep_rows = []

    cfgs = [cfg for cfg in LOCALIZATION_CONFIG_ORDER if cfg in set(pred_df['config'].astype(str))]
    if not cfgs:
        cfgs = sorted(pred_df['config'].dropna().astype(str).unique())

    for cfg in cfgs:
        cfg_pred = pred_df[(pred_df['config'] == cfg) & (pred_df['label'] == 1)].copy()
        cfg_trace = trace_df[(trace_df['config'] == cfg) & (trace_df['label'] == 1)].copy()
        if cfg_pred.empty or cfg_trace.empty:
            continue

        for stressor, part in cfg_pred.groupby('stressor', sort=False):
            stressor_rows.append({
                'feature_profile': profile,
                'profile_label': profile_label,
                'config': cfg,
                'config_label': LOCALIZATION_CONFIG_LABELS.get(cfg, cfg),
                'stressor': stressor,
                'mean_run_score': float(pd.to_numeric(part['run_score'], errors='coerce').mean()),
                'mean_run_score_wc': float(pd.to_numeric(part.get('run_score_wc', np.nan), errors='coerce').mean()) if 'run_score_wc' in part.columns else np.nan,
                'mean_peak_block_score': float(pd.to_numeric(part.get('peak_block_score', np.nan), errors='coerce').mean()) if 'peak_block_score' in part.columns else np.nan,
                'mean_run_alert': float(pd.to_numeric(part['run_alert'], errors='coerce').mean()),
                'n_cases': int(len(part)),
            })

        reps = []
        for case_id, blocks in cfg_trace.groupby('case_id', sort=False):
            picked, loc_source = localization_pick_blocks(blocks, LOCALIZATION_BLOCK_COUNT)
            if picked.empty:
                continue
            picked = picked.sort_values('block_idx').reset_index(drop=True)
            meta = cfg_pred[cfg_pred['case_id'] == case_id].iloc[0]
            rep_block_times = picked['block_end_s'].astype(float).tolist()
            diag_row = diag_df[(diag_df['config'] == cfg) & (diag_df['case_id'] == case_id)]
            dominant_tier = localization_majority(picked['dominant_tier'])
            dominant_mech = localization_majority(picked['dominant_mechanism'])
            if not diag_row.empty:
                dominant_tier = str(diag_row['dominant_tier'].iloc[0]) if 'dominant_tier' in diag_row.columns else dominant_tier
                dominant_mech = str(diag_row['dominant_mechanism'].iloc[0]) if 'dominant_mechanism' in diag_row.columns else dominant_mech

            reps.append({
                'feature_profile': profile,
                'profile_label': profile_label,
                'config': cfg,
                'config_label': LOCALIZATION_CONFIG_LABELS.get(cfg, cfg),
                'case_id': case_id,
                'workload': meta['workload'],
                'stressor': meta['stressor'],
                'run_score': float(meta['run_score']),
                'run_alert': int(meta['run_alert']),
                'localization_source': loc_source,
                'dominant_tier': dominant_tier,
                'dominant_mechanism': dominant_mech,
                'n_recorded_blocks': int(len(rep_block_times)),
                'first_block_s': rep_block_times[0] if len(rep_block_times) > 0 else np.nan,
                'second_block_s': rep_block_times[1] if len(rep_block_times) > 1 else np.nan,
                'third_block_s': rep_block_times[2] if len(rep_block_times) > 2 else np.nan,
                'fourth_block_s': rep_block_times[3] if len(rep_block_times) > 3 else np.nan,
                'fifth_block_s': rep_block_times[4] if len(rep_block_times) > 4 else np.nan,
            })

            for block in picked.itertuples(index=False):
                feat, feat_score, selection_mode = localization_pick_feature_name(block, monotonic_tokens, generic_tokens)
                if feat and selection_mode != 'fallback':
                    feature_rows.append({
                        'feature_profile': profile,
                        'profile_label': profile_label,
                        'config': cfg,
                        'config_label': LOCALIZATION_CONFIG_LABELS.get(cfg, cfg),
                        'case_id': case_id,
                        'workload': meta['workload'],
                        'stressor': meta['stressor'],
                        'block_idx': int(block.block_idx),
                        'block_end_s': float(block.block_end_s),
                        'feature_name': feat,
                        'feature_tier': localization_feature_tier(feat),
                        'feature_score': float(feat_score) if pd.notna(feat_score) else np.nan,
                        'selection_mode': selection_mode,
                    })

            timeline_rows.append({
                'feature_profile': profile,
                'profile_label': profile_label,
                'config': cfg,
                'config_label': LOCALIZATION_CONFIG_LABELS.get(cfg, cfg),
                'case_id': case_id,
                'workload': meta['workload'],
                'stressor': meta['stressor'],
                'run_score': float(meta['run_score']),
                'run_alert': int(meta['run_alert']),
                'localization_source': loc_source,
                'first_block_s': rep_block_times[0] if len(rep_block_times) > 0 else np.nan,
                'second_block_s': rep_block_times[1] if len(rep_block_times) > 1 else np.nan,
                'third_block_s': rep_block_times[2] if len(rep_block_times) > 2 else np.nan,
                'fourth_block_s': rep_block_times[3] if len(rep_block_times) > 3 else np.nan,
                'fifth_block_s': rep_block_times[4] if len(rep_block_times) > 4 else np.nan,
                'n_recorded_blocks': int(len(rep_block_times)),
            })

        rep_df = pd.DataFrame(reps)
        if not rep_df.empty:
            rep_df['stressor_order'] = rep_df['stressor'].map({name: idx for idx, name in enumerate(LOCALIZATION_STRESSOR_ORDER)}).fillna(999)
            rep_keep = rep_df.sort_values(['stressor_order', 'run_score'], ascending=[True, False]).groupby('stressor', as_index=False).head(1).reset_index(drop=True)
            rep_rows.extend(rep_keep.drop(columns=['stressor_order']).to_dict('records'))

    stressor_df = pd.DataFrame(stressor_rows)
    if not stressor_df.empty:
        stressor_df['stressor_order'] = stressor_df['stressor'].map({name: idx for idx, name in enumerate(LOCALIZATION_STRESSOR_ORDER)}).fillna(999)
        stressor_df = stressor_df.sort_values(['feature_profile', 'config', 'mean_run_score', 'stressor_order'], ascending=[True, True, False, True]).reset_index(drop=True)
        stressor_df['rank'] = stressor_df.groupby(['feature_profile', 'config']).cumcount() + 1
        stressor_df = stressor_df.drop(columns=['stressor_order'])

    feature_df = pd.DataFrame(feature_rows)
    if not feature_df.empty:
        feature_df = (
            feature_df.groupby(['feature_profile', 'profile_label', 'config', 'config_label', 'feature_name', 'feature_tier'], as_index=False)
            .agg(
                hit_count=('feature_name', 'size'),
                case_count=('case_id', 'nunique'),
                stressor_count=('stressor', 'nunique'),
                mean_feature_score=('feature_score', 'mean'),
                earliest_block_s=('block_end_s', 'min'),
            )
            .sort_values(['feature_profile', 'config', 'hit_count', 'case_count', 'mean_feature_score'], ascending=[True, True, False, False, False])
            .reset_index(drop=True)
        )
        feature_df['rank'] = feature_df.groupby(['feature_profile', 'config']).cumcount() + 1
        feature_df = feature_df[feature_df['rank'] <= LOCALIZATION_TOP_FEATURES].reset_index(drop=True)

    timeline_df = pd.DataFrame(timeline_rows)
    rep_df = pd.DataFrame(rep_rows)
    for df_name, frame in {
        'industry_localization_stressors.csv': stressor_df,
        'industry_localization_top_features.csv': feature_df,
        'industry_localization_timelines.csv': timeline_df,
        'industry_localization_representative_cases.csv': rep_df,
    }.items():
        frame.to_csv(loc_paper / df_name, index=False)

    return {
        'stressor_df': stressor_df,
        'feature_df': feature_df,
        'timeline_df': timeline_df,
        'rep_df': rep_df,
    }


INDUSTRY_LOCALIZATION_RESULTS = {}
all_stressors = []
all_features = []
all_timelines = []
all_reps = []
for profile in LOCALIZATION_PROFILE_ORDER:
    try:
        out = build_profile_localization(profile)
    except FileNotFoundError as exc:
        display(Markdown(f'`{PROFILE_LABEL.get(profile, profile)}` localization skipped: `{exc}`'))
        continue
    INDUSTRY_LOCALIZATION_RESULTS[profile] = out
    if not out['stressor_df'].empty:
        all_stressors.append(out['stressor_df'])
    if not out['feature_df'].empty:
        all_features.append(out['feature_df'])
    if not out['timeline_df'].empty:
        all_timelines.append(out['timeline_df'])
    if not out['rep_df'].empty:
        all_reps.append(out['rep_df'])

INDUSTRY_LOCALIZATION_STRESSORS = pd.concat(all_stressors, ignore_index=True) if all_stressors else pd.DataFrame()
INDUSTRY_LOCALIZATION_TOP_FEATURES = pd.concat(all_features, ignore_index=True) if all_features else pd.DataFrame()
INDUSTRY_LOCALIZATION_TIMELINE = pd.concat(all_timelines, ignore_index=True) if all_timelines else pd.DataFrame()
INDUSTRY_LOCALIZATION_REP_CASES = pd.concat(all_reps, ignore_index=True) if all_reps else pd.DataFrame()

globals()['INDUSTRY_LOCALIZATION_RESULTS'] = INDUSTRY_LOCALIZATION_RESULTS
globals()['INDUSTRY_LOCALIZATION_STRESSORS'] = INDUSTRY_LOCALIZATION_STRESSORS
globals()['INDUSTRY_LOCALIZATION_TOP_FEATURES'] = INDUSTRY_LOCALIZATION_TOP_FEATURES
globals()['INDUSTRY_LOCALIZATION_TIMELINE'] = INDUSTRY_LOCALIZATION_TIMELINE
globals()['INDUSTRY_LOCALIZATION_REP_CASES'] = INDUSTRY_LOCALIZATION_REP_CASES

if INDUSTRY_LOCALIZATION_STRESSORS.empty:
    display(Markdown('No localization tables were produced yet. Make sure the global result bundles include `case_predictions.csv` and `case_block_traces.csv`.'))
else:
    final_cfgs = [cfg for cfg in ['tier0_tier1_tier2', 'tier0_tier1_alt_tier2'] if cfg in set(INDUSTRY_LOCALIZATION_STRESSORS['config'])]
    final_cfg = final_cfgs[0] if final_cfgs else INDUSTRY_LOCALIZATION_STRESSORS['config'].iloc[-1]

    display(Markdown('#### Top stressors by profile and tier'))
    display(
        INDUSTRY_LOCALIZATION_STRESSORS[
            INDUSTRY_LOCALIZATION_STRESSORS['rank'] <= 5
        ][['profile_label', 'config_label', 'rank', 'stressor', 'mean_run_score', 'mean_run_alert', 'mean_peak_block_score']].round(3)
    )

    display(Markdown('#### Top localized features in the final head'))
    feat_show = INDUSTRY_LOCALIZATION_TOP_FEATURES[
        (INDUSTRY_LOCALIZATION_TOP_FEATURES['config'] == final_cfg) & (INDUSTRY_LOCALIZATION_TOP_FEATURES['rank'] <= 10)
    ][['profile_label', 'rank', 'feature_name', 'feature_tier', 'hit_count', 'case_count', 'mean_feature_score', 'earliest_block_s']]
    display(feat_show.round(3))

    display(Markdown('#### Representative first-through-fifth abnormal windows'))
    rep_show = INDUSTRY_LOCALIZATION_REP_CASES[
        INDUSTRY_LOCALIZATION_REP_CASES['config'] == final_cfg
    ][['profile_label', 'stressor', 'workload', 'first_block_s', 'second_block_s', 'third_block_s', 'fourth_block_s', 'fifth_block_s', 'dominant_tier', 'dominant_mechanism']]
    display(rep_show.round(1))

In [ ]:
# Plain-language: This cell localizes anomalies in time by marking the first several abnormal windows for representative cases in both mixed and full.

display(Markdown('### Industry-facing anomaly localization figures'))

if 'INDUSTRY_LOCALIZATION_RESULTS' not in globals() or not INDUSTRY_LOCALIZATION_RESULTS:
    display(Markdown('Run the localization tables cell first so the representative cases and localization times are available.'))
else:
    LOCALIZATION_WINDOWS_TO_SHOW = 5
    highlight_color = '#E15759'

    def _fmt_time(v):
        v = float(v)
        if np.isclose(v, round(v), atol=0.05):
            return f'{int(round(v))}s'
        return f'{v:.1f}s'

    def _zoom_window_around_picks(all_x, picked_x):
        all_x = np.asarray(all_x, dtype=float)
        picked_x = np.sort(np.unique(np.asarray(picked_x, dtype=float)))

        full_min = float(np.nanmin(all_x))
        full_max = float(np.nanmax(all_x))
        local_min = float(np.nanmin(picked_x))
        local_max = float(np.nanmax(picked_x))

        if picked_x.size > 1:
            step = float(np.nanmedian(np.diff(picked_x)))
            if not np.isfinite(step) or step <= 0:
                step = max((local_max - local_min) / max(len(picked_x) - 1, 1), 1.0)
        else:
            uniq = np.unique(all_x)
            if uniq.size > 1:
                step = float(np.nanmedian(np.diff(uniq)))
                if not np.isfinite(step) or step <= 0:
                    step = max((full_max - full_min) / 20.0, 1.0)
            else:
                step = 1.0

        local_span = max(local_max - local_min, step)
        left_pad = max(2.6 * step, 0.30 * local_span)
        right_pad = max(2.0 * step, 0.24 * local_span)

        zoom_left = max(full_min, local_min - left_pad)
        zoom_right = min(full_max, local_max + right_pad)

        if zoom_right - zoom_left < 4.8 * step:
            mid = 0.5 * (zoom_left + zoom_right)
            half = 2.5 * step
            zoom_left = max(full_min, mid - half)
            zoom_right = min(full_max, mid + half)

        return zoom_left, zoom_right

    displayed = []
    for profile in LOCALIZATION_PROFILE_ORDER:
        if profile not in INDUSTRY_LOCALIZATION_RESULTS:
            continue

        loc_out = profile_global_dir(profile)
        loc_paper = profile_paper_dir(profile)
        loc_fig_dir = loc_paper / 'figures'
        loc_fig_dir.mkdir(parents=True, exist_ok=True)
        trace_path = loc_out / 'case_block_traces.csv'
        if not trace_path.exists():
            continue

        trace_df = pd.read_csv(trace_path)
        rep_df = INDUSTRY_LOCALIZATION_RESULTS[profile]['rep_df'].copy()
        if rep_df.empty:
            continue

        trace_cfgs = [cfg for cfg in LOCALIZATION_CONFIG_ORDER if cfg in set(rep_df['config'])]
        if not trace_cfgs:
            trace_cfgs = sorted(rep_df['config'].dropna().astype(str).unique())

        for cfg in trace_cfgs:
            reps = rep_df[rep_df['config'] == cfg].copy()
            if reps.empty:
                continue

            reps['stressor_order'] = reps['stressor'].map(
                {name: idx for idx, name in enumerate(LOCALIZATION_STRESSOR_ORDER)}
            ).fillna(999)
            reps = reps.sort_values(['stressor_order', 'workload', 'case_id']).reset_index(drop=True)

            fig = plt.figure(figsize=(18.8, 4.25 * len(reps)))
            gs = fig.add_gridspec(
                len(reps),
                2,
                width_ratios=[3.7, 1.55],
                left=0.08,
                right=0.83,
                bottom=0.12,
                top=0.90,
                hspace=0.74,
                wspace=0.20,
            )

            main_axes = []
            zoom_axes = []

            for ridx, rep in enumerate(reps.itertuples(index=False)):
                ax = fig.add_subplot(gs[ridx, 0])
                ax_zoom = fig.add_subplot(gs[ridx, 1])
                main_axes.append(ax)
                zoom_axes.append(ax_zoom)

                blocks = trace_df[
                    (trace_df['config'] == rep.config) & (trace_df['case_id'] == rep.case_id)
                ].copy()
                blocks = blocks.sort_values('block_idx').reset_index(drop=True)

                picked, localization_source = localization_pick_blocks(blocks, LOCALIZATION_WINDOWS_TO_SHOW)
                if picked.empty:
                    continue

                picked = picked.sort_values('block_end_s').reset_index(drop=True)

                x = blocks['block_end_s'].to_numpy(dtype=float)
                y = blocks['score'].to_numpy(dtype=float)
                picked_x = picked['block_end_s'].to_numpy(dtype=float)

                for axis, face in [(ax, '#FBFCFE'), (ax_zoom, '#FFFFFF')]:
                    axis.set_facecolor(face)
                    for spine in axis.spines.values():
                        spine.set_color('#CBD5E1')
                        spine.set_linewidth(1.15)

                ax.plot(
                    x,
                    y,
                    color='#355C7D',
                    linewidth=3.0,
                    alpha=0.98,
                    solid_capstyle='round',
                    zorder=2,
                )

                threshold = pd.to_numeric(blocks.get('threshold', np.nan), errors='coerce').dropna()
                if not threshold.empty:
                    thr = float(threshold.iloc[0])
                    ax.axhline(
                        thr,
                        color='#C44E52',
                        linestyle=(0, (5, 3)),
                        linewidth=2.1,
                        alpha=0.95,
                        zorder=1,
                    )
                else:
                    thr = None

                span_start = float(picked['block_end_s'].min())
                span_end = float(picked['block_end_s'].max())
                ax.axvspan(span_start, span_end, color='#F59E0B', alpha=0.11, zorder=0)

                finite_y = pd.to_numeric(blocks['score'], errors='coerce').dropna().to_numpy(dtype=float)
                if finite_y.size:
                    ymin = float(np.nanmin(finite_y))
                    ymax = float(np.nanmax(finite_y))
                    if thr is not None:
                        ymin = min(ymin, thr)
                        ymax = max(ymax, thr)
                    span = max(ymax - ymin, 1e-6)
                    ax.set_ylim(ymin - 0.08 * span, ymax + 0.22 * span)

                x_min = float(np.nanmin(x))
                x_max = float(np.nanmax(x))
                x_span = max(x_max - x_min, 1e-9)
                ax.set_xlim(max(0.0, x_min - 0.02 * x_span), x_max + 0.02 * x_span)
                ax.margins(x=0)
                ax.xaxis.set_major_locator(MaxNLocator(nbins=7, min_n_ticks=5))

                time_handles = []
                for idx, block in enumerate(picked.itertuples(index=False), start=1):
                    bx = float(block.block_end_s)
                    by = float(block.score)

                    ax.scatter(
                        bx,
                        by,
                        color=highlight_color,
                        s=250,
                        edgecolor='white',
                        linewidth=1.35,
                        zorder=5,
                    )
                    ax.text(
                        bx,
                        by,
                        str(idx),
                        ha='center',
                        va='center',
                        fontsize=12.4,
                        fontweight='bold',
                        color='white',
                        zorder=6,
                    )

                    time_handles.append(
                        Line2D(
                            [0], [0],
                            marker='o',
                            linestyle='None',
                            markersize=10.2,
                            markerfacecolor=highlight_color,
                            markeredgecolor='white',
                            markeredgewidth=1.0,
                            label=f'{idx} = {_fmt_time(bx)}',
                        )
                    )

                zoom_left, zoom_right = _zoom_window_around_picks(x, picked_x)
                zoom_mask = (x >= zoom_left) & (x <= zoom_right)
                if zoom_mask.sum() < 2:
                    zoom_mask = np.ones_like(x, dtype=bool)

                x_zoom = x[zoom_mask]
                y_zoom = y[zoom_mask]

                ax_zoom.plot(
                    x_zoom,
                    y_zoom,
                    color='#355C7D',
                    linewidth=2.8,
                    alpha=0.98,
                    solid_capstyle='round',
                    zorder=2,
                )

                if thr is not None:
                    ax_zoom.axhline(
                        thr,
                        color='#C44E52',
                        linestyle=(0, (5, 3)),
                        linewidth=1.8,
                        alpha=0.95,
                        zorder=1,
                    )

                ax_zoom.axvspan(span_start, span_end, color='#F59E0B', alpha=0.11, zorder=0)

                for idx, block in enumerate(picked.itertuples(index=False), start=1):
                    bx = float(block.block_end_s)
                    by = float(block.score)
                    ax_zoom.scatter(
                        bx,
                        by,
                        color=highlight_color,
                        s=230,
                        edgecolor='white',
                        linewidth=1.2,
                        zorder=5,
                    )
                    ax_zoom.text(
                        bx,
                        by,
                        str(idx),
                        ha='center',
                        va='center',
                        fontsize=11.6,
                        fontweight='bold',
                        color='white',
                        zorder=6,
                    )

                zoom_y = pd.to_numeric(blocks.loc[zoom_mask, 'score'], errors='coerce').dropna().to_numpy(dtype=float)
                if zoom_y.size:
                    zoom_ymin = float(np.nanmin(zoom_y))
                    zoom_ymax = float(np.nanmax(zoom_y))
                    picked_y = picked['score'].to_numpy(dtype=float)
                    zoom_ymin = min(zoom_ymin, float(np.nanmin(picked_y)))
                    zoom_ymax = max(zoom_ymax, float(np.nanmax(picked_y)))
                    if thr is not None:
                        zoom_ymin = min(zoom_ymin, thr)
                        zoom_ymax = max(zoom_ymax, thr)
                    zoom_span = max(zoom_ymax - zoom_ymin, 1e-6)
                    ax_zoom.set_ylim(zoom_ymin - 0.10 * zoom_span, zoom_ymax + 0.20 * zoom_span)

                ax_zoom.set_xlim(zoom_left, zoom_right)
                ax_zoom.set_xticks(picked_x)
                ax_zoom.set_xticklabels([_fmt_time(v).replace('s', '') for v in picked_x])
                if len(picked_x) >= 4:
                    for lbl in ax_zoom.get_xticklabels():
                        lbl.set_rotation(18)
                        lbl.set_ha('right')
                        lbl.set_rotation_mode('anchor')

                ax_zoom.grid(alpha=0.16, linewidth=0.85, color='#CBD5E1')
                ax_zoom.set_title(
                    'Zoomed localization window',
                    fontsize=13.0,
                    fontweight='bold',
                    color='#0F172A',
                    pad=10,
                )

                point_legend = ax_zoom.legend(
                    handles=time_handles,
                    title='Highlighted times',
                    loc='upper left',
                    bbox_to_anchor=(1.02, 1.0),
                    frameon=True,
                    fancybox=True,
                    fontsize=11.0,
                    title_fontsize=11.7,
                    borderpad=0.46,
                    labelspacing=0.36,
                    handletextpad=0.55,
                    columnspacing=0.8,
                )
                point_legend.get_frame().set_facecolor('white')
                point_legend.get_frame().set_edgecolor('#E2E8F0')
                point_legend.get_frame().set_alpha(0.97)

                ax.set_ylabel('Score', fontsize=14.2, fontweight='bold', color='#0F172A', labelpad=10)
                ax.set_xlabel('Block end time (s)', fontsize=14.4, fontweight='bold', color='#0F172A', labelpad=10)
                ax.tick_params(axis='x', labelbottom=True, labelsize=12.8, colors='#0F172A')
                ax.tick_params(axis='y', labelsize=12.8, colors='#0F172A')
                ax.grid(alpha=0.18, linewidth=0.95, color='#CBD5E1')

                ax_zoom.set_xlabel('Block end time (s)', fontsize=13.4, fontweight='bold', color='#0F172A', labelpad=8)
                ax_zoom.tick_params(axis='x', labelsize=11.0, colors='#0F172A')
                ax_zoom.tick_params(axis='y', labelsize=11.0, colors='#0F172A')

                ax.set_title(
                    f"{PROFILE_LABEL.get(profile, profile)} | {LOCALIZATION_CONFIG_LABELS.get(cfg, cfg)} | {rep.stressor} on {rep.workload}",
                    fontsize=14.8,
                    fontweight='bold',
                    color='#0F172A',
                    pad=30,
                    loc='left',
                )
                ax.text(
                    0.0,
                    1.05,
                    f"Tier: {rep.dominant_tier} | Mechanism: {rep.dominant_mechanism} | highlighted windows: {len(picked)} of {LOCALIZATION_WINDOWS_TO_SHOW}",
                    transform=ax.transAxes,
                    ha='left',
                    va='bottom',
                    fontsize=12.2,
                    color='#475569',
                    fontweight='bold',
                )

            legend_handles = [
                Line2D([0], [0], color='#355C7D', linewidth=3.0, label='Score trace'),
                Line2D([0], [0], color='#C44E52', linewidth=2.1, linestyle=(0, (5, 3)), label='Threshold'),
                Patch(facecolor='#F59E0B', edgecolor='none', alpha=0.18, label='Localization span'),
                Line2D([0], [0], marker='o', linestyle='None', markersize=10.0, markerfacecolor=highlight_color, markeredgecolor='white', markeredgewidth=1.0, label='Numbered abnormal window'),
            ]

            fig.legend(
                handles=legend_handles,
                loc='lower center',
                bbox_to_anchor=(0.43, 0.015),
                ncol=4,
                frameon=False,
                fontsize=12.2,
                handlelength=2.4,
                columnspacing=1.5,
            )

            fig.suptitle(
                f"DICE anomaly localization windows: {PROFILE_LABEL.get(profile, profile)} {LOCALIZATION_CONFIG_LABELS.get(cfg, cfg)}",
                fontsize=18.2,
                fontweight='bold',
                y=0.992,
                color='#0F172A',
            )

            out_png = loc_fig_dir / f"fig_industry_localization_{cfg}.png"
            fig.savefig(out_png, dpi=220, bbox_inches='tight', facecolor='white')
            plt.close(fig)

            if cfg in {'tier0_tier1_tier2', 'tier0_tier1_alt_tier2'}:
                displayed.append((PROFILE_LABEL.get(profile, profile), out_png))

    if displayed:
        for profile_label, out_png in displayed:
            display(Markdown(f'#### {profile_label} final-head localization'))
            display(Image(filename=str(out_png)))
    else:
        display(Markdown('Localization figures were written to disk, but no final-head figures were available to display.'))



## 8C. Hardware-Aware Localization on the M2 Pro

This section translates DICE's recurring top features and stressors into likely M2 Pro subsystems.

- It stays realistic: the notebook localizes to **CPU**, **GPU/display**, **Neural Engine**, **unified memory/swap**, **storage**, or **runtime scheduling**.
- It does **not** claim to identify a single transistor path or exact physical defect site.
- For this laptop, memory and swap features are especially important because the machine has **16 GB unified memory**, and GPU/display features should be read with the external Dell monitor in mind.


In [ ]:
if not hardware_hotspot_df.empty:
    display(Markdown('#### Most visible hardware hotspots in the final head'))
    hotspot_show = hardware_hotspot_df.sort_values(
        ['profile_label', 'total_hit_count', 'n_top_features'],
        ascending=[True, False, False]
    ).copy()
    display(
        hotspot_show[
            ['profile_label', 'likely_subsystem', 'likely_hardware_block', 'total_hit_count', 'n_top_features', 'max_feature_score']
        ]
        .reset_index(drop=True)
        .round(3)
    )

    profiles = [p for p in PROFILE_ORDER if p in set(hardware_hotspot_df['feature_profile'])]
    fig, axes = plt.subplots(
        1,
        max(1, len(profiles)),
        figsize=(8.2 * max(1, len(profiles)), 5.8),
        squeeze=False,
        constrained_layout=True,
    )
    axes = axes[0]

    for ax, profile in zip(axes, profiles):
        part = hardware_hotspot_df[hardware_hotspot_df['feature_profile'] == profile].copy()
        part = part.sort_values(
            ['total_hit_count', 'n_top_features', 'likely_subsystem'],
            ascending=[True, True, False]
        ).reset_index(drop=True)

        values = pd.to_numeric(part['total_hit_count'], errors='coerce').to_numpy(dtype=float)
        max_val = float(np.nanmax(values)) if len(values) else 1.0
        right_pad = max(0.8, 0.18 * max_val)
        x_max = max_val + right_pad

        bars = ax.barh(
            part['likely_subsystem'],
            values,
            color=PROFILE_COLOR_MAP.get(profile, '#4C78A8'),
            alpha=0.95,
        )

        ax.set_xlim(0, x_max)

        for bar, row in zip(bars, part.itertuples(index=False)):
            val = float(row.total_hit_count)
            label_x = min(val + 0.10 * right_pad, x_max - 0.10 * right_pad)
            ax.text(
                label_x,
                bar.get_y() + bar.get_height() / 2.0,
                f"{val:.1f}",
                va='center',
                ha='left',
                fontsize=12.2,
                fontweight='bold',
                color='#0F172A',
            )

        ax.set_title(
            f"{PROFILE_LABEL.get(profile, profile)} final head",
            fontsize=15.2,
            fontweight='bold',
            color='#0F172A',
            pad=10,
        )
        ax.set_xlabel(
            'Total recurring feature hits',
            fontsize=13.2,
            fontweight='bold',
            color='#0F172A',
            labelpad=8,
        )
        ax.set_ylabel(
            'Likely subsystem',
            fontsize=13.2,
            fontweight='bold',
            color='#0F172A',
            labelpad=8,
        )
        ax.tick_params(axis='x', labelsize=12.0, colors='#0F172A')
        ax.tick_params(axis='y', labelsize=12.0, colors='#0F172A')
        ax.grid(axis='x', alpha=0.20)

        for spine in ax.spines.values():
            spine.set_color('#CBD5E1')
            spine.set_linewidth(1.1)

    fig.suptitle(
        'DICE hardware hotspots on the M2 Pro',
        fontsize=17.2,
        fontweight='bold',
        color='#0F172A',
    )
    fig.savefig(HARDWARE_HOTSPOTS_PNG, dpi=PRO_FIG_DPI, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    display(Image(filename=str(HARDWARE_HOTSPOTS_PNG)))




## 9. Tier and Mechanism Attribution Dashboard

These plots explain where the detector is getting its evidence.

- The tier view shows which sensing level contributes the most.
- The mechanism view shows which type of system behavior contributes the most.
- The confusion matrix shows which stressors are easiest or hardest for the diagnosis head to tell apart.


## 9A. Diagnosis Management Controls

DICE keeps **detection** and **diagnosis** separate on purpose.

- The detector still uses the full aligned feature set to produce residual scores, conformal p-values, and persistent alerts.
- The diagnosis head uses a **diagnosis-only weighting** so explanatory features are less likely to be dominated by non-causal stock counters.
- In the current notebook, DICE now applies three diagnosis controls:
  - it removes **monotonic counters** such as uptime and cumulative interrupt/syscall counters,
  - it **downweights generic memory-state features** such as `mem_free_bytes`, `mem_inactive_bytes`, and `swap_percent`,
  - and it compares **whole-run attribution** against a **post-alert window** attribution built from blocks immediately after the first alert.
- The notebook reports both modes, but it keeps the stronger one as the paper-facing default.


In [ ]:
# Plain-language: This cell turns on the diagnosis controls that make the attribution and triage outputs more reliable.
display(Markdown('### Diagnosis controls now active'))

diag_profile = globals().get('ACTIVE_RESULT_PROFILE', DISPLAY_FEATURE_PROFILE if 'DISPLAY_FEATURE_PROFILE' in globals() else 'mixed')
diag_out = globals().get('ACTIVE_OUT_FULL', OUT_FULL if 'OUT_FULL' in globals() else Path(DATASET_ROOT) / 'results_dice_full')

FINAL_DIAG_CONFIGS = ['tier0_tier1_tier2', 'tier0_tier1_alt_tier2']

def _pick_diag_final_config(df: pd.DataFrame) -> str | None:
    available = set(df['config'].dropna().astype(str)) if not df.empty and 'config' in df.columns else set()
    for cfg in FINAL_DIAG_CONFIGS:
        if cfg in available:
            return cfg
    return sorted(available)[-1] if available else None

def _fmt_diag_metric(val):
    return '—' if pd.isna(val) else f'{float(val):.3f}'

diag_controls = pd.DataFrame([
    {'Setting': 'Detector score path', 'Value': 'All aligned features remain active for residual scoring and conformal alerts.'},
    {'Setting': 'Default diagnosis mode', 'Value': 'Whole-run weighted residual attribution'},
    {'Setting': 'Comparison diagnosis mode', 'Value': f'Post-alert window ({FULL_MODULE.DIAG_POST_ALERT_BLOCKS} blocks)'},
    {'Setting': 'Excluded from diagnosis', 'Value': ', '.join(FULL_MODULE.DIAG_MONOTONIC_TOKENS)},
    {'Setting': 'Downweighted in diagnosis', 'Value': ', '.join(FULL_MODULE.DIAG_GENERIC_MEMORY_STATE_TOKENS)},
    {'Setting': 'Downweight factor', 'Value': FULL_MODULE.DIAG_GENERIC_MEMORY_STATE_FACTOR},
])
display(diag_controls)

display(Markdown(f"Active diagnosis bundle: **{diag_profile}** at `{diag_out}`"))

cmp_path = Path(diag_out) / 'diagnosis_mode_comparison.csv'
final_cfg = None

if cmp_path.exists():
    cmp_df = pd.read_csv(cmp_path)
    final_cfg = _pick_diag_final_config(cmp_df)

    if final_cfg is not None:
        cmp_view = cmp_df[cmp_df['config'].astype(str) == str(final_cfg)].copy()

        metric_cols = [
            'top1_acc', 'top2_acc', 'macro_f1',
            'coverage', 'selective_top1_acc', 'selective_top2_acc'
        ]
        cmp_view = cmp_view[cmp_view[metric_cols].notna().any(axis=1)].copy()

        target_label_map = {
            'feature_level': 'Feature-level',
            'hierarchical': 'Hierarchical',
            'family_level': 'Family-level',
            'supervised': 'Supervised',
        }
        mode_label_map = {
            'whole_run': 'Whole-run',
            'post_alert_window': 'Post-alert window',
        }

        cmp_view['diagnosis_target'] = cmp_view['diagnosis_target'].astype(str).map(target_label_map).fillna(cmp_view['diagnosis_target'])
        cmp_view['diagnosis_mode'] = cmp_view['diagnosis_mode'].astype(str).map(mode_label_map).fillna(cmp_view['diagnosis_mode'])

        cmp_display = cmp_view[
            [
                'diagnosis_target', 'diagnosis_mode',
                'top1_acc', 'top2_acc', 'macro_f1',
                'coverage', 'selective_top1_acc', 'selective_top2_acc'
            ]
        ].copy()

        for col in ['top1_acc', 'top2_acc', 'macro_f1', 'coverage', 'selective_top1_acc', 'selective_top2_acc']:
            cmp_display[col] = cmp_display[col].apply(_fmt_diag_metric)

        display(Markdown(f'### Final-head diagnosis mode comparison for `{final_cfg}`'))
        display(cmp_display.reset_index(drop=True))

case_path = Path(diag_out) / 'case_diagnosis_summary.csv'
if case_path.exists():
    case_df = pd.read_csv(case_path)

    if final_cfg is None:
        final_cfg = _pick_diag_final_config(case_df)

    if final_cfg is not None:
        final_cases = case_df[
            (case_df['config'].astype(str) == str(final_cfg)) & (pd.to_numeric(case_df['label'], errors='coerce') == 1)
        ].copy()

        if not final_cases.empty and 'top_feature_1' in final_cases.columns:
            top_feature_counts = (
                final_cases['top_feature_1']
                .dropna()
                .value_counts()
                .rename_axis('Top diagnosis feature')
                .reset_index(name='Cases')
            )
            display(Markdown(f'### Which features now dominate the final diagnosis head for `{final_cfg}`?'))
            display(top_feature_counts.head(12).reset_index(drop=True))


In [ ]:
# Plain-language: This cell summarizes which tiers and mechanism groups contribute most to DICE's diagnosis evidence.
display(Markdown('### Attribution summary tables'))

tier_csv = COMPARISON_DIR / 'tier_attribution_profiles.csv'
mechanism_csv = COMPARISON_DIR / 'mechanism_attribution_profiles.csv'
diagnosis_csv = COMPARISON_DIR / 'diagnosis_summary_profiles.csv'

ATTRIBUTION_DIAGNOSIS_FILES = [filename for _, filename in DIAGNOSIS_CANDIDATES]


def resolve_attribution_profiles():
    ready = []
    missing = {}

    for profile in PROFILE_ORDER:
        profile_missing = []
        gdir = profile_global_dir(profile)

        if not (gdir / 'case_diagnosis_summary.csv').exists():
            profile_missing.append('global/case_diagnosis_summary.csv')

        diagnosis_name = next(
            (name for name in ATTRIBUTION_DIAGNOSIS_FILES if (gdir / name).exists()),
            None,
        )
        if diagnosis_name is None:
            profile_missing.append(
                'global/stressor_hierarchical_diagnosis_metrics.csv or global/stressor_feature_diagnosis_metrics.csv or global/stressor_diagnosis_metrics.csv'
            )

        if profile_missing:
            missing[profile] = profile_missing
        else:
            ready.append(profile)

    return ready, missing


ready_profiles, missing_profiles = resolve_attribution_profiles()

if not ready_profiles:
    display(Markdown('No diagnosis bundles are ready yet. Run **Run End-to-End** first.'))
    if missing_profiles:
        display(Markdown(markdown_missing_profiles(missing_profiles)))
else:
    tier_rows = []
    mechanism_rows = []
    diagnosis_rows = []

    for profile in ready_profiles:
        case_diag = read_profile_csv(profile, 'global', 'case_diagnosis_summary.csv')
        anomaly = case_diag[case_diag['label'] == 1].copy()
        if anomaly.empty:
            continue

        tier_cols = [c for c in ['tier0_share', 'tier1_alt_share', 'tier2_share'] if c in anomaly.columns]
        mech_cols = [
            c for c in [
                'compute_share',
                'memory_io_share',
                'thermal_power_share',
                'scheduler_runtime_share',
                'platform_pressure_share',
            ]
            if c in anomaly.columns
        ]

        if tier_cols:
            tier_summary = (
                anomaly.groupby('config', sort=False)[tier_cols]
                .mean()
                .reset_index()
            )
            tier_summary['feature_profile'] = profile
            tier_rows.append(add_profile_and_config_labels(tier_summary))

        if mech_cols:
            mechanism_summary = (
                anomaly.groupby('config', sort=False)[mech_cols]
                .mean()
                .reset_index()
            )
            mechanism_summary['feature_profile'] = profile
            mechanism_rows.append(add_profile_and_config_labels(mechanism_summary))

        diagnosis_name = next(
            name for name in ATTRIBUTION_DIAGNOSIS_FILES
            if (profile_global_dir(profile) / name).exists()
        )
        diagnosis_raw = read_profile_csv(profile, 'global', diagnosis_name).copy()
        diagnosis_cols = [
            c
            for c in [
                'config',
                'top1_acc',
                'top2_acc',
                'balanced_acc',
                'macro_f1',
                'mean_margin_to_second',
                'median_margin_to_second',
            ]
            if c in diagnosis_raw.columns
        ]

        if 'config' in diagnosis_cols:
            diagnosis = diagnosis_raw[diagnosis_cols].copy()
            diagnosis['feature_profile'] = profile
            diagnosis_rows.append(add_profile_and_config_labels(diagnosis))

    if tier_rows:
        tier_summary = sort_profile_config(pd.concat(tier_rows, ignore_index=True))
        tier_summary.to_csv(tier_csv, index=False)
        display(Markdown('#### Mean tier share by head'))
        display(tier_summary.round(4))
    else:
        tier_summary = pd.DataFrame()
        display(Markdown('#### Mean tier share by head'))
        display(Markdown('No anomaly-attribution tier rows were available.'))

    if mechanism_rows:
        mechanism_summary = sort_profile_config(pd.concat(mechanism_rows, ignore_index=True))
        mechanism_summary.to_csv(mechanism_csv, index=False)
        display(Markdown('#### Mean mechanism share by head'))
        display(mechanism_summary.round(4))
    else:
        mechanism_summary = pd.DataFrame()
        display(Markdown('#### Mean mechanism share by head'))
        display(Markdown('No anomaly-attribution mechanism rows were available.'))

    if diagnosis_rows:
        diagnosis_summary = sort_profile_config(pd.concat(diagnosis_rows, ignore_index=True))
        diagnosis_summary.to_csv(diagnosis_csv, index=False)
        display(Markdown('#### Diagnosis accuracy by head'))
        display(diagnosis_summary.round(4))
    else:
        diagnosis_summary = pd.DataFrame()
        display(Markdown('#### Diagnosis accuracy by head'))
        display(Markdown('No diagnosis metric rows were available.'))

    if missing_profiles:
        display(Markdown(markdown_missing_profiles(missing_profiles)))



In [ ]:
# Plain-language: This cell reports exact, top-2, and abstain-aware diagnosis quality so readers can understand diagnosis confidence.
display(Markdown('### Exact, Top-2, and abstain-aware diagnosis tables'))

DIAGNOSIS_EXACT_CSV = COMPARISON_DIR / 'diagnosis_exact_profiles.csv'
DIAGNOSIS_TOP2_CSV = COMPARISON_DIR / 'diagnosis_top2_profiles.csv'
DIAGNOSIS_ABSTAIN_CSV = COMPARISON_DIR / 'diagnosis_abstain_profiles.csv'


def _collect_ready_profiles(required_global=(), required_holdout=(), required_paper=(), required_appendix=()):
    ready = []
    missing = {}

    for profile in PROFILE_ORDER:
        missing_paths = []

        for name in required_global:
            path = profile_global_dir(profile) / name
            if not path.exists():
                missing_paths.append(path)

        for name in required_holdout:
            path = profile_holdout_dir(profile) / name
            if not path.exists():
                missing_paths.append(path)

        for name in required_paper:
            path = profile_paper_dir(profile) / name
            if not path.exists():
                missing_paths.append(path)

        for name in required_appendix:
            path = profile_appendix_dir(profile) / name
            if not path.exists():
                missing_paths.append(path)

        if missing_paths:
            missing[profile] = missing_paths
        else:
            ready.append(profile)

    return ready, missing


def _safe_missing_profiles_markdown(missing):
    helper = globals().get('markdown_missing_profiles', None)
    if callable(helper):
        return helper(missing)

    lines = ['Missing required files for some profiles:']
    for profile, paths in missing.items():
        short_paths = ', '.join(str(p.name) for p in paths)
        lines.append(f"- `{PROFILE_LABEL.get(profile, profile)}`: {short_paths}")
    return '\n'.join(lines)


ready_profiles, missing_profiles = _collect_ready_profiles(required_global=['overall_metrics.csv'])
exact_rows = []
top2_rows = []
abstain_rows = []
notes = []

if not ready_profiles:
    display(Markdown('No diagnosis-ready profiles are available yet. Run **Run End-to-End** first.'))
    if missing_profiles:
        display(Markdown(_safe_missing_profiles_markdown(missing_profiles)))
else:
    for profile in ready_profiles:
        diag, diagnosis_source, diagnosis_path = read_profile_best_diagnosis(profile)
        if not diag.empty:
            diag = diag.copy()
            diag['feature_profile'] = profile
            for col in DIAGNOSIS_RESULT_COLUMNS:
                if col not in diag.columns:
                    diag[col] = np.nan
            diag = add_profile_and_config_labels(diag)
            diag['diagnosis_source'] = diagnosis_source

            exact_rows.append(
                diag[
                    [
                        'feature_profile', 'profile_label', 'config', 'config_label', 'diagnosis_source',
                        'top1_acc', 'balanced_acc', 'macro_f1'
                    ]
                ].copy()
            )

            top2_rows.append(
                diag[
                    [
                        'feature_profile', 'profile_label', 'config', 'config_label', 'diagnosis_source',
                        'top2_acc', 'mean_margin_to_second', 'median_margin_to_second'
                    ]
                ].copy()
            )

            diag_note = diagnosis_source
            if diagnosis_path is not None:
                diag_note = f"{diagnosis_source} (`{diagnosis_path.name}`)"
            notes.append(f"- `{PROFILE_LABEL.get(profile, profile)}` exact and top-2 tables use **{diag_note}**.")

        sweep, sweep_path = read_profile_abstain_sweep(profile)
        if not sweep.empty:
            sweep = sweep.copy()
            sweep = sweep[sweep['gate_label'].isin(['trained', 'conf_0.05', 'conf_0.15', 'conf_0.25'])]
            sweep['feature_profile'] = profile
            sweep = add_profile_and_config_labels(sweep)

            abstain_rows.append(
                sweep[
                    [
                        'feature_profile', 'profile_label', 'config', 'config_label', 'gate_label', 'confidence_threshold',
                        'coverage', 'abstain_rate', 'selective_top1_acc', 'selective_top2_acc', 'selective_macro_f1'
                    ]
                ].copy()
            )

            if sweep_path is not None:
                notes.append(
                    f"- `{PROFILE_LABEL.get(profile, profile)}` abstain-aware table uses hierarchical sweep `{sweep_path.name}`."
                )

    if exact_rows:
        exact_df = sort_profile_config(pd.concat(exact_rows, ignore_index=True))
        exact_df.to_csv(DIAGNOSIS_EXACT_CSV, index=False)
        display(exact_df[['profile_label', 'config_label', 'diagnosis_source', 'top1_acc', 'balanced_acc', 'macro_f1']].round(4))
    else:
        display(Markdown('Exact diagnosis table is unavailable because no diagnosis metrics were found.'))

    if top2_rows:
        top2_df = sort_profile_config(pd.concat(top2_rows, ignore_index=True))
        top2_df.to_csv(DIAGNOSIS_TOP2_CSV, index=False)
        display(top2_df[['profile_label', 'config_label', 'diagnosis_source', 'top2_acc', 'mean_margin_to_second', 'median_margin_to_second']].round(4))
    else:
        display(Markdown('Top-2 diagnosis table is unavailable because no diagnosis metrics were found.'))

    if abstain_rows:
        abstain_df = pd.concat(abstain_rows, ignore_index=True)
        abstain_df['feature_profile'] = pd.Categorical(abstain_df['feature_profile'], categories=PROFILE_ORDER, ordered=True)
        abstain_df['config'] = pd.Categorical(abstain_df['config'], categories=CONFIG_ORDER, ordered=True)
        gate_order = {'trained': 0, 'conf_0.05': 1, 'conf_0.15': 2, 'conf_0.25': 3}
        abstain_df['gate_order'] = abstain_df['gate_label'].map(gate_order).fillna(99)
        abstain_df = abstain_df.sort_values(['feature_profile', 'config', 'gate_order']).reset_index(drop=True)
        abstain_df.drop(columns=['gate_order']).to_csv(DIAGNOSIS_ABSTAIN_CSV, index=False)
        display(
            abstain_df[
                [
                    'profile_label', 'config_label', 'gate_label', 'confidence_threshold',
                    'coverage', 'abstain_rate', 'selective_top1_acc', 'selective_top2_acc', 'selective_macro_f1'
                ]
            ].round(4)
        )
    else:
        display(Markdown('Abstain-aware diagnosis table will appear after the hierarchical sweep CSV is generated.'))

    if notes:
        display(Markdown('**Diagnosis notes**\n' + '\n'.join(dict.fromkeys(notes))))

    if missing_profiles:
        display(Markdown(_safe_missing_profiles_markdown(missing_profiles)))


### Attribution comparison figures


In [ ]:
# Plain-language: This cell visualizes attribution patterns so readers can see how DICE explains different stressors.
display(Markdown('### Attribution comparison figures'))


tier_csv = COMPARISON_DIR / 'tier_attribution_profiles.csv'
mechanism_csv = COMPARISON_DIR / 'mechanism_attribution_profiles.csv'
attrib_png = COMPARISON_FIG / 'fig_attribution_profile_comparison.png'

if not tier_csv.exists() or not mechanism_csv.exists():
    display(Markdown('Run the attribution-summary cell first so the comparison CSVs exist.'))
else:
    tier_summary = pd.read_csv(tier_csv)
    mechanism_summary = pd.read_csv(mechanism_csv)

    final_tier = tier_summary[tier_summary['config'] == FINAL_CONFIG].copy()
    final_mech = mechanism_summary[mechanism_summary['config'] == FINAL_CONFIG].copy()

    profiles_present = set(pd.concat([
        final_tier['feature_profile'],
        final_mech['feature_profile'],
    ], ignore_index=True).dropna().astype(str))

    profiles = [p for p in PROFILE_ORDER if p in profiles_present]

    if not profiles:
        display(Markdown(f'No attribution rows were found for final config `{FINAL_CONFIG}`.'))
    else:
        tier_specs = [
            ('tier0_share', 'Tier-0'),
            ('tier1_alt_share', 'Tier-1'),
            ('tier2_share', 'Tier-2'),
        ]
        mech_specs = [
            ('memory_io_share', 'Memory / IO'),
            ('compute_share', 'Compute'),
            ('thermal_power_share', 'Thermal / Power'),
            ('scheduler_runtime_share', 'Runtime'),
            ('platform_pressure_share', 'Platform'),
        ]

        profile_colors = {
            p: PROFILE_COLOR.get(p, '#334155')
            for p in profiles
        }
        profile_labels = {
            p: PROFILE_LABEL.get(p, p.title())
            for p in profiles
        }

        def plot_butterfly(ax, df, specs, title):
            present_specs = [(col, label) for col, label in specs if col in df.columns]
            if not present_specs:
                ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes, fontsize=12)
                ax.set_axis_off()
                return

            labels = [label for _, label in present_specs]
            y = np.arange(len(labels), dtype=float)

            for yi in y:
                if int(yi) % 2 == 0:
                    ax.axhspan(yi - 0.44, yi + 0.44, color='#F8FAFC', zorder=0)

            profile_vals = {}
            for profile in profiles:
                row = df[df['feature_profile'] == profile]
                vals = []
                for col, _ in present_specs:
                    if row.empty or col not in row.columns or pd.isna(row.iloc[0][col]):
                        vals.append(np.nan)
                    else:
                        vals.append(float(row.iloc[0][col]))
                profile_vals[profile] = np.asarray(vals, dtype=float)

            finite_vals = []
            for vals in profile_vals.values():
                finite_vals.extend(np.abs(vals[np.isfinite(vals)]).tolist())
            max_val = max(finite_vals) if finite_vals else 0.05
            max_val = max(max_val, 0.05)
            pad = max(0.04, 0.22 * max_val)

            if len(profiles) >= 2:
                left_profile, right_profile = profiles[:2]
                left_vals = -np.nan_to_num(profile_vals[left_profile], nan=0.0)
                right_vals = np.nan_to_num(profile_vals[right_profile], nan=0.0)

                left_fill = left_vals
                right_fill = right_vals

                ax.barh(
                    y,
                    left_fill,
                    height=0.56,
                    color=profile_colors[left_profile],
                    alpha=0.88,
                    edgecolor='white',
                    linewidth=1.0,
                    zorder=3,
                )
                ax.barh(
                    y,
                    right_fill,
                    height=0.56,
                    color=profile_colors[right_profile],
                    alpha=0.88,
                    edgecolor='white',
                    linewidth=1.0,
                    zorder=3,
                )

                ax.axvline(0.0, color='#111827', linewidth=1.2, zorder=2)

                for yi, val in zip(y, left_vals):
                    if val != 0:
                        ax.text(
                            val - 0.01,
                            yi,
                            f'{abs(val):.0%}',
                            ha='right',
                            va='center',
                            fontsize=10.6,
                            fontweight='bold',
                            color=profile_colors[left_profile],
                        )

                for yi, val in zip(y, right_vals):
                    if val != 0:
                        ax.text(
                            val + 0.01,
                            yi,
                            f'{abs(val):.0%}',
                            ha='left',
                            va='center',
                            fontsize=10.6,
                            fontweight='bold',
                            color=profile_colors[right_profile],
                        )

                ax.set_xlim(-(max_val + pad), max_val + pad)
                ax.set_xlabel(
                    f'Evidence share ({profile_labels[left_profile]} left, {profile_labels[right_profile]} right)',
                    fontsize=12.4,
                )
            else:
                only_profile = profiles[0]
                vals = np.nan_to_num(profile_vals[only_profile], nan=0.0)

                ax.barh(
                    y,
                    vals,
                    height=0.56,
                    color=profile_colors[only_profile],
                    alpha=0.88,
                    edgecolor='white',
                    linewidth=1.0,
                    zorder=3,
                )

                for yi, val in zip(y, vals):
                    if val != 0:
                        ax.text(
                            val + 0.01,
                            yi,
                            f'{val:.0%}',
                            ha='left',
                            va='center',
                            fontsize=10.6,
                            fontweight='bold',
                            color=profile_colors[only_profile],
                        )

                ax.set_xlim(0.0, max_val + pad)
                ax.set_xlabel('Mean anomaly evidence share', fontsize=12.4)

            ax.set_yticks(y)
            ax.set_yticklabels(labels, fontsize=12.0)
            ax.invert_yaxis()
            ax.grid(axis='x', alpha=0.18)
            ax.tick_params(axis='x', labelsize=11.0)
            ax.tick_params(axis='y', length=0)
            ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{abs(x):.0%}'))
            ax.set_title(title, fontsize=14.0, fontweight='bold', pad=10)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_visible(False)

        fig, axes = plt.subplots(
            1,
            2,
            figsize=(15.0, 6.1),
            constrained_layout=False,
        )

        plot_butterfly(axes[0], final_tier, tier_specs, 'Final-head tier attribution')
        plot_butterfly(axes[1], final_mech, mech_specs, 'Final-head mechanism attribution')

        legend_handles = [
            Patch(facecolor=profile_colors[p], edgecolor='none', label=profile_labels[p])
            for p in profiles
        ]

        fig.subplots_adjust(left=0.12, right=0.98, top=0.84, bottom=0.24, wspace=0.34)

        fig.legend(
            handles=legend_handles,
            loc='lower center',
            bbox_to_anchor=(0.5, -0.01),
            ncol=max(1, len(legend_handles)),
            frameon=False,
            fontsize=11.2,
            columnspacing=1.8,
            handletextpad=0.6,
        )

        fig.suptitle('Mixed vs full: attribution comparison', fontsize=16.0, fontweight='bold', y=0.95)
        fig.savefig(attrib_png, dpi=PRO_FIG_DPI, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        display(Image(filename=str(attrib_png)))


## 10. Residual Evidence Concentration

This section shows how much anomaly evidence is captured by the top residual contributors.


In [ ]:
# Plain-language: This cell measures how concentrated the residual evidence is, which helps explain how focused or diffuse each anomaly looks.
display(Markdown('### Evidence concentration tables'))

evidence_csv = COMPARISON_DIR / 'residual_evidence_concentration_profiles.csv'
stressor_evidence_csv = COMPARISON_DIR / 'stressor_residual_evidence_concentration_profiles.csv'

profile_order = list(globals().get('PROFILE_ORDER', ['mixed', 'full']))


def resolve_evidence_profiles():
    ready = []
    missing = {}

    for profile in profile_order:
        profile_missing = []
        gdir = profile_global_dir(profile)

        if not (gdir / 'case_diagnosis_summary.csv').exists():
            profile_missing.append('global/case_diagnosis_summary.csv')

        if profile_missing:
            missing[profile] = profile_missing
        else:
            ready.append(profile)

    return ready, missing


def render_missing_profiles(missing):
    helper = globals().get('markdown_missing_profiles', None)
    if callable(helper):
        return helper(missing)

    lines = ['Missing bundles:']
    for profile, items in missing.items():
        joined = ', '.join(f'`{x}`' for x in items)
        lines.append(f'- `{profile}`: {joined}')
    return '\n'.join(lines)


ready_profiles, missing_profiles = resolve_evidence_profiles()

if not ready_profiles:
    display(Markdown('No diagnosis bundles are ready yet. Run **Run End-to-End** first.'))
    if missing_profiles:
        display(Markdown(render_missing_profiles(missing_profiles)))
else:
    evidence_rows = []
    stressor_rows = []

    mech_cols = [
        'compute_contrib',
        'memory_io_contrib',
        'thermal_power_contrib',
        'scheduler_runtime_contrib',
        'platform_pressure_contrib',
    ]

    for profile in ready_profiles:
        case_diag = read_profile_csv(profile, 'global', 'case_diagnosis_summary.csv')
        anom_diag = case_diag[case_diag['label'] == 1].copy()
        if anom_diag.empty:
            continue

        contrib_cols = [c for c in mech_cols if c in anom_diag.columns]
        if not contrib_cols:
            continue

        anom_diag['total_evidence'] = anom_diag[contrib_cols].sum(axis=1).replace(0.0, np.nan)

        for k in range(1, 6):
            cols = [c for c in [f'top_feature_score_{i}' for i in range(1, k + 1)] if c in anom_diag.columns]
            if cols:
                anom_diag[f'feature_top{k}_coverage'] = anom_diag[cols].sum(axis=1) / anom_diag['total_evidence']
            else:
                anom_diag[f'feature_top{k}_coverage'] = np.nan

        for k in range(1, 4):
            cols = [c for c in [f'top_mechanism_score_{i}' for i in range(1, k + 1)] if c in anom_diag.columns]
            if cols:
                anom_diag[f'mechanism_top{k}_coverage'] = anom_diag[cols].sum(axis=1) / anom_diag['total_evidence']
            else:
                anom_diag[f'mechanism_top{k}_coverage'] = np.nan

        coverage_cols = [
            'feature_top1_coverage', 'feature_top2_coverage', 'feature_top3_coverage',
            'feature_top4_coverage', 'feature_top5_coverage',
            'mechanism_top1_coverage', 'mechanism_top2_coverage', 'mechanism_top3_coverage',
        ]

        evidence_frontier = (
            anom_diag.groupby('config', sort=False)[coverage_cols]
            .mean()
            .reset_index()
        )
        evidence_frontier['feature_profile'] = profile
        evidence_rows.append(add_profile_and_config_labels(evidence_frontier))

        final_slice = anom_diag[anom_diag['config'] == FINAL_CONFIG].copy()
        if not final_slice.empty:
            stressor_cols = [
                'feature_top1_coverage', 'feature_top3_coverage', 'feature_top5_coverage',
                'mechanism_top1_coverage', 'mechanism_top2_coverage', 'mechanism_top3_coverage',
            ]
            stressor_evidence = (
                final_slice.groupby('stressor', sort=False)[stressor_cols]
                .mean()
                .reset_index()
            )
            stressor_evidence['feature_profile'] = profile
            stressor_rows.append(add_profile_and_config_labels(stressor_evidence))

    if evidence_rows:
        evidence_frontier = sort_profile_config(pd.concat(evidence_rows, ignore_index=True))
        evidence_frontier.to_csv(evidence_csv, index=False)
        display(Markdown('#### Mean evidence concentration by head'))
        display(evidence_frontier.round(4))
    else:
        evidence_frontier = pd.DataFrame()
        display(Markdown('#### Mean evidence concentration by head'))
        display(Markdown('No anomaly evidence-concentration rows were available.'))

    if stressor_rows:
        stressor_evidence = pd.concat(stressor_rows, ignore_index=True)
        stressor_evidence.to_csv(stressor_evidence_csv, index=False)
        display(Markdown('#### Final-head stressor evidence concentration'))
        display(stressor_evidence.round(4))
    else:
        stressor_evidence = pd.DataFrame()
        display(Markdown('#### Final-head stressor evidence concentration'))
        display(Markdown('No final-head stressor evidence rows were available.'))

    if missing_profiles:
        display(Markdown(render_missing_profiles(missing_profiles)))

### Evidence concentration comparison figure


In [ ]:
# Plain-language: This cell plots the evidence concentration comparison so the mixed and full profiles can be contrasted visually.

display(Markdown('### Evidence concentration comparison figure'))

evidence_csv = COMPARISON_DIR / 'residual_evidence_concentration_profiles.csv'
evidence_png = COMPARISON_FIG / 'fig_residual_evidence_concentration_profiles.png'

if not evidence_csv.exists():
    display(Markdown('Run the evidence-concentration cell first so the comparison CSV exists.'))
else:
    evidence = pd.read_csv(evidence_csv)
    final_head = evidence[evidence['config'] == FINAL_CONFIG].copy()

    profile_order = list(globals().get('PROFILE_ORDER', ['mixed', 'full']))
    profile_label_map = globals().get('PROFILE_LABEL', {'mixed': 'Mixed', 'full': 'Full'})
    profile_color_map = globals().get('PROFILE_COLOR', {'mixed': '#355C7D', 'full': '#C44E52'})
    profile_marker_map = globals().get('PROFILE_MARKER', {'mixed': 'o', 'full': 's'})
    stressor_order = ['ATOMIC', 'BRANCH', 'CACHE', 'MEMBW', 'TLB']

    profiles = [p for p in profile_order if p in set(final_head['feature_profile'])]

    if not profiles:
        display(Markdown(f'No evidence-concentration rows were found for final config `{FINAL_CONFIG}`.'))
    else:
        stressor_rows = []
        mech_cols = [
            'compute_contrib',
            'memory_io_contrib',
            'thermal_power_contrib',
            'scheduler_runtime_contrib',
            'platform_pressure_contrib',
        ]

        for profile in profiles:
            try:
                case_diag = read_profile_csv(profile, 'global', 'case_diagnosis_summary.csv')
            except Exception:
                continue

            anom_diag = case_diag[
                (case_diag['label'] == 1) &
                (case_diag['config'] == FINAL_CONFIG)
            ].copy()
            if anom_diag.empty:
                continue

            contrib_cols = [c for c in mech_cols if c in anom_diag.columns]
            if not contrib_cols:
                continue

            anom_diag['total_evidence'] = anom_diag[contrib_cols].sum(axis=1).replace(0.0, np.nan)

            for k in range(1, 6):
                cols = [c for c in [f'top_feature_score_{i}' for i in range(1, k + 1)] if c in anom_diag.columns]
                anom_diag[f'feature_top{k}_coverage'] = (
                    anom_diag[cols].sum(axis=1) / anom_diag['total_evidence']
                    if cols else np.nan
                )

            for k in range(1, 4):
                cols = [c for c in [f'top_mechanism_score_{i}' for i in range(1, k + 1)] if c in anom_diag.columns]
                anom_diag[f'mechanism_top{k}_coverage'] = (
                    anom_diag[cols].sum(axis=1) / anom_diag['total_evidence']
                    if cols else np.nan
                )

            stressor_summary = (
                anom_diag.groupby('stressor', sort=False)[[
                    'feature_top1_coverage',
                    'feature_top3_coverage',
                    'feature_top5_coverage',
                    'mechanism_top1_coverage',
                    'mechanism_top2_coverage',
                    'mechanism_top3_coverage',
                ]]
                .mean()
                .reset_index()
            )
            stressor_summary['feature_profile'] = profile
            stressor_rows.append(stressor_summary)

        stressor_evidence = (
            pd.concat(stressor_rows, ignore_index=True)
            if stressor_rows else pd.DataFrame()
        )

        TEXT_DARK = '#0F172A'
        TEXT_MID = '#475569'
        GRID_SOFT = '#D9E2EC'
        SPINE_SOFT = '#CBD5E1'
        PANEL_BG = '#FBFCFE'

        TITLE_FS = 15.8
        AXIS_FS = 13.8
        TICK_FS = 12.2
        LEGEND_FS = 12.2
        VALUE_FS = 11.4
        SUPTITLE_FS = 18.2

        def style_panel(ax):
            ax.set_facecolor(PANEL_BG)
            ax.grid(alpha=0.24, color=GRID_SOFT, linewidth=1.0)
            ax.tick_params(labelsize=TICK_FS, colors=TEXT_DARK)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_color(SPINE_SOFT)
            ax.spines['bottom'].set_color(SPINE_SOFT)
            ax.spines['left'].set_linewidth(1.2)
            ax.spines['bottom'].set_linewidth(1.2)

        fig, axes = plt.subplots(
            2,
            2,
            figsize=(16.6, 9.9),
            constrained_layout=False,
        )

        ax_feat = axes[0, 0]
        ax_mech = axes[0, 1]
        ax_stress_feat = axes[1, 0]
        ax_stress_mech = axes[1, 1]

        fig.subplots_adjust(left=0.08, right=0.98, top=0.89, bottom=0.20, hspace=0.42, wspace=0.28)

        # Top-left: feature concentration frontier
        for profile in profiles:
            row = final_head[final_head['feature_profile'] == profile]
            if row.empty:
                continue
            row = row.iloc[0]
            vals = np.array([float(row.get(f'feature_top{k}_coverage', np.nan)) for k in range(1, 6)], dtype=float)
            x = np.arange(1, 6, dtype=float)

            ax_feat.plot(
                x,
                vals,
                marker=profile_marker_map.get(profile, 'o'),
                markersize=10.5,
                linewidth=3.4,
                color=profile_color_map.get(profile, '#334155'),
                markeredgecolor='white',
                markeredgewidth=1.2,
                solid_capstyle='round',
                label=profile_label_map.get(profile, profile),
                zorder=3,
            )
            ax_feat.scatter(
                x,
                vals,
                s=135,
                color=profile_color_map.get(profile, '#334155'),
                edgecolor='white',
                linewidth=1.1,
                marker=profile_marker_map.get(profile, 'o'),
                zorder=4,
            )

        style_panel(ax_feat)
        ax_feat.set_xlabel('Top-k residual features', fontsize=AXIS_FS, fontweight='bold', color=TEXT_DARK, labelpad=10)
        ax_feat.set_ylabel('Mean anomaly evidence coverage', fontsize=AXIS_FS, fontweight='bold', color=TEXT_DARK, labelpad=10)
        ax_feat.set_xticks(range(1, 6))
        ax_feat.set_ylim(0.0, 1.02)
        ax_feat.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
        ax_feat.set_title('Overall feature concentration frontier', fontsize=TITLE_FS, fontweight='bold', color=TEXT_DARK, pad=12)

        # Top-right: mechanism concentration frontier
        for profile in profiles:
            row = final_head[final_head['feature_profile'] == profile]
            if row.empty:
                continue
            row = row.iloc[0]
            vals = np.array([float(row.get(f'mechanism_top{k}_coverage', np.nan)) for k in range(1, 4)], dtype=float)
            x = np.arange(1, 4, dtype=float)

            ax_mech.plot(
                x,
                vals,
                marker=profile_marker_map.get(profile, 'o'),
                markersize=10.5,
                linewidth=3.4,
                color=profile_color_map.get(profile, '#334155'),
                markeredgecolor='white',
                markeredgewidth=1.2,
                solid_capstyle='round',
                label=profile_label_map.get(profile, profile),
                zorder=3,
            )
            ax_mech.scatter(
                x,
                vals,
                s=135,
                color=profile_color_map.get(profile, '#334155'),
                edgecolor='white',
                linewidth=1.1,
                marker=profile_marker_map.get(profile, 'o'),
                zorder=4,
            )

        style_panel(ax_mech)
        ax_mech.set_xlabel('Top-k mechanism groups', fontsize=AXIS_FS, fontweight='bold', color=TEXT_DARK, labelpad=10)
        ax_mech.set_ylabel('Mean anomaly evidence coverage', fontsize=AXIS_FS, fontweight='bold', color=TEXT_DARK, labelpad=10)
        ax_mech.set_xticks(range(1, 4))
        ax_mech.set_ylim(0.0, 1.02)
        ax_mech.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
        ax_mech.set_title('Overall mechanism concentration frontier', fontsize=TITLE_FS, fontweight='bold', color=TEXT_DARK, pad=12)

        def plot_gap_lollipops(ax, df, metric_col, title):
            style_panel(ax)

            if df.empty or metric_col not in df.columns or len(profiles) < 2:
                ax.text(
                    0.5,
                    0.5,
                    'No stressor gap breakdown available',
                    ha='center',
                    va='center',
                    transform=ax.transAxes,
                    fontsize=13.0,
                    color=TEXT_MID,
                    fontweight='bold',
                )
                ax.set_axis_off()
                return

            left_profile, right_profile = profiles[:2]

            rows = []
            for stressor in stressor_order:
                left_hit = df[(df['feature_profile'] == left_profile) & (df['stressor'] == stressor)]
                right_hit = df[(df['feature_profile'] == right_profile) & (df['stressor'] == stressor)]
                if left_hit.empty or right_hit.empty:
                    continue

                left_val = float(left_hit.iloc[0][metric_col])
                right_val = float(right_hit.iloc[0][metric_col])

                rows.append(
                    {
                        'stressor': stressor,
                        'left_val': left_val,
                        'right_val': right_val,
                        'gap': right_val - left_val,
                        'gap_abs': abs(right_val - left_val),
                    }
                )

            plot_df = pd.DataFrame(rows)
            if plot_df.empty:
                ax.text(
                    0.5,
                    0.5,
                    'No stressor gap breakdown available',
                    ha='center',
                    va='center',
                    transform=ax.transAxes,
                    fontsize=13.0,
                    color=TEXT_MID,
                    fontweight='bold',
                )
                ax.set_axis_off()
                return

            plot_df = plot_df.sort_values(['gap_abs', 'stressor'], ascending=[False, True]).reset_index(drop=True)
            y = np.arange(len(plot_df), dtype=float)

            for yi in y:
                if int(yi) % 2 == 0:
                    ax.axhspan(yi - 0.42, yi + 0.42, color='#F8FAFC', zorder=0)

            for yi, row in zip(y, plot_df.itertuples(index=False)):
                lo = min(row.left_val, row.right_val)
                hi = max(row.left_val, row.right_val)

                ax.plot(
                    [lo, hi],
                    [yi, yi],
                    color='#CBD5E1',
                    linewidth=6.0,
                    solid_capstyle='round',
                    zorder=1,
                )
                ax.plot(
                    [lo, hi],
                    [yi, yi],
                    color='#94A3B8',
                    linewidth=1.15,
                    solid_capstyle='round',
                    zorder=1.1,
                )

                ax.scatter(
                    [row.left_val],
                    [yi],
                    s=190,
                    color=profile_color_map.get(left_profile, '#334155'),
                    marker=profile_marker_map.get(left_profile, 'o'),
                    edgecolor='white',
                    linewidth=1.2,
                    zorder=3,
                )
                ax.scatter(
                    [row.right_val],
                    [yi],
                    s=190,
                    color=profile_color_map.get(right_profile, '#334155'),
                    marker=profile_marker_map.get(right_profile, 'o'),
                    edgecolor='white',
                    linewidth=1.2,
                    zorder=3,
                )

                x_text = min(max(row.left_val, row.right_val) + 0.03, 0.94)
                ax.text(
                    x_text,
                    yi,
                    f'Δ {row.gap:+.0%}',
                    va='center',
                    ha='left',
                    fontsize=VALUE_FS,
                    fontweight='bold',
                    color=TEXT_MID,
                    bbox=dict(
                        boxstyle='round,pad=0.18',
                        fc='white',
                        ec='none',
                        alpha=0.92,
                    ),
                )

            ax.set_yticks(y)
            ax.set_yticklabels(plot_df['stressor'], fontsize=TICK_FS, fontweight='bold', color=TEXT_DARK)
            ax.invert_yaxis()
            ax.set_xlim(0.0, 1.0)
            ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
            ax.set_xlabel('Mean anomaly evidence coverage', fontsize=AXIS_FS, fontweight='bold', color=TEXT_DARK, labelpad=10)
            ax.set_title(title, fontsize=TITLE_FS, fontweight='bold', color=TEXT_DARK, pad=12)
            ax.tick_params(axis='x', labelsize=TICK_FS)
            ax.tick_params(axis='y', length=0)
            ax.spines['left'].set_visible(False)

        plot_gap_lollipops(
            ax_stress_feat,
            stressor_evidence,
            'feature_top3_coverage',
            'Largest stressor gaps in feature top-3 coverage',
        )
        plot_gap_lollipops(
            ax_stress_mech,
            stressor_evidence,
            'mechanism_top2_coverage',
            'Largest stressor gaps in mechanism top-2 coverage',
        )

        legend_handles = [
            Line2D(
                [0], [0],
                color=profile_color_map.get(p, '#334155'),
                marker=profile_marker_map.get(p, 'o'),
                linewidth=3.2,
                markersize=10.0,
                markeredgecolor='white',
                markeredgewidth=1.0,
                label=profile_label_map.get(p, p),
            )
            for p in profiles
        ]

        if len(profiles) >= 2:
            legend_handles.append(
                Line2D(
                    [0], [0],
                    color='#94A3B8',
                    linewidth=2.6,
                    label='Profile gap',
                )
            )

        fig.legend(
            handles=legend_handles,
            loc='lower center',
            bbox_to_anchor=(0.5, 0.03),
            ncol=max(2, len(legend_handles)),
            frameon=False,
            fontsize=LEGEND_FS,
            columnspacing=1.8,
            handlelength=2.8,
        )

        fig.suptitle(
            'Mixed vs full: residual evidence concentration',
            fontsize=SUPTITLE_FS,
            fontweight='bold',
            color=TEXT_DARK,
            y=0.965,
        )
        fig.savefig(evidence_png, dpi=PRO_FIG_DPI, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        display(Image(filename=str(evidence_png)))



## 11. Uncertainty and Confidence Intervals

This section now combines the **uncertainty-aware digital-twin experiment** with the bootstrap confidence intervals used for the reported paper metrics.


In [ ]:
# Plain-language: This cell summarizes the uncertainty-aware digital-twin experiment first, then adds bootstrap confidence intervals for the paper metrics.
display(Markdown('### Uncertainty-aware digital-twin summary'))

UNCERTAINTY_COMPARISON_CSV = COMPARISON_DIR / 'uncertainty_summary_profiles.csv'
UNCERTAINTY_HOLDOUT_COMPARISON_CSV = COMPARISON_DIR / 'uncertainty_holdout_profiles.csv'
BOOTSTRAP_SAMPLES = 1000
bootstrap_csv = COMPARISON_DIR / 'bootstrap_confidence_profiles.csv'

profile_order = list(globals().get('PROFILE_ORDER', ['mixed', 'full']))
profile_label_map = globals().get('PROFILE_LABEL', {'mixed': 'Mixed', 'full': 'Full'})
cfg_label_map = globals().get('CFG_LABEL', {'tier0': 'Tier-0', 'tier0_tier1': 'Tier-0/1', 'tier0_tier1_tier2': 'Tier-0/1/2'})

uncertainty_rows = []
uncertainty_holdout_rows = []
for profile in profile_order:
    summary_path = profile_paper_dir(profile) / 'uncertainty_summary.csv'
    holdout_path = profile_paper_dir(profile) / 'uncertainty_holdout_summary.csv'
    if summary_path.exists():
        df = pd.read_csv(summary_path)
        if not df.empty:
            df['feature_profile'] = profile
            df['profile_label'] = profile_label_map.get(profile, profile)
            df['config_label'] = df['config'].map(cfg_label_map).fillna(df['config'])
            uncertainty_rows.append(df)
    if holdout_path.exists():
        df = pd.read_csv(holdout_path)
        if not df.empty:
            df['feature_profile'] = profile
            df['profile_label'] = profile_label_map.get(profile, profile)
            df['config_label'] = df['config'].map(cfg_label_map).fillna(df['config'])
            uncertainty_holdout_rows.append(df)

if uncertainty_rows:
    uncertainty_profiles = pd.concat(uncertainty_rows, ignore_index=True)
    uncertainty_profiles.to_csv(UNCERTAINTY_COMPARISON_CSV, index=False)
    final_uncertainty = uncertainty_profiles[uncertainty_profiles['config'] == FINAL_CONFIG].copy()
    final_uncertainty = final_uncertainty.sort_values(['protocol', 'feature_profile']).reset_index(drop=True)
    display(final_uncertainty[[
        'profile_label', 'protocol', 'config_label', 'n_features',
        'roc_auc_mean_score', 'pr_auc_mean_score', 'uncertainty_log_auc',
        'benign_mean_log_score_std', 'anomaly_mean_log_score_std',
        'mean_alert_probability_benign', 'mean_alert_probability_anomaly'
    ]].rename(columns={
        'profile_label': 'Profile',
        'protocol': 'Protocol',
        'config_label': 'Head',
        'n_features': 'Features',
        'roc_auc_mean_score': 'ROC-AUC (ensemble mean)',
        'pr_auc_mean_score': 'AUC-PR (ensemble mean)',
        'uncertainty_log_auc': 'Uncertainty ROC-AUC (log spread)',
        'benign_mean_log_score_std': 'Benign log spread',
        'anomaly_mean_log_score_std': 'Anomaly log spread',
        'mean_alert_probability_benign': 'Mean alert prob. benign',
        'mean_alert_probability_anomaly': 'Mean alert prob. anomaly',
    }).round(4))
else:
    pd.DataFrame().to_csv(UNCERTAINTY_COMPARISON_CSV, index=False)
    display(Markdown('No uncertainty-summary CSVs were found yet. Run `tools/evaluate_dice_uncertainty.py` for the desired profiles.'))

if uncertainty_holdout_rows:
    uncertainty_holdout_profiles = pd.concat(uncertainty_holdout_rows, ignore_index=True)
    uncertainty_holdout_profiles.to_csv(UNCERTAINTY_HOLDOUT_COMPARISON_CSV, index=False)
    final_holdout_uncertainty = uncertainty_holdout_profiles[uncertainty_holdout_profiles['config'] == FINAL_CONFIG].copy()
    final_holdout_uncertainty = final_holdout_uncertainty.sort_values(['feature_profile', 'holdout_workload']).reset_index(drop=True)
    display(Markdown('#### Holdout uncertainty by workload'))
    display(final_holdout_uncertainty[[
        'profile_label', 'holdout_workload', 'config_label', 'mean_log_run_score_std',
        'benign_mean_log_score_std', 'anomaly_mean_log_score_std', 'uncertainty_log_auc'
    ]].rename(columns={
        'profile_label': 'Profile',
        'holdout_workload': 'Held-out workload',
        'config_label': 'Head',
        'mean_log_run_score_std': 'Mean log spread',
        'benign_mean_log_score_std': 'Benign log spread',
        'anomaly_mean_log_score_std': 'Anomaly log spread',
        'uncertainty_log_auc': 'Uncertainty ROC-AUC (log spread)',
    }).round(4))
else:
    pd.DataFrame().to_csv(UNCERTAINTY_HOLDOUT_COMPARISON_CSV, index=False)

display(Markdown('### Bootstrap confidence tables'))

def resolve_bootstrap_profiles():
    ready = []
    missing = {}

    for profile in profile_order:
        profile_missing = []
        gdir = profile_global_dir(profile)

        if not (gdir / 'case_predictions.csv').exists():
            profile_missing.append('global/case_predictions.csv')

        if profile_missing:
            missing[profile] = profile_missing
        else:
            ready.append(profile)

    return ready, missing


def render_missing_profiles(missing):
    helper = globals().get('markdown_missing_profiles', None)
    if callable(helper):
        return helper(missing)

    lines = ['Missing bundles:']
    for profile, items in missing.items():
        joined = ', '.join(f'`{x}`' for x in items)
        lines.append(f'- `{profile}`: {joined}')
    return '\n'.join(lines)


ready_profiles, missing_profiles = resolve_bootstrap_profiles()

if not ready_profiles:
    display(Markdown('No case-level prediction bundles are ready yet. Run **Run End-to-End** first.'))
    if missing_profiles:
        display(Markdown(render_missing_profiles(missing_profiles)))
else:
    rng = np.random.default_rng(0)
    rows = []

    for profile in ready_profiles:
        case_pred = read_profile_csv(profile, 'global', 'case_predictions.csv')
        final_df = case_pred[case_pred['config'] == FINAL_CONFIG].copy().reset_index(drop=True)
        if final_df.empty or 'label' not in final_df.columns:
            continue

        y = final_df['label'].to_numpy(dtype=int)

        metric_defs = [
            ('ROC-AUC', 'run_score', roc_auc_score),
            ('AUC-PR', 'run_score', average_precision_score),
            ('ROC-AUC (WC)', 'run_score_wc', roc_auc_score),
            ('AUC-PR (WC)', 'run_score_wc', average_precision_score),
        ]

        for metric_label, score_col, metric_fn in metric_defs:
            if score_col not in final_df.columns:
                continue

            score = final_df[score_col].to_numpy(dtype=float)
            valid = np.isfinite(score)
            y_valid = y[valid]
            score_valid = score[valid]

            if len(score_valid) == 0:
                continue

            samples = []
            for _ in range(BOOTSTRAP_SAMPLES):
                idx = rng.integers(0, len(score_valid), len(score_valid))
                y_boot = y_valid[idx]
                if len(np.unique(y_boot)) < 2:
                    continue
                samples.append(float(metric_fn(y_boot, score_valid[idx])))

            if not samples:
                continue

            samples_arr = np.asarray(samples, dtype=float)
            rows.append({
                'feature_profile': profile,
                'profile_label': profile_label_map.get(profile, profile),
                'metric': metric_label,
                'mean': float(samples_arr.mean()),
                'p05': float(np.quantile(samples_arr, 0.05)),
                'p50': float(np.quantile(samples_arr, 0.50)),
                'p95': float(np.quantile(samples_arr, 0.95)),
                'n_boot_valid': int(len(samples_arr)),
            })

    if not rows:
        display(Markdown('Bootstrap confidence intervals could not be computed yet.'))
    else:
        bootstrap_ci = pd.DataFrame(rows)
        bootstrap_ci.to_csv(bootstrap_csv, index=False)
        display(bootstrap_ci.round(4))

    if missing_profiles:
        display(Markdown(render_missing_profiles(missing_profiles)))


### Bootstrap comparison figure


In [ ]:
# Plain-language: This cell visualizes the bootstrap comparisons so readers can see how stable the main metrics are.
display(Markdown('### Bootstrap comparison figure'))


bootstrap_csv = COMPARISON_DIR / 'bootstrap_confidence_profiles.csv'
bootstrap_png = COMPARISON_FIG / 'fig_bootstrap_confidence_profiles.png'

if not bootstrap_csv.exists():
    display(Markdown('Run the bootstrap-summary cell first so the comparison CSV exists.'))
else:
    bootstrap_ci = pd.read_csv(bootstrap_csv)
    metric_order = ['ROC-AUC', 'AUC-PR', 'ROC-AUC (WC)', 'AUC-PR (WC)']
    bootstrap_ci['metric'] = pd.Categorical(bootstrap_ci['metric'], categories=metric_order, ordered=True)
    bootstrap_ci = bootstrap_ci.sort_values(['metric', 'feature_profile']).reset_index(drop=True)

    profiles = [p for p in PROFILE_ORDER if p in set(bootstrap_ci['feature_profile'])]

    if not profiles:
        display(Markdown('No bootstrap confidence rows were available.'))
    else:
        def lighten_color(color, amount):
            rgb = np.array(to_rgb(color), dtype=float)
            white = np.ones(3, dtype=float)
            return tuple(rgb * (1.0 - amount) + white * amount)

        profile_label_map = globals().get('PROFILE_LABEL', {'mixed': 'Mixed', 'full': 'Full'})
        profile_color_map = globals().get('PROFILE_COLOR', {'mixed': '#355C7D', 'full': '#C44E52'})
        profile_marker_map = globals().get('PROFILE_MARKER', {'mixed': 'o', 'full': 's'})

        metric_accent = {
            'ROC-AUC': '#1D4ED8',
            'AUC-PR': '#0F766E',
            'ROC-AUC (WC)': '#B45309',
            'AUC-PR (WC)': '#7C3AED',
        }

        fig, axes = plt.subplots(2, 2, figsize=(13.2, 7.8), constrained_layout=False)
        axes = axes.ravel()
        fig.subplots_adjust(left=0.10, right=0.96, top=0.87, bottom=0.22, wspace=0.24, hspace=0.32)

        for ax, metric in zip(axes, metric_order):
            d = bootstrap_ci[bootstrap_ci['metric'] == metric].copy()
            accent = metric_accent.get(metric, '#334155')

            ax.set_facecolor(lighten_color(accent, 0.95))
            for spine in ax.spines.values():
                spine.set_visible(False)

            # Background gauge bands
            ax.axvspan(0.00, 0.70, color='#E2E8F0', alpha=0.28, zorder=0)
            ax.axvspan(0.70, 0.85, color='#CBD5E1', alpha=0.28, zorder=0)
            ax.axvspan(0.85, 1.00, color='#94A3B8', alpha=0.18, zorder=0)

            y_positions = np.arange(len(profiles), dtype=float)[::-1]
            ax.set_ylim(-0.55, len(profiles) - 0.30)
            ax.set_xlim(0.0, 1.02)
            ax.set_yticks(y_positions)
            ax.set_yticklabels([profile_label_map.get(p, p) for p in profiles], fontsize=11.8, fontweight='bold')
            ax.set_xticks([0.0, 0.5, 0.75, 0.9, 1.0])
            ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
            ax.tick_params(axis='x', labelsize=10.8)
            ax.tick_params(axis='y', length=0)
            ax.grid(axis='x', alpha=0.16)

            for yi in y_positions:
                ax.plot([0.0, 1.0], [yi, yi], color='white', linewidth=16, solid_capstyle='round', alpha=0.92, zorder=1)
                ax.plot([0.0, 1.0], [yi, yi], color='#CBD5E1', linewidth=6, solid_capstyle='round', alpha=0.95, zorder=1.1)

            for yi, profile in zip(y_positions, profiles):
                row = d[d['feature_profile'] == profile]
                if row.empty:
                    continue
                row = row.iloc[0]

                color = profile_color_map.get(profile, '#334155')
                marker = profile_marker_map.get(profile, 'o')
                p05 = float(row['p05'])
                p50 = float(row['p50'])
                p95 = float(row['p95'])
                mean = float(row['mean'])

                ax.plot(
                    [p05, p95],
                    [yi, yi],
                    color=lighten_color(color, 0.40),
                    linewidth=13,
                    solid_capstyle='round',
                    zorder=2,
                )
                ax.plot(
                    [p05, p95],
                    [yi, yi],
                    color=color,
                    linewidth=5.0,
                    solid_capstyle='round',
                    zorder=3,
                )

                ax.vlines(
                    mean,
                    yi - 0.22,
                    yi + 0.22,
                    colors=color,
                    linewidth=2.3,
                    zorder=4,
                )

                ax.scatter(
                    [p50],
                    [yi],
                    s=150,
                    color=color,
                    marker=marker,
                    edgecolor='white',
                    linewidth=1.2,
                    zorder=5,
                )

                ax.text(
                    1.02,
                    yi,
                    f'{p50:.3f}',
                    ha='left',
                    va='center',
                    fontsize=10.4,
                    color='#334155',
                    transform=ax.get_yaxis_transform(),
                    clip_on=False,
                )

            ax.set_title(metric, fontsize=13.8, fontweight='bold', color=accent, pad=10)
            if metric in ['ROC-AUC (WC)', 'AUC-PR (WC)']:
                ax.set_xlabel('Metric value', fontsize=11.8)

        legend_handles = [
            Line2D([0], [0], color='#64748B', linewidth=6, label='90% bootstrap interval'),
            Line2D([0], [0], color='#111827', marker='o', linestyle='None', markersize=8, label='Median (p50)'),
            Line2D([0], [0], color='#111827', marker='|', linestyle='None', markersize=14, markeredgewidth=2.0, label='Mean'),
        ]

        fig.legend(
            handles=legend_handles,
            loc='lower center',
            bbox_to_anchor=(0.5, -0.01),
            ncol=3,
            frameon=False,
            fontsize=11.2,
            columnspacing=1.8,
            handlelength=2.4,
        )

        fig.suptitle('Mixed vs full: bootstrap confidence gauges', fontsize=16.2, fontweight='bold', y=0.95)
        fig.savefig(bootstrap_png, dpi=PRO_FIG_DPI, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        display(Image(filename=str(bootstrap_png)))


## 12. Paper Bundle and Appendix Exports

This section gathers the paper-ready artifacts, appendix exports, grounded LLM bundles, and reproducibility metadata after the main analysis is complete.


In [ ]:
# Plain-language: This cell gathers the main paper-ready tables and figures into one bundle for easy drafting.
display(Markdown('### Paper-ready bundle summary'))

bundle_csv = COMPARISON_DIR / 'paper_ready_bundle_summary.csv'
rows = []
for profile in PROFILE_ORDER:
    appendix_dir = profile_appendix_dir(profile)
    rows.append({
        'feature_profile': profile,
        'profile_label': PROFILE_LABEL[profile],
        'global_dir_exists': int(profile_global_dir(profile).exists()),
        'holdout_dir_exists': int(profile_holdout_dir(profile).exists()),
        'paper_dir_exists': int(profile_paper_dir(profile).exists()),
        'overall_metrics_exists': int((profile_global_dir(profile) / 'overall_metrics.csv').exists()),
        'holdout_summary_exists': int((profile_holdout_dir(profile) / 'holdout_robustness_summary.csv').exists()),
        'two_stage_exists': int((profile_paper_dir(profile) / 'two_stage_dice_summary.csv').exists()),
        'feature_budget_exists': int((profile_paper_dir(profile) / 'dse_feature_budget_summary.csv').exists()),
        'reliability_exists': int((profile_paper_dir(profile) / 'conformal_reliability_summary.csv').exists()),
        'uncertainty_exists': int((profile_paper_dir(profile) / 'uncertainty_summary.csv').exists()),
        'llm_asset_exists': int((appendix_dir / 'llm_case_cards.csv').exists()),
        'llm_scored_exists': int(any(appendix_dir.glob('llm_grounding_summary_*.csv'))),
    })
summary_df = pd.DataFrame(rows)
summary_df.to_csv(bundle_csv, index=False)
display(summary_df)


## 12A. Grounded LLM Triage Results


In [ ]:
# Plain-language: This cell surfaces the saved grounded LLM triage results in a reader-friendly way.

display(Markdown('### Grounded LLM triage results'))

LLM_GROUNDING_SUMMARY_CSV = COMPARISON_DIR / 'llm_grounding_summary_profiles.csv'
LLM_GROUNDING_ASSET_CSV = COMPARISON_DIR / 'llm_grounded_triage_assets_profiles.csv'
LLM_RUNTIME_SUMMARY_CSV = COMPARISON_DIR / 'llm_runtime_profiles.csv'
LLM_GROUNDING_NOTE_MD = COMPARISON_DIR / 'LLM_GROUNDED_TRIAGE_NOTE.md'


def _pct(v, decimals=0):
    if pd.isna(v):
        return '—'
    return f'{100.0 * float(v):.{decimals}f}%'


def _short_model(model_id):
    if pd.isna(model_id):
        return ''
    return str(model_id).split('/')[-1]


def _clean_preview(text, max_chars=320):
    if pd.isna(text):
        return ''
    text = ' '.join(str(text).replace('\r', '\n').split())
    return text if len(text) <= max_chars else text[: max_chars - 1] + '…'


def _ensure_cols(df, cols, default=np.nan):
    out = df.copy()
    for col in cols:
        if col not in out.columns:
            out[col] = default
    return out


assets_df = pd.read_csv(LLM_GROUNDING_ASSET_CSV) if LLM_GROUNDING_ASSET_CSV.exists() else pd.DataFrame()
summary_df = pd.read_csv(LLM_GROUNDING_SUMMARY_CSV) if LLM_GROUNDING_SUMMARY_CSV.exists() else pd.DataFrame()
runtime_df = pd.read_csv(LLM_RUNTIME_SUMMARY_CSV) if LLM_RUNTIME_SUMMARY_CSV.exists() else pd.DataFrame()

if assets_df.empty and summary_df.empty:
    display(Markdown('No grounded LLM triage outputs are visible yet. Run the 12A export/scoring cell first.'))
else:
    if LLM_GROUNDING_NOTE_MD.exists():
        note_text = LLM_GROUNDING_NOTE_MD.read_text().strip()
        note_text = note_text.replace('Grounded LLM triage note', '#### Saved scored run note')
        display(Markdown(note_text))

    if not summary_df.empty:
        scorecard = _ensure_cols(
            summary_df,
            [
                'feature_profile', 'profile_label', 'model_id', 'prompt_type', 'n_outputs',
                'grounded_core_rate', 'dominant_tier_rate', 'dominant_mechanism_rate',
                'top_feature_1_rate', 'top_mechanism_1_rate',
                'mean_cue_coverage', 'hallucination_rate'
            ],
        )

        if not runtime_df.empty:
            runtime_view = _ensure_cols(
                runtime_df,
                ['feature_profile', 'model_id', 'backend', 'seed', 'max_tokens', 'elapsed_s'],
            )

            if runtime_view['model_id'].notna().any() and scorecard['model_id'].notna().any():
                runtime_view = runtime_view[
                    ['feature_profile', 'model_id', 'backend', 'seed', 'max_tokens', 'elapsed_s']
                ].drop_duplicates(subset=['feature_profile', 'model_id'])
                scorecard = scorecard.merge(
                    runtime_view,
                    on=['feature_profile', 'model_id'],
                    how='left',
                )
            else:
                runtime_view = runtime_view[
                    ['feature_profile', 'backend', 'seed', 'max_tokens', 'elapsed_s']
                ].drop_duplicates(subset=['feature_profile'])
                scorecard = scorecard.merge(
                    runtime_view,
                    on=['feature_profile'],
                    how='left',
                )

        scorecard = _ensure_cols(scorecard, ['backend', 'seed', 'max_tokens', 'elapsed_s'], default=np.nan)

        scorecard_view = pd.DataFrame({
            'Profile': scorecard['profile_label'].fillna(scorecard['feature_profile']),
            'Model': scorecard['model_id'].apply(_short_model),
            'Prompt': scorecard['prompt_type'].fillna('—'),
            'Outputs': pd.to_numeric(scorecard['n_outputs'], errors='coerce'),
            'Backend': scorecard['backend'].fillna('—'),
            'Grounded core': scorecard['grounded_core_rate'].apply(lambda v: _pct(v, 0)),
            'Dominant tier': scorecard['dominant_tier_rate'].apply(lambda v: _pct(v, 0)),
            'Dominant mechanism': scorecard['dominant_mechanism_rate'].apply(lambda v: _pct(v, 0)),
            'Top mechanism': scorecard['top_mechanism_1_rate'].apply(lambda v: _pct(v, 0)),
            'Top feature': scorecard['top_feature_1_rate'].apply(lambda v: _pct(v, 0)),
            'Cue coverage': scorecard['mean_cue_coverage'].apply(lambda v: _pct(v, 0)),
            'Hallucination': scorecard['hallucination_rate'].apply(lambda v: _pct(v, 0)),
            'Elapsed (s)': pd.to_numeric(scorecard['elapsed_s'], errors='coerce').round(2),
        })

        display(Markdown('#### Scorecard'))
        display(scorecard_view.reset_index(drop=True))

    if not assets_df.empty:
        assets_df = _ensure_cols(
            assets_df,
            [
                'profile_label', 'n_case_cards', 'n_model_profiles', 'n_prompt_rows',
                'prompt_types', 'llm_output_csvs_found', 'runtime_mlx_lm', 'appendix_dir'
            ],
        )
        display(Markdown('#### Export bundle'))
        display(
            pd.DataFrame({
                'Profile': assets_df['profile_label'],
                'Case cards': pd.to_numeric(assets_df['n_case_cards'], errors='coerce'),
                'Prompt templates': pd.to_numeric(assets_df['n_model_profiles'], errors='coerce'),
                'Prompt rows': pd.to_numeric(assets_df['n_prompt_rows'], errors='coerce'),
                'Prompt types': assets_df['prompt_types'].fillna(''),
                'Scored output CSVs': pd.to_numeric(assets_df['llm_output_csvs_found'], errors='coerce'),
                'MLX available': pd.to_numeric(assets_df['runtime_mlx_lm'], errors='coerce'),
            }).reset_index(drop=True)
        )

    case_card_previews = []
    reviewer_note_previews = []
    grounding_detail_rows = []

    source_profiles = (
        assets_df[['feature_profile', 'profile_label', 'appendix_dir']].itertuples(index=False)
        if not assets_df.empty and {'feature_profile', 'profile_label', 'appendix_dir'}.issubset(assets_df.columns)
        else []
    )

    for row in source_profiles:
        appendix_dir = Path(row.appendix_dir)

        cards_path = appendix_dir / 'llm_case_cards.csv'
        if cards_path.exists():
            cards = pd.read_csv(cards_path)
            cards = _ensure_cols(
                cards,
                [
                    'case_id', 'workload', 'stressor',
                    'dominant_tier', 'dominant_mechanism',
                    'top_feature_1', 'top_mechanism_1'
                ],
                default='',
            )
            if not cards.empty:
                preview = cards[
                    [
                        'case_id', 'workload', 'stressor',
                        'dominant_tier', 'dominant_mechanism',
                        'top_feature_1', 'top_mechanism_1'
                    ]
                ].head(3).copy()
                preview.insert(0, 'Profile', row.profile_label)
                case_card_previews.append(preview)

        output_paths = sorted(appendix_dir.glob('llm_outputs_*reviewer_summary*.csv'))
        if output_paths:
            outputs = pd.read_csv(output_paths[0])
            outputs = _ensure_cols(outputs, ['case_id', 'response_text'], default='')
            if not outputs.empty:
                out_preview = outputs[['case_id', 'response_text']].head(3).copy()
                out_preview.insert(0, 'Profile', row.profile_label)
                out_preview['response_preview'] = out_preview['response_text'].apply(lambda t: _clean_preview(t, 340))
                reviewer_note_previews.append(out_preview[['Profile', 'case_id', 'response_preview']])

        detail_paths = sorted(appendix_dir.glob('llm_grounding_detail_*reviewer_summary*.csv'))
        if detail_paths:
            detail = pd.read_csv(detail_paths[0])
            detail = _ensure_cols(
                detail,
                [
                    'case_id', 'grounded_core', 'cue_coverage',
                    'mentions_dominant_tier', 'mentions_dominant_mechanism',
                    'mentions_top_feature_1', 'mentions_top_mechanism_1',
                    'hallucination_flag'
                ],
            )
            if not detail.empty:
                detail.insert(0, 'Profile', row.profile_label)
                focus = detail[
                    (~detail['grounded_core'].fillna(False))
                    | (pd.to_numeric(detail['cue_coverage'], errors='coerce') < 1.0)
                    | (detail['hallucination_flag'].fillna(False))
                ].copy()
                if not focus.empty:
                    grounding_detail_rows.append(
                        focus[
                            [
                                'Profile', 'case_id', 'grounded_core', 'cue_coverage',
                                'mentions_dominant_tier', 'mentions_dominant_mechanism',
                                'mentions_top_feature_1', 'mentions_top_mechanism_1',
                                'hallucination_flag'
                            ]
                        ]
                    )

    if case_card_previews:
        display(Markdown('#### Example grounded case cards'))
        display(pd.concat(case_card_previews, ignore_index=True))

    if reviewer_note_previews:
        display(Markdown('#### Example reviewer-note outputs'))
        display(pd.concat(reviewer_note_previews, ignore_index=True))

    if grounding_detail_rows:
        detail_show = pd.concat(grounding_detail_rows, ignore_index=True)
        detail_show['cue_coverage'] = pd.to_numeric(detail_show['cue_coverage'], errors='coerce').round(2)
        display(Markdown('#### Cases worth manual review'))
        display(detail_show.reset_index(drop=True))



## 12B. DICE Story, Practical Scenarios, and LLM Role

### DICE story from start to finish

1. **Collect deployment-available telemetry.** DICE starts from what the platform can expose in the field: Tier-0 OS telemetry, Tier-1 power/thermal/frequency proxies, and Tier-2 profiler evidence when available.
2. **Anchor the analysis to the real laptop hardware.** For this notebook, the collection host is a `16-inch 2023 MacBook Pro` with an `Apple M2 Pro`, `16 GB` unified memory, an integrated GPU, and a `16-core Neural Engine`. That lets DICE translate telemetry into subsystem-level localization on the actual laptop.
3. **Learn what benign behavior looks like.** DICE is trained only on nominal runs. It estimates robust medians/scales, then fits a compact linear behavioral micro-twin that predicts what the next benign telemetry vector should look like.
4. **Turn mismatch into evidence.** At runtime, DICE compares the observed telemetry against the micro-twin prediction. The residual is the evidence that something may be off.
5. **Aggregate evidence over decision blocks.** DICE summarizes residual magnitude over fixed windows, scores each block, and then applies split-conformal calibration so the benign block-level false-alarm rate is controlled at the chosen `alpha` under benign exchangeability.
6. **Emit a run-level alert only after persistence.** A single noisy block is not enough. DICE requires consecutive flagged blocks before it raises a run alert.
7. **Localize and explain the alert on the M2 Pro.** Once DICE decides something is abnormal, it marks the first few abnormal decision blocks, records when the anomaly first appears and persists, and converts the residual evidence into tier shares, mechanism shares, stressor-level diagnosis scores, and likely visible hardware subsystems such as CPU, GPU/display, ANE, unified memory, or runtime scheduling.
8. **Add a grounded LLM triage layer.** DICE exports structured case cards from its own evidence. A grounded LLM then turns those case cards into reviewer summaries, triage notes, and follow-up steps. The LLM is part of the end-to-end DICE workflow, but it is bounded by detector evidence rather than allowed to invent new facts.

### How to explain DICE to laypeople

- **Laptop or workstation health:** DICE watches everyday system telemetry, flags when a device starts behaving unlike its normal healthy pattern, records the first abnormal windows, and points to the subsystem where the anomaly is most visible.
- **M2 Pro hardware story:** On this MacBook Pro, DICE can most realistically point to CPU-side behavior, GPU/display-side behavior, Neural Engine activity, unified-memory pressure, swap/storage effects, or runtime scheduling structure.
- **Edge device or robot:** DICE acts like a lightweight always-on self-check that can warn when the system is drifting into a risky operating state.
- **Fleet screening:** DICE can run on each host locally, then send only compact summaries or alerts to a backend instead of streaming all raw telemetry.
- **Two-stage deployment:** A smaller Tier-0 screen can run all the time, and a richer Tier-0/1/2 analysis can be triggered only when the lightweight screen sees something suspicious.

### Where the LLM fits in DICE

- The **core DICE detector is not an LLM**. It is a benign-trained residual detector with conformal calibration.
- The **LLM is the final human-facing explanation layer**. It reads DICE evidence such as dominant tier, dominant mechanism, top residual cues, likely hardware subsystem, and confidence.
- In this notebook, the LLM section now always exports case cards, a model catalog, and prompt bundles for **both mixed and full**. If saved model outputs are present, it also scores grounding, cue coverage, and hallucination rate automatically.
- The safe way to describe it is: **DICE detects and structures the evidence; the grounded LLM explains the evidence without becoming the source of truth.**


In [ ]:
# Plain-language: This cell creates a one-slide summary of DICE that is easy to reuse in talks or quick overviews.
display(Markdown('### One-slide DICE explanation'))

dice_story = pd.DataFrame([
    {'Stage': 'Sense', 'Plain-language role': 'Collect OS, proxy, and profiler signals that are actually available in the field.'},
    {'Stage': 'Predict', 'Plain-language role': 'Use a compact digital twin to predict what healthy behavior should look like next.'},
    {'Stage': 'Compare', 'Plain-language role': 'Measure the mismatch between observed telemetry and healthy expected telemetry.'},
    {'Stage': 'Decide', 'Plain-language role': 'Use conformal calibration plus persistence so short spikes do not become false alarms.'},
    {'Stage': 'Diagnose', 'Plain-language role': 'Summarize which tiers, mechanisms, and stressor patterns best explain the abnormal run.'},
    {'Stage': 'Explain', 'Plain-language role': 'Turn the structured DICE case card into a grounded reviewer note, triage summary, or follow-up plan.'},
])
display(dice_story)

In [ ]:
# Plain-language: This cell packages grounded LLM assets, reads any saved scored outputs, and records the exact portable runtime settings used for the LLM results.
display(Markdown('### Grounded LLM triage bundle and scored outputs'))

LLM_HELPERS = load_grounded_llm_triage_helpers(REPO_ROOT)
discover_latest_llm_outputs = LLM_HELPERS['discover_latest_llm_outputs']
evaluate_llm_grounding_outputs = LLM_HELPERS['evaluate_llm_grounding_outputs']
export_llm_case_cards = LLM_HELPERS['export_llm_case_cards']
export_llm_diagnostic_model_catalog = LLM_HELPERS['export_llm_diagnostic_model_catalog']
export_llm_diagnostic_prompt_bundle = LLM_HELPERS['export_llm_diagnostic_prompt_bundle']
runtime_availability = LLM_HELPERS['runtime_availability']

LLM_GROUNDING_SUMMARY_CSV = COMPARISON_DIR / 'llm_grounding_summary_profiles.csv'
LLM_GROUNDING_ASSET_CSV = COMPARISON_DIR / 'llm_grounded_triage_assets_profiles.csv'
LLM_RUNTIME_SUMMARY_CSV = COMPARISON_DIR / 'llm_runtime_profiles.csv'
LLM_GROUNDING_NOTE_MD = COMPARISON_DIR / 'LLM_GROUNDED_TRIAGE_NOTE.md'
LLM_PRESENTATION_SNIPPET = COMPARISON_DIR / 'LLM_RESULTS_SNIPPET.md'

runtime_modules = runtime_availability()
display(Markdown('#### Local runtime availability'))
display(pd.DataFrame([runtime_modules]))

asset_rows = []
score_rows = []
preview_rows = []
runtime_rows = []
notes = []

for profile in PROFILE_ORDER:
    global_dir = profile_global_dir(profile)
    appendix_dir = profile_appendix_dir(profile)
    appendix_dir.mkdir(parents=True, exist_ok=True)
    profile_label = PROFILE_LABEL.get(profile, profile)

    if not (global_dir / 'case_diagnosis_summary.csv').exists():
        notes.append(
            f"- `{profile_label}`: diagnosis bundle is not ready yet, so grounded LLM case cards were not exported."
        )
        continue

    cards = export_llm_case_cards(global_dir, appendix_dir)
    models = export_llm_diagnostic_model_catalog(appendix_dir)
    prompt_bundle = export_llm_diagnostic_prompt_bundle(cards, models, appendix_dir)
    output_paths = discover_latest_llm_outputs(appendix_dir)

    asset_rows.append({
        'feature_profile': profile,
        'profile_label': profile_label,
        'n_case_cards': int(len(cards)),
        'n_model_profiles': int(len(models)),
        'n_prompt_rows': int(len(prompt_bundle)),
        'prompt_types': ', '.join(sorted(prompt_bundle['prompt_type'].unique())) if not prompt_bundle.empty else '',
        'llm_output_csvs_found': int(len(output_paths)),
        'runtime_mlx_lm': int(bool(runtime_modules.get('mlx_lm', False))),
        'appendix_dir': str(appendix_dir),
    })

    if not cards.empty:
        preview = cards[['case_id', 'workload', 'stressor', 'dominant_tier', 'dominant_mechanism', 'top_feature_1']].head(3).copy()
        preview.insert(0, 'profile_label', profile_label)
        preview_rows.append(preview)

    if output_paths:
        for output_csv in output_paths:
            llm_outputs = pd.read_csv(output_csv)
            stem = output_csv.stem.replace('llm_outputs_', '')
            llm_summary, _ = evaluate_llm_grounding_outputs(llm_outputs, cards, appendix_dir, stem=stem)
            runtime_json = appendix_dir / f'llm_runtime_{stem}.json'
            runtime_meta = {}
            if runtime_json.exists():
                runtime_meta = json.loads(runtime_json.read_text())
            runtime_rows.append({
                'feature_profile': profile,
                'profile_label': profile_label,
                'model_id': runtime_meta.get('model_id', llm_outputs['model_id'].iloc[0] if not llm_outputs.empty else ''),
                'backend': runtime_meta.get('backend', llm_outputs['backend'].iloc[0] if ('backend' in llm_outputs.columns and not llm_outputs.empty) else ''),
                'seed': runtime_meta.get('seed', llm_outputs['seed'].iloc[0] if ('seed' in llm_outputs.columns and not llm_outputs.empty) else np.nan),
                'max_tokens': runtime_meta.get('max_tokens', llm_outputs['max_tokens'].iloc[0] if ('max_tokens' in llm_outputs.columns and not llm_outputs.empty) else np.nan),
                'resolved_cached_revision': runtime_meta.get('resolved_cached_revision', runtime_meta.get('revision', '')),
                'n_outputs': runtime_meta.get('n_outputs', int(len(llm_outputs))),
                'elapsed_s': runtime_meta.get('elapsed_s', np.nan),
                'runtime_json': str(runtime_json) if runtime_json.exists() else '',
            })
            if not llm_summary.empty:
                llm_summary['feature_profile'] = profile
                llm_summary['profile_label'] = profile_label
                llm_summary['source_csv'] = str(output_csv)
                llm_summary['backend'] = runtime_meta.get('backend', llm_outputs['backend'].iloc[0] if ('backend' in llm_outputs.columns and not llm_outputs.empty) else '')
                llm_summary['seed'] = runtime_meta.get('seed', llm_outputs['seed'].iloc[0] if ('seed' in llm_outputs.columns and not llm_outputs.empty) else np.nan)
                llm_summary['max_tokens'] = runtime_meta.get('max_tokens', llm_outputs['max_tokens'].iloc[0] if ('max_tokens' in llm_outputs.columns and not llm_outputs.empty) else np.nan)
                llm_summary['resolved_cached_revision'] = runtime_meta.get('resolved_cached_revision', runtime_meta.get('revision', ''))
                score_rows.append(llm_summary)
            notes.append(f"- `{profile_label}`: scored grounded LLM outputs from `{output_csv.name}`.")
    else:
        notes.append(
            f"- `{profile_label}`: exported {len(cards)} grounded case cards, {len(models)} model profiles, and {len(prompt_bundle)} prompts to `{appendix_dir}`; no saved LLM output CSV was found yet."
        )

if asset_rows:
    llm_asset_summary = pd.DataFrame(asset_rows).sort_values('feature_profile').reset_index(drop=True)
    llm_asset_summary.to_csv(LLM_GROUNDING_ASSET_CSV, index=False)
    display(Markdown('#### Grounded LLM asset bundle'))
    display(llm_asset_summary)
else:
    llm_asset_summary = pd.DataFrame(columns=['feature_profile', 'profile_label', 'n_case_cards', 'n_model_profiles', 'n_prompt_rows', 'prompt_types', 'llm_output_csvs_found', 'runtime_mlx_lm', 'appendix_dir'])
    llm_asset_summary.to_csv(LLM_GROUNDING_ASSET_CSV, index=False)
    display(Markdown('#### Grounded LLM asset bundle'))
    display(Markdown('No grounded case-card exports were written because no ready diagnosis bundles were found.'))

if preview_rows:
    display(Markdown('#### Grounded case-card preview'))
    display(pd.concat(preview_rows, ignore_index=True))

if runtime_rows:
    llm_runtime_summary = pd.DataFrame(runtime_rows).sort_values(['feature_profile', 'model_id']).reset_index(drop=True)
    llm_runtime_summary.to_csv(LLM_RUNTIME_SUMMARY_CSV, index=False)
    display(Markdown('#### Scored runtime summary'))
    display(llm_runtime_summary[[
        'profile_label', 'model_id', 'backend', 'seed', 'max_tokens', 'resolved_cached_revision', 'n_outputs', 'elapsed_s'
    ]])
else:
    pd.DataFrame(columns=['feature_profile', 'profile_label', 'model_id', 'backend', 'seed', 'max_tokens', 'resolved_cached_revision', 'n_outputs', 'elapsed_s', 'runtime_json']).to_csv(LLM_RUNTIME_SUMMARY_CSV, index=False)

if score_rows:
    llm_summary = pd.concat(score_rows, ignore_index=True)
    llm_summary.to_csv(LLM_GROUNDING_SUMMARY_CSV, index=False)
    display(Markdown('#### Scored LLM grounding results'))
    display(llm_summary[[
        'profile_label', 'model_id', 'backend', 'seed', 'prompt_type', 'n_outputs', 'grounded_core_rate',
        'dominant_tier_rate', 'dominant_mechanism_rate', 'mean_cue_coverage', 'hallucination_rate'
    ]].round(4))

    snippet_lines = [
        'Grounded LLM end-of-results note:',
        'DICE now includes a portable local scored baseline that reads the released evidence bundle and converts it into grounded reviewer notes.',
    ]
    for row in llm_summary.itertuples(index=False):
        snippet_lines.append(
            f"- {row.profile_label}, {row.model_id}, {row.prompt_type}: grounded-core={row.grounded_core_rate:.3f}, cue-coverage={row.mean_cue_coverage:.3f}, hallucination-rate={row.hallucination_rate:.3f}."
        )
else:
    placeholder_cols = [
        'feature_profile', 'profile_label', 'model_id', 'backend', 'seed', 'prompt_type', 'n_outputs',
        'grounded_core_rate', 'dominant_tier_rate', 'dominant_mechanism_rate',
        'mean_cue_coverage', 'hallucination_rate', 'source_csv',
    ]
    pd.DataFrame(columns=placeholder_cols).to_csv(LLM_GROUNDING_SUMMARY_CSV, index=False)
    display(Markdown('#### Scored LLM grounding results'))
    display(Markdown(
        'No saved LLM output CSVs were found in this environment. The notebook still exported the grounded case cards, model catalog, and prompt bundles for both mixed and full, so this table will auto-populate the moment a local model run is added.'
    ))

    snippet_lines = [
        'Grounded LLM end-of-results note:',
        'DICE now always exports grounded case cards and prompt bundles for both mixed and full profiles.',
        'In the current environment, no local LLM runtime or saved output CSV was available, so scored grounding metrics are pending even though the evidence bundle is ready for immediate local inference.',
    ]

LLM_GROUNDING_NOTE_MD.write_text('Grounded LLM triage note\n\n' + '\n'.join(notes) + '\n')
LLM_PRESENTATION_SNIPPET.write_text('\n'.join(snippet_lines) + '\n')

display(Markdown('\n'.join(notes) if notes else 'No grounded LLM notes were generated.'))
display(Markdown(f'Wrote: `{LLM_GROUNDING_ASSET_CSV}`'))
display(Markdown(f'Wrote: `{LLM_RUNTIME_SUMMARY_CSV}`'))
display(Markdown(f'Wrote: `{LLM_GROUNDING_SUMMARY_CSV}`'))
display(Markdown(f'Wrote: `{LLM_PRESENTATION_SNIPPET}`'))
display(Markdown(f'Wrote: `{LLM_GROUNDING_NOTE_MD}`'))


## 13. Reproducibility Manifest

This final section records the dataset hash, environment hash, notebook runtime summary, and execution configuration so the released analysis can be checked across machines. The draft-facing paper bundle is expected under the **mixed** profile unless you intentionally switch the primary profile above.


In [ ]:
# Plain-language: This cell loads the reproducibility manifest so readers can see exactly how this notebook run was configured.
manifest = json.loads(MANIFEST.read_text())
print('Manifest path:', MANIFEST)
git_meta = manifest.get('git', {})
platform_meta = manifest.get('platform', {})
print('Git commit:', git_meta.get('short_commit', git_meta.get('commit', 'n/a')))
print('Git branch:', git_meta.get('branch', 'n/a'))
print('Git tree clean:', git_meta.get('is_clean', 'n/a'))
print('Python version:', platform_meta.get('python_version', 'n/a'))
print('Platform:', platform_meta.get('platform', 'n/a'))
print('Dataset SHA256:', manifest['dataset_digest']['sha256'])
print('Environment file SHA256:', manifest['environment_files']['environment_yml']['sha256'])
print('Requirements SHA256:', manifest['environment_files']['requirements_txt']['sha256'])
print('Run config:', manifest.get('run_config', {}))
print('Primary paper profile:', globals().get('PRIMARY_PAPER_PROFILE', 'mixed'))
if NOTEBOOK_RUNTIME.exists():
    runtime_summary = json.loads(NOTEBOOK_RUNTIME.read_text())
    print('Notebook runtime summary:', runtime_summary)
print('Main paper artifacts:', PAPER_FULL)
print('Appendix artifacts :', APPENDIX_FULL)

artifact_manifest = write_saved_artifact_manifest()
print('Saved artifact manifest:', RESULT_ARTIFACT_MANIFEST_CSV)
print('Saved PNG count:', int((artifact_manifest['suffix'] == '.png').sum()))
print('Saved CSV count:', int((artifact_manifest['suffix'] == '.csv').sum()))
if not artifact_manifest.empty:
    display(artifact_manifest.head(20))


In [ ]:
# Plain-language: This placeholder cell is intentionally left blank so you can add follow-up notes or checks without disturbing the main flow.